# Quantum diffusion: one reproducible research notebook

This is the **active research entry point**. All scientific implementation is embedded below. No project Python modules or old notebooks are imported. The earlier paper's invalid circuits remain historical evidence; the verified dense references and optimized variants here are new implementations.

**Use:** save this notebook, edit only the configuration cell, then Run All. Compatible completed stages load automatically. Each new stage saves raw results, circuits, code, environment, and calibration before the next stage begins. Reopening is historical reuse, not an independent replication. Change `replication_id` and stochastic seeds for independent simulator samples; hardware spending also requires space in the explicit campaign cap.

The existing IBM pilot is completed. The default `hardware.mode=collect_only` reloads matching results and prevents new submissions when parameters change. Use `submit_new` only for a deliberate new bounded campaign. Set `hardware.enabled=False` for offline work. Physical execution uses the named saved account and a durable one-job / 60-second pilot limit. Pending or uncertain submissions are never automatically repeated. Account secrets belong in IBM's saved account store, never this notebook.

The small dense algorithms use classical matrix preprocessing and exact states for validation. They do not establish speedup. Z-basis distributions do not establish state fidelity or solution signs. Fake snapshots and approximate noise models are labeled separately from physical results.

In [ ]:
# CELL_ID: configuration
from pathlib import Path
WORKSPACE = Path.cwd().resolve()
NOTEBOOK = WORKSPACE / 'Quantum_Diffusion_Master.ipynb'
if not NOTEBOOK.is_file():
    raise FileNotFoundError('Open a kernel in the folder containing Quantum_Diffusion_Master.ipynb.')
CONFIG = {
    'replication_id': 'pilot_v1',
    'coherent_screening': {'enabled':True},
    'confirmation': {'prepare':True,'submit':False,'collect_existing':True,'protocol':'vqls_tomography','account':'lab_zhang','backend':'ibm_rensselaer',
        'campaign_id':'common_preparation_tomography_2026_10_04','replication_id':'block_0'},
    'coherent_confirmation': {'prepare':True,'submit':False,'collect_existing':True,'protocol':'coherent_bias',
        'account':'lab_zhang','backend':'ibm_rensselaer',
        'campaign_id':'coherent_bias_2026_10_04','replication_id':'block_0'},
    'problem': {'n':4, 'family':'cycle', 'gamma':8.0, 'seed':17},
    'solvers': {
        'hhl': {'n_qpe':5},
        'qsvt': {'degree':15, 'phase_tolerance':0.02, 'seed':11},
        'vqls_statevector': {'reps':2, 'max_evals':250, 'seed':11},
        'vqls_measured': {'reps':2, 'max_evals':60, 'seed':11, 'shots':2048, 'tol':0.001},
    },
    'ideal_samples': {'shots':4096,'seed':901},
    'analysis': {'bootstrap_samples':400,'seed':1911},
    'fake_backend': {'enabled':True, 'class_name':'FakeGuadalupeV2'},
    'transpile': {'optimization_level':2,'seed_transpiler':1729,
                  'max_two_qubit_gates_per_circuit':1500},
    'snapshot_simulation': {'shots':4096,'seed':1701,'max_active_qubits':12,'method':'density_matrix'},
    'hardware': {'enabled':True, 'mode':'collect_only', 'archive_job_metadata':True, 'account':'lab_zhang', 'backend':'ibm_rensselaer',
        'shots':4096, 'max_execution_time_seconds':60, 'max_jobs':1,
        'max_circuits_per_job':8, 'max_two_qubit_gates_per_circuit':1500,
        'campaign_id':'notebook_pilot_v1', 'replication_id':'pilot_v1'},
    # Journal study (frozen design in the journal_design cell). Planning, retrospective
    # and optimizer stages are local. Hardware blocks run by themselves in the frozen order
    # B1 -> B2 -> B3 (journal_hardware_run): saved blocks are reloaded, and the next missing
    # block is submitted once, only after ibm_rensselaer has been recalibrated since the
    # previous block. Nothing here needs editing between blocks. T1 on the Heron devices
    # also reloads automatically but submits only with transfer_submit=True.
    'journal': {'enabled':True,
        'planning': {'enabled':True, 'devices':['ibm_rensselaer','ibm_kingston','ibm_boston'], 'blocks':['B1']},
        'retrospective': {'enabled':True},
        'optimizer_study': {'enabled':True},
        'hardware': {'enabled': True, 'submit': True, 'collect_existing': True,
                    'devices': ['ibm_rensselaer', 'ibm_kingston', 'ibm_boston'], 'transfer_submit': True,
                    'wait_minutes': 30, 'allow_time_block': False,
                    'approval_note': 'Batuhan, 15 Oct 2026: run B1-B3 automatically on Run All, '
                                     'each job once, one block per provider calibration'},
        'analysis': {'bootstrap_samples':400, 'seed':20261011}},
}
# collect_only reopens existing job records. Changed parameters without a matching
# saved job return disabled. A deliberate new bounded pilot uses mode=submit_new.
# Available premium choices: ibm_pittsburgh, ibm_boston, ibm_fez,
# ibm_marrakesh, ibm_kingston with account='lab_premium'.
# lab_zhang is explicitly restricted to ibm_rensselaer.
# Bootstrap changes affect analysis only. Shot changes reuse the logical solver.
# A changed fake backend keeps the mathematical solution and ideal samples.
# Each fake class uses its packaged snapshot; live refresh is a new treatment.
OUTPUTS = WORKSPACE if (WORKSPACE / 'portable_restore.json').exists() else WORKSPACE / 'outputs'
print('Outputs:', OUTPUTS)

## Storage and provenance

The cache identifies the mathematical input, executed cell source, dependency versions, solver settings, backend calibration, and stochastic treatment. Calibration retrieval time is recorded separately from calibration content. Completed artifacts are checksum-verified before loading. Partial results retain their evidence; they are not silently accepted as successful results.

Every stage has a `code_and_outputs.zip` with a self-contained source notebook. `outputs/invocations/` records each launch and reuse event. `outputs/calibrations/` contains immutable complete device snapshots plus observations. Ideal simulation explicitly declares that it has no device noise.

In [ ]:
# CELL_ID: cache
import contextlib
import ast
import copy
import hashlib
import inspect
import json
import os
import platform
import re
import shutil
import sys
import uuid
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path, PurePosixPath
import numpy as np

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def portable(value):
    if isinstance(value, dict):
        return {str(k): portable(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [portable(v) for v in value]
    if isinstance(value, np.ndarray):
        return portable(value.tolist())
    if isinstance(value, np.generic):
        return portable(value.item())
    if isinstance(value, complex):
        return {'real': float(value.real), 'imag': float(value.imag)}
    if isinstance(value, (datetime, Path)):
        return str(value)
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f'Unsupported artifact type {type(value).__name__}; do not serialize arbitrary account objects.')

def canonical(value):
    return json.dumps(portable(value), sort_keys=True, separators=(',', ':'), allow_nan=False)

def content_hash(value):
    return hashlib.sha256(canonical(value).encode('utf8')).hexdigest()

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(portable(value), indent=2, allow_nan=False), encoding='utf8')
    os.replace(temporary, path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf8'))

def environment_record():
    names = ['numpy', 'scipy', 'qiskit', 'qiskit-aer', 'qiskit-ibm-runtime', 'matplotlib', 'nbformat', 'ipykernel']
    versions = {}
    for name in names:
        try: versions[name] = version(name)
        except PackageNotFoundError: versions[name] = None
    return {'python': platform.python_version(), 'platform': platform.platform(), 'packages': versions}

def executed_cell_sources():
    # IPython records executed source, including edits not yet saved to disk.
    sources = dict(globals().get('_EXECUTED_SOURCES', {}))
    try:
        ip = get_ipython()
        for source in ip.history_manager.input_hist_raw:
            if source.startswith('# CELL_ID: '):
                role = source.splitlines()[0].split(':', 1)[1].strip()
                sources[role] = source
    except (NameError, AttributeError):
        pass
    return sources

def code_fingerprint(roles):
    sources = executed_cell_sources()
    missing = set(roles) - sources.keys()
    if missing:
        raise RuntimeError(f'Execute definition cells before use: {sorted(missing)}')
    return {role: hashlib.sha256(sources[role].encode()).hexdigest() for role in sorted(roles)}

class CacheIntegrityError(RuntimeError):
    pass

_SECRET_KEYS = {'token', 'api_key', 'apikey', 'api_token', 'access_token', 'refresh_token',
                'password', 'secret', 'client_secret', 'authorization', 'qiskit_ibm_token'}

def _sensitive_name(name):
    name = str(name).lower().replace('-', '_')
    return name in _SECRET_KEYS or name.endswith(('_api_key', '_api_token', '_access_token', '_password', '_client_secret'))

def _credential_value(value):
    if value is None or value is False:
        return False
    if isinstance(value, str):
        normalized = value.strip().upper()
        return normalized not in {'', '...', 'YOUR_TOKEN', 'YOUR_API_KEY', 'YOUR_PASSWORD', '<TOKEN>', '<API_KEY>'}
    return not isinstance(value, (int, float, bool))

def assert_no_credentials(value):
    """Reject credential-bearing structured artifacts; never enumerate account stores."""
    if isinstance(value, dict):
        for key, child in value.items():
            if _sensitive_name(key) and _credential_value(child):
                raise CacheIntegrityError('Export contains a credential field; use a saved account alias.')
            assert_no_credentials(child)
    elif isinstance(value, (list, tuple)):
        for child in value:
            assert_no_credentials(child)

def assert_source_has_no_credentials(source):
    """Find literal credential assignments/dictionary entries without evaluating source."""
    try:
        tree = ast.parse(source)
    except SyntaxError:
        # Non-Python notebook magics still receive a conservative literal check.
        if re.search(r'(?im)\b(?:api[_-]?key|api_token|access_token|password|client_secret|qiskit_ibm_token|token)\s*[:=]\s*[\'\"][^\'\"\n]{8,}', source):
            raise CacheIntegrityError('Non-Python source contains a possible credential literal; inspect before export.')
        return
    for node in ast.walk(tree):
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            value = node.value
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            if isinstance(value, ast.Constant):
                for target in targets:
                    name = target.id if isinstance(target, ast.Name) else None
                    if isinstance(target, ast.Subscript) and isinstance(target.slice, ast.Constant):
                        name = target.slice.value
                    if name is not None and _sensitive_name(name) and _credential_value(value.value):
                        raise CacheIntegrityError('Notebook source contains a literal credential; use a saved account alias.')
        if isinstance(node, ast.Dict):
            for key, value in zip(node.keys, node.values):
                if isinstance(key, ast.Constant) and isinstance(value, ast.Constant):
                    if _sensitive_name(key.value) and _credential_value(value.value):
                        raise CacheIntegrityError('Notebook source contains a literal credential dictionary entry.')
        if isinstance(node, ast.Call):
            for keyword in node.keywords:
                if keyword.arg and _sensitive_name(keyword.arg) and isinstance(keyword.value, ast.Constant):
                    if _credential_value(keyword.value.value):
                        raise CacheIntegrityError('Notebook source contains a literal credential argument.')

def _safe_component(value, name='identifier'):
    if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,127}', value):
        raise CacheIntegrityError(f'Invalid {name}; path separators and traversal are forbidden.')
    return value

def _safe_hash(value):
    if not isinstance(value, str) or not re.fullmatch(r'[0-9a-f]{64}', value):
        raise CacheIntegrityError('Expected a complete SHA-256 identifier.')
    return value

def _inside_file(directory, relative):
    if not isinstance(relative, str) or '\\' in relative or ':' in relative:
        raise CacheIntegrityError('Invalid artifact path.')
    path = PurePosixPath(relative)
    if path.is_absolute() or not path.parts or any(part in ('', '.', '..') for part in path.parts):
        raise CacheIntegrityError('Artifact path escapes its record.')
    target = Path(directory).joinpath(*path.parts)
    if target.is_symlink() or not target.resolve().is_relative_to(Path(directory).resolve()):
        raise CacheIntegrityError('Symlink or artifact path escapes its record.')
    return target

def _calibration_payload_valid(payload):
    if not isinstance(payload, dict) or payload.get('kind') not in ('ideal', 'fake', 'hardware'):
        raise CacheIntegrityError('Backend kind must be ideal, fake, or hardware.')
    assert_no_credentials(payload)
    if payload['kind'] == 'ideal':
        if payload.get('noise_mode') != 'ideal':
            raise CacheIntegrityError('Ideal execution requires an explicit no-noise declaration.')
    else:
        properties = payload.get('properties')
        qubits = properties.get('qubits') if isinstance(properties, dict) else None
        valid_field = lambda field: (isinstance(field, dict) and isinstance(field.get('name'), str)
            and bool(field['name']) and isinstance(field.get('value'), (int, float))
            and not isinstance(field['value'], bool) and np.isfinite(field['value']))
        if not isinstance(qubits, list) or not qubits or not any(
                isinstance(q, list) and any(valid_field(field) for field in q) for q in qubits):
            raise CacheIntegrityError('A fake or real device requires nonempty archived calibration fields.')
        if not isinstance(payload.get('backend'), str) or not payload['backend'].strip():
            raise CacheIntegrityError('A device calibration requires its backend identity.')
        timestamp = payload.get('calibration_last_update', properties.get('last_update_date'))
        if not timestamp or str(timestamp).strip().lower() in ('none', 'null', 'unknown'):
            raise CacheIntegrityError('A device calibration requires its provider calibration timestamp.')

class StageStore:
    """Content-addressed stages with immutable attempts, checksums and code bundles."""
    def __init__(self, root, notebook_path, configuration):
        assert_no_credentials(configuration)
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.notebook_path = Path(notebook_path).resolve()
        self.configuration = copy.deepcopy(configuration)
        self.environment = environment_record()
        self.invocation_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
        self.events = []
        atomic_json(self.root/'invocations'/self.invocation_id/'configuration.json', configuration)
        atomic_json(self.root/'invocations'/self.invocation_id/'environment.json', self.environment)

    def key(self, stage, spec, roles=(), dependencies=(), calibration_ids=()):
        _safe_component(stage, 'stage')
        assert_no_credentials(spec)
        return content_hash({'schema': 3, 'stage': stage, 'spec': spec,
            'code': code_fingerprint(roles), 'environment': self.environment,
            'dependencies': list(dependencies), 'calibration_ids': sorted(calibration_ids)})

    def _event(self, event):
        entry = {'utc': utc_now(), **event}
        self.events.append(entry)
        atomic_json(self.root/'invocations'/self.invocation_id/'events'/f'{len(self.events):05d}.json', entry)

    def _load(self, stage, key, record_event=False):
        _safe_component(stage, 'stage')
        _safe_hash(key)
        base = self.root/'cache'/stage/key
        pointer = base/'complete.json'
        if not pointer.exists():
            return None
        try:
            reference = read_json(pointer)
            _safe_component(reference['attempt_id'], 'attempt')
        except (OSError, ValueError, KeyError, TypeError) as error:
            raise CacheIntegrityError('Invalid completed-stage pointer.') from error
        attempt = base/'attempts'/reference['attempt_id']
        if not attempt.resolve().is_relative_to(base.resolve()) or attempt.is_symlink():
            raise CacheIntegrityError('Attempt path escapes its stage.')
        if not (attempt/'manifest.json').is_file() or file_hash(attempt/'manifest.json') != reference.get('manifest_sha256'):
            raise CacheIntegrityError('Completed manifest was altered; cached results will not be silently reused.')
        manifest = read_json(attempt/'manifest.json')
        if manifest.get('status') != 'complete' or manifest.get('schema') != 3:
            raise CacheIntegrityError('Completion pointer refers to incomplete work.')
        if (manifest.get('stage'), manifest.get('key'), manifest.get('attempt_id')) != (stage, key, reference['attempt_id']):
            raise CacheIntegrityError('Manifest identity differs from its completion pointer.')
        for identifier in manifest.get('calibration_ids', []):
            self.calibration(identifier)
        if 'result.json' not in manifest.get('artifacts', {}) or 'source/Quantum_Diffusion_Master.ipynb' not in manifest['artifacts']:
            raise CacheIntegrityError('Completed stage lacks its required result or executed-source archive.')
        for name, expected in manifest['artifacts'].items():
            p = _inside_file(attempt, name)
            if not p.is_file() or file_hash(p) != expected:
                raise CacheIntegrityError(f'Corrupt or missing cached artifact: {name}')
        bundle = attempt/'code_and_outputs.zip'
        if not bundle.is_file() or file_hash(bundle) != reference.get('bundle_sha256'):
            raise CacheIntegrityError('Completed code/output bundle is missing or corrupt.')
        if record_event:
            self._event({'stage': stage, 'key': key, 'action': 'loaded_existing', 'original_completed_utc': manifest['completed_utc'],
                         'original_invocation_id': manifest['invocation_id'], 'attempt_id': manifest['attempt_id']})
        return {'stage': stage, 'key': key, 'directory': attempt, 'result': read_json(attempt/'result.json'),
                'manifest': manifest, 'cache_hit': True}

    def lookup(self, stage, key):
        return self._load(stage, key, record_event=True)

    def calibration(self, calibration_id):
        _safe_hash(calibration_id)
        directory = self.root/'calibrations'/calibration_id
        try:
            record = read_json(directory/'snapshot.json')
            payload = record['payload']
        except (OSError, ValueError, TypeError, KeyError) as error:
            raise CacheIntegrityError('Mandatory calibration snapshot is missing or unreadable.') from error
        if content_hash(payload) != calibration_id:
            raise CacheIntegrityError('Calibration content does not match its immutable identifier.')
        _calibration_payload_valid(payload)
        return record

    def register_calibration(self, payload, observation=None):
        _calibration_payload_valid(payload)
        assert_no_credentials(observation)
        identifier = content_hash(payload)
        directory = self.root/'calibrations'/identifier
        directory.mkdir(parents=True, exist_ok=True)
        snapshot = directory/'snapshot.json'
        if not snapshot.exists():
            # Identical content-addressed payloads may race. Publish the first complete
            # snapshot atomically without replacing an existing first-archive envelope.
            temporary = directory/(uuid.uuid4().hex+'.tmp')
            atomic_json(temporary, {'first_archived_utc': utc_now(), 'payload': payload})
            try:
                os.link(temporary, snapshot)
            except FileExistsError:
                pass
            finally:
                temporary.unlink(missing_ok=True)
        self.calibration(identifier)
        atomic_json(directory/'observations'/f'{uuid.uuid4().hex}.json', {'retrieved_utc': utc_now(), **(observation or {})})
        return identifier

    def _source_notebook(self):
        notebook = read_json(self.notebook_path)
        executed = executed_cell_sources()
        for cell in notebook['cells']:
            if cell['cell_type'] != 'code': continue
            source = ''.join(cell['source'])
            if source.startswith('# CELL_ID: '):
                role = source.splitlines()[0].split(':', 1)[1].strip()
                if role in executed: cell['source'] = executed[role].splitlines(True)
            cell['outputs'] = []
            cell['execution_count'] = None
            assert_source_has_no_credentials(''.join(cell['source']))
        assert_no_credentials(notebook)
        return notebook

    def _export_sources(self, directory):
        atomic_json(directory/'source'/'Quantum_Diffusion_Master.ipynb', self._source_notebook())
        atomic_json(directory/'source'/'configuration.json', self.configuration)
        atomic_json(directory/'source'/'environment.json', self.environment)
        atomic_json(directory/'source'/'executed_cell_hashes.json', code_fingerprint(executed_cell_sources().keys()))
        (directory/'source'/'requirements.txt').write_text('\n'.join(
            f'{k}=={v}' for k,v in self.environment['packages'].items() if v)+'\n',encoding='utf8')

    def _export_files(self, directory):
        allowed = {'.json', '.ipynb', '.qpy', '.npy', '.npz', '.csv', '.tsv', '.txt', '.md', '.py', '.png', '.pdf', '.svg'}
        result = {}
        for path in sorted(directory.rglob('*')):
            if path.is_symlink():
                raise CacheIntegrityError('Artifact exports cannot include symlinks.')
            if not path.is_file() or path.suffix.lower() == '.zip' or '.tmp' in path.name:
                continue
            if path.suffix.lower() not in allowed or path.name.lower() in {'.env', 'credentials.json', 'qiskit-ibm.json'}:
                raise CacheIntegrityError('Unapproved artifact type/name in export; use the scientific artifact allowlist.')
            relative = path.relative_to(directory).as_posix()
            _inside_file(directory, relative)
            if path.suffix.lower() in {'.json', '.ipynb'}:
                structured = read_json(path)
                assert_no_credentials(structured)
                if path.suffix.lower() == '.ipynb':
                    for cell in structured.get('cells', []):
                        if cell.get('cell_type') == 'code':
                            assert_source_has_no_credentials(''.join(cell.get('source', '')))
            if path.suffix.lower() == '.py':
                assert_source_has_no_credentials(path.read_text(encoding='utf8'))
            if path.suffix.lower() in {'.txt', '.md', '.csv', '.tsv', '.svg'}:
                text = path.read_text(encoding='utf8')
                if re.search(r'(?im)\b(?:api[_-]?key|api_token|access_token|password|client_secret|qiskit_ibm_token|token)\s*[:=]\s*[\'\"][^\'\"\n]{8,}', text):
                    raise CacheIntegrityError('Text artifact contains a possible credential literal.')
            result[relative] = path
        return result

    def _dependency_records(self, dependencies):
        found = {}
        visiting = set()
        def visit(dependency):
            identity = (dependency['stage'], dependency['key'])
            if identity in visiting:
                raise CacheIntegrityError('Dependency cycle in cached stage graph.')
            if identity in found:
                return
            visiting.add(identity)
            record = self._load(*identity)
            if record is None:
                raise CacheIntegrityError('A declared dependency is missing or incomplete.')
            for child in record['manifest'].get('dependencies', []):
                visit(child)
            visiting.remove(identity)
            found[identity] = record
        for dependency in dependencies:
            visit(dependency)
        return found

    def bundle(self, directory, calibration_ids=(), dependencies=()):
        directory = Path(directory)
        if not (directory/'source'/'Quantum_Diffusion_Master.ipynb').exists():
            self._export_sources(directory)
        members = dict(self._export_files(directory))
        identifiers = set(calibration_ids)
        for (stage, key), record in self._dependency_records(dependencies).items():
            identifiers.update(record['manifest'].get('calibration_ids', []))
            for name, path in self._export_files(record['directory']).items():
                members[f'dependencies/{stage}/{key}/{name}'] = path
        for identifier in sorted(identifiers):
            self.calibration(identifier)
            members[f'calibrations/{identifier}/snapshot.json'] = self.root/'calibrations'/identifier/'snapshot.json'
        export_manifest = {'schema': 1, 'archive_layout': 'flattened transitive dependency artifacts; no nested archives',
                           'members': {name: file_hash(path) for name, path in sorted(members.items())}}
        target = directory/'code_and_outputs.zip'
        temporary = directory/('bundle.'+uuid.uuid4().hex+'.tmp')
        try:
            with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
                for name, path in sorted(members.items()):
                    archive.write(path, name)
                archive.writestr('bundle_manifest.json', canonical(export_manifest))
            os.replace(temporary, target)
        finally:
            temporary.unlink(missing_ok=True)
        return target

    def _assert_no_prior_submission(self, base):
        for name in ('submission_attempt.json', 'submission_intent.json', 'job.json'):
            if any((base/'attempts').glob('*/'+name)):
                raise CacheIntegrityError('A prior submission attempt exists without a reusable completion. Reconcile its job or use a deliberate new replication; no automatic duplicate was started.')

    def run(self, stage, spec, function, roles=(), dependencies=(), calibration_ids=()):
        for identifier in calibration_ids: self.calibration(identifier)
        key = self.key(stage,spec,roles,dependencies,calibration_ids)
        hit = self.lookup(stage,key)
        if hit is not None: return hit
        base = self.root/'cache'/stage/key
        base.mkdir(parents=True,exist_ok=True)
        lock = base/'active.lock'
        try: lock.mkdir()
        except FileExistsError:
            raise RuntimeError(f'Stage is already running or was interrupted. Inspect {lock}; no duplicate execution was started.')
        try:
            # Another process may have completed between the first lookup and lock acquisition.
            hit = self.lookup(stage,key)
            if hit is not None:
                return hit
            self._assert_no_prior_submission(base)
            self._dependency_records(dependencies)
            attempt_id = uuid.uuid4().hex
            directory = base/'attempts'/attempt_id
            directory.mkdir(parents=True)
            manifest = {'schema':3,'stage':stage,'key':key,'attempt_id':attempt_id,'status':'running',
                'started_utc':utc_now(),'spec':spec,'code':code_fingerprint(roles),
                'dependencies':list(dependencies),'calibration_ids':list(calibration_ids),
                'invocation_id':self.invocation_id}
            atomic_json(directory/'manifest.json',manifest)
            atomic_json(directory/'request.json', {'spec':spec,'configuration':self.configuration,'environment':self.environment})
            # Refuse unsafe source before starting any expensive or physical work.
            self._export_sources(directory)
            result = function(directory)
            assert_no_credentials(result)
            manifest['key_input_dependencies']=list(dependencies)
            dynamic=result.get('cache_dependencies',[]) if isinstance(result,dict) else []
            dependencies=list({(d['stage'],d['key']):d for d in [*dependencies,*dynamic]}.values())
            self._dependency_records(dependencies)
            manifest['dependencies']=dependencies
            atomic_json(directory/'result.json',result)
            for identifier in calibration_ids:
                self.calibration(identifier)
            manifest.update(status='complete',completed_utc=utc_now())
            manifest['artifacts']={name:file_hash(path) for name,path in self._export_files(directory).items()
                if name!='manifest.json'}
            atomic_json(directory/'manifest.json',manifest)
            bundle = self.bundle(directory,calibration_ids,dependencies)
            atomic_json(base/'complete.json',{'attempt_id':attempt_id,'manifest_sha256':file_hash(directory/'manifest.json'),
                                            'bundle_sha256':file_hash(bundle)})
            self._event({'stage':stage,'key':key,'action':'computed_and_saved','directory':str(directory.relative_to(self.root))})
            return {'stage':stage,'key':key,'directory':directory,'result':portable(result),'manifest':manifest,'cache_hit':False}
        except Exception as error:
            if 'manifest' in locals():
                manifest.update(status='failed',completed_utc=utc_now(),error_type=type(error).__name__)
                # Error strings from remote services can include account identifiers; type is durable.
                atomic_json(directory/'manifest.json',manifest)
                try:
                    self.bundle(directory,calibration_ids,dependencies)
                except Exception as export_error:
                    manifest['failure_bundle_status'] = 'unavailable'
                    manifest['failure_bundle_error_type'] = type(export_error).__name__
                    atomic_json(directory/'manifest.json',manifest)
            self._event({'stage':stage,'key':key,'action':'failed_saved','error_type':type(error).__name__})
            raise
        finally:
            lock.rmdir()

    def dependency(self, record):
        return {'stage':record['stage'],'key':record['key']}

    def export_run(self, records, extra_files=()):
        """One portable raw archive; restore_run_export rebuilds derived stage ZIPs."""
        invocation = self.root/'invocations'/self.invocation_id
        source = invocation/'portable_source'
        self._export_sources(source)
        dependencies = [self.dependency(record) for record in records]
        all_records = self._dependency_records(dependencies)
        members = {'Quantum_Diffusion_Master.ipynb': source/'source'/'Quantum_Diffusion_Master.ipynb',
                   'configuration.json': source/'source'/'configuration.json',
                   'environment.json': source/'source'/'environment.json'}
        calibrations = set()
        pointers = []
        for (stage, key), record in all_records.items():
            for name, path in self._export_files(record['directory']).items():
                members[path.relative_to(self.root).as_posix()] = path
            pointer = self.root/'cache'/stage/key/'complete.json'
            members[pointer.relative_to(self.root).as_posix()] = pointer
            pointers.append({'stage': stage, 'key': key})
            calibrations.update(record['manifest'].get('calibration_ids', []))
        for identifier in sorted(calibrations):
            self.calibration(identifier)
            for name, path in self._export_files(self.root/'calibrations'/identifier).items():
                members[path.relative_to(self.root).as_posix()] = path
        for item in extra_files:
            path = Path(item).resolve()
            if not path.is_relative_to(self.root) or not path.is_file() or path.is_symlink():
                raise CacheIntegrityError('Extra exports must be scientific files inside the output root.')
            # Apply the same extension and credential checks to each extra file.
            checked = self._export_files(path.parent)
            if path.name not in checked:
                raise CacheIntegrityError('Nested archives are excluded from portable exports.')
            members[path.relative_to(self.root).as_posix()] = path
        instructions = ('This archive preserves scientific manifests, raw artifacts, executed code and calibration snapshots.\n'
            'It deliberately omits derived per-stage ZIP files. In the master notebook, run the definition cells, then call\n'
            'restored_store = restore_run_export(archive_path, empty_output_directory).\n'
            'Restoration verifies every checksum and rebuilds derived ZIPs in dependency order without running experiments.\n'
            'Set the notebook output directory to that directory before continuing. Completion pointers retain the original\n'
            'pointer and restoration provenance; scientific manifest hashes and original execution IDs remain unchanged.\n')
        instruction_path = source/'RESTORE_INSTRUCTIONS.md'
        instruction_path.write_text(instructions, encoding='utf8')
        members['RESTORE_INSTRUCTIONS.md'] = instruction_path
        manifest = {'schema': 1, 'kind': 'portable_stage_cache', 'invocation_id': self.invocation_id,
                    'restore_requires_bundle_rebuild': True, 'stages': pointers,
                    'members': {name: file_hash(path) for name, path in sorted(members.items())}}
        target = invocation/'portable_code_and_outputs.zip'
        temporary = invocation/(uuid.uuid4().hex+'.tmp')
        try:
            with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
                for name, path in sorted(members.items()): archive.write(path, name)
                archive.writestr('portable_manifest.json', canonical(manifest))
            os.replace(temporary, target)
        finally:
            temporary.unlink(missing_ok=True)
        self._event({'action': 'portable_export_saved', 'path': str(target.relative_to(self.root)), 'sha256': file_hash(target)})
        return target

    def summary(self):
        return {'invocation_id':self.invocation_id,'loaded':sum(e['action']=='loaded_existing' for e in self.events),
                'computed':sum(e['action']=='computed_and_saved' for e in self.events),'events':self.events}

def restore_run_export(archive_path, output_root):
    """Verify and restore our portable archive into an empty directory, without execution."""
    destination = Path(output_root).resolve()
    if destination.exists() and any(destination.iterdir()):
        raise CacheIntegrityError('Restore into an empty output directory; existing work is never overwritten.')
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise CacheIntegrityError('Duplicate archive members are forbidden.')
        descriptor = json.loads(archive.read('portable_manifest.json'))
        if descriptor.get('kind') != 'portable_stage_cache' or descriptor.get('schema') != 1:
            raise CacheIntegrityError('Unsupported portable archive format.')
        expected = descriptor.get('members', {})
        if set(names) != set(expected) | {'portable_manifest.json'}:
            raise CacheIntegrityError('Portable archive member inventory differs from its checksum manifest.')
        # Verify ALL names and contents before writing even one extracted artifact.
        for name, digest in expected.items():
            _inside_file(destination, name)
            _safe_hash(digest)
            if name.lower().endswith('.zip') or hashlib.sha256(archive.read(name)).hexdigest() != digest:
                raise CacheIntegrityError('Portable archive contains a nested ZIP or checksum mismatch.')
        destination.mkdir(parents=True, exist_ok=True)
        for name in expected:
            path = _inside_file(destination, name)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(name))
    restored = StageStore(destination, destination/'Quantum_Diffusion_Master.ipynb', read_json(destination/'configuration.json'))
    atomic_json(destination/'portable_restore.json', {'restored_utc':utc_now(),'archive_sha256':file_hash(archive_path),
        'output_directory_is_notebook_directory':True})
    identities = {(item['stage'], item['key']) for item in descriptor['stages']}
    finished, visiting = set(), set()
    def rebuild(identity):
        if identity in finished: return
        if identity in visiting or identity not in identities:
            raise CacheIntegrityError('Missing or cyclic portable dependency.')
        visiting.add(identity)
        stage, key = identity
        _safe_component(stage, 'stage'); _safe_hash(key)
        base = destination/'cache'/stage/key
        original = read_json(base/'complete.json')
        _safe_component(original['attempt_id'], 'attempt')
        directory = base/'attempts'/original['attempt_id']
        if file_hash(directory/'manifest.json') != original.get('manifest_sha256'):
            raise CacheIntegrityError('Original scientific manifest checksum changed during export.')
        manifest = read_json(directory/'manifest.json')
        for child in manifest.get('dependencies', []): rebuild((child['stage'], child['key']))
        bundle = restored.bundle(directory, manifest.get('calibration_ids', ()), manifest.get('dependencies', ()))
        atomic_json(base/'complete.json', {**original, 'bundle_sha256': file_hash(bundle),
            'restoration': {'restored_utc': utc_now(), 'archive_sha256': file_hash(archive_path),
                            'original_completion_pointer': original, 'reason': 'derived ZIP regenerated from verified raw artifacts'}})
        restored._load(stage, key)
        visiting.remove(identity); finished.add(identity)
    for identity in sorted(identities): rebuild(identity)
    restored._event({'action': 'portable_cache_restored', 'stage_count': len(finished), 'archive_sha256': file_hash(archive_path)})
    return restored


## Fixed mathematical problem and separate output metrics

The same matrix and right-hand side are used for every solver. The graph parameter gamma changes the mathematical problem; it is never tuned separately for competing algorithms. Residual and vector diagnostics require complete complex states and are labeled local. Hardware reports postselection, accepted shots, and probability distributions from actual counts.

In [ ]:
# CELL_ID: problems
"""Explicit fixed mathematical instances; gamma is a model parameter."""
from dataclasses import dataclass
import hashlib
import json
import time

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import cg, spsolve


@dataclass(frozen=True)
class Problem:
    name: str
    A: np.ndarray
    b: np.ndarray
    metadata: dict

    @property
    def instance_id(self):
        h = hashlib.sha256()
        for array in (self.A, self.b):
            a = np.asarray(array, dtype='<f8')
            h.update(str(a.shape).encode())
            h.update(a.tobytes())
        return h.hexdigest()[:20]


def graph_problem(n=4, family='cycle', gamma=2.0, seed=0):
    if not isinstance(n, int) or n < 3:
        raise ValueError('Graph size must be an integer >= 3.')
    if not np.isfinite(gamma) or gamma <= 0:
        raise ValueError('gamma must be positive and finite.')
    w = np.zeros((n, n))
    rng = np.random.default_rng(seed)
    if family not in ('cycle', 'path', 'weighted_cycle'):
        raise ValueError('Unknown graph family.')
    for i in range(n-1):
        value = float(rng.uniform(.5, 1.5)) if family == 'weighted_cycle' else 1.0
        w[i, i+1] = w[i+1, i] = value
    if family != 'path':
        value = float(rng.uniform(.5, 1.5)) if family == 'weighted_cycle' else 1.0
        w[0, -1] = w[-1, 0] = value
    L = np.diag(w.sum(axis=1)) - w
    A = L + gamma*np.eye(n)
    b = np.full(n, .1)
    b[0] = 1.
    eig = np.linalg.eigvalsh(A)
    return Problem(f'{family}_n{n}_gamma{gamma:g}_seed{seed}', A, b, {
        'family': family, 'n': n, 'gamma': gamma, 'graph_seed': seed,
        'lambda_min': float(eig[0]), 'lambda_max': float(eig[-1]),
        'condition_number_2': float(eig[-1]/eig[0]),
        'rhs': 'b[0]=1; b[1:]=0.1', 'matrix_definition': 'A=diag(W1)-W+gamma I'})


def rod_problem(n=4, left=100., right=0.):
    """n interior unknowns, prescribed endpoints excluded from x; no source."""
    A = 2*np.eye(n)-np.eye(n,k=1)-np.eye(n,k=-1)
    b = np.zeros(n)
    b[0] += left
    b[-1] += right
    return Problem(f'rod_interior_n{n}', A, b, {'n':n, 'family':'rod',
        'left_boundary':left,'right_boundary':right,
        'condition_number_2':float(np.linalg.cond(A)),
        'matrix_definition':'tridiagonal(-1,2,-1), interior unknowns only'})


def pad_problem(problem):
    """Embed A as A direct-sum lambda_max I and b as (b,0), preserving x."""
    n = len(problem.b)
    m = 1 << (n-1).bit_length()
    if m == n:
        return problem
    fill = float(np.linalg.eigvalsh(problem.A)[-1])
    A = fill*np.eye(m)
    A[:n,:n] = problem.A
    b = np.pad(problem.b, (0,m-n))
    return Problem(problem.name+'_padded', A, b,
        dict(problem.metadata, physical_dimension=n, encoded_dimension=m,
             padding_diagonal=fill))


def classical_baselines(problem):
    """Recorded local timings are diagnostics, not robust speedup estimates."""
    A = csr_matrix(problem.A)
    t = time.perf_counter()
    direct = spsolve(A, problem.b)
    direct_seconds = time.perf_counter()-t
    iterations = [0]
    def count(_): iterations[0] += 1
    t = time.perf_counter()
    iterative, info = cg(A, problem.b, rtol=1e-11, atol=0., callback=count,
                         maxiter=max(100,10*len(problem.b)))
    cg_seconds = time.perf_counter()-t
    return direct, {
        'sparse_direct_seconds':direct_seconds,
        'sparse_direct_residual':float(np.linalg.norm(A@direct-problem.b)/np.linalg.norm(problem.b)),
        'cg_seconds':cg_seconds,'cg_info':int(info),'cg_iterations':iterations[0],
        'cg_residual':float(np.linalg.norm(A@iterative-problem.b)/np.linalg.norm(problem.b)),
        'timing_scope':'one local solve after sparse conversion; no timing uncertainty'}


In [ ]:
# CELL_ID: metrics
"""Keep state diagnostics, reference-free reconstruction, and Z distributions distinct."""
import numpy as np


def normalized(v):
    v = np.asarray(v, dtype=complex)
    norm = np.linalg.norm(v)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError('A finite nonzero vector is required.')
    return v/norm


def reconstruct_from_state(A, b, state):
    """Residual-minimizing scale uses A,b and measured/reconstructed u only.

    This costs classical matrix-vector work and presupposes complete complex
    state access. It is a local diagnostic, NOT a hardware norm estimator.
    """
    u = normalized(state)
    Au = np.asarray(A)@u
    scale = np.vdot(Au,b)/np.vdot(Au,Au)
    return scale*u


def state_metrics(A, b, state, reference):
    u, target = normalized(state), normalized(reference)
    xhat = reconstruct_from_state(A,b,u)
    return {'state_fidelity_diagnostic':float(np.clip(abs(np.vdot(target,u))**2,0,1)),
        'residual_reference_free_scale':float(np.linalg.norm(A@xhat-b)/np.linalg.norm(b)),
        'relative_vector_error_diagnostic':float(np.linalg.norm(xhat-reference)/np.linalg.norm(reference)),
        'reconstruction_scope':'complex state available in simulation; scale from A and b; classical O(N^2) dense work'}


def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0 or not 0 <= successes <= total:
        raise ValueError('Require 0 <= successes <= positive total.')
    p = successes/total
    den = 1+z*z/total
    center = (p+z*z/(2*total))/den
    half = z*np.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return [float(max(0,center-half)),float(min(1,center+half))]


def conditional_counts(counts, system_qubits, postselect_bits):
    """Qiskit integer bit i corresponds to logical qubit i, one meas register."""
    selected = np.zeros(1 << len(system_qubits),dtype=np.int64)
    total = 0
    for bitstring, count in counts.items():
        if not isinstance(count,(int,np.integer)) or count < 0:
            raise ValueError('Counts must be nonnegative integers.')
        value = int(bitstring.replace(' ',''),2) if isinstance(bitstring,str) else int(bitstring)
        total += int(count)
        if all(((value >> int(i)) & 1) == int(v) for i,v in postselect_bits.items()):
            idx = sum(((value >> q) & 1) << j for j,q in enumerate(system_qubits))
            selected[idx] += count
    if total == 0:
        raise ValueError('No shots.')
    return selected,total


def probability_metrics(counts, target_probabilities, system_qubits, postselect_bits,
                        bootstrap_samples=400, seed=0):
    selected,total = conditional_counts(counts,system_qubits,postselect_bits)
    accepted = int(selected.sum())
    result = {'total_shots':total,'accepted_shots':accepted,
        'postselection_rate':accepted/total,
        'postselection_wilson95':wilson_interval(accepted,total),
        'conditional_counts':selected.tolist(),
        'uncertainty_scope':'single-job finite shots only; excludes drift, training and calibration uncertainty'}
    if not accepted:
        return dict(result,status='zero_postselection',total_variation=None,
                    classical_distribution_fidelity=None,tv_plugin_bootstrap95=None)
    q = np.asarray(target_probabilities,dtype=float)
    if q.shape != selected.shape or np.any(q < 0) or not np.isclose(q.sum(),1.):
        raise ValueError('Target must be a normalized probability vector of matching dimension.')
    p = selected/accepted
    tv = float(.5*np.abs(p-q).sum())
    fidelity = float(np.clip(np.sqrt(p*q).sum()**2,0,1))
    interval = None
    if bootstrap_samples:
        rng = np.random.default_rng(seed)
        draws = rng.multinomial(accepted,p,size=bootstrap_samples)/accepted
        vals = .5*np.abs(draws-q).sum(axis=1)
        interval = np.quantile(vals,[.025,.975]).tolist()
    return dict(result,status='ok',total_variation=tv,
        classical_distribution_fidelity=fidelity,tv_plugin_bootstrap95=interval,
        interpretation='Z-basis distribution similarity does not certify quantum-state fidelity or signs')


## Verified references and equivalent optimized circuits

HHL performs phase estimation, reciprocal rotation, and uncomputation, selecting success=1 and all phase bits=0. Its reciprocal scale depends on phase precision and is disclosed in metadata. QSVT uses a checked Hermitian block encoding and numerically fitted inverse phases with an independent dense validation grid; the grid is not a rigorous uniform-error certificate.

The optimized HHL multiplexes reciprocal rotations with UCRY. The optimized QSVT shares the block unitary between the real-part branches using sign-controlled phase rotations. Whole-state and unitary checks below verify equivalence to the inline references.

In [ ]:
# CELL_ID: reference_solvers
"""Auditable *small dense* linear-solver circuit references.

These routines use dense eigendecompositions, dense oracle synthesis, and exact
statevectors. They are validation instruments, not scalable input oracles and not
evidence of a computational advantage. No routine consults a classical solution
of A x = b to choose an output's sign, phase, or scale.
"""


import math
from typing import Any

import numpy as np
from scipy.linalg import expm
from scipy.optimize import least_squares, minimize
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate, RYGate, StatePreparation, UnitaryGate
from qiskit.quantum_info import Statevector


class PostselectionError(RuntimeError):
    """The specified successful work-register branch has zero probability."""


class PhaseSynthesisError(RuntimeError):
    """An inverse polynomial has not passed independent phase validation."""


def _validated_system(A: np.ndarray, b: np.ndarray | None = None):
    raw = np.asarray(A)
    if raw.ndim != 2 or raw.shape[0] != raw.shape[1]:
        raise ValueError("A must be a square matrix.")
    n = raw.shape[0]
    if n < 2 or n & (n - 1) or n > 64:
        raise ValueError("Dense references support power-of-two dimensions 2 through 64.")
    if not np.isfinite(raw).all() or np.max(np.abs(np.imag(raw))) > 1e-12:
        raise ValueError("This reference requires a finite real matrix.")
    matrix = np.asarray(raw.real, dtype=float)
    if not np.allclose(matrix, matrix.T, atol=1e-12, rtol=1e-12):
        raise ValueError("A must be real symmetric.")
    eigenvalues = np.linalg.eigvalsh(matrix)
    if eigenvalues[0] <= 0:
        raise ValueError("A must be strictly positive definite.")
    if b is None:
        return matrix, eigenvalues
    vector = np.asarray(b, dtype=complex)
    if vector.shape != (n,) or not np.isfinite(vector).all():
        raise ValueError("b must be a finite vector of length A.shape[0].")
    if np.max(np.abs(vector.imag)) > 1e-12:
        raise ValueError("This reference requires a real right-hand side.")
    norm = float(np.linalg.norm(vector))
    if norm == 0:
        raise ValueError("The zero right-hand side cannot be amplitude encoded.")
    return matrix, vector / norm, norm, eigenvalues


def _positive_integer(value: int, name: str) -> int:
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")
    return int(value)


def _circuit_metadata(circuit: QuantumCircuit) -> dict[str, Any]:
    return {
        "num_qubits": circuit.num_qubits,
        "logical_depth": circuit.depth(),
        "logical_operations": {str(k): int(v) for k, v in circuit.count_ops().items()},
        "count_convention": "before transpilation; includes state preparation; no measurements or resets; dense gates are composite oracles",
        "execution": "exact statevector simulation",
        "input_model": "dense classical matrix and amplitude-prepared b; classical eigendecomposition/oracle synthesis costs excluded from gate counts",
        "scalability_claim": False,
    }


def hhl_reference(
    A: np.ndarray,
    b: np.ndarray,
    n_qpe: int = 4,
    cutoff: float | None = None,
) -> dict[str, Any]:
    """Execute a dense HHL circuit with genuine phase estimation.

    Qubit order is [system least-significant bits, phase little-endian bits,
    success most-significant bit]. Controlled U^(2^j) with
    U=exp(+2*pi*i*A/(2*lambda_max)) precedes an inverse QFT including swaps.
    The phase integer k therefore estimates lambda_k=2*lambda_max*k/2^m.
    Zero phase is rejected. Other bins below the optional absolute eigenvalue
    cutoff are rejected. C=2*lambda_max/2^m keeps every retained C/lambda_k <=1.

    Success requires both success=1 AND every phase qubit=0 after inverse QPE.
    Finite phase resolution biases this branch: its probability does not recover
    the exact solution norm unless all relevant phases are exactly represented.
    """
    matrix, rhs, rhs_norm, eigenvalues = _validated_system(A, b)
    m = _positive_integer(n_qpe, "n_qpe")
    system_qubits = int(math.log2(len(rhs)))
    if m > 10 or system_qubits + m + 1 > 18:
        raise ValueError("This dense validation routine limits QPE to 10 and total qubits to 18.")
    lambda_max = float(eigenvalues[-1])
    bins = 1 << m
    spacing = 2.0 * lambda_max / bins
    if cutoff is None:
        cutoff = spacing
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be a positive finite eigenvalue threshold.")
    system = list(range(system_qubits))
    phase = list(range(system_qubits, system_qubits + m))
    success = system_qubits + m
    qpe = QuantumCircuit(system_qubits + m, name="QPE")
    for j, control in enumerate(phase):
        qpe.h(control)
        evolution = expm(2j * np.pi * matrix * (1 << j) / (2.0 * lambda_max))
        qpe.append(UnitaryGate(evolution, label=f"U^{1 << j}").control(), [control] + system)
    qpe.append(QFTGate(m).inverse(), phase)

    circuit = QuantumCircuit(system_qubits + m + 1, name="HHL dense reference")
    circuit.append(StatePreparation(rhs), system)
    circuit.compose(qpe, qubits=system + phase, inplace=True)
    retained_bins = []
    for k in range(1, bins):
        estimate = k * spacing
        if estimate + 1e-14 * lambda_max < cutoff:
            continue
        retained_bins.append(k)
        angle = 2 * np.arcsin(min(1.0, spacing / estimate))
        rotation = RYGate(float(angle)).control(m, ctrl_state=k)
        circuit.append(rotation, phase + [success])
    circuit.compose(qpe.inverse(), qubits=system + phase, inplace=True)
    full_state = Statevector.from_instruction(circuit).data
    offset = 1 << success
    amplitudes = full_state[offset : offset + len(rhs)].copy()
    probability = float(np.vdot(amplitudes, amplitudes).real)
    success_marginal = float(np.vdot(full_state[offset:], full_state[offset:]).real)
    if probability < 1e-28:
        raise PostselectionError("HHL has zero probability for success=1 and phase=0; inspect cutoff and QPE resolution.")
    exact_grid = bool(np.allclose(eigenvalues / spacing, np.rint(eigenvalues / spacing), atol=1e-9, rtol=0))
    metadata = _circuit_metadata(circuit)
    metadata.update({
        "algorithm": "HHL dense reference",
        "n_qpe": m,
        "lambda_min": float(eigenvalues[0]),
        "lambda_max": lambda_max,
        "condition_number": float(eigenvalues[-1] / eigenvalues[0]),
        "phase_fraction_definition": "lambda / (2 lambda_max)",
        "phase_interval": [0.0, 0.5],
        "eigenvalue_grid_spacing": spacing,
        "reciprocal_rotation_constant": spacing,
        "cutoff": float(cutoff),
        "retained_phase_bins": retained_bins,
        "spectrum_exactly_on_phase_grid": exact_grid,
        "minimum_eigenvalue_resolved": bool(eigenvalues[0] >= spacing * (1 - 1e-12)),
        "success_probability": probability,
        "success_qubit_marginal_probability": success_marginal,
        "postselection": "success=1 AND all QPE work qubits=0",
        "system_qubits": system,
        "qpe_qubits": phase,
        "success_qubit": success,
        "postselect_bits": {**{q: 0 for q in phase}, success: 1},
        "norm_estimate_exact_grid_formula": rhs_norm * math.sqrt(probability) / spacing,
        "norm_formula_is_exact": exact_grid and bool(cutoff <= eigenvalues[0] + 1e-12 * lambda_max),
        "cost_trace": [],
    })
    return {"state": amplitudes / math.sqrt(probability), "circuit": circuit, "metadata": metadata}


def _real_ansatz(num_qubits: int, parameters: np.ndarray, reps: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits, name="real RY-CX ansatz")
    index = 0
    for layer in range(reps + 1):
        for qubit in range(num_qubits):
            circuit.ry(float(parameters[index]), qubit)
            index += 1
        if layer < reps:
            for qubit in range(num_qubits - 1):
                circuit.cx(qubit, qubit + 1)
    return circuit


def vqls_statevector(
    A: np.ndarray,
    b: np.ndarray,
    reps: int = 2,
    max_evals: int = 300,
    seed: int = 0,
    tol: float = 1e-8,
    initial_parameters: np.ndarray | None = None,
) -> dict[str, Any]:
    """Optimize global VQLS cost using exact classical statevectors.

    C(v)=1-|<b_normalized|A|v>|^2 / <v|A^2|v>. COBYLA receives a
    deterministic seeded RY/CX ansatz, bounded angles, and an evaluation budget.
    Every cost evaluation is recorded. The returned state uses the best evaluated
    parameters, including when the optimizer terminates at a worse point.
    This is explicitly NOT a hardware-evaluated or shot-evaluated VQLS optimizer.
    """
    matrix, rhs, _, eigenvalues = _validated_system(A, b)
    if isinstance(reps, bool) or not isinstance(reps, (int, np.integer)) or reps < 0:
        raise ValueError("reps must be a nonnegative integer.")
    budget = _positive_integer(max_evals, "max_evals")
    if not np.isfinite(tol) or tol <= 0:
        raise ValueError("tol must be finite and positive.")
    n_qubits = int(math.log2(len(rhs)))
    n_parameters = n_qubits * (int(reps) + 1)
    if budget < n_parameters + 2:
        raise ValueError(f"COBYLA requires at least {n_parameters + 2} objective evaluations for this ansatz.")
    if initial_parameters is None:
        initial = np.random.default_rng(seed).uniform(-np.pi, np.pi, n_parameters)
    else:
        initial = np.asarray(initial_parameters, dtype=float).copy()
        if initial.shape != (n_parameters,) or not np.isfinite(initial).all() or np.any(np.abs(initial) > np.pi):
            raise ValueError("initial_parameters must be finite, in [-pi,pi], and match the ansatz size.")
    trace: list[dict[str, Any]] = []
    best_cost = float("inf")
    best_parameters = initial.copy()

    def objective(parameters):
        nonlocal best_cost, best_parameters
        parameters = np.clip(np.asarray(parameters), -np.pi, np.pi)
        state = Statevector.from_instruction(_real_ansatz(n_qubits, parameters, int(reps))).data
        image = matrix @ state
        denominator = float(np.vdot(image, image).real)
        cost = float(np.clip(1.0 - abs(np.vdot(rhs, image)) ** 2 / denominator, 0.0, 1.0))
        trace.append({"evaluation": len(trace) + 1, "cost": cost, "parameters": np.asarray(parameters).tolist()})
        if cost < best_cost:
            best_cost, best_parameters = cost, np.asarray(parameters).copy()
        return cost

    result = minimize(objective, initial, method="COBYLA", bounds=[(-np.pi, np.pi)] * n_parameters,
                      options={"maxiter": budget, "tol": float(tol), "rhobeg": 0.5, "catol": float(tol)})
    circuit = _real_ansatz(n_qubits, best_parameters, int(reps))
    state = Statevector.from_instruction(circuit).data
    metadata = _circuit_metadata(circuit)
    metadata.update({
        "algorithm": "VQLS statevector-optimized reference",
        "objective_evaluation_method": "classical exact statevector and dense matrix products",
        "hardware_optimizer": False,
        "ansatz": "RY on every qubit; linear nearest-neighbor CX after each nonfinal layer",
        "reps": int(reps), "parameter_count": n_parameters, "seed": int(seed),
        "initial_parameters": initial.tolist(), "parameter_bounds": [-math.pi, math.pi],
        "optimizer": "COBYLA", "max_evals": budget, "objective_evaluations": len(trace),
        "optimizer_success": bool(result.success), "optimizer_message": str(result.message),
        "optimizer_tolerance": float(tol), "best_cost": best_cost,
        "cost_trace": trace, "success_probability": 1.0, "postselection": "none",
        "system_qubits": list(range(n_qubits)), "postselect_bits": {},
        "condition_number": float(eigenvalues[-1] / eigenvalues[0]),
    })
    return {"state": state, "circuit": circuit, "parameters": best_parameters, "metadata": metadata}


def hermitian_block_encoding(A: np.ndarray, alpha: float | None = None) -> dict[str, Any]:
    """Construct and verify U=[[B,sqrt(I-B^2)],[sqrt(I-B^2),-B]].

    B=A/alpha, alpha>=||A||_2. The block flag is the highest qubit. This
    Hermitian unitary block encodes A/alpha; it does NOT alone implement A^-1.
    """
    matrix, eigenvalues = _validated_system(A)
    if alpha is None:
        alpha = float(eigenvalues[-1])
    alpha = float(alpha)
    if not np.isfinite(alpha) or alpha < eigenvalues[-1] * (1 - 1e-12):
        raise ValueError("alpha must be finite and at least the spectral norm of A.")
    values, vectors = np.linalg.eigh(matrix / alpha)
    B = matrix / alpha
    complement = (vectors * np.sqrt(np.clip(1 - values ** 2, 0, None))) @ vectors.T
    unitary = np.block([[B, complement], [complement, -B]])
    error = float(np.linalg.norm(unitary.T @ unitary - np.eye(len(unitary)), ord=2))
    if error > 1e-10:
        raise ValueError(f"Block encoding failed unitarity validation: {error}.")
    return {"unitary": unitary, "scaled_matrix": B, "alpha": alpha,
            "metadata": {"unitarity_error": error, "block_error": float(np.linalg.norm(unitary[:len(B), :len(B)] - B)),
                         "construction": "dense eigendecomposition", "inverse_implemented": False}}


def _phase_response(phases: np.ndarray, x: np.ndarray, jacobian: bool = False):
    """Re <0|D(phi0) U(x) ... U(x) D(phid)|0>, with optional Jacobian."""
    x = np.asarray(x, dtype=float)
    U = np.empty((len(x), 2, 2), dtype=complex)
    U[:, 0, 0], U[:, 1, 1] = x, -x
    U[:, 0, 1] = U[:, 1, 0] = np.sqrt(np.maximum(0, 1 - x * x))
    rotations = [np.diag([np.exp(1j * p), np.exp(-1j * p)]) for p in phases]
    prefix = [np.broadcast_to(np.eye(2, dtype=complex), U.shape)]
    for rotation in rotations[:-1]:
        prefix.append(prefix[-1] @ rotation @ U)
    response = (prefix[-1] @ rotations[-1])[:, 0, 0].real
    if not jacobian:
        return response
    suffix = [None] * len(phases)
    suffix[-1] = np.broadcast_to(np.eye(2, dtype=complex), U.shape)
    for i in range(len(phases) - 2, -1, -1):
        suffix[i] = U @ rotations[i + 1] @ suffix[i + 1]
    Z = np.diag([1j, -1j])
    jac = np.column_stack([(prefix[i] @ rotations[i] @ Z @ suffix[i])[:, 0, 0].real
                           for i in range(len(phases))])
    return response, jac


def synthesize_inverse_phases(
    interval_min: float,
    degree: int = 15,
    tolerance: float = 0.02,
    seed: int = 0,
    restarts: int = 4,
    max_nfev: int = 500,
) -> dict[str, Any]:
    """Fit a real QSVT inverse polynomial and reject failed validation.

    Target P(x)=c/x on [a,1], c=a/2, with odd degree. Numerical nonlinear
    phase fitting is intentionally limited to tiny diagnostic workloads. The
    error is independently checked on 4,097 points, including interval endpoints;
    this is a numerical grid validation, not a proof of a uniform error bound.
    Unitarity of the phase sequence ensures |P(x)|<=1 on [-1,1].
    """
    a = float(interval_min)
    degree = _positive_integer(degree, "degree")
    restarts = _positive_integer(restarts, "restarts")
    max_nfev = _positive_integer(max_nfev, "max_nfev")
    if not 0 < a <= 1 or not np.isfinite(a):
        raise ValueError("interval_min must lie in (0,1].")
    if degree % 2 != 1 or degree > 63:
        raise ValueError("Inverse phase synthesis requires odd degree <=63.")
    if not np.isfinite(tolerance) or tolerance <= 0:
        raise ValueError("tolerance must be finite and positive.")
    scale = a / 2
    # Endpoint-dense fit grid; the independent validation grid is uniformly spaced.
    fit_grid = a + (1 - a) * (1 - np.cos(np.linspace(0, np.pi, max(64, 8 * degree)))) / 2
    validation_grid = np.linspace(a, 1, 4097)
    rng = np.random.default_rng(seed)
    best = None
    attempts = []
    for restart in range(restarts):
        initial = rng.uniform(-np.pi, np.pi, degree + 1)
        result = least_squares(lambda p: _phase_response(p, fit_grid) - scale / fit_grid,
                               initial, jac=lambda p: _phase_response(p, fit_grid, True)[1],
                               max_nfev=max_nfev, ftol=1e-11, xtol=1e-11, gtol=1e-11)
        phases = (result.x + np.pi) % (2 * np.pi) - np.pi
        observed_error = float(np.max(np.abs(_phase_response(phases, validation_grid) - scale / validation_grid)))
        attempts.append({"restart": restart, "nfev": int(result.nfev), "njev": int(result.njev or 0),
                         "validation_max_error": observed_error, "optimizer_success": bool(result.success)})
        if best is None or observed_error < best[0]:
            best = (observed_error, phases)
        if observed_error <= tolerance:
            break
    if best[0] > tolerance:
        raise PhaseSynthesisError(f"Degree {degree} inverse phase fit failed: grid error {best[0]:.6g} exceeds {tolerance:.6g} on [{a:.6g},1]. Increase degree or relax a declared tolerance.")
    bounded_grid = np.linspace(-1, 1, 4097)
    observed_magnitude = float(np.max(np.abs(_phase_response(best[1], bounded_grid))))
    if observed_magnitude > 1 + 1e-10:
        raise PhaseSynthesisError("Phase sequence failed boundedness check.")
    return {"phases": best[1], "metadata": {
        "degree": degree, "interval": [a, 1.0], "inverse_polynomial_scale": scale,
        "phase_seed": int(seed), "phase_synthesis_attempts": attempts,
        "phase_validation_grid_points": 4097, "phase_validation_max_error": best[0],
        "phase_validation_tolerance": float(tolerance),
        "validation_type": "independent dense numerical grid, not a rigorous interval bound",
        "observed_max_magnitude_on_unit_interval": observed_magnitude,
        "phases": best[1].tolist(),
        "sequence_convention": "D(phi0) U D(phi1) ... U D(phid), D(phi)=exp(i phi Z); real part extracted coherently",
    }}


def qsvt_reference(
    A: np.ndarray,
    b: np.ndarray,
    degree: int = 15,
    phase_tolerance: float = 0.02,
    seed: int = 0,
    phase_restarts: int = 4,
) -> dict[str, Any]:
    """Execute a phase-validated dense QSVT inverse with real-part extraction.

    A Hermitian block encoding U of B=A/alpha is interleaved with block-flag
    phase rotations. A second flag coherently selects W(phi) and W(-phi),
    making its zero branch Re W. Both flags must be zero. For real symmetric
    B the projected block is the real odd polynomial P(B), fitted to c/B.
    This routine rejects phase fits that miss the declared numerical tolerance.
    """
    matrix, rhs, rhs_norm, eigenvalues = _validated_system(A, b)
    encoding = hermitian_block_encoding(matrix)
    alpha = encoding["alpha"]
    phases_result = synthesize_inverse_phases(float(eigenvalues[0] / alpha), degree,
                                               phase_tolerance, seed, phase_restarts)
    phases = phases_result["phases"]
    n = int(math.log2(len(rhs)))
    system = list(range(n))
    block_flag, real_flag = n, n + 1
    signal_qubits = system + [block_flag]

    def sequence(sequence_phases):
        qc = QuantumCircuit(n + 1, name="QSVT phase sequence")
        # Circuit instructions multiply from right to left.
        qc.rz(float(-2 * sequence_phases[-1]), block_flag)
        oracle = UnitaryGate(encoding["unitary"], label="U(A/alpha)")
        for p in sequence_phases[-2::-1]:
            qc.append(oracle, signal_qubits)
            qc.rz(float(-2 * p), block_flag)
        return qc

    W = sequence(phases)
    W_conjugate = sequence(-phases)
    circuit = QuantumCircuit(n + 2, name="QSVT dense inverse reference")
    circuit.append(StatePreparation(rhs), system)
    circuit.h(real_flag)
    circuit.append(W.to_gate().control(1, ctrl_state=0), [real_flag] + signal_qubits)
    circuit.append(W_conjugate.to_gate().control(1, ctrl_state=1), [real_flag] + signal_qubits)
    circuit.h(real_flag)
    full_state = Statevector.from_instruction(circuit).data
    amplitudes = full_state[:len(rhs)].copy()
    probability = float(np.vdot(amplitudes, amplitudes).real)
    if probability < 1e-28:
        raise PostselectionError("QSVT successful flag branch has zero probability.")
    # Independent eigenspace prediction checks circuit convention and extraction.
    values, vectors = np.linalg.eigh(matrix / alpha)
    expected = vectors @ (_phase_response(phases, values) * (vectors.T @ rhs))
    circuit_error = float(np.linalg.norm(amplitudes - expected))
    if circuit_error > 1e-9:
        raise PhaseSynthesisError(f"QSVT circuit disagrees with its polynomial: {circuit_error}.")
    metadata = _circuit_metadata(circuit)
    metadata.update(phases_result["metadata"])
    metadata.update({
        "algorithm": "QSVT dense phase-validated inverse reference",
        "alpha": alpha, "condition_number": float(eigenvalues[-1] / eigenvalues[0]),
        "block_encoding_unitarity_error": encoding["metadata"]["unitarity_error"],
        "circuit_polynomial_action_error": circuit_error,
        "controlled_block_encoding_calls": 2 * int(degree),
        "block_encoding_calls_per_branch": int(degree),
        "real_part_extraction_overhead": "two controlled sequences, one extra flag and two Hadamards",
        "success_probability": probability,
        "postselection": "block flag=0 AND real-part flag=0",
        "system_qubits": system, "postselect_bits": {block_flag: 0, real_flag: 0},
        "norm_estimate_inverse_polynomial_formula": rhs_norm * math.sqrt(probability) / (phases_result["metadata"]["inverse_polynomial_scale"] * alpha),
        "norm_formula_is_exact": False,
        "norm_formula_note": "The correct exact-inverse formula is ||b|| sqrt(p)/(c alpha); finite polynomial error biases it.",
        "cost_trace": [],
    })
    return {"state": amplitudes / math.sqrt(probability), "circuit": circuit, "metadata": metadata}


qsvt_inverse = qsvt_reference


In [ ]:
# CELL_ID: optimized_solvers
# Optimized genuine HHL/QSVT circuits. This cell follows the inline reference
# solver definitions; it imports no project modules or external code files.
# Dependencies supplied by that cell: _validated_system, _positive_integer,
# _circuit_metadata, PostselectionError, PhaseSynthesisError,
# hermitian_block_encoding, synthesize_inverse_phases, and _phase_response.
# Primary API references (accessed 2026-09-13):
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.UCRYGate
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RZZGate
import math
from typing import Any
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, RYGate, StatePreparation, UnitaryGate, UCRYGate
from qiskit.quantum_info import Operator, Statevector

def optimized_hhl_reference(
    A: np.ndarray,
    b: np.ndarray,
    n_qpe: int = 4,
    cutoff: float | None = None,
) -> dict[str, Any]:
    """Execute a dense HHL circuit with genuine phase estimation.

    Qubit order is [system least-significant bits, phase little-endian bits,
    success most-significant bit]. Controlled U^(2^j) with
    U=exp(+2*pi*i*A/(2*lambda_max)) precedes an inverse QFT including swaps.
    The phase integer k therefore estimates lambda_k=2*lambda_max*k/2^m.
    Zero phase is rejected. Other bins below the optional absolute eigenvalue
    cutoff are rejected. C=2*lambda_max/2^m keeps every retained C/lambda_k <=1.

    Success requires both success=1 AND every phase qubit=0 after inverse QPE.
    Finite phase resolution biases this branch: its probability does not recover
    the exact solution norm unless all relevant phases are exactly represented.
    """
    matrix, rhs, rhs_norm, eigenvalues = _validated_system(A, b)
    m = _positive_integer(n_qpe, "n_qpe")
    system_qubits = int(math.log2(len(rhs)))
    if m > 10 or system_qubits + m + 1 > 18:
        raise ValueError("This dense validation routine limits QPE to 10 and total qubits to 18.")
    lambda_max = float(eigenvalues[-1])
    bins = 1 << m
    spacing = 2.0 * lambda_max / bins
    if cutoff is None:
        cutoff = spacing
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be a positive finite eigenvalue threshold.")
    system = list(range(system_qubits))
    phase = list(range(system_qubits, system_qubits + m))
    success = system_qubits + m
    qpe = QuantumCircuit(system_qubits + m, name="QPE")
    for j, control in enumerate(phase):
        qpe.h(control)
        evolution = expm(2j * np.pi * matrix * (1 << j) / (2.0 * lambda_max))
        qpe.append(UnitaryGate(evolution, label=f"U^{1 << j}").control(), [control] + system)
    qpe.append(QFTGate(m).inverse(), phase)

    circuit = QuantumCircuit(system_qubits + m + 1, name="HHL multiplexed dense reference")
    circuit.append(StatePreparation(rhs), system)
    circuit.compose(qpe, qubits=system + phase, inplace=True)
    retained_bins = []
    reciprocal_angles = [0.0] * bins
    for k in range(1, bins):
        estimate = k * spacing
        if estimate + 1e-14 * lambda_max < cutoff:
            continue
        retained_bins.append(k)
        angle = 2 * np.arcsin(min(1.0, spacing / estimate))
        reciprocal_angles[k] = float(angle)
    # UCRY uses target first, then little-endian controls.
    circuit.append(UCRYGate(reciprocal_angles), [success] + phase)
    circuit.compose(qpe.inverse(), qubits=system + phase, inplace=True)
    full_state = Statevector.from_instruction(circuit).data
    offset = 1 << success
    amplitudes = full_state[offset : offset + len(rhs)].copy()
    probability = float(np.vdot(amplitudes, amplitudes).real)
    success_marginal = float(np.vdot(full_state[offset:], full_state[offset:]).real)
    if probability < 1e-28:
        raise PostselectionError("HHL has zero probability for success=1 and phase=0; inspect cutoff and QPE resolution.")
    exact_grid = bool(np.allclose(eigenvalues / spacing, np.rint(eigenvalues / spacing), atol=1e-9, rtol=0))
    metadata = _circuit_metadata(circuit)
    metadata.update({
        "algorithm": "HHL dense reference",
        "implementation": "uniformly controlled reciprocal RY multiplexor",
        "reciprocal_rotation_angles": reciprocal_angles,
        "reciprocal_synthesis": "one UCRY, same full reciprocal transfer function as separate controlled RY rotations",
        "n_qpe": m,
        "lambda_min": float(eigenvalues[0]),
        "lambda_max": lambda_max,
        "condition_number": float(eigenvalues[-1] / eigenvalues[0]),
        "phase_fraction_definition": "lambda / (2 lambda_max)",
        "phase_interval": [0.0, 0.5],
        "eigenvalue_grid_spacing": spacing,
        "reciprocal_rotation_constant": spacing,
        "cutoff": float(cutoff),
        "retained_phase_bins": retained_bins,
        "spectrum_exactly_on_phase_grid": exact_grid,
        "minimum_eigenvalue_resolved": bool(eigenvalues[0] >= spacing * (1 - 1e-12)),
        "success_probability": probability,
        "success_qubit_marginal_probability": success_marginal,
        "postselection": "success=1 AND all QPE work qubits=0",
        "system_qubits": system,
        "qpe_qubits": phase,
        "success_qubit": success,
        "postselect_bits": {**{q: 0 for q in phase}, success: 1},
        "norm_estimate_exact_grid_formula": rhs_norm * math.sqrt(probability) / spacing,
        "norm_formula_is_exact": exact_grid and bool(cutoff <= eigenvalues[0] + 1e-12 * lambda_max),
        "cost_trace": [],
    })
    return {"state": amplitudes / math.sqrt(probability), "circuit": circuit, "metadata": metadata}


def optimized_qsvt_reference(
    A: np.ndarray,
    b: np.ndarray,
    degree: int = 15,
    phase_tolerance: float = 0.02,
    seed: int = 0,
    phase_restarts: int = 4,
) -> dict[str, Any]:
    """Execute a phase-validated dense QSVT inverse with real-part extraction.

    A Hermitian block encoding U of B=A/alpha is interleaved with block-flag
    phase rotations. A second flag coherently selects W(phi) and W(-phi),
    making its zero branch Re W. Both flags must be zero. For real symmetric
    B the projected block is the real odd polynomial P(B), fitted to c/B.
    This routine rejects phase fits that miss the declared numerical tolerance.
    """
    matrix, rhs, rhs_norm, eigenvalues = _validated_system(A, b)
    encoding = hermitian_block_encoding(matrix)
    alpha = encoding["alpha"]
    phases_result = synthesize_inverse_phases(float(eigenvalues[0] / alpha), degree,
                                               phase_tolerance, seed, phase_restarts)
    phases = phases_result["phases"]
    n = int(math.log2(len(rhs)))
    system = list(range(n))
    block_flag, real_flag = n, n + 1
    signal_qubits = system + [block_flag]

    circuit = QuantumCircuit(n + 2, name="QSVT shared-oracle dense inverse")
    circuit.append(StatePreparation(rhs), system)
    circuit.h(real_flag)
    # exp(i phi Z_block Z_real) selects D(+phi) for real=0 and D(-phi)
    # for real=1. U is identical in both branches and is applied just once.
    circuit.rzz(float(-2 * phases[-1]), block_flag, real_flag)
    oracle = UnitaryGate(encoding["unitary"], label="U(A/alpha)")
    for p in phases[-2::-1]:
        circuit.append(oracle, signal_qubits)
        circuit.rzz(float(-2 * p), block_flag, real_flag)
    circuit.h(real_flag)
    full_state = Statevector.from_instruction(circuit).data
    amplitudes = full_state[:len(rhs)].copy()
    probability = float(np.vdot(amplitudes, amplitudes).real)
    if probability < 1e-28:
        raise PostselectionError("QSVT successful flag branch has zero probability.")
    # Independent eigenspace prediction checks circuit convention and extraction.
    values, vectors = np.linalg.eigh(matrix / alpha)
    expected = vectors @ (_phase_response(phases, values) * (vectors.T @ rhs))
    circuit_error = float(np.linalg.norm(amplitudes - expected))
    if circuit_error > 1e-9:
        raise PhaseSynthesisError(f"QSVT circuit disagrees with its polynomial: {circuit_error}.")
    metadata = _circuit_metadata(circuit)
    metadata.update(phases_result["metadata"])
    metadata.update({
        "algorithm": "QSVT dense phase-validated inverse reference",
        "alpha": alpha, "condition_number": float(eigenvalues[-1] / eigenvalues[0]),
        "block_encoding_unitarity_error": encoding["metadata"]["unitarity_error"],
        "circuit_polynomial_action_error": circuit_error,
        "implementation": "shared unconditioned block encoding with signed RZZ phases",
        "controlled_block_encoding_calls": 0,
        "uncontrolled_block_encoding_calls": int(degree),
        "signed_phase_rzz_calls": int(degree) + 1,
        "block_encoding_calls_per_branch": int(degree),
        "real_part_extraction_overhead": "one shared sequence, d+1 signed RZZ phases, one extra flag and two Hadamards",
        "success_probability": probability,
        "postselection": "block flag=0 AND real-part flag=0",
        "system_qubits": system, "postselect_bits": {block_flag: 0, real_flag: 0},
        "norm_estimate_inverse_polynomial_formula": rhs_norm * math.sqrt(probability) / (phases_result["metadata"]["inverse_polynomial_scale"] * alpha),
        "norm_formula_is_exact": False,
        "norm_formula_note": "The correct exact-inverse formula is ||b|| sqrt(p)/(c alpha); finite polynomial error biases it.",
        "cost_trace": [],
    })
    return {"state": amplitudes / math.sqrt(probability), "circuit": circuit, "metadata": metadata}


def _operator_or_state_error_up_to_global_phase(reference, candidate):
    """Diagnostic only: verify circuit equivalence, never choose solver scale."""
    reference = np.asarray(reference, dtype=complex)
    candidate = np.asarray(candidate, dtype=complex)
    overlap = np.vdot(reference.ravel(), candidate.ravel())
    if abs(overlap) < 1e-30:
        return float(np.linalg.norm(reference - candidate))
    phase = overlap / abs(overlap)
    return float(np.linalg.norm(candidate - phase * reference))


def _offline_basis_resources(circuit, seed_transpiler=73):
    """Offline all-to-all CX basis counts, not a claim about a physical device."""
    compiled = transpile(circuit, basis_gates=["rz", "sx", "x", "cx"],
                         optimization_level=2, seed_transpiler=seed_transpiler)
    gates = [item for item in compiled.data if item.operation.name not in ("measure", "barrier", "delay", "reset")]
    return {
        "basis": ["rz", "sx", "x", "cx"], "optimization_level": 2,
        "seed_transpiler": int(seed_transpiler), "topology": "all-to-all, no hardware backend",
        "qubits": compiled.num_qubits, "depth": compiled.depth(),
        "two_qubit_gates": sum(len(item.qubits) == 2 for item in gates),
        "one_qubit_gates": sum(len(item.qubits) == 1 for item in gates),
        "operation_counts": {str(k): int(v) for k, v in compiled.count_ops().items()},
        "counts_include": "complete state preparation, solver and ancilla uncomputation; no measurement",
    }


def verify_optimized_solvers(include_resource_counts=True):
    """Check exact transfer functions and whole states against inline originals.

    This runs entirely locally. The baseline functions hhl_reference and
    qsvt_reference must remain bound to their original inline implementations.
    Two non-diagonal SPD fixtures have dimensions 4 and 8. Tests compare the
    complete, unpostselected state, the successful state, and success rates.
    An exhaustive multiplexor operator test also includes rejected phase bins.
    """
    report = {"status": "running", "hardware_calls": 0, "checks": [], "comparisons": []}

    # Exhaustively compare the reciprocal gate on every basis state, including
    # success=1 input. This catches control order and relative-phase mistakes.
    for m in [1, 3, 4]:
        bins = 1 << m
        # Explicit cutoff keeps bin zero and, when m>1, bin one rejected.
        first_retained = 1 if m == 1 else 2
        angles = [0.0 if k < first_retained else float(2 * np.arcsin(1 / k)) for k in range(bins)]
        separate = QuantumCircuit(m + 1)
        for k, angle in enumerate(angles):
            if angle:
                separate.append(RYGate(angle).control(m, ctrl_state=k), list(range(m)) + [m])
        multiplexed = QuantumCircuit(m + 1)
        multiplexed.append(UCRYGate(angles), [m] + list(range(m)))
        error = _operator_or_state_error_up_to_global_phase(Operator(separate).data, Operator(multiplexed).data)
        assert error < 1e-10, f"UCRY full operator mismatch at m={m}: {error}"
        report["checks"].append({"check": "reciprocal_operator_equivalence", "n_qpe": m, "error": error})

    for dimension in [4, 8]:
        rng = np.random.default_rng(712 + dimension)
        eigenvectors, _ = np.linalg.qr(rng.normal(size=(dimension, dimension)))
        eigenvalues = (np.array([1.0, 1.25, 1.5, 2.0]) if dimension == 4
                       else np.linspace(1.0, 2.0, dimension))
        A = (eigenvectors * eigenvalues) @ eigenvectors.T
        b = rng.normal(size=dimension)
        implementations = [
            ("hhl", hhl_reference, optimized_hhl_reference, {"n_qpe": 4, "cutoff": 0.5}),
            ("qsvt", qsvt_reference, optimized_qsvt_reference,
             {"degree": 7, "phase_tolerance": 0.02, "seed": 0}),
        ]
        for algorithm, original_function, optimized_function, parameters in implementations:
            original = original_function(A, b, **parameters)
            optimized = optimized_function(A, b, **parameters)
            reference_full = Statevector.from_instruction(original["circuit"]).data
            optimized_full = Statevector.from_instruction(optimized["circuit"]).data
            full_error = _operator_or_state_error_up_to_global_phase(reference_full, optimized_full)
            output_error = _operator_or_state_error_up_to_global_phase(original["state"], optimized["state"])
            success_error = abs(original["metadata"]["success_probability"] - optimized["metadata"]["success_probability"])
            assert full_error < 1e-9, f"{algorithm}, N={dimension}: whole-state error {full_error}"
            assert output_error < 1e-9, f"{algorithm}, N={dimension}: selected-state error {output_error}"
            assert success_error < 1e-10, f"{algorithm}, N={dimension}: success-rate error {success_error}"
            assert original["metadata"]["postselect_bits"] == optimized["metadata"]["postselect_bits"]
            comparison = {"algorithm": algorithm, "dimension": dimension, "parameters": parameters,
                          "fixture": "seeded real orthogonal eigenbasis; non-diagonal SPD matrix",
                          "full_state_error_up_to_phase": full_error,
                          "selected_state_error_up_to_phase": output_error,
                          "absolute_success_probability_difference": success_error}
            if algorithm == "qsvt":
                unitary_error = _operator_or_state_error_up_to_global_phase(
                    Operator(original["circuit"]).data, Operator(optimized["circuit"]).data)
                assert unitary_error < 1e-8, f"QSVT complete-unitary mismatch: {unitary_error}"
                comparison["complete_unitary_error_up_to_phase"] = unitary_error
                assert optimized["metadata"]["controlled_block_encoding_calls"] == 0
                assert optimized["metadata"]["uncontrolled_block_encoding_calls"] == parameters["degree"]
            if include_resource_counts:
                comparison["original_resources"] = _offline_basis_resources(original["circuit"])
                comparison["optimized_resources"] = _offline_basis_resources(optimized["circuit"])
                before = comparison["original_resources"]["two_qubit_gates"]
                after = comparison["optimized_resources"]["two_qubit_gates"]
                comparison["two_qubit_gate_reduction_fraction"] = 1 - after / before
            report["comparisons"].append(comparison)
    report["status"] = "passed"
    return report


## VQLS cost from measured expectations

The objective estimates M=A|b̂⟩⟨b̂|A and D=A² using grouped Pauli basis measurements. It preserves bit order, raw counts, and within-group covariance. The ratio cost is finite-shot biased and is not clipped to [0,1]. A separate final confirmation uses an independent seed and is charged in addition to the optimization budget.

Local measured optimization uses actual sampler counts. The physical pilot evaluates the final measured-trained ansatz and its five cost settings; it does not claim that optimization occurred on the QPU. Every local objective sample is immediately checkpointed and packaged.

The collected pilot cost settings had different physical layouts; their aggregate is a diagnostic only. The phase-validation cells check the common-state theorem and reject a hardware fidelity interpretation. A separate prospective stage constructs all five settings from one saved physical preparation, verifies the prefix and basis transformations locally, and submits no job.

In [ ]:
# CELL_ID: measured_cost
# Measured global VQLS cost: all definitions live in this notebook cell.
# Requires the earlier inline _real_ansatz(num_qubits, parameters, reps) helper.
# No IBM connection is made here. The default executor is a local shot sampler.
import hashlib
import math
import time
from collections.abc import Mapping

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector


_MVQLS_OBSERVABLE_CACHE = {}


def _mvqls_integer(value, name, minimum=1):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < minimum:
        raise ValueError(f"{name} must be an integer >= {minimum}.")
    return int(value)


def _mvqls_system(A, b):
    matrix = np.asarray(A, dtype=complex)
    vector = np.asarray(b, dtype=complex)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("A must be a square matrix.")
    dim = matrix.shape[0]
    if dim < 2 or dim > 64 or dim & (dim - 1):
        raise ValueError("This dense measured-cost reference supports dimensions 2,4,...,64.")
    if not np.isfinite(matrix).all() or not np.allclose(matrix, matrix.conj().T, rtol=1e-12, atol=1e-12):
        raise ValueError("A must be finite and Hermitian.")
    eig = np.linalg.eigvalsh(matrix)
    if eig[0] <= 0:
        raise ValueError("A must be positive definite.")
    if vector.shape != (dim,) or not np.isfinite(vector).all() or np.linalg.norm(vector) == 0:
        raise ValueError("b must be a finite, nonzero vector with matching dimension.")
    return matrix, vector / np.linalg.norm(vector), eig


def _mvqls_model(A, b):
    """Cache classical observable decompositions, never cached objective samples."""
    matrix, rhs, eig = _mvqls_system(A, b)
    digest = hashlib.sha256()
    digest.update(np.asarray(matrix, dtype='<c16').tobytes())
    digest.update(np.asarray(rhs, dtype='<c16').tobytes())
    key = digest.hexdigest()
    if key in _MVQLS_OBSERVABLE_CACHE:
        return _MVQLS_OBSERVABLE_CACHE[key]
    started = time.perf_counter()
    image = matrix @ rhs
    dense = {"numerator": np.outer(image, image.conj()), "denominator": matrix @ matrix}
    n = int(math.log2(len(rhs)))
    identity = 'I' * n
    coefficients = {"numerator": {}, "denominator": {}}
    validation = {}
    for name, observable in dense.items():
        # atol=rtol=0: no intentional Pauli truncation; roundoff is checked below.
        operator = SparsePauliOp.from_operator(Operator(observable), atol=0.0, rtol=0.0)
        imaginary = float(np.max(np.abs(operator.coeffs.imag)))
        scale = max(1.0, float(np.linalg.norm(observable, ord=2)))
        if imaginary > 1e-11 * scale:
            raise ValueError(f"{name} has unexpectedly complex Hermitian Pauli coefficients.")
        for label, coefficient in zip(operator.paulis.to_labels(), operator.coeffs.real):
            if coefficient != 0.0:
                coefficients[name][label] = float(coefficient)
        reconstructed = SparsePauliOp.from_list(list(coefficients[name].items())).to_matrix()
        error = float(np.linalg.norm(reconstructed - observable, ord=2))
        if error > 1e-10 * scale:
            raise ValueError(f"{name} failed dense-to-Pauli reconstruction: {error}.")
        validation[name] = {"pauli_terms": len(coefficients[name]),
                            "operator_reconstruction_error_2": error,
                            "maximum_coefficient_imaginary_roundoff": imaginary}
    labels = (set(coefficients['numerator']) | set(coefficients['denominator'])) - {identity}
    # A deterministic first-fit qubit-wise-commuting grouping of the union.
    # Reuse each measured Pauli for both numerator and denominator.
    groups = []
    for label in sorted(labels, key=lambda label: (-sum(c != 'I' for c in label), label)):
        for group in groups:
            if all(x == 'I' or y == 'I' or x == y for x, y in zip(group['basis'], label)):
                group['basis'] = ''.join(y if x == 'I' else x for x, y in zip(group['basis'], label))
                group['labels'].append(label)
                break
        else:
            groups.append({'basis': label, 'labels': [label]})
    final_groups = []
    for group in groups:
        setting = group['basis'].replace('I', 'Z')
        terms = []
        for label in group['labels']:
            mask = sum(1 << q for q, axis in enumerate(reversed(label)) if axis != 'I')
            terms.append({'label': label, 'parity_mask': mask,
                          'numerator_coefficient': coefficients['numerator'].get(label, 0.0),
                          'denominator_coefficient': coefficients['denominator'].get(label, 0.0)})
        final_groups.append({'setting': setting, 'terms': terms})
    final_groups.sort(key=lambda g: g['setting'])
    if len({g['setting'] for g in final_groups}) != len(final_groups):
        raise ValueError("Duplicate measurement settings detected.")
    model = {'model_id': key, 'num_qubits': n, 'dimension': len(rhs),
             'groups': final_groups, 'coefficients': coefficients,
             'identity_offsets': {name: coefficients[name].get(identity, 0.0) for name in coefficients},
             'numerator_matrix': dense['numerator'], 'denominator_matrix': dense['denominator'],
             'lambda_min_A': float(eig[0]), 'lambda_max_A': float(eig[-1]),
             'denominator_exact_lower_bound': float(eig[0] ** 2),
             'denominator_numeric_floor': max(np.finfo(float).tiny, 1e-12 * float(eig[-1] ** 2)),
             'decomposition_validation': validation,
             'classical_decomposition_seconds': time.perf_counter() - started,
             'observable_definition': 'M=A|b_normalized><b_normalized|A; D=A^2; A Hermitian',
             'decomposition_scope': 'dense exact Pauli decomposition up to floating-point roundoff; no intentional truncation; exponential preprocessing/measurement settings possible',
             'bit_order': 'Pauli label leftmost is highest qubit; measured classical bit q equals logical qubit q',
             'measurement_basis': 'X: H then Z-readout; Y: Sdg then H then Z-readout; Z: Z-readout'}
    # Bounded cache: eviction does not alter a plan already returned to the caller.
    if len(_MVQLS_OBSERVABLE_CACHE) >= 32:
        _MVQLS_OBSERVABLE_CACHE.pop(next(iter(_MVQLS_OBSERVABLE_CACHE)))
    _MVQLS_OBSERVABLE_CACHE[key] = model
    return model


def _mvqls_basis_circuits(ansatz, groups):
    circuits = []
    for group in groups:
        circuit = ansatz.copy()
        circuit.name = 'vqls_cost_' + group['setting']
        for q, axis in enumerate(reversed(group['setting'])):
            if axis == 'X':
                circuit.h(q)
            elif axis == 'Y':
                circuit.sdg(q)
                circuit.h(q)
        circuit.measure_all()
        circuit.metadata = dict(circuit.metadata or {}, vqls_measurement_setting=group['setting'])
        circuits.append(circuit)
    return circuits


def measurement_plan(A, b, params, reps):
    """Build measurable global-VQLS cost circuits for one parameter vector.

    The returned circuits use one ``meas`` register and no ancilla. All Pauli
    terms in a group are qubit-wise compatible. The count-based analysis must
    retain covariance between terms sharing a setting. M,D construction is
    classical preprocessing, not free/scalable quantum data access.
    """
    reps = _mvqls_integer(reps, 'reps', minimum=0)
    model = _mvqls_model(A, b)
    raw_parameters = np.asarray(params)
    if np.iscomplexobj(raw_parameters) and np.any(raw_parameters.imag != 0):
        raise ValueError('Ansatz angles must be real; complex values are not discarded.')
    parameters = np.asarray(raw_parameters.real, dtype=float)
    n_parameters = model['num_qubits'] * (reps + 1)
    if parameters.shape != (n_parameters,) or not np.isfinite(parameters).all():
        raise ValueError(f"params must contain {n_parameters} finite real ansatz angles.")
    ansatz = _real_ansatz(model['num_qubits'], parameters, reps)
    circuits = _mvqls_basis_circuits(ansatz, model['groups'])
    param_id = hashlib.sha256(np.asarray(parameters, dtype='<f8').tobytes()).hexdigest()
    return {'plan_id': model['model_id'] + ':' + str(reps) + ':' + param_id,
            'model_id': model['model_id'], 'model': model,
            'num_qubits': model['num_qubits'], 'reps': reps,
            'parameters': parameters.tolist(), 'ansatz': ansatz,
            'groups': model['groups'], 'circuits': circuits,
            'settings': [group['setting'] for group in model['groups']],
            'circuits_per_evaluation': len(circuits),
            'logical_circuit_depths': [circuit.depth() for circuit in circuits],
            'denominator_guard_sigma': 3.0,
            'output_scope': 'counts-based cost; not a reconstructed solution vector'}


def _mvqls_clean_counts(raw, num_qubits):
    if not isinstance(raw, Mapping) or not raw:
        raise ValueError("Each setting needs a nonempty raw-count mapping.")
    cleaned = {}
    for key, count in raw.items():
        if isinstance(count, (bool, np.bool_)) or not isinstance(count, (int, np.integer)) or count < 0:
            raise ValueError("Raw counts must be nonnegative integers; quasiprobabilities are not counts.")
        if isinstance(key, str):
            # One logical register only: multiple-register strings require an
            # explicit bit-map adaptation in the executor before this analysis.
            if not key or any(c not in '01' for c in key) or len(key) != num_qubits:
                raise ValueError("Count strings must be binary and match the single measured register width.")
            value = int(key, 2)
        elif isinstance(key, (int, np.integer)) and not isinstance(key, (bool, np.bool_)):
            value = int(key)
        else:
            raise ValueError("Count keys must be binary strings or integers.")
        if value < 0 or value >= 1 << num_qubits:
            raise ValueError("Count key is outside the measured register.")
        bitstring = format(value, f'0{num_qubits}b')
        cleaned[bitstring] = cleaned.get(bitstring, 0) + int(count)
    if sum(cleaned.values()) <= 0:
        raise ValueError("The setting has no shots.")
    return cleaned


def analyze_cost(plan, counts):
    """Estimate M,D and unclipped C=1-M/D from measured bitstrings.

    Numerator/denominator estimates are linear sample means. Their ratio is
    generally biased. Error bars use an asymptotic delta method with within-
    setting covariance, assuming independent shots and independently executed
    settings; they exclude optimizer selection, readout error and drift.
    Invalid/uncertain denominators return a failure status, never a clipped cost.
    """
    if not isinstance(counts, Mapping) or set(counts) != set(plan['settings']):
        raise ValueError("counts must contain exactly one mapping per planned setting.")
    model = plan['model']
    estimate = np.array([model['identity_offsets']['numerator'], model['identity_offsets']['denominator']], dtype=float)
    covariance = np.zeros((2, 2), dtype=float)
    group_records, cleaned_all = [], {}
    sufficient = True
    for group in plan['groups']:
        setting = group['setting']
        cleaned = _mvqls_clean_counts(counts[setting], plan['num_qubits'])
        cleaned_all[setting] = cleaned
        shots = sum(cleaned.values())
        first, second = np.zeros(2), np.zeros((2, 2))
        for bitstring, count in cleaned.items():
            value = int(bitstring, 2)
            scores = np.zeros(2)
            for term in group['terms']:
                sign = -1.0 if (value & term['parity_mask']).bit_count() % 2 else 1.0
                scores += sign * np.array([term['numerator_coefficient'], term['denominator_coefficient']])
            first += count * scores
            second += count * np.outer(scores, scores)
        mean = first / shots
        estimate += mean
        cov_mean = None
        if shots >= 2:
            # Unbiased sample covariance divided by number of shots.
            cov_mean = (second / shots - np.outer(mean, mean)) / (shots - 1)
            cov_mean = (cov_mean + cov_mean.T) / 2
            # Remove only negative floating-point variance, not observed data.
            cov_mean[0, 0] = max(0.0, cov_mean[0, 0])
            cov_mean[1, 1] = max(0.0, cov_mean[1, 1])
            covariance += cov_mean
        else:
            sufficient = False
        group_records.append({'setting': setting, 'shots': shots,
                              'numerator_contribution': float(mean[0]),
                              'denominator_contribution': float(mean[1]),
                              'sample_mean_covariance_MD': None if cov_mean is None else cov_mean.tolist()})
    numerator, denominator = map(float, estimate)
    denom_se = math.sqrt(max(0.0, covariance[1, 1])) if sufficient else None
    positive = bool(denominator > model['denominator_numeric_floor'])
    stable = bool(positive and sufficient and denominator - plan['denominator_guard_sigma'] * denom_se > model['denominator_numeric_floor'])
    cost = float(1.0 - numerator / denominator) if positive else None
    cost_se = None
    if positive and sufficient:
        gradient = np.array([-1 / denominator, numerator / denominator ** 2])
        cost_se = float(math.sqrt(max(0.0, float(gradient @ covariance @ gradient))))
    status = ('ok' if stable else 'insufficient_shots' if not sufficient
              else 'nonpositive_denominator' if not positive else 'denominator_not_resolved')
    return {'status': status, 'cost_unclipped': cost,
            'cost_standard_error_delta': cost_se,
            'cost_normal95_diagnostic': None if cost_se is None else [cost - 1.96 * cost_se, cost + 1.96 * cost_se],
            'numerator_estimate': numerator, 'denominator_estimate': denominator,
            'denominator_standard_error': denom_se,
            'denominator_exact_lower_bound': model['denominator_exact_lower_bound'],
            'denominator_numeric_floor': model['denominator_numeric_floor'],
            'denominator_guard_sigma': plan['denominator_guard_sigma'],
            'cost_is_outside_physical_interval': bool(cost is not None and (cost < 0 or cost > 1)),
            'covariance_MD': covariance.tolist() if sufficient else None,
            'group_statistics': group_records, 'raw_counts': cleaned_all,
            'total_shots': sum(g['shots'] for g in group_records),
            'measurement_settings': len(group_records),
            'ratio_is_unbiased': False,
            'uncertainty_scope': 'delta-method shot uncertainty with within-setting covariance; excludes selection, calibration, readout bias and temporal drift',
            'guard_note': 'Positive/three-SE denominator guard is a numerical/statistical diagnostic, not a rigorous positivity certificate; no exact eigenvalue replacement or cost clipping.'}


def local_measured_vqls_executor(plan, *, shots, seed, evaluation):
    """Actual local StatevectorSampler measurements, with independent group seeds."""
    shots = _mvqls_integer(shots, 'shots', minimum=2)
    counts, group_seeds = {}, {}
    for index, (setting, circuit) in enumerate(zip(plan['settings'], plan['circuits'])):
        group_seed = int(np.random.SeedSequence([int(seed), int(evaluation), index]).generate_state(1)[0])
        sampler = StatevectorSampler(default_shots=shots, seed=group_seed)
        result = sampler.run([circuit], shots=shots).result()
        counts[setting] = result[0].data.meas.get_counts()
        group_seeds[setting] = group_seed
    return {'counts': counts,
            'metadata': {'execution': 'local StatevectorSampler finite-shot measurements',
                         'hardware_execution': False, 'group_seeds': group_seeds,
                         'shots_per_setting_requested': shots, 'evaluation': int(evaluation)}}


class MeasuredVQLSEvaluationError(RuntimeError):
    """Execution or denominator guard failed; partial experiment is preserved."""


def measured_vqls(A, b, reps=2, max_evals=100, seed=0, shots=4096,
                  evaluate_callback=None, initial_parameters=None, tol=1e-3):
    """Optimize a counted-measurement global cost, preserving every evaluation.

    Callback contract: executor(plan, shots=..., seed=..., evaluation=...) returns
    {'counts': {setting: raw_counts}, 'metadata': {...}}. A bare counts mapping
    is also accepted. It receives all logical circuits and parameters, so the
    notebook's executor can cache/transpile and persist records as they arrive.
    This function does not contact IBM; the default executor is purely local.

    A bad denominator stops optimization and returns the partial trace instead
    of fabricating a cost or silently retrying. Best observed noisy cost is an
    optimizer selection statistic, not an independently confirmed final score.
    A fresh, independently seeded final cost evaluation is always attempted for
    the best available point and recorded separately from max_evals.
    """
    reps = _mvqls_integer(reps, 'reps', minimum=0)
    budget = _mvqls_integer(max_evals, 'max_evals')
    seed = _mvqls_integer(seed, 'seed', minimum=0)
    shots = _mvqls_integer(shots, 'shots', minimum=2)
    model = _mvqls_model(A, b)
    n_parameters = model['num_qubits'] * (reps + 1)
    if budget < n_parameters + 2:
        raise ValueError(f"COBYLA needs at least {n_parameters + 2} evaluations for this ansatz.")
    if not np.isfinite(tol) or tol <= 0:
        raise ValueError('tol must be finite and positive.')
    if initial_parameters is not None:
        supplied_initial = np.asarray(initial_parameters)
        if np.iscomplexobj(supplied_initial) and np.any(supplied_initial.imag != 0):
            raise ValueError('initial_parameters must be real.')
    initial = (np.random.default_rng(seed).uniform(-np.pi, np.pi, n_parameters)
               if initial_parameters is None else np.asarray(supplied_initial.real, dtype=float).copy())
    if initial.shape != (n_parameters,) or not np.isfinite(initial).all() or np.any(np.abs(initial) > np.pi):
        raise ValueError('initial_parameters must match the ansatz, be finite and lie in [-pi,pi].')
    executor = evaluate_callback or local_measured_vqls_executor
    trace, best = [], {'cost': float('inf'), 'parameters': None, 'evaluation': None}
    started = time.perf_counter()

    def evaluate(parameters, evaluation, role):
        # Periodic angle wrapping preserves the prepared state up to global
        # phase, unlike clipping a COBYLA point to a different physical point.
        parameters = (np.asarray(parameters, dtype=float) + np.pi) % (2 * np.pi) - np.pi
        plan = measurement_plan(A, b, parameters, reps)
        eval_seed = int(np.random.SeedSequence([seed, int(evaluation), 7321]).generate_state(1)[0])
        entry = {'evaluation': int(evaluation), 'role': role, 'parameters': parameters.tolist(),
                 'plan_id': plan['plan_id'], 'model_id': plan['model_id'],
                 'sampling_seed': eval_seed, 'shots_per_setting_requested': shots,
                 'measurement_settings': plan['settings'], 'circuits_per_evaluation': len(plan['circuits'])}
        call_started = time.perf_counter()
        response = None
        try:
            response = executor(plan, shots=shots, seed=eval_seed, evaluation=evaluation)
            if isinstance(response, Mapping) and 'counts' in response:
                raw, executor_metadata = response['counts'], response.get('metadata', {})
            else:
                raw, executor_metadata = response, {}
            # Preserve returned raw evidence before validation, including a
            # malformed/partial callback result that analysis may reject.
            entry['raw_counts'] = raw
            entry['executor_metadata'] = executor_metadata
            analysis = analyze_cost(plan, raw)
            entry['analysis'] = analysis
            entry['status'] = analysis['status']
            if analysis['status'] != 'ok':
                raise MeasuredVQLSEvaluationError('VQLS measurement guard: ' + analysis['status'])
        except Exception as error:
            entry.setdefault('status', 'execution_or_analysis_failed')
            entry['error_type'], entry['error_message'] = type(error).__name__, str(error)
            raise MeasuredVQLSEvaluationError(str(error)) from error
        finally:
            entry['wall_seconds'] = time.perf_counter() - call_started
            trace.append(entry)
        return float(analysis['cost_unclipped']), parameters

    def objective(parameters):
        index = sum(e['role'] == 'optimization' for e in trace) + 1
        if index > budget:
            raise MeasuredVQLSEvaluationError('Optimizer exceeded the declared objective-evaluation budget.')
        value, prepared = evaluate(parameters, index, 'optimization')
        if value < best['cost']:
            best.update(cost=value, parameters=prepared.copy(), evaluation=index)
        return value

    optimizer_result, failure = None, None
    try:
        optimizer_result = minimize(objective, initial, method='COBYLA',
                                    options={'maxiter': budget, 'tol': float(tol), 'rhobeg': 0.5})
    except MeasuredVQLSEvaluationError as error:
        failure = str(error)
    confirmation = None
    if best['parameters'] is not None:
        try:
            evaluate(best['parameters'], budget + 1, 'independent_final_confirmation')
        except MeasuredVQLSEvaluationError as error:
            failure = failure or ('Final confirmation: ' + str(error))
        confirmation = trace[-1]
    final_parameters = best['parameters']
    circuit = None if final_parameters is None else _real_ansatz(model['num_qubits'], final_parameters, reps)
    model_record = {k: v for k, v in model.items() if k not in ('numerator_matrix', 'denominator_matrix')}
    metadata = {'algorithm': 'VQLS measured-Pauli-cost optimizer',
                'objective_evaluation_method': 'Pauli basis rotations and counted measurements of M,D',
                'execution': 'local finite-shot optimization' if evaluate_callback is None else 'callback-defined counted-measurement optimization; inspect each evaluation metadata',
                'hardware_optimizer': bool(trace) and all(e.get('executor_metadata', {}).get('hardware_execution') is True for e in trace),
                'seed': seed, 'initial_parameters': initial.tolist(), 'ansatz': 'inline _real_ansatz RY/CX',
                'reps': reps, 'max_evals': budget,
                'objective_evaluations': sum(e['role'] == 'optimization' for e in trace),
                'confirmation_evaluations': sum(e['role'] != 'optimization' for e in trace),
                'total_measurement_circuits_requested': sum(e['circuits_per_evaluation'] for e in trace),
                'total_shots_observed': sum(e.get('analysis', {}).get('total_shots', 0) for e in trace),
                'optimizer': 'COBYLA; finite-shot unclipped cost; periodic angle wrapping',
                'optimizer_success': False if optimizer_result is None else bool(optimizer_result.success),
                'optimizer_message': failure if optimizer_result is None else str(optimizer_result.message),
                'best_observed_cost': None if final_parameters is None else best['cost'],
                'best_evaluation': best['evaluation'], 'cost_trace': trace,
                'success_probability': 1.0, 'postselection': 'none',
                'system_qubits': list(range(model['num_qubits'])), 'postselect_bits': {},
                'local_wall_seconds': time.perf_counter() - started,
                'measurement_cost_note': 'Every optimizer call uses shots per setting, not shots total; one extra final confirmation is charged separately.',
                'inference_note': 'Final ansatz is provided without using the classical solution; complex/signed solution reconstruction requires a separate protocol.',
                'ansatz_scope': 'Real-amplitude RY/CX ansatz; complex A,b are supported by the cost algebra but arbitrary complex solution states are outside this ansatz.',
                'scope': 'Small dense classical Pauli preprocessing plus measured expectations; no quantum speedup or hardware execution is implied.'}
    return {'status': 'ok' if failure is None and final_parameters is not None else 'partial_or_failed',
            'failure': failure, 'parameters': final_parameters, 'circuit': circuit,
            'metadata': metadata, 'measurement_model': model_record,
            'evaluations': trace, 'independent_final_cost': confirmation,
            'state': None}


def validate_measured_vqls_component():
    """Bounded local validation; returns an auditable JSON-compatible report.

    Call explicitly from the notebook's validation cell. This performs actual
    local sampling but no IBM calls, file writes, or external module imports.
    Synthetic large-count fixtures below are clearly separated from sampler
    measurements and validate arbitrary complex-state measurement algebra.
    """
    from qiskit.circuit.library import StatePreparation
    checks = []
    rng = np.random.default_rng(82931)
    max_exact_error, max_count_error = 0.0, 0.0
    saw_Y = False
    for dim in (2, 4, 8):
        for complex_fixture in (False, True):
            raw = rng.normal(size=(dim, dim))
            if complex_fixture:
                raw = raw + 1j * rng.normal(size=(dim, dim))
            basis, _ = np.linalg.qr(raw)
            A = (basis * np.linspace(1, 2, dim)) @ basis.conj().T
            b = rng.normal(size=dim)
            state = rng.normal(size=dim)
            if complex_fixture:
                b = b + 1j * rng.normal(size=dim)
                state = state + 1j * rng.normal(size=dim)
            state = state / np.linalg.norm(state)
            n = int(math.log2(dim))
            plan = measurement_plan(A, b, rng.normal(size=n), reps=0)
            preparation = QuantumCircuit(n)
            preparation.append(StatePreparation(state), list(range(n)))
            # Only this test overrides the real production ansatz, to exercise Y.
            circuits = _mvqls_basis_circuits(preparation, plan['groups'])
            estimate = np.array([plan['model']['identity_offsets']['numerator'],
                                 plan['model']['identity_offsets']['denominator']])
            synthetic = {}
            synthetic_shots = 1 << 24
            for group, circuit in zip(plan['groups'], circuits):
                saw_Y = saw_Y or 'Y' in group['setting']
                probabilities = Statevector.from_instruction(circuit.remove_final_measurements(inplace=False)).probabilities()
                for term in group['terms']:
                    # Direct left-to-right Pauli/bitstring interpretation is
                    # independent of the production integer-parity mask.
                    expectation = 0.0
                    for index, probability in enumerate(probabilities):
                        bits = format(index, f'0{n}b')
                        parity = sum(int(bit) for axis, bit in zip(term['label'], bits) if axis != 'I')
                        expectation += probability * (-1 if parity % 2 else 1)
                    estimate += expectation * np.array([term['numerator_coefficient'], term['denominator_coefficient']])
                scaled = probabilities / probabilities.sum() * synthetic_shots
                integer_counts = np.floor(scaled).astype(np.int64)
                remainder = synthetic_shots - int(integer_counts.sum())
                if remainder:
                    integer_counts[np.argsort(-(scaled - integer_counts))[:remainder]] += 1
                synthetic[group['setting']] = {format(i, f'0{n}b'): int(v) for i, v in enumerate(integer_counts) if v}
            bhat = b / np.linalg.norm(b)
            image = A @ state
            dense_expectation = np.array([abs(np.vdot(bhat, image)) ** 2, np.vdot(image, image).real])
            error = float(np.max(np.abs(estimate - dense_expectation)))
            analysis = analyze_cost(plan, synthetic)
            count_error = float(np.max(np.abs(np.array([analysis['numerator_estimate'], analysis['denominator_estimate']]) - dense_expectation)))
            assert error < 1e-10, ('complex/Pauli exact expectation', dim, error)
            assert count_error < 1e-5, ('integer count endian/basis', dim, count_error)
            assert analysis['status'] == 'ok'
            max_exact_error, max_count_error = max(max_exact_error, error), max(max_count_error, count_error)
            checks.append({'name': f'Pauli_and_count_algebra_N{dim}_complex{complex_fixture}',
                           'passed': True, 'settings': len(plan['settings']),
                           'exact_expectation_error': error, 'synthetic_count_error': count_error})
    assert saw_Y
    plan = measurement_plan(np.diag([1., 2.]), [1., 1.], [.7], 0)
    covariance_counts = {'X': {'0': 65, '1': 35}, 'Z': {'0': 80, '1': 20}}
    analyzed = analyze_cost(plan, covariance_counts)
    varX, varZ = 4 * .65 * .35 / 99, 4 * .8 * .2 / 99
    covariance_expected = np.array([[varX + .75 ** 2 * varZ, .75 * 1.5 * varZ],
                                    [.75 * 1.5 * varZ, 1.5 ** 2 * varZ]])
    np.testing.assert_allclose(analyzed['covariance_MD'], covariance_expected, atol=1e-14)
    unclipped = analyze_cost(plan, {'X': {'0': 100}, 'Z': {'0': 100}})
    assert unclipped['cost_unclipped'] < 0 and unclipped['cost_is_outside_physical_interval']
    checks.append({'name': 'shared_setting_covariance_and_unclipped_ratio', 'passed': True,
                   'negative_cost_preserved': unclipped['cost_unclipped']})

    guard_plan = measurement_plan(np.array([[1.5, .8], [.8, .5]]), [1, 2], [.2], 0)
    bad = analyze_cost(guard_plan, {s: {'1': 100} for s in guard_plan['settings']})
    unresolved = analyze_cost(guard_plan, {s: {'0': 20, '1': 80} for s in guard_plan['settings']})
    assert bad['status'] == 'nonpositive_denominator' and bad['cost_unclipped'] is None
    assert unresolved['status'] == 'denominator_not_resolved'
    checks.append({'name': 'denominator_positivity_and_statistical_guards', 'passed': True})

    image = np.diag([1., 2.]) @ Statevector.from_instruction(plan['ansatz']).data
    exact_cost = float(1 - abs(np.vdot(np.ones(2) / np.sqrt(2), image)) ** 2 / np.vdot(image, image).real)
    rmse_by_shots = {}
    for shots in (256, 8192):
        errors = []
        for trial in range(12):
            result = local_measured_vqls_executor(plan, shots=shots, seed=8100 + trial, evaluation=1)
            analysis = analyze_cost(plan, result['counts'])
            assert analysis['status'] == 'ok'
            errors.append(analysis['cost_unclipped'] - exact_cost)
        rmse_by_shots[str(shots)] = float(np.sqrt(np.mean(np.square(errors))))
    assert rmse_by_shots['8192'] < rmse_by_shots['256']
    assert rmse_by_shots['8192'] < .025
    checks.append({'name': 'actual_sampler_cost_converges_with_shots_12_seeds', 'passed': True,
                   'root_mean_squared_error': rmse_by_shots})

    first = measured_vqls(np.diag([1., 2.]), [1, 1], reps=0, max_evals=35, seed=3, shots=8192)
    second = measured_vqls(np.diag([1., 2.]), [1, 1], reps=0, max_evals=35, seed=3, shots=8192)
    assert first['status'] == second['status'] == 'ok'
    np.testing.assert_array_equal(first['parameters'], second['parameters'])
    assert len(first['evaluations']) == len(second['evaluations'])
    for left, right in zip(first['evaluations'], second['evaluations']):
        assert left['parameters'] == right['parameters'] and left['raw_counts'] == right['raw_counts']
    output = Statevector.from_instruction(first['circuit']).data
    target = np.array([1., .5]) / np.linalg.norm([1., .5])
    fidelity = float(abs(np.vdot(target, output)) ** 2)
    assert fidelity > .98
    assert first['metadata']['objective_evaluations'] <= 35
    assert first['metadata']['confirmation_evaluations'] == 1
    assert first['independent_final_cost']['role'] == 'independent_final_confirmation'
    assert all(e['raw_counts'] for e in first['evaluations'])
    checks.append({'name': 'shot_optimizer_determinism_budget_and_complete_trace', 'passed': True,
                   'objective_evaluations': first['metadata']['objective_evaluations'],
                   'heldout_confirmation_cost': first['independent_final_cost']['analysis']['cost_unclipped'],
                   'statevector_fidelity_for_validation_only': fidelity})

    def guard_failure_executor(plan, *, shots, seed, evaluation):
        return {'counts': {s: {'1': shots} for s in plan['settings']},
                'metadata': {'execution': 'synthetic failure fixture', 'hardware_execution': False}}
    failed = measured_vqls(np.array([[1.5, .8], [.8, .5]]), [1, 2], reps=0,
                          max_evals=5, shots=100, evaluate_callback=guard_failure_executor)
    assert failed['status'] == 'partial_or_failed'
    assert len(failed['evaluations']) == 1
    assert failed['evaluations'][0]['analysis']['status'] == 'nonpositive_denominator'
    assert failed['evaluations'][0]['raw_counts']
    checks.append({'name': 'failed_objective_preserves_counts_and_stops', 'passed': True})
    return {'status': 'passed', 'checks_passed': len(checks), 'checks': checks,
            'maximum_exact_expectation_error': max_exact_error,
            'maximum_synthetic_integer_count_error': max_count_error,
            'hardware_calls': 0,
            'scope': 'Bounded deterministic local tests; synthetic-count algebra fixtures are separate from actual StatevectorSampler shot measurements.'}


In [ ]:
# CELL_ID: phase_validation
"""Phase-sensitive validation of the notebook's counted global VQLS witness.

This cell performs no device access. It uses the earlier measured_cost cell only
to test that cell's actual grouping, basis rotations, bit ordering and estimator.
The exact-state/density-matrix functions and applicability audit are independent.
Reference for global VQLS cost: https://arxiv.org/abs/1909.05820 .
The mixed-state extension below is derived explicitly here; it is not attributed
to an unverified experimental certification claim in that reference.
"""
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector


MIXED_STATE_WITNESS_PROOF = r"""
Let A be positive definite with eigenvalues in [a,b], let beta be the normalized
right-hand side, and let x=A^{-1}beta/||A^{-1}beta||. Define P=I-|beta><beta|,
H=A P A, D=A^2, and let rho be any positive-semidefinite trace-one density matrix.
Set E=Tr(rho H), d=Tr(rho D), C=E/d, and F=<x|rho|x> (squared pure-target
fidelity convention). H has kernel span{x}. The nonzero eigenvalues of H=K^dag K,
K=P A, equal those of K K^dag=P A^2 P on beta-perpendicular. For every vector y
in that subspace, <y|P A^2 P|y>=<y|A^2|y> >= a^2 ||y||^2. Therefore
H >= a^2(I-|x><x|), so E >= a^2(1-F). Since d<=b^2, E=C d<=C b^2. Consequently
F >= max(0,1-E/a^2) >= max(0,1-kappa(A)^2 C).
This operator proof covers arbitrary mixed rho, not only pure ansatz states.
For a pure normalized state v, minimizing ||s A v - b_rhs||/||b_rhs|| over a
complex scalar s yields residual sqrt(C). A mixed rho has no single vector
residual, so no such residual is assigned to mixed states.

Applicability: M=A|beta><beta|A and D must be expectations on one common rho.
Different physical preparation circuits/layouts or drift between settings can
produce an aggregate that is not Tr(rho M), Tr(rho D) for a common state.
Basis rotations/readout also change the implemented POVM unless controlled.
Finite-shot M_hat/D_hat is biased; negative or >1 measured costs are not clipped.
The notebook's delta-method intervals are diagnostics, not rigorous coverage
bounds, and do not control readout bias, preparation mismatch or drift. Thus
plugging raw hardware costs or their delta-method intervals into the exact
inequality does NOT establish a certified hardware fidelity lower bound.
"""


def _phase_witness_inputs(A, b):
    A = np.asarray(A, dtype=complex)
    b = np.asarray(b, dtype=complex)
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] < 1 or not np.isfinite(A).all():
        raise ValueError('A must be a finite square matrix.')
    if not np.allclose(A, A.conj().T, atol=1e-12, rtol=1e-12):
        raise ValueError('A must be Hermitian.')
    eigenvalues = np.linalg.eigvalsh(A)
    if eigenvalues[0] <= 0:
        raise ValueError('A must be positive definite.')
    if b.shape != (len(A),) or not np.isfinite(b).all() or np.linalg.norm(b) == 0:
        raise ValueError('b must be finite, nonzero, and dimensionally compatible.')
    beta = b / np.linalg.norm(b)
    M = np.outer(A @ beta, (A @ beta).conj())
    D = A @ A
    H = D - M
    return A, b, beta, M, D, H, eigenvalues


def exact_phase_witness_diagnostics(A, b, state_or_density):
    """Small dense mathematical diagnostics, never inferred from raw hardware cost.

    An explicitly supplied complex state or density matrix is required. Solving
    classically for the target is used only to check the theorem's diagnostic
    fidelity; it is not used by any solver to set phase, sign, or scale.
    """
    A, b, beta, M, D, H, eigenvalues = _phase_witness_inputs(A, b)
    supplied = np.asarray(state_or_density, dtype=complex)
    pure = supplied.ndim == 1
    vector = None
    if pure:
        if supplied.shape != (len(A),) or not np.isfinite(supplied).all() or np.linalg.norm(supplied) == 0:
            raise ValueError('A finite nonzero state of matching dimension is required.')
        vector = supplied / np.linalg.norm(supplied)
        rho = np.outer(vector, vector.conj())
    else:
        rho = supplied
        if rho.shape != A.shape or not np.isfinite(rho).all():
            raise ValueError('Density matrix dimensions or entries are invalid.')
        if not np.allclose(rho, rho.conj().T, atol=1e-12, rtol=0) or not np.isclose(np.trace(rho), 1, atol=1e-12, rtol=0):
            raise ValueError('Density matrix must be Hermitian with trace one.')
        if np.linalg.eigvalsh(rho)[0] < -1e-12:
            raise ValueError('Density matrix must be positive semidefinite.')
    numerator = float(np.trace(rho @ M).real)
    denominator = float(np.trace(rho @ D).real)
    energy = float(np.trace(rho @ H).real)
    cost = energy / denominator
    target = np.linalg.solve(A, beta)
    target /= np.linalg.norm(target)
    fidelity = float(np.vdot(target, rho @ target).real)
    projector = np.outer(target, target.conj())
    psd_margin = float(np.linalg.eigvalsh(H - eigenvalues[0] ** 2 * (np.eye(len(A)) - projector))[0])
    residual = None
    if pure:
        image = A @ vector
        scale = np.vdot(image, b) / np.vdot(image, image)
        residual = float(np.linalg.norm(scale * image - b) / np.linalg.norm(b))
    kappa = float(eigenvalues[-1] / eigenvalues[0])
    # Only bound values from exact matrices have roundoff clamped. Measured cost
    # estimates are never passed here, and their out-of-range values stay intact.
    return {'state_kind': 'pure' if pure else 'density_matrix',
        'cost_exact_float': cost, 'numerator_exact': numerator, 'denominator_exact': denominator,
        'witness_energy_exact': energy, 'condition_number': kappa,
        'target_fidelity_exact_diagnostic': fidelity,
        'fidelity_bound_from_exact_energy': float(np.clip(1 - energy / eigenvalues[0] ** 2, 0, 1)),
        'fidelity_bound_from_exact_cost': float(np.clip(1 - kappa ** 2 * cost, 0, 1)),
        'operator_inequality_minimum_eigenvalue_roundoff': psd_margin,
        'pure_reference_free_residual': residual,
        'mixed_vector_residual_defined': False,
        'z_probabilities': np.diag(rho).real.tolist(),
        'hardware_fidelity_certification': False,
        'scope': 'exact supplied state/density matrix; classical small-system validation only'}


def audit_phase_witness_applicability(compiled_entries, *, stationarity_evidence=None,
                                      measurement_error_control=None, expected_settings=None):
    """Reject a fidelity interpretation without common preparation and mapping.

    Records may provide physical_preparation_sha256 and preparation_layout for
    a demonstrably shared compiled preparation prefix. Final layout is reported
    as a diagnostic but cannot by itself prove identical preparation. Supplying
    these declarations still does not create a finite-shot certificate.
    """
    entries = [e for e in compiled_entries if e.get('kind') == 'cost']
    layouts = {str(e.get('setting')): list(e.get('final_layout') or []) for e in entries}
    preparation_layouts = {str(e.get('setting')): e.get('preparation_layout') for e in entries}
    prefix_ids = [e.get('physical_preparation_sha256') for e in entries]
    reasons = []
    settings = [str(e.get('setting')) for e in entries]
    if len(settings) != len(set(settings)):
        reasons.append('Cost-setting compilation records contain repeated settings.')
    if expected_settings is None:
        reasons.append('The complete required cost-setting set is not supplied for validation.')
    elif set(settings) != set(expected_settings):
        reasons.append('Cost-setting compilation records do not match the required measurement plan.')
    if not entries:
        reasons.append('No cost-setting compilation records were supplied.')
    if len({tuple(value) for value in layouts.values()}) > 1:
        reasons.append('Cost settings have heterogeneous ordered physical final layouts.')
    if not prefix_ids or any(not value for value in prefix_ids) or len(set(prefix_ids)) != 1:
        reasons.append('An identical compiled physical preparation prefix is not established for every setting.')
    if any(value is None for value in preparation_layouts.values()) or len({tuple(value or []) for value in preparation_layouts.values()}) != 1:
        reasons.append('A common ordered physical preparation layout is not established.')
    if not stationarity_evidence:
        reasons.append('A common stationary prepared state across settings is not established.')
    if measurement_error_control not in ('ideal_measurements', 'independently_validated_error_bounds'):
        reasons.append('Basis-rotation and readout errors have no validated treatment for this witness.')
    return {'classification': 'heterogeneous_layout_diagnostic_only' if len({tuple(value) for value in layouts.values()}) > 1
                            else 'assumptions_not_established' if reasons else 'common_state_assumptions_declared',
        'ordered_physical_layout_by_setting': layouts,
        'expected_settings': None if expected_settings is None else sorted(expected_settings),
        'preparation_layout_by_setting': preparation_layouts,
        'common_state_theorem_assumptions_declared': not reasons,
        'reasons': reasons,
        'finite_shot_certification_implemented': False,
        'hardware_fidelity_lower_bound': None,
        'scope': 'Applicability audit only. Even matching declarations require finite-sample and measurement-error bounds before any certification.'}


def _phase_pauli_matrix(label):
    matrices = {'I': np.eye(2), 'X': np.array([[0, 1], [1, 0]]),
                'Y': np.array([[0, -1j], [1j, 0]]), 'Z': np.diag([1, -1])}
    result = np.ones((1, 1), dtype=complex)
    for axis in label:
        result = np.kron(result, matrices[axis])
    return result


def _phase_integer_fixture(probabilities, total):
    probabilities = np.asarray(probabilities, dtype=float)
    scaled = probabilities / probabilities.sum() * total
    values = np.floor(scaled).astype(np.int64)
    remainder = total - int(values.sum())
    if remainder:
        values[np.argsort(-(scaled - values), kind='stable')[:remainder]] += 1
    width = int(np.log2(len(values)))
    return {format(i, f'0{width}b'): int(value) for i, value in enumerate(values) if value}


def validate_phase_sensitive_witness(*, compiled_entries=None, seed=20260917,
                                     shots_per_setting=8192, mixed_trials=160):
    """Local algebra, ordering, phase-blindness and mixed-state theorem checks.

    Requires measurement_plan, _mvqls_basis_circuits and analyze_cost from the
    earlier inline measured_cost cell. Synthetic integer counts are explicitly
    algebra fixtures; separate local StatevectorSampler measurements are kept
    under local_finite_shot_check. No hardware/fake backend is contacted.
    """
    needed = ('measurement_plan', '_mvqls_basis_circuits', 'analyze_cost')
    if any(name not in globals() for name in needed):
        raise RuntimeError('Execute the inline measured_cost definitions before this validation cell.')
    A = np.array([[2.1, -.30, .20, .10], [-.30, 1.4, -.15, .05],
                  [.20, -.15, 2.7, -.40], [.10, .05, -.40, 3.4]])
    target = np.array([1., 2., 3., 4.]) / np.sqrt(30)
    b = A @ target
    twisted = target * np.exp(1j * np.pi / 2 * np.arange(4))
    diagnostics = {
        'solution': exact_phase_witness_diagnostics(A, b, target),
        'global_phase_only': exact_phase_witness_diagnostics(A, b, np.exp(.73j) * target),
        'relative_phases': exact_phase_witness_diagnostics(A, b, twisted),
        'fully_dephased_mixture': exact_phase_witness_diagnostics(A, b, np.diag(abs(target) ** 2)),
    }
    for value in diagnostics.values():
        np.testing.assert_allclose(value['z_probabilities'], abs(target) ** 2, atol=1e-14)
    assert abs(diagnostics['solution']['cost_exact_float']) < 1e-12
    assert abs(diagnostics['global_phase_only']['cost_exact_float']) < 1e-12
    assert diagnostics['relative_phases']['cost_exact_float'] > .1
    assert diagnostics['fully_dephased_mixture']['cost_exact_float'] > .1
    assert diagnostics['fully_dephased_mixture']['pure_reference_free_residual'] is None
    for name in ('solution', 'global_phase_only', 'relative_phases'):
        value = diagnostics[name]
        assert abs(value['pure_reference_free_residual'] ** 2 - value['cost_exact_float']) < 1e-12

    plan = measurement_plan(A, b, np.array([.37, -.83]), reps=0)
    preparation = QuantumCircuit(2)
    preparation.append(StatePreparation(twisted), [0, 1])
    circuits = _mvqls_basis_circuits(preparation, plan['groups'])
    expected_M = diagnostics['relative_phases']['numerator_exact']
    expected_D = diagnostics['relative_phases']['denominator_exact']
    estimate = np.array([plan['model']['identity_offsets']['numerator'], plan['model']['identity_offsets']['denominator']])
    wrong_endian_estimate = estimate.copy()
    max_term_error, synthetic_counts, setting_probabilities = 0.0, {}, {}
    for group, circuit in zip(plan['groups'], circuits):
        probabilities = Statevector.from_instruction(circuit.remove_final_measurements(inplace=False)).probabilities()
        setting_probabilities[group['setting']] = probabilities.tolist()
        for term in group['terms']:
            label = term['label']
            direct = float(np.vdot(twisted, _phase_pauli_matrix(label) @ twisted).real)
            measured, reversed_bits = 0.0, 0.0
            for integer, probability in enumerate(probabilities):
                parity = (integer & term['parity_mask']).bit_count() % 2
                wrong_integer = int(format(integer, '02b')[::-1], 2)
                wrong_parity = (wrong_integer & term['parity_mask']).bit_count() % 2
                measured += probability * (1 - 2 * parity)
                reversed_bits += probability * (1 - 2 * wrong_parity)
            max_term_error = max(max_term_error, abs(direct - measured))
            coefficients = np.array([term['numerator_coefficient'], term['denominator_coefficient']])
            estimate += measured * coefficients
            wrong_endian_estimate += reversed_bits * coefficients
        synthetic_counts[group['setting']] = _phase_integer_fixture(probabilities, 1 << 24)
    np.testing.assert_allclose(estimate, [expected_M, expected_D], atol=1e-11)
    assert max_term_error < 1e-12
    analyzed = analyze_cost(plan, synthetic_counts)
    assert analyzed['status'] == 'ok'
    synthetic_error = abs(analyzed['cost_unclipped'] - diagnostics['relative_phases']['cost_exact_float'])
    assert synthetic_error < 1e-6
    wrong_cost = 1 - wrong_endian_estimate[0] / wrong_endian_estimate[1]
    wrong_cost_error = abs(wrong_cost - diagnostics['relative_phases']['cost_exact_float'])
    assert wrong_cost_error > .01, 'Fixture must detect a reversed-qubit implementation.'

    # Actual local finite-shot experiment; never called a hardware certificate.
    actual_counts = {}
    for index, (setting, circuit) in enumerate(zip(plan['settings'], circuits)):
        sampler = StatevectorSampler(seed=seed + index)
        actual_counts[setting] = sampler.run([circuit], shots=shots_per_setting).result()[0].data.meas.get_counts()
    sampled = analyze_cost(plan, actual_counts)
    assert sampled['status'] == 'ok'

    rng = np.random.default_rng(seed)
    minimum_fidelity_margin, minimum_operator_margin = float('inf'), float('inf')
    for trial in range(mixed_trials):
        dimension = 2 if trial % 2 == 0 else 4
        unitary, _ = np.linalg.qr(rng.normal(size=(dimension, dimension)) + 1j * rng.normal(size=(dimension, dimension)))
        eigenvalues = np.geomspace(1, 1 + (trial % 11), dimension)
        matrix = (unitary * eigenvalues) @ unitary.conj().T
        rhs = rng.normal(size=dimension) + 1j * rng.normal(size=dimension)
        x = np.linalg.solve(matrix, rhs)
        x /= np.linalg.norm(x)
        factor = rng.normal(size=(dimension, dimension)) + 1j * rng.normal(size=(dimension, dimension))
        rho = factor @ factor.conj().T
        rho /= np.trace(rho)
        mixing = [.001, .03, .2, 1.][trial % 4]
        rho = (1 - mixing) * np.outer(x, x.conj()) + mixing * rho
        value = exact_phase_witness_diagnostics(matrix, rhs, rho)
        margin = value['target_fidelity_exact_diagnostic'] - value['fidelity_bound_from_exact_cost']
        energy_margin = value['target_fidelity_exact_diagnostic'] - value['fidelity_bound_from_exact_energy']
        assert margin >= -1e-11 and energy_margin >= -1e-11
        assert value['operator_inequality_minimum_eigenvalue_roundoff'] >= -1e-9
        minimum_fidelity_margin = min(minimum_fidelity_margin, margin)
        minimum_operator_margin = min(minimum_operator_margin, value['operator_inequality_minimum_eigenvalue_roundoff'])

    # Deliberately incompatible measurement marginals: retain the negative ratio.
    unclipped_plan = measurement_plan(np.diag([1., 2.]), [1., 1.], [.2], reps=0)
    outside = analyze_cost(unclipped_plan, {'X': {'0': 100}, 'Z': {'0': 100}})
    assert outside['cost_unclipped'] < 0 and outside['cost_is_outside_physical_interval']
    missing_metadata_audit = audit_phase_witness_applicability(compiled_entries or [])
    assert missing_metadata_audit['hardware_fidelity_lower_bound'] is None
    sweep = []
    for angle in np.linspace(0, 2 * np.pi, 17):
        value = exact_phase_witness_diagnostics(A, b, target * np.exp(1j * angle * np.arange(4)))
        sweep.append({'phase_increment_radians': float(angle), 'z_total_variation_from_target': 0.0,
                      'global_cost': value['cost_exact_float'],
                      'reference_free_residual': value['pure_reference_free_residual'],
                      'target_fidelity': value['target_fidelity_exact_diagnostic']})
    return {'status': 'passed', 'created_utc': datetime.now(timezone.utc).isoformat(), 'hardware_calls': 0,
        'fixture': {'A': A.tolist(), 'b': b.tolist(), 'target': target.tolist(),
                    'description': 'Non-diagonal SPD matrix without qubit-exchange symmetry; unequal state probabilities.'},
        'phase_examples': diagnostics, 'phase_sweep': sweep,
        'bit_order_validation': {'maximum_Pauli_expectation_error': max_term_error,
            'synthetic_count_cost_error': synthetic_error, 'deliberately_reversed_bits_cost_error': float(wrong_cost_error),
            'settings': plan['settings'], 'setting_probabilities': setting_probabilities,
            'synthetic_counts': synthetic_counts, 'synthetic_counts_are_experiments': False,
            'synthetic_counts_note': 'Deterministic rounded exact probabilities; algebra fixture, not a shot experiment.'},
        'local_finite_shot_check': {'executor': 'StatevectorSampler', 'shots_per_setting': shots_per_setting,
            'seed': seed, 'counts': actual_counts, 'analysis': sampled, 'certified_fidelity_lower_bound': None},
        'mixed_state_theorem': {'proof': MIXED_STATE_WITNESS_PROOF, 'random_density_matrices_checked': mixed_trials,
            'minimum_numerical_bound_margin': minimum_fidelity_margin,
            'minimum_operator_inequality_eigenvalue_roundoff': minimum_operator_margin,
            'claim': 'Exact common-state expectations only; numerical checks supplement the operator proof.'},
        'unclipped_measured_cost_guard': {'synthetic_inconsistent_cost': outside['cost_unclipped'],
            'outside_physical_interval': outside['cost_is_outside_physical_interval'],
            'certified_fidelity_lower_bound': None},
        'historical_hardware_applicability': missing_metadata_audit,
        'conclusion': 'Z distributions alone are phase blind. The counted Pauli witness is phase sensitive under a common-state measurement model. Raw noisy or heterogeneous-layout aggregates are not fidelity certificates.'}


## Device calibration, compilation and bounded hardware execution

Fake and real devices must provide nonempty calibration before compilation or submission. Physical jobs retain preparation, immediately-before-submission, job-associated, and after-collection evidence. If IBM cannot return exact job-associated calibration, that missing evidence is explicit and raw measurements remain preserved.

The pilot limit applies across notebook restarts. Existing jobs load by their saved identity before any new submission. A job whose submission outcome is uncertain requires reconciliation; rerunning never creates a duplicate. Raw Sampler counts use no measurement mitigation.

In [ ]:
# CELL_ID: backends
# Public IBM API references checked 2026-09-13:
# https://quantum.cloud.ibm.com/docs/api/qiskit-ibm-runtime/runtime-job-v2
# https://quantum.cloud.ibm.com/docs/en/guides/max-execution-time
import io
import time
from qiskit import qpy
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

HARDWARE_ALLOWLIST = {
    'lab_zhang': ['ibm_rensselaer'],
    'lab_premium': ['ibm_pittsburgh', 'ibm_boston', 'ibm_fez', 'ibm_marrakesh', 'ibm_kingston'],
}

def _public_json(value):
    # Restricted to public backend/primitive fields, never a service/account object.
    return json.loads(json.dumps(value, default=str, allow_nan=False))

def _backend_name(backend):
    name = backend.name
    return name() if callable(name) else name

def _target_snapshot(target):
    instructions = []
    for name in sorted(target.operation_names):
        operation = target.operation_from_name(name)
        for qubits, properties in target[name].items():
            instructions.append({'name': name, 'qubits': None if qubits is None else list(qubits),
                'operation_class': getattr(operation, '__name__', type(operation).__name__),
                'parameters': ([] if isinstance(operation, type) else [str(p) for p in getattr(operation, 'params', [])]),
                'duration_seconds': getattr(properties, 'duration', None),
                'error': getattr(properties, 'error', None),
                'calibration': str(getattr(properties, 'calibration', None))})
    return _public_json({'num_qubits': target.num_qubits, 'dt': target.dt,
        'granularity': target.granularity, 'min_length': target.min_length,
        'pulse_alignment': target.pulse_alignment, 'acquire_alignment': target.acquire_alignment,
        'qubit_properties': [None if p is None else {'t1': p.t1, 't2': p.t2, 'frequency': p.frequency}
                             for p in (target.qubit_properties or [])],
        'instructions': instructions,
        'scope': 'all public Target instruction locations, errors, durations, parameters and qubit properties; public BackendProperties archived separately'})

def capture_backend(store, backend, kind='fake', observation=None):
    """Archive public properties/configuration/target before using a device."""
    if kind not in ('fake', 'hardware'):
        raise ValueError('Use an explicit ideal calibration registration for ideal simulation.')
    try:
        properties = backend.properties(refresh=True)
    except TypeError:
        properties = backend.properties()
    if properties is None:
        raise ValueError('Backend provides no properties; execution is stopped before any submission.')
    raw = _public_json(properties.to_dict())
    # Preserve retrieved calibration immediately even if later configuration or
    # Target retrieval fails. Never replace a missing snapshot with an empty one.
    observation_dir = store.root/'calibration_observations'/uuid.uuid4().hex
    atomic_json(observation_dir/'properties.json', raw)
    configuration = _public_json(backend.configuration().to_dict())
    atomic_json(observation_dir/'configuration.json', configuration)
    target = _target_snapshot(backend.target)
    atomic_json(observation_dir/'target.json', target)
    payload = {'kind': kind, 'backend': _backend_name(backend),
               'backend_class': type(backend).__module__ + '.' + type(backend).__name__,
               'properties': raw, 'configuration': configuration, 'target': target,
               'calibration_last_update': str(properties.last_update_date)}
    identifier = store.register_calibration(payload, {'source': 'backend public snapshot', **(observation or {})})
    atomic_json(observation_dir/'registered.json', {'calibration_id': identifier, 'retrieved_utc': utc_now()})
    return identifier

def archive_noise_model(directory, noise_model, calibration_id, details):
    """Persist exact generated Aer error channels alongside their source ID."""
    atomic_json(Path(directory)/'noise_model.json', noise_model.to_dict(serializable=True))
    atomic_json(Path(directory)/'noise_model_provenance.json', {
        'calibration_id': calibration_id, 'noise_model_sha256': file_hash(Path(directory)/'noise_model.json'),
        'details': details, 'archived_utc': utc_now()})

def circuit_resources(circuit):
    gates = [i for i in circuit.data if i.operation.name not in ('measure', 'barrier', 'reset', 'delay')]
    active = sorted({circuit.find_bit(q).index for i in gates for q in i.qubits})
    return {'allocated_physical_qubits': circuit.num_qubits, 'active_gate_qubits': active,
        'active_gate_qubit_count': len(active), 'depth_including_measurements': circuit.depth(),
        'two_qubit_gates': sum(len(i.qubits) == 2 for i in gates),
        'basis_operation_counts': dict(circuit.count_ops()),
        'measurement_count': sum(i.operation.name == 'measure' for i in circuit.data)}

def _qpy_bytes(circuits):
    stream = io.BytesIO()
    qpy.dump(list(circuits), stream)
    return stream.getvalue()

def _semantic_circuit_signature(circuit):
    """Hash operations, not generated circuit names or QPY container details."""
    operations = []
    for item in circuit.data:
        operation = item.operation
        if getattr(operation, 'blocks', ()):
            raise ValueError('The bounded pilot supports static circuits only.')
        parameters = []
        for value in operation.params:
            if hasattr(value, 'parameters') and value.parameters:
                raise ValueError('All pilot parameters must be bound before submission.')
            try:
                parameters.append(portable(value))
            except TypeError:
                parameters.append(float(value))
        if type(operation).__module__ == 'qiskit.circuit.gate' and operation.definition is not None:
            identity = {'definition': _semantic_circuit_signature(operation.definition)}
        else:
            identity = {'name': operation.name, 'parameters': parameters,
                        'num_qubits': operation.num_qubits, 'num_clbits': operation.num_clbits}
        operations.append({'operation': identity,
            'qubits': [circuit.find_bit(q).index for q in item.qubits],
            'clbits': [circuit.find_bit(c).index for c in item.clbits]})
    return {'num_qubits': circuit.num_qubits, 'num_clbits': circuit.num_clbits,
            'cregs': [(r.name, len(r)) for r in circuit.cregs],
            'global_phase': float(circuit.global_phase), 'operations': operations}

def compile_for_backend(store, circuits, entries, backend, calibration_id, transpile_config, dependencies=()):
    """Cache measured ISA circuits; a resource rejection still retains its QPY."""
    if len(circuits) != len(entries):
        raise ValueError('Each circuit requires one explicit metadata entry.')
    encoded = _qpy_bytes(circuits)
    settings = {'optimization_level': int(transpile_config.get('optimization_level', 2)),
                'seed_transpiler': int(transpile_config.get('seed_transpiler', 1729)),
                'max_two_qubit_gates': min(1500, int(transpile_config.get('max_two_qubit_gates', transpile_config.get('max_two_qubit_gates_per_circuit', 1500))))}
    # QPY can regenerate internal gate names/UUIDs when a circuit is loaded.
    # Cache the full mathematical instruction signature; retain byte hashes for integrity.
    spec = {'logical_circuits': [_semantic_circuit_signature(c) for c in circuits], 'entries': entries,
            'backend': _backend_name(backend), 'transpile': settings}
    def compute(directory):
        (directory/'logical_measured.qpy').write_bytes(encoded)
        pm = generate_preset_pass_manager(backend=backend, optimization_level=settings['optimization_level'],
                                          seed_transpiler=settings['seed_transpiler'])
        compiled, compiled_entries = [], []
        for index, (logical, entry) in enumerate(zip(circuits, entries)):
            if len(logical.cregs) != 1 or logical.cregs[0].name != 'meas':
                raise ValueError('Use exactly one classical register named meas.')
            started = time.perf_counter()
            isa = pm.run(logical)
            # Persist each circuit immediately, including expensive rejected ones.
            with (directory/f'isa_circuit_{index}.qpy').open('wb') as stream:
                qpy.dump([isa], stream)
            cost = circuit_resources(isa)
            metadata = {**entry, 'resources': cost, 'transpile_seconds': time.perf_counter()-started,
                'initial_layout': isa.layout.initial_index_layout(),
                'final_layout': isa.layout.final_index_layout()}
            atomic_json(directory/f'compiled_entry_{index}.json', metadata)
            compiled.append(isa)
            compiled_entries.append(metadata)
        with (directory/'isa_circuits.qpy').open('wb') as stream:
            qpy.dump(compiled, stream)
        accepted = all(e['resources']['two_qubit_gates'] <= settings['max_two_qubit_gates'] for e in compiled_entries)
        return {'status': 'compiled' if accepted else 'resource_cap_exceeded',
                'calibration_id': calibration_id, 'entries': compiled_entries,
                'isa_circuits_sha256': file_hash(directory/'isa_circuits.qpy'),
                'limits': settings, 'hardware_jobs_submitted': 0}
    return store.run('compile', spec, compute, roles=('backends',), dependencies=dependencies,
                     calibration_ids=(calibration_id,))

def _checked_hardware_configuration(configuration):
    account = configuration.get('account', 'lab_zhang')
    backend = configuration.get('backend', 'ibm_rensselaer')
    if backend not in HARDWARE_ALLOWLIST.get(account, []):
        raise ValueError('Account/backend pair is outside the user-specified allowlist.')
    shots = int(configuration.get('shots', 4096))
    duration = int(configuration.get('max_execution_time_seconds', 60))
    if int(configuration.get('max_jobs', 1)) != 1:
        raise ValueError('This bounded pilot supports exactly one allowed job per campaign; use enabled=False to disable it.')
    if not 1 <= shots <= 4096 or not 1 <= duration <= 60:
        raise ValueError('Pilot limits: 1–4096 shots and at most 60 seconds QPU usage per job.')
    return {'account': account, 'backend': backend, 'shots': shots,
            'max_execution_time_seconds': duration,
            'campaign_id': str(configuration.get('campaign_id', 'bounded_pilot_v1')),
            'replication_id': str(configuration.get('replication_id', 'replicate_0'))}

def _archive_job_calibrations(store, job, backend, registry, compile_calibration_id):
    """Record availability separately; missing job properties are never invented."""
    observation = {'queried_utc': utc_now(), 'job_id': job.job_id(), 'job_specific_available': False}
    ids = [compile_calibration_id]
    try:
        properties = job.properties(refresh=True)
        if properties is not None and properties.to_dict().get('qubits'):
            raw = _public_json(properties.to_dict())
            atomic_json(registry/'backend_properties_job_start.json', raw)
            base = store.calibration(compile_calibration_id)['payload']
            payload = {**base, 'properties': raw, 'calibration_last_update': str(properties.last_update_date),
                       'snapshot_scope': 'job-start BackendProperties; configuration/Target retained from compilation',
                       'job_id': job.job_id()}
            identifier = store.register_calibration(payload, {'source': 'job.properties(refresh=True)', 'job_id': job.job_id()})
            observation.update(job_specific_available=True, job_calibration_id=identifier)
            ids.append(identifier)
        else:
            observation['job_specific_unavailable_reason'] = 'service_returned_no_properties'
    except Exception as error:
        observation['job_specific_unavailable_reason'] = type(error).__name__
    try:
        after = capture_backend(store, backend, 'hardware', {'phase': 'after_collection', 'job_id': job.job_id()})
        observation['after_collection_calibration_id'] = after
        ids.append(after)
    except Exception as error:
        observation['after_collection_unavailable_reason'] = type(error).__name__
    atomic_json(registry/'calibration_availability.json', observation)
    return ids, observation

def _collect_registered_pilot(store, service, backend, registry):
    request = read_json(registry/'request.json')
    job = service.job(read_json(registry/'job.json')['job_id'])
    status = str(job.status()).upper().split('.')[-1]
    atomic_json(registry/'latest_job_status.json', {'job_id': job.job_id(), 'status': status, 'queried_utc': utc_now()})
    if status not in ('DONE', 'ERROR', 'CANCELLED'):
        return {'status': 'pending', 'job_id': job.job_id(), 'job_status': status, 'registry': str(registry),
                'compilation': request['compilation'], 'compile_calibration_id': request['compile_calibration_id'],
                'submitted_new_job': False}
    rows = []
    validation_errors = []
    if status == 'DONE':
        pubs = job.result()
        # Raw counts are durable before any diagnostics or optional metadata calls.
        saved_counts = []
        for index, pub in enumerate(pubs):
            counts = pub.data.meas.get_counts()
            atomic_json(registry/f'raw_counts_hardware_{index}.json', counts)
            saved_counts.append(counts)
        if len(pubs) != len(request['compiled_entries']):
            validation_errors.append('Returned PUB count differs from submitted circuit count.')
        for index, (pub, counts) in enumerate(zip(pubs, saved_counts)):
            atomic_json(registry/f'pub_metadata_{index}.json', _public_json(pub.metadata))
            if any(not isinstance(value, (int, np.integer)) or value < 0 for value in counts.values()) or sum(counts.values()) != request['settings']['shots']:
                validation_errors.append(f'PUB {index} has invalid counts or differs from the requested shot budget.')
            entry = request['compiled_entries'][index] if index < len(request['compiled_entries']) else None
            rows.append({'entry': entry, 'counts_file': f'raw_counts_hardware_{index}.json',
                         'counts_sha256': file_hash(registry/f'raw_counts_hardware_{index}.json')})
    ids, availability = _archive_job_calibrations(store, job, backend, registry, request['compile_calibration_id'])
    ids.append(request['before_submission_calibration_id'])
    atomic_json(registry/'raw_counts_validation.json', {'valid': not validation_errors, 'errors': validation_errors})
    try:
        atomic_json(registry/'job_metrics.json', _public_json(job.metrics()))
    except Exception as error:
        atomic_json(registry/'job_metrics_availability.json', {'available': False, 'reason': type(error).__name__})
    dependencies = request['dependencies']
    def collect(directory):
        for source in registry.iterdir():
            if source.is_file() and source.suffix in ('.json', '.qpy'):
                shutil.copy2(source, directory/source.name)
        return {'status': ('invalid_hardware_counts' if validation_errors else 'completed') if status == 'DONE' else 'hardware_job_failed',
                'job_id': job.job_id(), 'job_status': status, 'rows': rows,
                'calibration_availability': availability, 'registry': str(registry),
                'compilation': request['compilation'], 'compile_calibration_id': request['compile_calibration_id'],
                'hardware_jobs_submitted': 1, 'metric_scope': 'raw Z-basis counts; derive metrics separately without resubmission'}
    record = store.run('hardware_results', {'job_id': job.job_id(), 'request_id': request['request_id']},
                       collect, roles=('backends',), dependencies=dependencies, calibration_ids=tuple(set(ids)))
    atomic_json(registry/'completed_reference.json', store.dependency(record))
    return record

def hardware_pilot(store, circuits, entries, logical_dependencies, hardware_config, transpile_config):
    """At most one hardware job per campaign; reload before any backend access.

    The physical-job identity deliberately excludes bootstrap/analysis options.
    A persistent intent without a job identifier is uncertain and never retried
    automatically. A known pending job is only queried/collected on later calls.
    ``enabled=False`` performs no device work. Setting enabled=True is reserved
    for an already authorized bounded pilot after local validation.
    """
    settings = _checked_hardware_configuration(hardware_config)
    circuit_limit = min(8, int(hardware_config.get('max_circuits_per_job', 8)))
    if not 1 <= len(circuits) <= circuit_limit or len(circuits) != len(entries):
        raise ValueError('Pilot requires one metadata entry per circuit, with 1–8 circuits.')
    request_id = content_hash({'hardware': settings, 'logical_circuits': [_semantic_circuit_signature(c) for c in circuits]})
    registry = store.root/'hardware_jobs'/request_id
    completed = registry/'completed_reference.json'
    if completed.exists():
        reference = read_json(completed)
        result = store.lookup(reference['stage'], reference['key'])
        if result is None:
            raise CacheIntegrityError('Hardware completion reference is missing; refuse to submit a replacement.')
        return result
    if (registry/'submission_intent.json').exists() and not (registry/'job.json').exists():
        return {'status': 'submission_uncertain', 'registry': str(registry),
                'submitted_new_job': False, 'action': 'Inspect the saved intent and IBM job list; no automatic duplicate.'}
    if not (registry/'job.json').exists() and not hardware_config.get('enabled', False):
        return {'status': 'disabled', 'submitted_new_job': False, 'request_id': request_id}
    campaign = store.root/'hardware_campaigns'/content_hash({k: settings[k] for k in ('account', 'backend', 'campaign_id')})
    if not (registry/'job.json').exists() and (campaign/'claimed_request.json').exists():
        return {'status': 'campaign_job_already_claimed', 'submitted_new_job': False,
                'claimed_request': read_json(campaign/'claimed_request.json')}
    service = QiskitRuntimeService(name=settings['account'])
    backend = service.backend(settings['backend'])
    if (registry/'job.json').exists():
        return _collect_registered_pilot(store, service, backend, registry)
    if not backend.status().operational:
        raise RuntimeError('Backend is not operational; no job submitted.')
    calibration_id = capture_backend(store, backend, 'hardware', {'phase': 'before_transpilation'})
    compiled = compile_for_backend(store, circuits, entries, backend, calibration_id,
                                   transpile_config, dependencies=logical_dependencies)
    if compiled['result']['status'] != 'compiled':
        return {'status': 'resource_cap_exceeded', 'compiled_record': compiled, 'submitted_new_job': False}
    isa_path = compiled['directory']/'isa_circuits.qpy'
    if file_hash(isa_path) != compiled['result']['isa_circuits_sha256']:
        raise CacheIntegrityError('Prepared circuit changed before submission.')
    with isa_path.open('rb') as stream:
        isa_circuits = qpy.load(stream)
    gate_limit = min(1500, int(hardware_config.get('max_two_qubit_gates_per_circuit', 1500)))
    if len(isa_circuits) != len(circuits) or any(circuit_resources(c)['two_qubit_gates'] > gate_limit or c.num_qubits > backend.num_qubits for c in isa_circuits):
        raise ValueError('Actual loaded circuits violate the pilot resource limits.')
    before_id = capture_backend(store, backend, 'hardware', {'phase': 'immediately_before_submission'})
    if before_id != calibration_id:
        return {'status': 'calibration_changed_before_submission', 'submitted_new_job': False,
                'compilation': store.dependency(compiled), 'compile_calibration_id': calibration_id,
                'new_calibration_id': before_id,
                'action': 'Re-run to compile against the newly archived calibration; no circuit was submitted.'}
    registry.mkdir(parents=True, exist_ok=True)
    campaign.mkdir(parents=True, exist_ok=True)
    # Exclusive create claims the one-job campaign across concurrent notebook kernels.
    with (campaign/'claimed_request.json').open('x', encoding='utf8') as stream:
        json.dump({'request_id': request_id, 'claimed_utc': utc_now()}, stream)
    dependencies = [*logical_dependencies, store.dependency(compiled)]
    atomic_json(registry/'request.json', {'request_id': request_id, 'settings': settings,
        'compiled_entries': compiled['result']['entries'], 'compile_calibration_id': calibration_id,
        'compilation': store.dependency(compiled),
        'before_submission_calibration_id': before_id, 'dependencies': dependencies,
        'isa_circuits_sha256': file_hash(isa_path)})
    shutil.copy2(isa_path, registry/'isa_circuits.qpy')
    atomic_json(registry/'submission_intent.json', {'status': 'submitting_or_unknown',
        'written_before_remote_submission_utc': utc_now(), 'request_id': request_id,
        'limits': {'max_jobs': 1, 'max_execution_time_seconds': settings['max_execution_time_seconds'],
                   'max_circuits': circuit_limit, 'max_two_qubit_gates_per_circuit': gate_limit},
        'calibration_ids': [calibration_id, before_id]})
    store.bundle(registry, calibration_ids=(calibration_id, before_id), dependencies=dependencies)
    sampler = SamplerV2(mode=backend, options={'default_shots': settings['shots'],
                        'max_execution_time': settings['max_execution_time_seconds']})
    job = sampler.run(isa_circuits, shots=settings['shots'])
    atomic_json(registry/'job.json', {'job_id': job.job_id(), 'submitted_utc': utc_now()})
    store.bundle(registry, calibration_ids=(calibration_id, before_id), dependencies=dependencies)
    return {'status': 'submitted', 'job_id': job.job_id(), 'registry': str(registry),
            'compilation': store.dependency(compiled), 'compile_calibration_id': calibration_id,
            'request_id': request_id, 'submitted_new_job': True,
            'action': 'Rerun this stage to query or collect the same saved job.'}

def refresh_saved_job_calibration(store, hardware_registry):
    """Explicit read-only refresh for job calibration that was unavailable earlier.

    A separate cached artifact preserves the original completed hardware result.
    No sampler is created and no circuit is submitted.
    """
    registry = Path(hardware_registry).resolve()
    expected_root = (store.root/'hardware_jobs').resolve()
    if not registry.is_relative_to(expected_root):
        raise ValueError('The registry must belong to this notebook run store.')
    request = read_json(registry/'request.json')
    settings = request['settings']
    if settings['backend'] not in HARDWARE_ALLOWLIST.get(settings['account'], []):
        raise ValueError('Saved account/backend pair is outside the allowlist.')
    service = QiskitRuntimeService(name=settings['account'])
    backend = service.backend(settings['backend'])
    job = service.job(read_json(registry/'job.json')['job_id'])
    observation = registry/'calibration_refreshes'/uuid.uuid4().hex
    observation.mkdir(parents=True)
    ids, available = _archive_job_calibrations(store, job, backend, observation, request['compile_calibration_id'])
    def save_refresh(directory):
        for p in observation.iterdir():
            if p.is_file(): shutil.copy2(p, directory/p.name)
        return {'status': 'calibration_refreshed', 'job_id': job.job_id(), 'availability': available,
                'hardware_jobs_submitted': 0}
    return store.run('hardware_calibration_refresh', {'job_id': job.job_id(), 'availability': available},
                     save_refresh, roles=('backends',), calibration_ids=tuple(set(ids)))


In [ ]:
# CELL_ID: noise_helpers
from qiskit import QuantumRegister
def compress_physical_circuit(circuit: QuantumCircuit, max_active_qubits: int = 12):
    """Relabel active physical qubits; preserve classical bit order exactly."""
    if not 1 <= max_active_qubits <= 14:
        raise ValueError("The density-matrix reference allows max_active_qubits from 1 to 14.")
    active = sorted({circuit.find_bit(q).index for item in circuit.data
                     if item.operation.name != "barrier" for q in item.qubits})
    if not active:
        raise ValueError("Circuit contains no active qubits.")
    if len(active) > max_active_qubits:
        raise ValueError(f"Circuit uses {len(active)} active physical qubits, exceeding cap {max_active_qubits}.")
    if any(getattr(item.operation, "blocks", ()) for item in circuit.data):
        raise ValueError("Dynamic control-flow circuits require a separate validated simulator path.")
    physical_to_compact = {physical: i for i, physical in enumerate(active)}
    compact = QuantumCircuit(QuantumRegister(len(active), "sim_q"), *circuit.cregs,
                             name=circuit.name + "_active")
    if compact.num_clbits != circuit.num_clbits:
        raise ValueError("Standalone classical bits are not supported; use the prepared meas register.")
    compact.global_phase = circuit.global_phase
    for item in circuit.data:
        physical = [circuit.find_bit(q).index for q in item.qubits]
        if item.operation.name == "barrier":
            relevant = [physical_to_compact[q] for q in physical if q in physical_to_compact]
            if relevant:
                compact.barrier(*relevant)
            continue
        compact.append(item.operation, [physical_to_compact[q] for q in physical],
                       [circuit.find_bit(c).index for c in item.clbits])
    return compact, physical_to_compact


def remap_properties(properties_dict, physical_to_compact):
    """Subset public BackendProperties fields, retaining their physical identity."""
    source = copy.deepcopy(properties_dict)
    if not source.get("qubits"):
        raise ValueError("Archived backend properties are empty; a calibrated noise comparison is unavailable.")
    active = sorted(physical_to_compact, key=physical_to_compact.get)
    if any(q >= len(source["qubits"]) for q in active):
        raise ValueError("Circuit physical qubit is absent from the calibration snapshot.")
    source["qubits"] = [source["qubits"][q] for q in active]
    gates = []
    for gate in source.get("gates", []):
        if all(q in physical_to_compact for q in gate["qubits"]):
            gate["qubits"] = [physical_to_compact[q] for q in gate["qubits"]]
            gates.append(gate)
    source["gates"] = gates
    return source


def validate_measurement_map(circuit, entry):
    """Check that metrics' logical indices are the actual output classical bits."""
    if len(circuit.cregs) != 1 or circuit.cregs[0].name != "meas":
        raise ValueError("Prepared ISA circuit must have exactly one classical register named meas.")
    measured = {}
    for item in circuit.data:
        if item.operation.name == "measure":
            classical = circuit.find_bit(item.clbits[0]).index
            if classical in measured:
                raise ValueError("Repeated measurement of a classical bit is not supported.")
            measured[classical] = circuit.find_bit(item.qubits[0]).index
    meta = entry["solver_metadata"]
    required = {int(q) for q in meta["system_qubits"]} | {int(q) for q in meta["postselect_bits"]}
    final_layout = entry["final_layout"]
    for logical in required:
        if logical not in measured or logical >= len(final_layout) or measured[logical] != final_layout[logical]:
            raise ValueError(f"Measurement bit {logical} disagrees with the archived final layout.")
    return measured


def snapshot_noise_model(selected_properties, physical_to_compact, configuration):
    """Build calibrated subset noise when public thermal inputs are missing.

    Aer 0.17.2's BackendProperties path requests frequency in the delay-noise
    setup even at zero temperature. Missing frequencies must not be fabricated.
    The fallback retains average gate errors as depolarizing channels and the
    archived readout confusion probabilities; it adds no separate T1/T2 channel.
    """
    compact_to_physical = {v: k for k, v in physical_to_compact.items()}
    missing = []
    for compact, fields in enumerate(selected_properties["qubits"]):
        present = {field["name"] for field in fields if field.get("value") is not None}
        absent = sorted({"T1", "T2", "frequency"} - present)
        if absent:
            missing.append({"physical_qubit": compact_to_physical[compact],
                            "compact_qubit": compact, "missing_properties": absent})
    thermal = not missing
    properties = BackendProperties.from_dict(selected_properties)
    model = NoiseModel.from_backend_properties(properties, gate_error=True, readout_error=True,
                                               thermal_relaxation=thermal, dt=configuration.get("dt"))
    return model, {
        "noise_model_scope": ("snapshot-derived gate/depolarizing, T1/T2 relaxation and readout errors"
                              if thermal else "snapshot-derived gate-depolarizing and readout errors only"),
        "thermal_relaxation_enabled": thermal,
        "missing_thermal_input_properties": missing,
        "noise_fallback_reason": (None if thermal else
                                  "Public snapshot lacks inputs required by Aer's BackendProperties thermal path; no missing physical properties were invented."),
        "additional_noise_exclusions": ([] if thermal else
                                        ["separate T1/T2 relaxation channels", "thermal relaxation during delays"]),
        "gate_error_interpretation": "Reported average gate errors are approximated as depolarizing channels; these aggregate errors may already include relaxation contributions.",
    }


In [ ]:
# CELL_ID: collection_hardening
# Collection and metadata retrieval only: these helpers never construct a sampler.
# Official encoder: https://quantum.cloud.ibm.com/docs/en/guides/save-jobs
from qiskit_ibm_runtime import RuntimeEncoder

def _collection_runtime_json(value):
    # Preserve typed execution-span metadata using IBM's public encoder. Do not
    # decode arbitrary tagged objects when reopening an artifact: plain JSON is enough.
    encoded = json.loads(json.dumps(value, cls=RuntimeEncoder, allow_nan=False))
    assert_no_credentials(encoded)
    return encoded

def _collection_optional(directory, name, producer):
    try:
        value = _collection_runtime_json(producer())
        atomic_json(Path(directory)/(name+'.json'), value)
        availability = {'available': True, 'encoding': 'qiskit_ibm_runtime.RuntimeEncoder'}
    except Exception as error:
        availability = {'available': False, 'reason': type(error).__name__}
    atomic_json(Path(directory)/(name+'_availability.json'), availability)
    return availability

def _collection_input_options(job):
    # Do not serialize account objects, circuits/PUBs, arbitrary job inputs or credentials.
    inputs = job.inputs
    if not isinstance(inputs, dict):
        raise ValueError('Job inputs are not an options mapping.')
    scalar_fields = ('version', 'resilience_level', 'default_shots', 'default_precision', 'shots')
    option_fields = ('default_shots', 'default_precision', 'init_qubits', 'dynamical_decoupling',
        'twirling', 'execution', 'measurement_type', 'meas_type', 'resilience_level', 'resilience',
        'optimization_level', 'max_execution_time')
    selected = {key: inputs[key] for key in scalar_fields if key in inputs}
    if isinstance(inputs.get('options'), dict):
        selected['options'] = {key: inputs['options'][key] for key in option_fields if key in inputs['options']}
    assert_no_credentials(selected)
    return {'selected_inputs': selected, 'scope': 'explicit primitive options allowlist; accounts, service objects, PUBs and circuits excluded'}

def _collection_validate(directory, request, rows):
    """Validate raw evidence without modifying it; inspect every PUB and mapping."""
    directory = Path(directory)
    errors, checks = [], []
    entries = request.get('compiled_entries', [])
    isa_path = directory/'isa_circuits.qpy'
    expected_hash = request.get('isa_circuits_sha256')
    actual_hash = file_hash(isa_path) if isa_path.is_file() else None
    if actual_hash is None or actual_hash != expected_hash:
        errors.append('Saved physical QPY is missing or differs from the submitted checksum.')
        circuits = []
    else:
        try:
            with isa_path.open('rb') as stream:
                circuits = qpy.load(stream)
        except Exception as error:
            errors.append('Physical QPY could not be read: '+type(error).__name__)
            circuits = []
    if len(circuits) != len(entries):
        errors.append('Physical circuit count differs from submitted entry count.')
    if len(rows) != len(entries):
        errors.append('Returned PUB count differs from submitted entry count.')
    seen_names = set()
    for index, row in enumerate(rows):
        item = {'pub_index': index, 'valid': True, 'errors': []}
        try:
            if index >= len(entries) or index >= len(circuits):
                raise ValueError('PUB has no corresponding submitted circuit and entry.')
            entry, circuit = entries[index], circuits[index]
            if content_hash(row.get('entry')) != content_hash(entry):
                raise ValueError('Returned entry differs from immutable submission metadata.')
            name = row.get('counts_file')
            if not isinstance(name, str) or Path(name).name != name or name in seen_names:
                raise ValueError('Count filename is invalid or repeated.')
            seen_names.add(name)
            counts_path = _inside_file(directory, name)
            if file_hash(counts_path) != row.get('counts_sha256'):
                raise ValueError('Count checksum differs from archived raw evidence.')
            counts = read_json(counts_path)
            if not isinstance(counts, dict) or not counts:
                raise ValueError('A PUB has no count mapping.')
            width = circuit.num_clbits
            if width < 1 or any(not isinstance(key, str) or len(key) != width or set(key)-{'0','1'} for key in counts):
                raise ValueError('Count keys do not have the exact binary width of the meas register.')
            if any(isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 0 for value in counts.values()):
                raise ValueError('Counts must be nonnegative integers.')
            if sum(counts.values()) != request['settings']['shots']:
                raise ValueError('Returned shot total differs from the submitted budget.')
            meta = entry['solver_metadata']
            system = [int(bit) for bit in meta['system_qubits']]
            post = {int(bit): value for bit,value in meta['postselect_bits'].items()}
            if len(system) != len(set(system)) or set(system) & set(post):
                raise ValueError('System and postselection bit roles are repeated or overlap.')
            if any(bit < 0 or bit >= width for bit in [*system,*post]) or any(value not in (0,1) for value in post.values()):
                raise ValueError('Invalid classical bit index or postselection value.')
            mapping = validate_measurement_map(circuit, entry)
            if set(mapping) != set(range(width)):
                raise ValueError('Each output classical bit must be measured exactly once.')
            # Work-register outputs matter too: do not validate only the bits
            # later used by a particular estimator or postselection rule.
            layout = entry['final_layout']
            if len(layout) < width or any(mapping[bit] != layout[bit] for bit in range(width)):
                raise ValueError('A measured output bit differs from its archived final physical layout.')
            if len(set(mapping.values())) != width:
                raise ValueError('Distinct output bits must measure distinct physical qubits.')
            item.update(classical_width=width, shots=int(sum(counts.values())),
                classical_to_physical={str(key): value for key,value in mapping.items()},
                final_logical_to_physical=entry['final_layout'], counts_sha256=file_hash(counts_path))
        except Exception as error:
            item['valid'] = False
            # These messages originate in this local validation path, not remote accounts.
            item['errors'] = [str(error) if isinstance(error, ValueError) else type(error).__name__]
            errors.append('PUB '+str(index)+': '+item['errors'][0])
        checks.append(item)
    return {'valid': not errors, 'errors': errors, 'pub_checks': checks,
        'submitted_circuit_count': len(entries), 'returned_pub_count': len(rows),
        'submitted_qpy_sha256': expected_hash, 'observed_qpy_sha256': actual_hash,
        'scope': 'raw count integrity, exact classical widths and physical measurement maps; does not certify a shared preparation across VQLS bases'}

def _collection_copy_journal(registry, directory):
    # Historical submission evidence is deliberately nested: a failed local
    # collection is recoverable and must not be mistaken for a fresh submission.
    target = Path(directory)/'historical_submission'
    for name in ('job.json', 'submission_intent.json', 'request.json', 'completed_reference.json'):
        source = Path(registry)/name
        if source.is_file():
            target.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target/name)

def _collect_registered_pilot(store, service, backend, registry):
    """Read a known job only. Preserve earlier completion references and counts."""
    registry = Path(registry).resolve()
    if not registry.is_relative_to((store.root/'hardware_jobs').resolve()):
        raise ValueError('Saved job registry is outside this store.')
    request = read_json(registry/'request.json')
    job_id = read_json(registry/'job.json')['job_id']
    settings = request['settings']
    if settings['backend'] not in HARDWARE_ALLOWLIST.get(settings['account'], []):
        raise ValueError('Saved account/backend pair is outside the allowlist.')
    reference_path = registry/'hardened_collection_reference.json'
    if reference_path.exists():
        reference = read_json(reference_path)
        record = store.lookup(reference['stage'], reference['key'])
        if record is None:
            raise CacheIntegrityError('Saved collection reference is incomplete; no submission is possible in this helper.')
        return record
    job = service.job(job_id)
    if job.job_id() != job_id:
        raise CacheIntegrityError('Retrieved job identifier differs from the journal.')
    status = str(job.status()).upper().split('.')[-1]
    if status not in ('DONE', 'ERROR', 'CANCELLED'):
        atomic_json(registry/'latest_read_only_job_status.json', {'job_id':job_id,'status':status,'queried_utc':utc_now()})
        return {'status':'pending','job_id':job_id,'job_status':status,'submitted_new_job':False,
            'registry':str(registry.relative_to(store.root)),'compilation':request['compilation'],
            'compile_calibration_id':request['compile_calibration_id']}
    observation = registry/'collections'/uuid.uuid4().hex
    observation.mkdir(parents=True)
    rows, retrieval_errors = [], []
    atomic_json(observation/'job_status.json', {'job_id':job_id,'status':status,'retrieved_utc':utc_now()})
    if (registry/'isa_circuits.qpy').is_file():
        shutil.copy2(registry/'isa_circuits.qpy', observation/'isa_circuits.qpy')
    atomic_json(observation/'submission_request.json', request)
    try:
        if status == 'DONE':
            try:
                pubs = job.result()
            except Exception as error:
                pubs = None
                retrieval_errors.append('result_retrieval_'+type(error).__name__)
            if pubs is not None:
                saved_pubs = []
                # No optional metadata access until every obtainable count table is durable.
                for index, pub in enumerate(pubs):
                    name = f'raw_counts_hardware_{index}.json'
                    try:
                        counts = pub.data.meas.get_counts()
                        atomic_json(observation/name, counts)
                        rows.append({'entry':request['compiled_entries'][index] if index < len(request['compiled_entries']) else None,
                            'counts_file':name,'counts_sha256':file_hash(observation/name)})
                        saved_pubs.append((index,pub))
                    except Exception as error:
                        retrieval_errors.append('PUB_'+str(index)+'_counts_'+type(error).__name__)
                for index, pub in saved_pubs:
                    _collection_optional(observation, f'pub_metadata_{index}', lambda pub=pub: pub.metadata)
                _collection_optional(observation, 'primitive_result_metadata', lambda: pubs.metadata)
        _collection_optional(observation, 'job_metrics', job.metrics)
        _collection_optional(observation, 'job_input_options', lambda: _collection_input_options(job))
    finally:
        # Calibration archival still runs when optional metadata or result reading fails.
        ids, availability = _archive_job_calibrations(store, job, backend, observation, request['compile_calibration_id'])
        ids.append(request['before_submission_calibration_id'])
    validation = _collection_validate(observation, request, rows)
    if retrieval_errors:
        validation['valid'] = False
        validation['errors'].extend(retrieval_errors)
    if status != 'DONE':
        validation.update(valid=False, not_applicable_reason='job_did_not_complete_successfully')
    atomic_json(observation/'raw_counts_validation.json', validation)
    atomic_json(observation/'collection_rows.json', rows)
    _collection_copy_journal(registry, observation)
    result_status = ('completed' if validation['valid'] else 'invalid_hardware_counts') if status == 'DONE' else 'hardware_job_failed'
    def collect(directory):
        for source in observation.iterdir():
            if source.is_dir(): shutil.copytree(source,directory/source.name)
            elif source.is_file(): shutil.copy2(source,directory/source.name)
        return {'status':result_status,'job_id':job_id,'job_status':status,'rows':rows,
            'calibration_availability':availability,'registry':str(registry.relative_to(store.root)),
            'compilation':request['compilation'],'compile_calibration_id':request['compile_calibration_id'],
            'validation':validation,'hardware_jobs_submitted':1,'hardware_jobs_submitted_this_call':0,
            'submitted_new_job':False,'metric_scope':'validated raw Z-basis counts; VQLS multi-basis physical-prefix consistency requires separate analysis'}
    spec = {'job_id':job_id,'request_id':request['request_id'],
            'observation_id':observation.name,'operation':'retrieve_existing_job_only'}
    record = store.run('read_only_hardware_collection',spec,collect,roles=['collection_hardening','noise_helpers','backends'],
        dependencies=request['dependencies'],calibration_ids=sorted(set(ids)))
    # Preserve completed_reference.json: it may identify the historical first collection.
    # Transient invalid retrievals remain retryable via a fresh observation on the same job.
    if result_status in ('completed','hardware_job_failed'):
        atomic_json(reference_path,store.dependency(record))
    return record

def audit_saved_hardware_record(record, store):
    """Purely local validation of an existing record under a new cached stage."""
    directory = Path(record['directory'])
    request_path = directory/'submission_request.json'
    if not request_path.exists(): request_path = directory/'request.json'
    request = read_json(request_path)
    rows = record['result'].get('rows',[])
    validation = _collection_validate(directory,request,rows)
    def save(directory):
        atomic_json(directory/'validation.json',validation)
        return {'status':'validated' if validation['valid'] else 'invalid','job_id':record['result']['job_id'],
            'all_counts_and_maps_valid':validation['valid'],
            'validation':validation,'hardware_jobs_submitted_this_call':0,'network_accessed':False,
            'original_result':store.dependency(record)}
    return store.run('hardware_evidence_audit',{'job_id':record['result']['job_id'],'validation':validation},save,
        roles=['collection_hardening','noise_helpers'],dependencies=[store.dependency(record)],
        calibration_ids=record['manifest']['calibration_ids'])

def archive_saved_hardware_metadata(record, store, service=None, *, service_factory=None, refresh=False):
    """Cache read-only metadata by original evidence, job, code and environment.

    Default repeated Run All calls return the saved observation before accessing
    the service. Missing metadata is recorded honestly and is cached too. Set
    refresh=True deliberately to make a separate immutable observation; this
    never changes original counts/calibration or submits a job. A service is
    needed only for an uncached observation, so cache replay may pass None.
    A zero-argument service_factory delays even service construction until a
    cache miss. Supply either a service or a factory, never both.
    """
    if service is not None and service_factory is not None:
        raise ValueError('Supply a runtime service or service_factory, not both.')
    job_id = record['result']['job_id']
    dependency = store.dependency(record)
    spec = {'job_id':job_id,'operation':'read_only_metadata_observation',
            'original_result':dependency,'refresh_requested':bool(refresh)}
    if refresh:
        spec['observation_id'] = uuid.uuid4().hex
    def save(directory):
        active_service = service if service is not None else service_factory() if service_factory is not None else None
        if active_service is None:
            raise ValueError('A runtime service is required for an uncached read-only metadata observation.')
        job = active_service.job(job_id)
        if job.job_id() != job_id:
            raise CacheIntegrityError('Retrieved job identifier differs from saved hardware evidence.')
        availability = {}
        try:
            pubs = job.result() if str(job.status()).upper().split('.')[-1] == 'DONE' else None
            if pubs is None: raise ValueError('Job has no completed result metadata.')
            availability['primitive_result_metadata'] = _collection_optional(directory,'primitive_result_metadata',lambda:pubs.metadata)
        except Exception as error:
            availability['primitive_result_metadata'] = {'available':False,'reason':type(error).__name__}
            atomic_json(directory/'primitive_result_metadata_availability.json',availability['primitive_result_metadata'])
        availability['job_metrics'] = _collection_optional(directory,'job_metrics',job.metrics)
        availability['job_input_options'] = _collection_optional(directory,'job_input_options',lambda:_collection_input_options(job))
        return {'status':'metadata_observed','job_id':job_id,'availability':availability,
            'retrieved_utc':utc_now(),'hardware_jobs_submitted_this_call':0,'original_result':dependency,
            'network_accessed_during_original_observation':True,
            'calibration_scope':'original result calibration IDs retained; this helper does not refresh device calibration'}
    return store.run('hardware_metadata_refresh',spec,save,
        roles=['collection_hardening'],dependencies=[dependency],
        calibration_ids=record['manifest']['calibration_ids'])


In [ ]:
# CELL_ID: workflow
from qiskit import qpy, transpile
from qiskit.primitives import StatevectorSampler
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.models import BackendProperties
import qiskit_ibm_runtime.fake_provider as fake_provider

def save_qpy(path, circuits):
    with Path(path).open('wb') as stream: qpy.dump(circuits, stream)

def load_qpy(path):
    with Path(path).open('rb') as stream: return qpy.load(stream)

def plan_record(plan):
    return {k:v for k,v in plan.items() if k not in ('circuits','ansatz')}

def make_problem_stage(store, specification):
    def compute(directory):
        problem = pad_problem(graph_problem(**specification))
        baseline = classical_baselines(problem)
        # Keep exact input arrays and the reference independently of solver results.
        np.savez_compressed(directory/'problem_arrays.npz', A=problem.A, b=problem.b)
        return {'name':problem.name,'instance_id':problem.instance_id,'A':problem.A,'b':problem.b,
                'metadata':problem.metadata,'classical_baselines':baseline}
    return store.run('problem', specification, compute, roles=['problems'])

def cached_cost_executor(store, ideal_id):
    def execute(plan, *, shots, seed, evaluation):
        specification = {'plan_id':plan['plan_id'],'settings':plan['settings'],
            'shots_per_setting':int(shots),'seed':int(seed),'evaluation':int(evaluation),'execution':'ideal_counted_measurement',
            'replication_id':store.configuration['replication_id']}
        def compute(directory):
            atomic_json(directory/'measurement_plan.json', plan_record(plan))
            save_qpy(directory/'measurement_circuits.qpy', plan['circuits'])
            response = local_measured_vqls_executor(plan, shots=shots, seed=seed, evaluation=evaluation)
            atomic_json(directory/'raw_counts.json', response['counts'])
            response['metadata'].update(hardware_execution=False, calibration_id=ideal_id)
            return response
        record = store.run('cost_samples', specification, compute,
            roles=['measured_cost','reference_solvers','workflow'],calibration_ids=[ideal_id])
        # A cache hit reuses the same historical sample and contributes no new shots.
        return {'counts':record['result']['counts'], 'metadata':dict(record['result']['metadata'],
            stage=record['stage'],key=record['key'],sample_reused=record['cache_hit'])}
    return execute

def make_solver_stage(store, problem_record, algorithm, parameters, ideal_id):
    data=problem_record['result']; A=np.asarray(data['A']); b=np.asarray(data['b'])
    def compute(directory):
        implementations={'hhl':optimized_hhl_reference,'qsvt':optimized_qsvt_reference,
                         'vqls_statevector':vqls_statevector,'vqls_measured':measured_vqls}
        if algorithm=='vqls_measured':
            result=implementations[algorithm](A,b,**parameters,evaluate_callback=cached_cost_executor(store,ideal_id))
            atomic_json(directory/'optimizer_trace.json',result['evaluations'])
            if result['status']!='ok':
                atomic_json(directory/'partial_optimization.json',{k:v for k,v in result.items() if k!='circuit'})
                raise RuntimeError('Measured optimization did not complete; preserved evaluations can be reloaded.')
            # Exact state is a separately labeled local diagnostic after measured optimization.
            state=Statevector.from_instruction(result['circuit']).data
        else:
            result=implementations[algorithm](A,b,**parameters);state=result['state']
        circuit=result['circuit'].copy();circuit.measure_all()
        save_qpy(directory/'logical_measured.qpy',[circuit])
        np.savez_compressed(directory/'selected_state.npz',state=state)
        reference=np.linalg.solve(A,b)
        metadata=result['metadata']
        answer={'algorithm':algorithm,'parameters':parameters,'instance_id':data['instance_id'],
            'solver_metadata':metadata,'target_probabilities':abs(normalized(reference))**2,
            'state_diagnostics':state_metrics(A,b,state,reference),
            'logical_measured_sha256':file_hash(directory/'logical_measured.qpy'),
            'best_parameters':result.get('parameters',metadata.get('parameters')),
            'inference_scope':'all state diagnostics here use local exact complex state; hardware Z counts do not establish these values'}
        if algorithm=='vqls_measured':
            answer['independent_final_cost']=result['independent_final_cost']
            answer['measurement_model']=result['measurement_model']
            answer['cache_dependencies']=[{'stage':e['executor_metadata']['stage'],'key':e['executor_metadata']['key']}
                for e in result['evaluations'] if 'stage' in e.get('executor_metadata',{})]
        return answer
    roles=['reference_solvers','optimized_solvers','metrics','workflow']
    if algorithm=='vqls_measured':roles+=['measured_cost']
    specification={'algorithm':algorithm,'parameters':parameters}
    if algorithm=='vqls_measured':specification['replication_id']=store.configuration['replication_id']
    return store.run('solver',specification,compute,roles=roles,
        dependencies=[store.dependency(problem_record)],calibration_ids=[ideal_id])

def ideal_samples_stage(store, solver, ideal_id, settings):
    def compute(directory):
        circuit=load_qpy(solver['directory']/'logical_measured.qpy')[0]
        counts=StatevectorSampler(seed=settings['seed']).run([circuit],shots=settings['shots']).result()[0].data.meas.get_counts()
        atomic_json(directory/'raw_counts.json',counts)
        return {'counts':counts,'shots':settings['shots'],'seed':settings['seed'],'execution':'ideal_logical_shots',
                'hardware_execution':False}
    return store.run('ideal_samples',{**settings,'replication_id':store.configuration['replication_id']},compute,
        roles=['workflow'],dependencies=[store.dependency(solver)],calibration_ids=[ideal_id])

def analysis_stage(store, samples, solver, settings):
    def compute(directory):
        meta=solver['result']['solver_metadata']
        answer=probability_metrics(samples['result']['counts'],solver['result']['target_probabilities'],
            meta['system_qubits'],meta['postselect_bits'],**settings)
        return answer
    return store.run('distribution_analysis',settings,compute,roles=['metrics','workflow'],
        dependencies=[store.dependency(samples),store.dependency(solver)],
        calibration_ids=samples['manifest']['calibration_ids'])

def snapshot_samples_stage(store, compilation, calibration_id, settings, noisy=True):
    payload=store.calibration(calibration_id)['payload']
    circuit=load_qpy(compilation['directory']/'isa_circuits.qpy')[0]
    validate_measurement_map(circuit,compilation['result']['entries'][0])
    compact,mapping=compress_physical_circuit(circuit,settings['max_active_qubits'])
    selected=remap_properties(payload['properties'],mapping)
    noise,noise_metadata=snapshot_noise_model(selected,mapping,payload['configuration'])
    # Enumerate gate/readout calibration coverage instead of silently assuming absent errors are zero.
    calibrated_gates={(g['gate'],tuple(g['qubits'])) for g in selected.get('gates',[]) 
        if any(p['name']=='gate_error' and p.get('value') is not None for p in g.get('parameters',[]))}
    missing_gates=sorted({(i.operation.name,tuple(compact.find_bit(q).index for q in i.qubits))
        for i in compact.data if i.operation.name not in ('barrier','measure','rz','delay')
        and (i.operation.name,tuple(compact.find_bit(q).index for q in i.qubits)) not in calibrated_gates})
    measured={compact.find_bit(i.qubits[0]).index for i in compact.data if i.operation.name=='measure'}
    missing_readout=[q for q in sorted(measured) if not any(p['name']=='readout_error' and p.get('value') is not None for p in selected['qubits'][q])]
    if noisy and (missing_gates or missing_readout):
        raise ValueError(f'Incomplete noise coverage: gates {missing_gates}, readout qubits {missing_readout}')
    noise_record=noise.to_dict(serializable=True)
    # Aer assigns random labels to error objects. They do not alter the channels.
    # Archive the original model, but exclude only those labels from semantic identity.
    semantic_noise=copy.deepcopy(noise_record)
    for error in semantic_noise.get('errors',[]):error.pop('id',None)
    spec={**settings,'noisy':noisy,'noise_model_sha256':content_hash(semantic_noise) if noisy else None,
          'replication_id':store.configuration['replication_id']}
    def compute(directory):
        atomic_json(directory/'noise_model.json',noise_record if noisy else {'noise_model':'none'})
        atomic_json(directory/'active_properties.json',selected)
        atomic_json(directory/'physical_to_compact.json',mapping)
        save_qpy(directory/'compact_isa.qpy',[compact])
        simulator=AerSimulator(method=settings['method'],noise_model=noise if noisy else None,
            max_parallel_threads=1,max_parallel_experiments=1,max_memory_mb=8192)
        started=time.perf_counter()
        result=simulator.run(compact,shots=settings['shots'],seed_simulator=settings['seed']).result()
        if not result.success:raise RuntimeError('Simulation failed; partial artifacts saved.')
        counts={str(k):int(v) for k,v in result.get_counts(0).items()}
        atomic_json(directory/'raw_counts.json',counts)
        if sum(counts.values())!=settings['shots']:raise ValueError('Returned shot count differs from request.')
        return {'counts':counts,'execution':'calibration_snapshot_noise' if noisy else 'compiled_ideal',
            'hardware_execution':False,'shots':settings['shots'],'seed':settings['seed'],
            'simulation_seconds':time.perf_counter()-started,'aer_metadata':result.results[0].metadata,
            'physical_to_compact':mapping,'noise_metadata':noise_metadata if noisy else {'noise_model_scope':'none'},
            'noise_exclusions':['drift','crosstalk','leakage','unmodeled coherent errors','scheduled idle noise'],
            'virtual_rz_policy':'ideal frame change','missing_gate_calibrations':missing_gates,'missing_readout':missing_readout}
    return store.run('snapshot_samples',spec,compute,roles=['workflow','noise_helpers'],
        dependencies=[store.dependency(compilation)],calibration_ids=[calibration_id])

def history_index(workspace):
    # Import old validated results as dated history, never as new optimized solver results.
    base=Path(workspace)/'results'/'reference_v1'
    if not base.exists():return {'available':False,'records':[]}
    rows=[]
    for path in sorted(base.glob('*/*/record.json')):
        record=read_json(path)
        rows.append({'path':path.relative_to(workspace).as_posix(),'sha256':file_hash(path),
                     'status':record.get('status'),'algorithm':record.get('algorithm'),
                     'scope':'historical local exact-state / synthetic ideal shots; previous source version'})
    return {'available':True,'records':rows,'automatically_recomputed':False}


In [ ]:
# CELL_ID: common_preparation
"""Offline future VQLS measurements sharing one saved physical preparation.

These circuits are prospective artifacts, never results of a physical experiment.
The helper neither creates a runtime service nor submits or retranspiles circuits.
An archived Target establishes support at the snapshot time only; a future run
must capture current calibration and verify that support still holds.
"""
from qiskit.quantum_info import SparsePauliOp, Statevector


def _common_supported_instruction(target, name, qubits):
    return any(item['name'] == name and
               (item.get('qubits') is None or list(item['qubits']) == list(qubits))
               for item in target.get('instructions', []))


def _common_logical_state(circuit, physical_layout):
    """Recover two logical amplitudes in Qiskit's little-endian ordering."""
    compact, physical_to_compact = compress_physical_circuit(circuit, max_active_qubits=2)
    if set(physical_to_compact) != set(physical_layout):
        raise ValueError('This reference requires exactly the two prepared physical system qubits.')
    state = Statevector.from_instruction(compact).data
    logical = np.empty(4, dtype=complex)
    for integer in range(4):
        physical_integer = sum(((integer >> logical_q) & 1) << physical_to_compact[physical_q]
                               for logical_q, physical_q in enumerate(physical_layout))
        logical[integer] = state[physical_integer]
    return logical


def build_common_preparation_measurements(store, compilation, entry_index, plan):
    """Append every basis suffix to one unmodified saved VQLS inference prefix.

    Only two-system-qubit, ancilla-free, bound, static unitary preparations are
    supported. Each logical output bit retains its saved ordered physical qubit.
    The StageStore result is reproducible and reusable with no backend access.
    """
    if isinstance(entry_index, bool) or not isinstance(entry_index, (int, np.integer)):
        raise ValueError('entry_index must be an integer.')
    entries = compilation['result']['entries']
    if not 0 <= entry_index < len(entries):
        raise ValueError('entry_index is outside the saved compilation.')
    entry = entries[entry_index]
    if entry.get('kind') != 'inference' or not str(entry.get('algorithm', '')).startswith('vqls'):
        raise ValueError('Select a saved VQLS inference entry.')
    if plan.get('num_qubits') != 2 or plan['ansatz'].num_qubits != 2:
        raise ValueError('This explicit common-preparation reference supports two system qubits.')
    meta = entry['solver_metadata']
    if list(meta['system_qubits']) != [0, 1] or meta['postselect_bits']:
        raise ValueError('The preparation must be an ancilla-free two-qubit inference circuit.')
    groups = plan['groups']
    settings = [group['setting'] for group in groups]
    if not settings or len(settings) != len(set(settings)) or settings != list(plan['settings']):
        raise ValueError('The plan must contain each required setting exactly once.')
    if any(len(setting) != 2 or set(setting) - {'X', 'Y', 'Z'} for setting in settings):
        raise ValueError('Every measurement setting must contain exactly two Pauli axes.')
    calibration_id = compilation['result']['calibration_id']
    calibration = store.calibration(calibration_id)['payload']
    target = calibration.get('target')
    if not target or not target.get('instructions'):
        raise ValueError('A complete archived instruction Target is required.')
    isa_path = Path(compilation['directory'])/'isa_circuits.qpy'
    if file_hash(isa_path) != compilation['result']['isa_circuits_sha256']:
        raise CacheIntegrityError('Saved compilation QPY checksum does not match its result.')
    spec = {'compilation': store.dependency(compilation), 'entry_index': int(entry_index),
            'plan_id': plan['plan_id'], 'groups': groups,
            'logical_ansatz': _semantic_circuit_signature(plan['ansatz']),
            'observable_matrices': content_hash({name: plan['model'][name] for name in
                                                ('numerator_matrix', 'denominator_matrix')}),
            'construction': 'saved physical prefix plus RZ/SX basis suffixes; no retranspilation'}

    def build(directory):
        with isa_path.open('rb') as stream:
            circuits = qpy.load(stream)
        if len(circuits) != len(entries):
            raise ValueError('Saved QPY circuit count differs from the compilation metadata.')
        source = circuits[entry_index]
        if source.num_qubits != target['num_qubits'] or source.num_clbits != 2:
            raise ValueError('Saved circuit width differs from the archived Target or two-bit readout.')
        mapping = validate_measurement_map(source, entry)
        layout = list(entry['final_layout'])
        if len(layout) != 2 or len(set(layout)) != 2 or mapping != {0: layout[0], 1: layout[1]}:
            raise ValueError('Both logical bits must retain their distinct archived physical measurements.')
        measurement_indices = [i for i, item in enumerate(source.data) if item.operation.name == 'measure']
        if len(measurement_indices) != 2:
            raise ValueError('Exactly two final measurements are required.')
        first_measurement = measurement_indices[0]
        if any(item.operation.name not in ('measure', 'barrier') for item in source.data[first_measurement:]):
            raise ValueError('Measurements must be terminal; mid-circuit measurement is unsupported.')
        prefix = source.copy(name='vqls_common_physical_preparation')
        del prefix.data[first_measurement:]
        if prefix.parameters:
            raise ValueError('All physical preparation parameters must already be bound.')
        for item in prefix.data:
            name = item.operation.name
            if name == 'barrier':
                continue
            if name in ('reset', 'delay', 'measure') or getattr(item.operation, 'blocks', ()) or item.clbits:
                raise ValueError('This validation requires a static unitary preparation without reset or delay.')
            qubits = [prefix.find_bit(q).index for q in item.qubits]
            if not _common_supported_instruction(target, name, qubits):
                raise ValueError('A saved preparation instruction is absent from the archived Target.')
        for physical in layout:
            for name in ('rz', 'sx', 'measure'):
                if not _common_supported_instruction(target, name, [physical]):
                    raise ValueError(f'Archived Target lacks {name} support on physical qubit {physical}.')
        # Support for arbitrary RZ angles, rather than one fixed RZ, is required.
        for physical in layout:
            rz_rows = [row for row in target['instructions'] if row['name'] == 'rz' and
                       (row.get('qubits') is None or row['qubits'] == [physical])]
            variable_angle = False
            for row in rz_rows:
                parameters = row.get('parameters', [])
                if len(parameters) == 1:
                    try:
                        float(parameters[0])
                    except (TypeError, ValueError):
                        variable_angle = True
            if not variable_angle:
                raise ValueError('Archived RZ instruction does not document a variable angle.')

        prefix_signature = _semantic_circuit_signature(prefix)
        preparation_hash = content_hash(prefix_signature)
        prefix_bytes = _qpy_bytes([prefix])
        (directory/'physical_preparation.qpy').write_bytes(prefix_bytes)
        atomic_json(directory/'physical_preparation_signature.json', prefix_signature)
        state = _common_logical_state(prefix, layout)
        logical_state = Statevector.from_instruction(plan['ansatz']).data
        overlap = np.vdot(logical_state, state)
        phase_aligned_error = float(np.linalg.norm(state - logical_state * overlap / abs(overlap))) if abs(overlap) else float('inf')
        if phase_aligned_error > 1e-10:
            raise ValueError('Saved physical preparation does not implement the supplied frozen logical ansatz.')
        candidates, new_entries, validation_rows = [], [], []
        prefix_count = len(prefix.data)
        for group in groups:
            setting = group['setting']
            candidate = prefix.copy(name='vqls_common_cost_' + setting)
            candidate.barrier(*layout, label='common_preparation_boundary')
            for logical_q, axis in enumerate(reversed(setting)):
                physical_q = layout[logical_q]
                if axis == 'Y':
                    candidate.rz(-np.pi / 2, physical_q)  # Sdg up to an irrelevant global phase.
                if axis in ('X', 'Y'):
                    # Chronological RZ(pi/2), SX, RZ(pi/2) implements H up to global phase.
                    candidate.rz(np.pi / 2, physical_q)
                    candidate.sx(physical_q)
                    candidate.rz(np.pi / 2, physical_q)
            for logical_q, physical_q in enumerate(layout):
                candidate.measure(physical_q, logical_q)
            candidate.metadata = {'kind': 'future_common_preparation_cost', 'setting': setting,
                                  'physical_preparation_sha256': preparation_hash,
                                  'preparation_layout': layout, 'historical_hardware_result': False}
            extracted = candidate.copy(name=prefix.name)
            del extracted.data[prefix_count:]
            extracted.metadata = copy.deepcopy(prefix.metadata)
            # QPY includes serializer details that need not be byte-stable on repeated dumps.
            # Compare every ordered operation, parameter, physical bit and global phase;
            # retain the one archived QPY byte hash for file integrity only.
            if _semantic_circuit_signature(extracted) != prefix_signature:
                raise ValueError('A measurement setting modified the shared physical prefix.')
            measured = candidate.copy()
            del measured.data[-2:]
            probabilities = abs(_common_logical_state(measured, layout)) ** 2
            reference = QuantumCircuit(2)
            reference.initialize(logical_state, [0, 1])
            for logical_q, axis in enumerate(reversed(setting)):
                if axis == 'Y': reference.sdg(logical_q)
                if axis in ('X', 'Y'): reference.h(logical_q)
            expected_probabilities = Statevector.from_instruction(reference).probabilities()
            probability_error = float(np.max(abs(probabilities - expected_probabilities)))
            pauli_errors = []
            for term in group['terms']:
                operator = SparsePauliOp.from_list([(term['label'], 1.0)]).to_matrix()
                direct = float(np.vdot(state, operator @ state).real)
                from_probabilities = sum(float(probability) * (1 - 2 * ((integer & term['parity_mask']).bit_count() % 2))
                                         for integer, probability in enumerate(probabilities))
                pauli_errors.append(abs(direct - from_probabilities))
            maximum_pauli_error = float(max(pauli_errors, default=0.0))
            if probability_error > 1e-10 or maximum_pauli_error > 1e-10:
                raise ValueError('Basis suffix or classical-bit order failed its exact local validation.')
            new_entry = {'algorithm': entry['algorithm'], 'kind': 'cost', 'setting': setting,
                         'instance_id': entry.get('instance_id'), 'plan_id': plan['plan_id'],
                         'solver_metadata': {'system_qubits': [0, 1], 'postselect_bits': {}},
                         'final_layout': layout, 'preparation_layout': layout,
                         'physical_preparation_sha256': preparation_hash,
                         'physical_preparation_qpy_sha256': hashlib.sha256(prefix_bytes).hexdigest(),
                         'prefix_instruction_count': prefix_count, 'resources': circuit_resources(candidate),
                         'calibration_id': calibration_id, 'historical_hardware_result': False}
            if validate_measurement_map(candidate, new_entry) != mapping:
                raise ValueError('A basis circuit changed the output physical mapping.')
            candidates.append(candidate)
            new_entries.append(new_entry)
            validation_rows.append({'setting': setting, 'shared_prefix_operations_parameters_bits_and_phase_equal': True,
                                    'ordered_physical_layout': layout, 'exact_basis_probabilities': probabilities.tolist(),
                                    'maximum_probability_error': probability_error,
                                    'maximum_Pauli_expectation_error': maximum_pauli_error})
        with (directory/'isa_circuits.qpy').open('wb') as stream:
            qpy.dump(candidates, stream)
        with (directory/'isa_circuits.qpy').open('rb') as stream:
            recovered = qpy.load(stream)
        for candidate, recovered_circuit in zip(candidates, recovered):
            if _semantic_circuit_signature(candidate) != _semantic_circuit_signature(recovered_circuit):
                raise ValueError('Measurement circuit changed during its QPY round trip.')
        local_validation = {'status': 'passed', 'settings': validation_rows,
                            'preparation_state_error_after_global_phase_alignment': phase_aligned_error,
                            'qpy_round_trip_validated': True, 'repeated_qpy_byte_equality_required': False, 'hardware_calls': 0,
                            'scope': 'exact local algebra on frozen circuits; no physical-device measurement or noise validation'}
        atomic_json(directory/'local_validation.json', local_validation)
        topology = {'backend': calibration.get('backend'), 'physical_width': source.num_qubits,
                    'ordered_system_physical_qubits': layout, 'calibration_id': calibration_id,
                    'target_scope': calibration.get('snapshot_scope', 'archived compilation snapshot'),
                    'gates_on_selected_qubits': [row for row in target['instructions'] if row.get('qubits') is None or
                                                  set(row['qubits']).issubset(set(layout))]}
        atomic_json(directory/'archived_topology_and_gates.json', topology)
        return {'status': 'prepared_for_future_confirmation', 'entries': new_entries,
                'calibration_id': calibration_id, 'backend': calibration.get('backend'),
                'isa_circuits_sha256': file_hash(directory/'isa_circuits.qpy'),
                'physical_preparation_sha256': preparation_hash,
                'physical_preparation_qpy_sha256': file_hash(directory/'physical_preparation.qpy'),
                'ordered_physical_layout': layout, 'validation': local_validation,
                'hardware_jobs_submitted': 0, 'hardware_calls': 0,
                'physical_evidence': False, 'fidelity_certificate': None,
                'remaining_requirements': ['capture and validate current calibration/Target before any future execution',
                    'randomized repeated settings with stationarity checks', 'basis-rotation/readout treatment and finite-sample bounds'],
                'scope': 'future measurement circuits sharing one saved physical preparation; no retranspilation and no hardware execution'}

    return store.run('common_preparation', spec, build,
                     roles=['common_preparation', 'backends', 'noise_helpers'],
                     dependencies=[store.dependency(compilation)], calibration_ids=[calibration_id])


## Validation and execution

Validation checks run locally and are themselves saved. A successful rerun loads compatible results without invoking their expensive calculation. Changing a code definition creates a new identity for stages that use it. Failed scientific guards preserve counts and diagnostic artifacts.

In [ ]:
# CELL_ID: coherent_screening
"""Small dense coherent screening; all candidate successes and failures persist.

No network or hardware submission occurs here. This reference-only screening
uses the complete projected operator, with a scale fixed from A before seeing b.
The existing HHL/QSVT helpers remain unchanged. Gamma is never tuned.
"""

def bounded_support_hhl(A, b, n_qpe=4):
    """HHL with C=lambda_min/2 and rejected estimates below C.

    C and the cutoff are spectral parameters independent of b. The retained
    rotation amplitude C/lambda_est <= 1. Finite-QPE side lobes remain present:
    cutoff and C changes are a different treatment, not free amplification.
    """
    matrix, rhs, rhs_norm, eigenvalues = _validated_system(A, b)
    m = _positive_integer(n_qpe, 'n_qpe')
    n = int(math.log2(len(rhs)))
    if m > 10 or n + m + 1 > 18:
        raise ValueError('Dense screening limits QPE to 10 and total qubits to 18.')
    lam_max = float(eigenvalues[-1]); C = float(eigenvalues[0]/2)
    spacing = 2*lam_max/(1 << m)
    system = list(range(n)); phase = list(range(n,n+m)); success = n+m
    qpe = QuantumCircuit(n+m, name='screen QPE')
    for j, control in enumerate(phase):
        qpe.h(control)
        evolution = expm(2j*np.pi*matrix*(1 << j)/(2*lam_max))
        qpe.append(UnitaryGate(evolution).control(), [control]+system)
    qpe.append(QFTGate(m).inverse(), phase)
    angles = [0.0]*(1 << m)
    retained = []
    for k in range(1,1 << m):
        estimate = k*spacing
        if estimate + 1e-14*lam_max < C:
            continue
        retained.append(k)
        ratio = C/estimate
        if ratio > 1+1e-12:
            raise ArithmeticError('Invalid reciprocal rotation amplitude.')
        angles[k] = float(2*np.arcsin(min(1.0,ratio)))
    circuit = QuantumCircuit(n+m+1, name='bounded support HHL')
    circuit.append(StatePreparation(rhs),system)
    circuit.compose(qpe,qubits=system+phase,inplace=True)
    circuit.append(UCRYGate(angles),[success]+phase)
    circuit.compose(qpe.inverse(),qubits=system+phase,inplace=True)
    full = Statevector.from_instruction(circuit).data
    amplitudes = full[(1 << success):(1 << success)+len(rhs)]
    probability = float(np.vdot(amplitudes,amplitudes).real)
    if probability < 1e-28:
        raise PostselectionError('Zero bounded-support HHL postselection.')
    metadata = dict(_circuit_metadata(circuit), algorithm='HHL bounded spectral support',
        n_qpe=m, lambda_min=float(eigenvalues[0]), lambda_max=lam_max,
        eigenvalue_grid_spacing=spacing, reciprocal_rotation_constant=C,
        cutoff=C, retained_phase_bins=retained, reciprocal_rotation_angles=angles,
        success_probability=probability, system_qubits=system, qpe_qubits=phase,
        success_qubit=success, postselect_bits={**{q:0 for q in phase},success:1},
        postselection='success=1 AND all QPE work qubits=0',
        scaling_policy='C=lambda_min(A)/2; reject estimated eigenvalues below C',
        scaling_uses_rhs=False, norm_formula_is_exact=False)
    return {'circuit':circuit,'state':amplitudes/np.sqrt(probability),'metadata':metadata}


def _coherent_projected_operator(prepared_circuit, dimension, metadata):
    """Remove exactly the known input preparation, then test every input column.

    Qiskit little-endian system bits are logical qubits 0..log2(N)-1.
    The required ancilla branch is projected after the entire inverse sequence.
    """
    n = int(math.log2(dimension))
    first = prepared_circuit.data[0]
    if (first.operation.name != 'state_preparation'
            or [prepared_circuit.find_bit(q).index for q in first.qubits] != list(range(n))):
        raise ValueError('Expected one explicit initial system StatePreparation.')
    transform = prepared_circuit.copy()
    transform.data.pop(0)
    expected_ancillas = set(range(n,transform.num_qubits))
    selected = {int(q):int(v) for q,v in metadata['postselect_bits'].items()}
    if set(selected) != expected_ancillas:
        raise ValueError('Projected operator requires every ancilla explicitly selected.')
    offset = sum(v << q for q,v in selected.items())
    operator = np.empty((dimension,dimension),dtype=complex)
    for column in range(dimension):
        initial = np.zeros(1 << transform.num_qubits,dtype=complex)
        initial[column] = 1.0
        full = Statevector(initial).evolve(transform).data
        operator[:,column] = full[offset:offset+dimension]
    return operator, transform


def _coherent_spectral_prediction(A, metadata, algorithm):
    values,vectors = np.linalg.eigh(A)
    if algorithm == 'hhl':
        bins = 1 << metadata['n_qpe']
        phases = values/(2*metadata['lambda_max'])
        estimates = np.arange(bins)*metadata['eigenvalue_grid_spacing']
        amplitudes = np.zeros(bins)
        amplitudes[metadata['retained_phase_bins']] = (
            metadata['reciprocal_rotation_constant']/estimates[metadata['retained_phase_bins']])
        # QPE followed by the inverse QPE weights each retained reciprocal by
        # |QPE_amplitude(k|lambda)|^2, rather than its amplitude or nearest bin.
        t = np.arange(bins)
        delta = phases[:,None,None]-np.arange(bins)[None,:,None]/bins
        qpe_prob = abs(np.exp(2j*np.pi*delta*t).mean(axis=-1))**2
        response = qpe_prob@amplitudes
        beta = metadata['reciprocal_rotation_constant']
    else:
        response = _phase_response(np.asarray(metadata['phases']), values/metadata['alpha'])
        beta = metadata['inverse_polynomial_scale']*metadata['alpha']
    return (vectors*response)@vectors.T, float(beta), response


def _coherent_candidate(A,b,candidate,settings,directory):
    """Candidate-level failures are completed scientific records, not omitted."""
    started = time.perf_counter()
    algorithm = candidate['algorithm']
    common = {'candidate_id':candidate['candidate_id'],'algorithm':algorithm,
        'parameters':candidate,'hardware_execution':False,
        'source_independence':'beta, phases, rotations depend on A only; full projected K tests every input',
        'screening_scope':'dense local operator correctness, not scalability or noisy hardware accuracy'}
    try:
        if algorithm == 'hhl':
            function = optimized_hhl_reference if candidate['scaling']=='bin_spacing' else bounded_support_hhl
            answer = function(A,b,n_qpe=candidate['n_qpe'])
        else:
            answer = optimized_qsvt_reference(A,b,degree=candidate['degree'],
                phase_tolerance=settings['phase_tolerance'],seed=settings['phase_seed'])
        metadata = answer['metadata']; circuit = answer['circuit']
        projected, transform = _coherent_projected_operator(circuit,len(b),metadata)
        prediction,beta,response = _coherent_spectral_prediction(A,metadata,algorithm)
        target_operator = beta*np.linalg.inv(A)
        absolute_error = float(np.linalg.norm(projected-target_operator,2))
        relative_error = absolute_error/float(np.linalg.norm(target_operator,2))
        spectral_error = float(np.linalg.norm(projected-prediction,2))
        reference = np.linalg.solve(A,b)
        diagnostics = state_metrics(A,b,answer['state'],reference)
        rhs = normalized(b)
        input_action_error = float(np.linalg.norm(projected@rhs-
            answer['state']*np.sqrt(metadata['success_probability'])))
        rng = np.random.default_rng(9137)
        probe_errors = []
        for _ in range(8):
            probe = normalized(rng.normal(size=len(b))+1j*rng.normal(size=len(b)))
            probe_errors.append(float(np.linalg.norm(projected@probe-prediction@probe)))
        reasons = []
        if relative_error > settings['operator_relative_tolerance']:
            reasons.append('projected_operator_relative_error_exceeds_limit')
        if diagnostics['residual_reference_free_scale'] > settings['residual_tolerance']:
            reasons.append('fixed_input_residual_exceeds_limit')
        if spectral_error > 1e-9 or input_action_error > 1e-9 or max(probe_errors) > 1e-9:
            reasons.append('circuit_transfer_function_validation_failed')
        np.savez_compressed(directory/'projected_operator.npz',K=projected,
            predicted_K=prediction,target_K=target_operator,eigenvalue_response=response,
            selected_state=answer['state'],A=A,b=b)
        measured = circuit.copy(); measured.measure_all()
        save_qpy(directory/'logical_measured.qpy',[measured])
        save_qpy(directory/'unprepared_transform.qpy',[transform])
        resources = _offline_basis_resources(circuit,settings['seed_transpiler'])
        return dict(common,status='accepted' if not reasons else 'rejected',
            rejection_reasons=reasons,solver_metadata=metadata,state_diagnostics=diagnostics,
            beta=beta,scaling_fitted_to_rhs=False,operator_relative_error=relative_error,
            operator_absolute_error=absolute_error,operator_error_norm='spectral 2-norm',
            spectral_prediction_error=spectral_error,input_action_error=input_action_error,
            random_complex_rhs_transfer_errors=probe_errors,
            complete_projected_operator_columns=len(b),success_probability=metadata['success_probability'],
            target_probabilities=abs(normalized(reference))**2,offline_resources=resources,
            offline_two_qubit_gates_per_success=resources['two_qubit_gates']/metadata['success_probability'],
            circuit_sha256=file_hash(directory/'logical_measured.qpy'),
            selection_policy='physical compilation <=300 two-qubit gates, then gates/success; no hardware execution here',
            elapsed_seconds=time.perf_counter()-started)
    except (PhaseSynthesisError,PostselectionError) as error:
        # These declared scientific failures do not invalidate other candidates.
        return dict(common,status='rejected',rejection_reasons=['solver_construction_rejected'],
            failure_type=type(error).__name__,failure_detail=str(error),
            elapsed_seconds=time.perf_counter()-started)


def coherent_screening_stages(store,problem_record,ideal_id,settings=None):
    defaults = {'hhl_qpe_bits':[3,4,5], 'hhl_scalings':['bin_spacing','bounded_support'],
        'qsvt_degrees':[3,5,7,9,15], 'phase_tolerance':0.02,'phase_seed':11,
        'operator_relative_tolerance':0.03,'residual_tolerance':0.03,'seed_transpiler':1729,
        'selection_max_two_qubit_gates':300}
    settings = {**defaults,**(settings or {})}
    data = problem_record['result']; A=np.asarray(data['A']); b=np.asarray(data['b'])
    candidates=[]
    for m in settings['hhl_qpe_bits']:
        for scaling in settings['hhl_scalings']:
            candidates.append({'candidate_id':f'hhl_m{m}_{scaling}',
                'algorithm':'hhl','n_qpe':int(m),'scaling':scaling})
    for degree in settings['qsvt_degrees']:
        candidates.append({'candidate_id':f'qsvt_d{degree}','algorithm':'qsvt','degree':int(degree)})
    records=[]
    for candidate in candidates:
        record=store.run('coherent_candidate',{'candidate':candidate,'screening':{k:v for k,v in settings.items() if k not in ('hhl_qpe_bits','hhl_scalings','qsvt_degrees')}},
            lambda directory,candidate=candidate:_coherent_candidate(A,b,candidate,settings,directory),
            roles=['coherent_screening','reference_solvers','optimized_solvers','metrics','workflow'],
            dependencies=[store.dependency(problem_record)],calibration_ids=[ideal_id])
        records.append(record)
    def summarize(directory):
        return {'status':'complete','instance_id':data['instance_id'],'settings':settings,
            'candidate_count':len(records),'hardware_execution':False,
            'candidates':[dict(r['result'],stage=r['stage'],key=r['key']) for r in records],
            'passing_candidates':[r['result']['candidate_id'] for r in records if r['result']['status']=='accepted'],
            'selection_status':'requires archived-target compilation before any physical selection',
            'failure_policy':'every declared candidate, including phase-fit rejection, remains in this summary'}
    summary=store.run('coherent_screening_summary',{'settings':settings},summarize,
        roles=['coherent_screening'],dependencies=[store.dependency(r) for r in records],calibration_ids=[ideal_id])
    return {'candidates':records,'summary':summary}


def load_coherent_candidates(screening,passing_only=True):
    circuits=[];entries=[];records=[]
    for record in screening['candidates']:
        data=record['result']
        if passing_only and data['status']!='accepted':
            continue
        path=record['directory']/'logical_measured.qpy'
        if not path.exists():
            continue
        circuits.extend(load_qpy(path))
        entries.append(dict(data,kind='inference'))
        records.append(record)
    return circuits,entries,records


In [ ]:
# CELL_ID: confirmation_campaign
# Explicit bounded confirmation; the historical eight-PUB pilot stays unchanged.
from qiskit.circuit.library import StatePreparation

def prepare_vqls_confirmation(store, problem_record, solver_record, backend, calibration_id, *, seed=20261004):
    A=np.asarray(problem_record['result']['A']); b=np.asarray(problem_record['result']['b'])
    plan=measurement_plan(A,b,solver_record['result']['best_parameters'],2)
    if plan['num_qubits']!=2:raise ValueError('This frozen confirmation is defined for two system qubits.')
    logical=load_qpy(solver_record['directory']/'logical_measured.qpy')
    inference=compile_for_backend(store,logical,[dict(solver_record['result'],kind='inference')],backend,
        calibration_id,{'optimization_level':2,'seed_transpiler':1729,'max_two_qubit_gates':300},
        dependencies=[store.dependency(problem_record),store.dependency(solver_record)])
    if inference['result']['status']!='compiled':raise ValueError('Inference compilation exceeded the declared gate cap.')
    settings=[a+b for a in 'XYZ' for b in 'XYZ']
    existing={g['setting']:g for g in plan['groups']}
    tomography_plan={**plan,'plan_id':plan['plan_id']+':all_nine_bases',
        'settings':settings,'groups':[existing.get(s,{'setting':s,'terms':[]}) for s in settings]}
    common=build_common_preparation_measurements(store,inference,0,tomography_plan)
    layout=common['result']['ordered_physical_layout']
    spec={'instance':store.dependency(problem_record),'solver':store.dependency(solver_record),
          'common':store.dependency(common),'seed':seed,'tomography_repeats':2,
          'readout_patterns':[0,1,2,3],'controls':['input','classical_solution'],
          'shots_per_circuit':4096,'max_circuits':24,'max_two_qubit_gates':300}
    def prepare(directory):
        templates=load_qpy(common['directory']/'isa_circuits.qpy')
        staged=[]
        for repeat in range(2):
            for circuit,entry in zip(templates,common['result']['entries']):
                staged.append((circuit.copy(),{**entry,'kind':'tomography','repeat':repeat,
                    'technical_repeat':True,'independent_calibration_block':False}))
        for integer in range(4):
            circuit=templates[0].copy_empty_like(name='readout_pattern_'+str(integer))
            circuit.global_phase=0
            for logical_q,physical_q in enumerate(layout):
                if (integer>>logical_q)&1:
                    # SX twice is X and avoids assuming an optional native X instruction.
                    circuit.sx(physical_q);circuit.sx(physical_q)
                circuit.measure(physical_q,logical_q)
            staged.append((circuit,{'kind':'readout_calibration','prepared_basis':integer,
                'algorithm':'readout_control','final_layout':layout,'preparation_layout':layout,
                'expected_state_real':np.eye(4)[integer].tolist(),
                'solver_metadata':{'system_qubits':[0,1],'postselect_bits':{}},
                'scope':'full two-qubit assignment calibration including preparation error'}))
        target=np.linalg.solve(A,b);target=target/np.linalg.norm(target)
        for label,state in [('input',b/np.linalg.norm(b)),('classical_solution',target)]:
            logical_control=QuantumCircuit(2)
            logical_control.append(StatePreparation(state),[0,1]);logical_control.measure_all()
            manager=generate_preset_pass_manager(backend=backend,optimization_level=2,
                seed_transpiler=1729,initial_layout=layout)
            native=manager.run(logical_control)
            if native.layout.final_index_layout()!=layout:
                raise ValueError('Control routing changed the frozen ordered pair.')
            staged.append((native,{'kind':'control','control_name':label,
                'algorithm':'reference_control','final_layout':layout,'preparation_layout':layout,'setting':'ZZ',
                'expected_state_real':state.tolist(),
                'solver_metadata':{'system_qubits':[0,1],'postselect_bits':{}},
                'expected_probabilities':(abs(state)**2).tolist(),
                'classical_solution_access':label=='classical_solution'}))
        # Two interleaved temporal groups; repeats are not independent drift blocks.
        rng=np.random.default_rng(seed);order=[]
        for indices in [list(range(9))+[18,19,22],list(range(9,18))+[20,21,23]]:
            order.extend(rng.permutation(indices).tolist())
        circuits=[];entries=[]
        target_snapshot=store.calibration(calibration_id)['payload']['target']
        for position,index in enumerate(order):
            circuit,entry=staged[index]
            for instruction in circuit.data:
                name=instruction.operation.name
                if name=='barrier':continue
                qubits=[circuit.find_bit(q).index for q in instruction.qubits]
                if not _common_supported_instruction(target_snapshot,name,qubits):
                    raise ValueError('Prepared instruction absent from archived Target.')
            if entry['kind'] in ('control','readout_calibration'):
                validation_circuit=circuit.remove_final_measurements(inplace=False)
                for physical in layout:validation_circuit.id(physical)
                observed=_common_logical_state(validation_circuit,layout)
                expected=np.asarray(entry['expected_state_real'],dtype=complex)
                overlap=np.vdot(expected,observed)
                error=float(np.linalg.norm(observed-expected*overlap/abs(overlap))) if abs(overlap) else 2.0
                if error>1e-10:raise ValueError('Reference/readout preparation failed its exact local state check.')
                entry={**entry,'local_state_error_up_to_global_phase':error}
            resources=circuit_resources(circuit)
            if resources['two_qubit_gates']>300:raise ValueError('Native gate cap exceeded.')
            validate_measurement_map(circuit,entry)
            entry={**entry,'schedule_position':position,'source_template_index':index,
                   'resources':resources,'calibration_id':calibration_id}
            circuits.append(circuit);entries.append(entry)
        if len(circuits)!=24:raise AssertionError('Confirmation must contain exactly 24 circuits.')
        with (directory/'isa_circuits.qpy').open('wb') as stream:qpy.dump(circuits,stream)
        with (directory/'isa_circuits.qpy').open('rb') as stream:round_trip=qpy.load(stream)
        for original,recovered in zip(circuits,round_trip):
            if _semantic_circuit_signature(original)!=_semantic_circuit_signature(recovered):
                raise ValueError('Physical circuit QPY round trip changed its operations.')
        audit={'status':'validated','common_prefix_verified':True,
               'common_ordered_mapping_verified':True,'ordered_physical_layout':layout,
               'physical_preparation_sha256':common['result']['physical_preparation_sha256'],
               'stationarity_established':False,'measurement_error_bounded':False,
               'scope':'structural circuit contract only; no hardware fidelity certificate'}
        atomic_json(directory/'preparation_audit.json',audit)
        atomic_json(directory/'measurement_plan.json',plan_record(plan))
        atomic_json(directory/'schedule.json',entries)
        return {'status':'prepared','protocol':'vqls_tomography','entries':entries,'calibration_id':calibration_id,
                'scientific_identity':{'problem':store.dependency(problem_record),'solver':store.dependency(solver_record)},
                'isa_circuits_sha256':file_hash(directory/'isa_circuits.qpy'),
                'preparation_audit':audit,'target_state_real':target.tolist(),
                'declared_budget':{'jobs':1,'circuits':24,'shots_per_circuit':4096,
                    'raw_shots':98304,'max_execution_time_seconds':60},
                'hardware_jobs_submitted':0,'cache_dependencies':[store.dependency(inference),store.dependency(common)]}
    record=store.run('confirmation_preparation',spec,prepare,
        roles=['confirmation_campaign','common_preparation','backends','noise_helpers','measured_cost'],
        dependencies=[store.dependency(problem_record),store.dependency(solver_record),store.dependency(common)],
        calibration_ids=[calibration_id])
    return record,plan

def _confirmation_registry(store,configuration):
    # Identity deliberately binds the campaign label before a volatile calibration.
    settings={'account':configuration['account'],'backend':configuration['backend'],
              'campaign_id':configuration['campaign_id'],'replication_id':configuration.get('replication_id','block_0'),
              'protocol':configuration.get('protocol','vqls_tomography')}
    if settings['account']!='lab_zhang' or settings['backend']!='ibm_rensselaer':
        raise ValueError('This confirmation uses the dedicated Rensselaer allocation only.')
    request_id=content_hash({'bounded_confirmation_v1':settings})
    return store.root/'hardware_jobs'/request_id,request_id,settings

def load_confirmation_if_known(store,configuration,*,scientific_identity,collect=True):
    registry,request_id,settings=_confirmation_registry(store,configuration)
    if (registry/'request.json').is_file():
        if read_json(registry/'request.json').get('scientific_identity')!=scientific_identity:
            raise CacheIntegrityError('Campaign label already belongs to another problem/solver. Use a deliberate new campaign; no silent result substitution.')
    for name in ['hardened_collection_reference.json','completed_reference.json']:
        if (registry/name).is_file():
            ref=read_json(registry/name);record=store.lookup(ref['stage'],ref['key'])
            if record is None:raise CacheIntegrityError('Known confirmation has a missing completion; no replacement submission.')
            return record
    if (registry/'job.json').is_file():
        if not collect:
            request=read_json(registry/'request.json')
            cached=read_json(registry/'latest_read_only_job_status.json') if (registry/'latest_read_only_job_status.json').is_file() else {}
            return {'status':'pending','job_id':read_json(registry/'job.json')['job_id'],'submitted_new_job':False,
                'job_status':cached.get('status','NOT_REFRESHED'),'status_is_cached':True,
                'compilation':request['compilation'],'compile_calibration_id':request['compile_calibration_id'],
                'registry':str(registry.relative_to(store.root))}
        service=QiskitRuntimeService(name=settings['account']);backend=service.backend(settings['backend'])
        return _collect_registered_pilot(store,service,backend,registry)
    if (registry/'submission_intent.json').is_file():
        return {'status':'submission_uncertain','registry':str(registry),'submitted_new_job':False}
    campaign=store.root/'hardware_campaigns'/content_hash({k:settings[k] for k in ('account','backend','campaign_id')})
    if (campaign/'claimed_request.json').exists():
        return {'status':'campaign_claim_requires_reconciliation','submitted_new_job':False,
                'claimed_request':read_json(campaign/'claimed_request.json')}
    return None

def submit_prepared_confirmation(store,prepared,backend,configuration):
    known=load_confirmation_if_known(store,configuration,scientific_identity=prepared['result']['scientific_identity'],collect=configuration.get('collect_existing',True))
    if known is not None:return known
    registry,request_id,settings=_confirmation_registry(store,configuration)
    expected_circuits={'vqls_tomography':24,'coherent_bias':12}.get(settings['protocol'])
    if expected_circuits is None or prepared['result'].get('protocol')!=settings['protocol']:
        raise ValueError('Prepared protocol must match the declared bounded campaign.')
    budget=prepared['result']['declared_budget']
    if budget!={'jobs':1,'circuits':expected_circuits,'shots_per_circuit':4096,'raw_shots':expected_circuits*4096,'max_execution_time_seconds':60}:
        raise ValueError('Prepared budget differs from the fixed campaign inventory.')
    settings.update(shots=4096,max_execution_time_seconds=60)
    if not configuration.get('submit',False):return {'status':'prepared_only','submitted_new_job':False,
        'compilation':store.dependency(prepared),'compile_calibration_id':prepared['result']['calibration_id']}
    if _backend_name(backend)!=settings['backend'] or not backend.status().operational:
        raise ValueError('Dedicated backend is not operational; no submission.')
    cal_id=prepared['result']['calibration_id']
    # Validate every saved native instruction and every map before claiming usage.
    path=prepared['directory']/'isa_circuits.qpy'
    if file_hash(path)!=prepared['result']['isa_circuits_sha256']:raise CacheIntegrityError('Prepared QPY changed.')
    with path.open('rb') as stream:circuits=qpy.load(stream)
    entries=prepared['result']['entries']
    if len(circuits)!=expected_circuits or len(entries)!=expected_circuits:raise ValueError('Prepared inventory differs from the declared campaign.')
    for circuit,entry in zip(circuits,entries):
        if circuit.num_qubits!=backend.num_qubits or circuit_resources(circuit)['two_qubit_gates']>300:
            raise ValueError('Native resource cap or backend width mismatch.')
        validate_measurement_map(circuit,entry)
        for instruction in circuit.data:
            if instruction.operation.name=='barrier':continue
            qargs=tuple(circuit.find_bit(q).index for q in instruction.qubits)
            if not backend.target.instruction_supported(operation_name=instruction.operation.name,qargs=qargs):
                raise ValueError('A native instruction is unsupported by the current backend Target.')
    before=capture_backend(store,backend,'hardware',{'phase':'confirmation_immediately_before_submission','campaign_id':settings['campaign_id']})
    if before!=cal_id:
        return {'status':'calibration_changed_before_submission','submitted_new_job':False,
                'compile_calibration_id':cal_id,'new_calibration_id':before}
    durations=[float(circuit.estimate_duration(backend.target,unit='s')) for circuit in circuits]
    rep_delay=float(backend.default_rep_delay)
    reset_duration=max(float(properties.duration or 0) for properties in backend.target['reset'].values())
    estimated_seconds=2+4096*sum(duration+rep_delay+reset_duration for duration in durations)
    timing={'circuit_longest_path_seconds':durations,'default_rep_delay_seconds':rep_delay,
            'reset_duration_seconds':reset_duration,'assumed_single_job_overhead_seconds':2,
            'estimated_qpu_seconds':estimated_seconds,
            'scope':'Approximation using Target durations and one overhead; provider splitting and scheduling may differ.'}
    if not np.isfinite(estimated_seconds) or estimated_seconds>60:
        raise ValueError('Estimated workload exceeds the declared 60-second cap; no submission.')
    options={'default_shots':4096,'max_execution_time':60,
             'twirling':{'enable_gates':False,'enable_measure':False},
             'dynamical_decoupling':{'enable':False},'execution':{'init_qubits':True,'meas_type':'classified'},
             'environment':{'job_tags':['diffusion_confirmation',settings['campaign_id'],request_id[:16]]}}
    # Construct/validate options before persisting an uncertain remote intent.
    sampler=SamplerV2(mode=backend,options=options)
    campaign=store.root/'hardware_campaigns'/content_hash({k:settings[k] for k in ('account','backend','campaign_id')})
    registry.mkdir(parents=True,exist_ok=True);campaign.mkdir(parents=True,exist_ok=True)
    if (campaign/'claimed_request.json').exists():
        return {'status':'campaign_job_already_claimed','submitted_new_job':False,
                'claimed_request':read_json(campaign/'claimed_request.json')}
    dependencies=[store.dependency(prepared)]
    request={'request_id':request_id,'settings':settings,'compiled_entries':entries,
        'scientific_identity':prepared['result']['scientific_identity'],
        'compile_calibration_id':cal_id,'before_submission_calibration_id':before,
        'compilation':store.dependency(prepared),'dependencies':dependencies,
        'isa_circuits_sha256':file_hash(path),'requested_sampler_options':options,
        'budget':prepared['result']['declared_budget'],'timing_estimate':timing,
        'user_authorization':'User requested any necessary runs and configurations; bounded measurement-repair confirmation, 4 October 2026.'}
    with (campaign/'claimed_request.json').open('x',encoding='utf8') as stream:
        json.dump({'request_id':request_id,'claimed_utc':utc_now(),'max_jobs':1},stream)
    atomic_json(registry/'request.json',request);shutil.copy2(path,registry/'isa_circuits.qpy')
    atomic_json(registry/'submission_intent.json',{'status':'submitting_or_unknown','request_id':request_id,
        'written_before_remote_submission_utc':utc_now(),'limits':prepared['result']['declared_budget'],
        'calibration_ids':[cal_id,before]})
    store.bundle(registry,calibration_ids=[cal_id,before],dependencies=dependencies)
    job=sampler.run(circuits,shots=4096)
    atomic_json(registry/'job.json',{'job_id':job.job_id(),'submitted_utc':utc_now()})
    store.bundle(registry,calibration_ids=[cal_id,before],dependencies=dependencies)
    return {'status':'submitted','job_id':job.job_id(),'submitted_new_job':True,
            'registry':str(registry.relative_to(store.root)),'compilation':store.dependency(prepared),
            'compile_calibration_id':cal_id,'request_id':request_id}

def run_confirmation_campaign(store,problem_record,solver_record,configuration):
    identity={'problem':store.dependency(problem_record),'solver':store.dependency(solver_record)}
    known=load_confirmation_if_known(store,configuration,scientific_identity=identity,collect=configuration.get('collect_existing',True))
    if known is not None:return known
    if not configuration.get('prepare',True):return {'status':'disabled','submitted_new_job':False}
    service=QiskitRuntimeService(name=configuration['account']);backend=service.backend(configuration['backend'])
    cal_id=capture_backend(store,backend,'hardware',{'phase':'confirmation_before_compilation',
                                                  'campaign_id':configuration['campaign_id']})
    preparation,plan=prepare_vqls_confirmation(store,problem_record,solver_record,backend,cal_id)
    result=submit_prepared_confirmation(store,preparation,backend,configuration)
    return result


In [ ]:
# CELL_ID: confirmation_analysis
"""Frozen-state two-qubit confirmation analysis. No network or sampling calls.

All labels use Qiskit's q1,q0 order. Calibration columns are measured outcome
probabilities conditional on the prepared computational-basis integer. The
linear readout inverse is a model-dependent diagnostic, not corrected counts.
"""
import itertools
import json
from pathlib import Path
import numpy as np


_CONFIRM_SETTINGS = tuple(''.join(p) for p in itertools.product('XYZ', repeat=2))
_CONFIRM_LABELS = tuple(''.join(p) for p in itertools.product('IXYZ', repeat=2))


def _confirmation_matrix_json(matrix):
    value = np.asarray(matrix, dtype=complex)
    return {'real': value.real.tolist(), 'imag': value.imag.tolist()}


def _confirmation_counts_vector(counts):
    clean = _mvqls_clean_counts(counts, 2)
    return np.asarray([clean.get(format(i, '02b'), 0) for i in range(4)], dtype=np.int64)


def _confirmation_counts_dict(vector):
    return {format(i, '02b'): int(value) for i, value in enumerate(vector) if value}


def _confirmation_pauli(label):
    matrices = {'I': np.eye(2), 'X': np.array([[0, 1], [1, 0]]),
                'Y': np.array([[0, -1j], [1j, 0]]), 'Z': np.diag([1, -1])}
    return np.kron(matrices[label[0]], matrices[label[1]])


def _confirmation_pauli_signs(label):
    mask = sum(1 << q for q, axis in enumerate(reversed(label)) if axis != 'I')
    return np.asarray([1 - 2 * ((integer & mask).bit_count() % 2) for integer in range(4)])


def _confirmation_validate_rows(rows, *, require_provenance=True):
    if len(rows) != 24:
        raise ValueError('The frozen confirmation inventory requires exactly 24 records.')
    tomography = [r for r in rows if r.get('kind') == 'tomography']
    readout = [r for r in rows if r.get('kind') == 'readout_calibration']
    controls = [r for r in rows if r.get('kind') == 'control']
    if len(tomography) != 18 or len(readout) != 4 or len(controls) != 2:
        raise ValueError('Require 18 tomography records, four readout patterns and two controls.')
    observed = [(r.get('setting'), r.get('repeat')) for r in tomography]
    repeats = sorted({r.get('repeat') for r in tomography})
    if len(repeats) != 2 or any(isinstance(x, bool) or not isinstance(x, int) for x in repeats):
        raise ValueError('Exactly two explicitly indexed technical repeats are required.')
    if len(set(observed)) != 18 or set(observed) != set(itertools.product(_CONFIRM_SETTINGS, repeats)):
        raise ValueError('Every tomography basis must appear once in each technical repeat.')
    if sorted(r.get('prepared_basis') for r in readout) != [0, 1, 2, 3]:
        raise ValueError('Readout patterns must be prepared integers 0,1,2,3 exactly once.')
    if any(r.get('setting') != 'ZZ' for r in controls):
        raise ValueError('The two declared controls are measured in ZZ.')
    if any(not r.get('control_name') for r in controls) or len({r['control_name'] for r in controls}) != 2:
        raise ValueError('Controls require two distinct explicit control_name labels.')
    for row in rows:
        vector = np.asarray(row['counts_vector'])
        if vector.shape != (4,) or vector.dtype.kind not in 'iu' or np.any(vector < 0) or vector.sum() <= 0:
            raise ValueError('Every record needs four nonnegative integer outcome counts.')
        if row.get('kind') == 'control':
            truth = np.asarray(row.get('expected_probabilities'), dtype=float)
            if truth.shape != (4,) or np.any(truth < 0) or not np.isfinite(truth).all() or not np.isclose(truth.sum(), 1.):
                raise ValueError('Controls need their predeclared expected probability vectors.')
    if require_provenance:
        prefixes = [r.get('physical_preparation_sha256') for r in tomography]
        layouts = [r.get('preparation_layout') for r in tomography]
        if any(not x for x in prefixes) or len(set(prefixes)) != 1:
            raise ValueError('All tomography records must preserve the same verified preparation hash.')
        if any(x is None or len(x) != 2 or x[0] == x[1] for x in layouts) or len({tuple(x) for x in layouts}) != 1:
            raise ValueError('A common ordered two-qubit physical mapping is required.')
        if any(r.get('preparation_layout') != layouts[0] for r in [*readout, *controls]):
            raise ValueError('Readout patterns and controls must use the same ordered physical qubits.')
    return tomography, readout, controls, repeats


def _confirmation_tomography(tomography, target, *, response_inverse=None):
    """Linear inversion; preserve negative eigenvalues and out-of-range fidelities."""
    estimates, term_records = {}, {}
    for label in _CONFIRM_LABELS:
        if label == 'II':
            estimates[label] = 1.0
            term_records[label] = {'estimate': 1.0, 'sources': [], 'shot_standard_error': 0.0}
            continue
        compatible = [r for r in tomography if all(p == 'I' or p == s for p, s in zip(label, r['setting']))]
        total_shots = sum(int(np.sum(r['counts_vector'])) for r in compatible)
        signs, means, weights, sources, variance = _confirmation_pauli_signs(label), [], [], [], 0.0
        for row in compatible:
            counts = np.asarray(row['counts_vector'], dtype=float)
            shots = int(counts.sum())
            observed = counts / shots
            probabilities = observed if response_inverse is None else response_inverse @ observed
            mean = float(signs @ probabilities)
            weight = shots / total_shots
            means.append(mean)
            weights.append(weight)
            sources.append({'setting': row['setting'], 'repeat': row['repeat'], 'shots': shots,
                            'weight': weight, 'estimate': mean, 'record_index': row['record_index']})
            # This SE is only supplied for raw Bernoulli Pauli samples; calibration
            # uncertainty is instead propagated by the full-count bootstrap.
            if response_inverse is None and shots > 1:
                variance += weight ** 2 * max(0., 1 - mean ** 2) / (shots - 1)
        estimates[label] = float(np.dot(weights, means))
        term_records[label] = {'estimate': estimates[label], 'sources': sources,
            'shot_standard_error': float(np.sqrt(variance)) if response_inverse is None else None}
    rho = sum(estimates[label] * _confirmation_pauli(label) for label in _CONFIRM_LABELS) / 4
    rho = (rho + rho.conj().T) / 2  # Remove floating-point anti-Hermitian roundoff only.
    eigenvalues, eigenvectors = np.linalg.eigh(rho)
    fidelity = float(np.vdot(target, rho @ target).real)
    expected = {label: float(np.vdot(target, _confirmation_pauli(label) @ target).real) for label in _CONFIRM_LABELS}
    clipped = np.maximum(eigenvalues, 0)
    clipped /= clipped.sum()
    descriptive = (eigenvectors * clipped) @ eigenvectors.conj().T
    return {'density_matrix_linear': _confirmation_matrix_json(rho),
        'eigenvalues_linear': eigenvalues.tolist(), 'trace_linear': float(np.trace(rho).real),
        'linear_target_fidelity_unclipped': fidelity,
        'nonphysical_linear_estimate': bool(eigenvalues[0] < -1e-10 or fidelity < -1e-10 or fidelity > 1 + 1e-10),
        'pauli_estimates': estimates, 'pauli_details': term_records,
        'pauli_target_differences': {label: estimates[label] - expected[label] for label in _CONFIRM_LABELS},
        'pauli_target_expectations': expected,
        'maximum_absolute_pauli_target_difference': max(abs(estimates[label] - expected[label]) for label in _CONFIRM_LABELS),
        'descriptive_psd_clipping': {'method': 'clip negative eigenvalues then renormalize; nonlinear descriptive estimate, not raw data',
            'density_matrix': _confirmation_matrix_json(descriptive),
            'target_fidelity': float(np.vdot(target, descriptive @ target).real)},
        'duplicate_single_pauli_rule': 'Shot-weighted pooling across compatible bases and both repeats; equal weights when shot counts are equal.',
        'interpretation': 'Linear tomography assumes common stationary preparation and ideal measurement axes. Fidelity is an uncorrected model diagnostic, never a hardware certificate.',
        'certified_fidelity_lower_bound': None}


def _confirmation_readout(readout, *, inverse_condition_limit=10000.0):
    ordered = sorted(readout, key=lambda r: r['prepared_basis'])
    response = np.column_stack([np.asarray(r['counts_vector']) / np.sum(r['counts_vector']) for r in ordered])
    condition = float(np.linalg.cond(response))
    columns = []
    for row in ordered:
        counts = np.asarray(row['counts_vector'])
        shots = int(counts.sum())
        columns.append({'prepared_basis': row['prepared_basis'], 'shots': shots,
            'probabilities': (counts / shots).tolist(),
            'outcome_wilson95': [wilson_interval(int(value), shots) for value in counts],
            'record_index': row['record_index']})
    valid = np.isfinite(condition) and condition <= inverse_condition_limit
    return {'response_matrix': response.tolist(),
        'matrix_convention': 'rows measured integer 0..3; columns prepared integer 0..3; q1q0 bit order',
        'condition_number_2': condition if np.isfinite(condition) else None,
        'inverse_condition_limit': inverse_condition_limit, 'inverse_allowed': bool(valid),
        'columns': columns,
        'uncertainty_scope': 'Marginal Wilson intervals are not simultaneous. Response includes state-preparation error; treating it as readout-only is an assumption.',
        'mitigation_model': 'One stationary, basis-independent correlated assignment matrix on the same ordered qubits; does not correct basis rotations or coherent preparation errors.'}, np.linalg.inv(response) if valid else None


def _confirmation_cost_from_probabilities(plan, probabilities):
    """Linear quasi-expectations only; never present inverse weights as counts."""
    model = plan['model']
    numerator = float(model['identity_offsets']['numerator'])
    denominator = float(model['identity_offsets']['denominator'])
    for group in plan['groups']:
        for term in group['terms']:
            expectation = float(_confirmation_pauli_signs(term['label']) @ probabilities[group['setting']])
            numerator += expectation * term['numerator_coefficient']
            denominator += expectation * term['denominator_coefficient']
    valid = denominator > model['denominator_numeric_floor']
    cost = 1 - numerator / denominator if valid else None
    return {'numerator': numerator, 'denominator': denominator,
        'cost_unclipped': float(cost) if valid else None,
        'status': 'model_diagnostic' if valid else 'nonpositive_or_unresolved_numeric_denominator',
        'cost_outside_physical_interval': bool(cost < 0 or cost > 1) if valid else None,
        'certified_fidelity_lower_bound': None,
        'interpretation': 'Linear inverse-assignment quasi-expectations; systematic model error uncontrolled; full bootstrap reported separately.'}


def _confirmation_point_analysis(rows, plan, target, *, readout_inverse=True, detailed=True):
    tomography, readout, controls, repeats = _confirmation_validate_rows(rows, require_provenance=False)
    aggregated = {setting: np.sum([r['counts_vector'] for r in tomography if r['setting'] == setting], axis=0) for setting in _CONFIRM_SETTINGS}
    counts = {setting: _confirmation_counts_dict(aggregated[setting]) for setting in plan['settings']}
    cost = analyze_cost(plan, counts)
    cost['certified_fidelity_lower_bound'] = None
    cost['scope'] = 'Same compiled preparation across settings; sampling-only aggregate under common-state/measurement assumptions, not certified hardware fidelity.'
    response, inverse = _confirmation_readout(readout)
    raw = _confirmation_tomography(tomography, target)
    result = {'raw_tomography': raw, 'unmitigated_cost': cost, 'readout_calibration': response,
              'mitigated_diagnostic': None}
    if readout_inverse and inverse is not None:
        quasi = {setting: inverse @ (value / value.sum()) for setting, value in aggregated.items()}
        result['mitigated_diagnostic'] = {'tomography': _confirmation_tomography(tomography, target, response_inverse=inverse),
            'setting_quasiprobabilities': {setting: value.tolist() for setting, value in quasi.items()},
            'has_negative_quasiprobabilities': any(np.any(value < -1e-12) for value in quasi.values()),
            'cost': _confirmation_cost_from_probabilities(plan, quasi),
            'scope': 'Optional model-dependent linear inverse, separately reported from raw measurements; negative values retained.'}
    if detailed:
        target_probabilities = abs(target) ** 2
        def distribution(vector, expected):
            value = probability_metrics(_confirmation_counts_dict(vector), expected, [0, 1], {}, bootstrap_samples=0)
            shots = int(np.sum(vector))
            value['p0'] = int(vector[0]) / shots
            value['p2'] = int(vector[2]) / shots
            value['p0_wilson95'] = wilson_interval(int(vector[0]), shots)
            value['p2_wilson95'] = wilson_interval(int(vector[2]), shots)
            value['two_output_95_hoeffding_radius'] = float(np.sqrt(np.log(4 / .05) / (2 * shots)))
            value['precision_target_003_reached'] = value['two_output_95_hoeffding_radius'] <= .03
            return value
        result['z_distribution'] = distribution(aggregated['ZZ'], target_probabilities)
        result['technical_repeats'] = [{'repeat': repeat,
            'tomography': _confirmation_tomography([r for r in tomography if r['repeat'] == repeat], target),
            'z_distribution': distribution(next(r['counts_vector'] for r in tomography if r['repeat'] == repeat and r['setting'] == 'ZZ'), target_probabilities)} for repeat in repeats]
        result['controls'] = [{'control_name': row['control_name'], 'record_index': row['record_index'],
                              'distribution': distribution(row['counts_vector'], row['expected_probabilities'])} for row in controls]
    return result


def analyze_confirmation_counts(rows, plan, target_state, *, bootstrap_samples=200,
                                seed=20261004, readout_inverse=True):
    """Pure local analysis of already available records. Synthetic fixtures stay labeled."""
    _confirmation_validate_rows(rows, require_provenance=False)
    target = np.asarray(target_state, dtype=complex)
    if target.shape != (4,) or not np.isfinite(target).all() or np.linalg.norm(target) <= 0:
        raise ValueError('A finite, nonzero four-amplitude diagnostic target is required.')
    target = target / np.linalg.norm(target)
    if plan['num_qubits'] != 2 or not set(plan['settings']).issubset(_CONFIRM_SETTINGS):
        raise ValueError('The cost plan must describe the same two logical qubits.')
    if isinstance(bootstrap_samples, bool) or not isinstance(bootstrap_samples, int) or bootstrap_samples < 0:
        raise ValueError('bootstrap_samples must be a nonnegative integer.')
    result = _confirmation_point_analysis(rows, plan, target, readout_inverse=readout_inverse)
    rng = np.random.default_rng(seed)
    names = ('raw_fidelity', 'raw_cost', 'raw_min_eigenvalue', 'z_tv', 'mitigated_fidelity', 'mitigated_cost')
    draws = {name: [] for name in names}
    control_tv = {r['control_name']: [] for r in rows if r['kind'] == 'control'}
    pauli_draws = {label: [] for label in _CONFIRM_LABELS}
    for _ in range(bootstrap_samples):
        synthetic = []
        for row in rows:
            vector = np.asarray(row['counts_vector'])
            sample = rng.multinomial(int(vector.sum()), vector / vector.sum())
            synthetic.append(dict(row, counts_vector=sample))
        point = _confirmation_point_analysis(synthetic, plan, target, readout_inverse=readout_inverse, detailed=False)
        draws['raw_fidelity'].append(point['raw_tomography']['linear_target_fidelity_unclipped'])
        draws['raw_min_eigenvalue'].append(point['raw_tomography']['eigenvalues_linear'][0])
        draws['raw_cost'].append(point['unmitigated_cost'].get('cost_unclipped'))
        zz = sum((r['counts_vector'] for r in synthetic if r['kind'] == 'tomography' and r['setting'] == 'ZZ'), np.zeros(4))
        draws['z_tv'].append(float(.5 * np.abs(zz / zz.sum() - abs(target) ** 2).sum()))
        for label in _CONFIRM_LABELS:
            pauli_draws[label].append(point['raw_tomography']['pauli_estimates'][label])
        for row in synthetic:
            if row['kind'] == 'control':
                value = row['counts_vector'] / np.sum(row['counts_vector'])
                control_tv[row['control_name']].append(float(.5 * np.abs(value - row['expected_probabilities']).sum()))
        mitigated = point['mitigated_diagnostic']
        draws['mitigated_fidelity'].append(None if mitigated is None else mitigated['tomography']['linear_target_fidelity_unclipped'])
        draws['mitigated_cost'].append(None if mitigated is None else mitigated['cost']['cost_unclipped'])
    def interval(values):
        usable = [value for value in values if value is not None and np.isfinite(value)]
        return {'percentile95': np.quantile(usable, [.025, .975]).tolist() if usable else None,
                'valid_replicates': len(usable), 'invalid_replicates': len(values) - len(usable)}
    result['bootstrap'] = {'replicates': bootstrap_samples, 'seed': seed,
        'scope': 'Independent multinomial resampling within each of the 24 observed count records, retaining technical-repeat and calibration structure. Shot-only plug-in intervals conditional on the assignment model; excludes drift, axis error and model bias. Not simultaneous or certification bounds.',
        'metrics': {name: interval(values) for name, values in draws.items()},
        'pauli_expectations': {name: interval(values) for name, values in pauli_draws.items()},
        'control_total_variation': {name: interval(values) for name, values in control_tv.items()}}
    result['z_distribution']['tv_plugin_bootstrap95'] = result['bootstrap']['metrics']['z_tv']['percentile95']
    for control in result['controls']:
        control['distribution']['tv_plugin_bootstrap95'] = result['bootstrap']['control_total_variation'][control['control_name']]['percentile95']
    result['status'] = 'analyzed'
    result['target_role'] = 'Classically solved target used only for diagnostic comparison, not algorithm input or a speedup claim.'
    result['claim_scope'] = 'One frozen two-qubit preparation in one job/time block; two technical repeats are not independent calibration blocks or a solver ranking.'
    result['total_raw_shots'] = sum(int(np.sum(r['counts_vector'])) for r in rows)
    result['certified_fidelity_lower_bound'] = None
    return result


def analyze_confirmation_stage(store, raw_record, plan, target_state, *, preparation_audit,
                               bootstrap_samples=200, seed=20261004, readout_inverse=True):
    """Cache analysis with every raw record and calibration as dependencies.

    Requires an upstream physical-QPY prefix and ordered mapping audit, not
    merely matching declarations in metadata. This function never submits or
    collects jobs and cannot authorize additional QPU use.
    """
    if preparation_audit.get('status') != 'validated' or not preparation_audit.get('common_prefix_verified') or not preparation_audit.get('common_ordered_mapping_verified'):
        raise ValueError('A validated physical-circuit common-prefix/mapping audit is required before pooling.')
    root = Path(raw_record['directory']).resolve()
    supplied = raw_record['result'].get('rows')
    if not isinstance(supplied, list):
        raise ValueError('The raw hardware result must contain rows.')
    rows, provenance = [], []
    for index, original in enumerate(supplied):
        row = {**original.get('entry', {}), **{k:v for k,v in original.items() if k!='entry'}}
        relative = row.get('counts_file')
        if not isinstance(relative, str):
            raise ValueError('Every physical record requires its original counts_file.')
        path = (root / relative).resolve()
        if not path.is_relative_to(root):
            raise ValueError('Raw counts path escapes its immutable hardware artifact.')
        if file_hash(path)!=original.get('counts_sha256'):
            raise CacheIntegrityError('Raw count checksum changed before confirmation analysis.')
        counts = json.loads(path.read_text(encoding='utf-8'))
        row['counts_vector'] = _confirmation_counts_vector(counts)
        row['record_index'] = index
        rows.append(row)
        provenance.append({'record_index': index, 'counts_file': relative, 'sha256': file_hash(path),
            'metadata': {key: value for key, value in original.items() if key != 'counts'}})
    _confirmation_validate_rows(rows, require_provenance=True)
    target = np.asarray(target_state, dtype=complex)
    spec = {'schema': 1, 'plan_id': plan['plan_id'], 'target': _confirmation_matrix_json(target),
        'bootstrap_samples': bootstrap_samples, 'seed': seed, 'readout_inverse': readout_inverse,
        'preparation_audit': preparation_audit, 'inventory': provenance,
        'estimator': 'full-two-qubit-linear-tomography-with-raw-and-separate-inverse-readout-diagnostics'}
    calibration_ids = list(raw_record['manifest'].get('calibration_ids', []))
    if not calibration_ids:
        raise ValueError('The physical result must retain calibration provenance.')
    def compute(directory):
        atomic_json(directory / 'raw_counts_and_metadata.json', [dict(row, counts_vector=row['counts_vector'].tolist()) for row in rows])
        atomic_json(directory / 'preparation_audit.json', preparation_audit)
        result = analyze_confirmation_counts(rows, plan, target_state, bootstrap_samples=bootstrap_samples,
                                             seed=seed, readout_inverse=readout_inverse)
        result['raw_record'] = store.dependency(raw_record)
        result['raw_provenance'] = provenance
        result['calibration_ids'] = calibration_ids
        result['preparation_audit'] = preparation_audit
        result['hardware_calls_from_analysis'] = 0
        return result
    return store.run('confirmation_analysis', spec, compute,
        roles=('confirmation_analysis', 'metrics', 'measured_cost'),
        dependencies=(store.dependency(raw_record),), calibration_ids=calibration_ids)


def validate_confirmation_analysis(seed=20261004):
    """Seeded synthetic tests, never hardware results or algorithm experiments."""
    rng = np.random.default_rng(seed)
    # Asymmetric complex product state detects bit-order and Y-rotation errors.
    q0 = np.array([np.cos(.37), np.exp(.61j) * np.sin(.37)])
    q1 = np.array([np.cos(.83), np.exp(-.47j) * np.sin(.83)])
    target = np.kron(q1, q0)
    A = np.array([[2.1, -.3, .2, .1], [-.3, 1.4, -.15, .05], [.2, -.15, 2.7, -.4], [.1, .05, -.4, 3.4]])
    b = A @ target
    plan = measurement_plan(A, b, [.2, -.4], reps=0)
    # A fixed stationary correlated response; columns sum to one.
    response = np.array([[.95, .02, .03, .01], [.02, .93, .01, .04],
                         [.02, .01, .93, .03], [.01, .04, .03, .92]])
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    Sdg = np.diag([1, -1j])
    rotations = {'X': H, 'Y': H @ Sdg, 'Z': np.eye(2)}
    rows = []
    shots = 1 << 20
    for repeat in (0, 1):
        for setting in _CONFIRM_SETTINGS:
            rotation = np.kron(rotations[setting[0]], rotations[setting[1]])
            true_probabilities = abs(rotation @ target) ** 2
            rows.append({'kind': 'tomography', 'setting': setting, 'repeat': repeat,
                'counts_vector': rng.multinomial(shots, response @ true_probabilities)})
    for basis in range(4):
        rows.append({'kind': 'readout_calibration', 'prepared_basis': basis,
            'counts_vector': rng.multinomial(shots, response[:, basis])})
    for label, vector in [('rhs', b / np.linalg.norm(b)), ('classical_target', target)]:
        expected = abs(vector) ** 2
        rows.append({'kind': 'control', 'control_name': label, 'setting': 'ZZ',
            'expected_probabilities': expected.tolist(), 'counts_vector': rng.multinomial(shots, response @ expected)})
    for index, row in enumerate(rows):
        row['record_index'] = index
    result = analyze_confirmation_counts(rows, plan, target, bootstrap_samples=20, seed=seed)
    mitigated = result['mitigated_diagnostic']['tomography']
    matrix = mitigated['density_matrix_linear']
    reconstructed = np.asarray(matrix['real']) + 1j * np.asarray(matrix['imag'])
    error = float(np.linalg.norm(reconstructed - np.outer(target, target.conj().T)))
    assert error < .007, error
    assert abs(mitigated['linear_target_fidelity_unclipped'] - 1) < .004
    assert result['raw_tomography']['linear_target_fidelity_unclipped'] < mitigated['linear_target_fidelity_unclipped'] - .01
    assert abs(result['mitigated_diagnostic']['cost']['cost_unclipped']) < .006
    # Endian contrast is deliberately large; IX != XI for this complex product state.
    assert abs(mitigated['pauli_estimates']['IX'] - mitigated['pauli_estimates']['XI']) > .2
    # Incompatible marginals must expose a nonphysical raw density, not hide it.
    inconsistent = [dict(row, counts_vector=row['counts_vector'].copy()) for row in rows]
    for row in inconsistent:
        if row['kind'] == 'tomography':
            row['counts_vector'] = np.array([shots, 0, 0, 0])
    bad = analyze_confirmation_counts(inconsistent, plan, target, bootstrap_samples=0)
    assert bad['raw_tomography']['nonphysical_linear_estimate']
    assert bad['raw_tomography']['eigenvalues_linear'][0] < -.1
    # Singular calibration must be preserved and block the inverse.
    singular = [dict(row, counts_vector=row['counts_vector'].copy()) for row in rows]
    for row in singular:
        if row['kind'] == 'readout_calibration':
            row['counts_vector'] = np.array([shots, 0, 0, 0])
    failed = analyze_confirmation_counts(singular, plan, target, bootstrap_samples=0)
    assert failed['mitigated_diagnostic'] is None
    assert not failed['readout_calibration']['inverse_allowed']
    assert result['certified_fidelity_lower_bound'] is None
    return {'status': 'passed', 'fixture_scope': 'Seeded synthetic multinomial algebra/analysis checks; no hardware samples or quantum experiments.',
        'seed': seed, 'shots_per_synthetic_record': shots, 'hardware_calls': 0,
        'mitigated_density_frobenius_error': error,
        'complex_state_endian_contrast_IX_minus_XI': mitigated['pauli_estimates']['IX'] - mitigated['pauli_estimates']['XI'],
        'readout_response_condition': result['readout_calibration']['condition_number_2'],
        'nonphysical_fixture_min_eigenvalue': bad['raw_tomography']['eigenvalues_linear'][0],
        'singular_response_inverse_blocked': True,
        'representative_analysis': result}


In [ ]:
# CELL_ID: coherent_confirmation
def _native_output_probabilities(circuit,logical_to_physical):
    unmeasured=circuit.remove_final_measurements(inplace=False)
    # Explicit identities retain output wires that are idle in the native prefix.
    for physical in logical_to_physical:unmeasured.id(physical)
    compact,mapping=compress_physical_circuit(unmeasured,max_active_qubits=12)
    state=Statevector.from_instruction(compact)
    return state.probabilities(qargs=[mapping[q] for q in logical_to_physical])

def _conditional_exact_probabilities(probabilities,metadata):
    system=metadata['system_qubits'];post={int(q):int(v) for q,v in metadata['postselect_bits'].items()}
    distribution=np.zeros(1<<len(system))
    for integer,probability in enumerate(probabilities):
        if all(((integer>>q)&1)==v for q,v in post.items()):
            node=sum(((integer>>q)&1)<<j for j,q in enumerate(system))
            distribution[node]+=probability
    acceptance=float(distribution.sum())
    if acceptance<=0:raise ValueError('Local coherent circuit has no accepted branch.')
    return distribution/acceptance,acceptance

def prepare_coherent_confirmation(store,problem_record,native_screen,*,seed=20261005):
    if set(native_screen['result']['selected'])!={'hhl','qsvt'}:
        raise ValueError('Both methods must pass the frozen local/native gates before this paired bias pilot.')
    ref=native_screen['result']['compilation'];compiled=store.lookup(ref['stage'],ref['key'])
    calibration_id=compiled['result']['calibration_id']
    spec={'screen':store.dependency(native_screen),'problem':store.dependency(problem_record),
          'settings':['ZZ','XX','XZ'],'repeats':2,'seed':seed,'raw_shots':49152,
          'selection_rule':'native two-qubit gates divided by ideal acceptance after operator/residual guards'}
    def prepare(directory):
        circuits=load_qpy(compiled['directory']/'isa_circuits.qpy')
        if file_hash(compiled['directory']/'isa_circuits.qpy')!=compiled['result']['isa_circuits_sha256']:
            raise CacheIntegrityError('Screened native circuits changed.')
        A=np.asarray(problem_record['result']['A']);b=np.asarray(problem_record['result']['b'])
        target=normalized(np.linalg.solve(A,b));target_snapshot=store.calibration(calibration_id)['payload']['target']
        H=np.array([[1,1],[1,-1]])/np.sqrt(2);I=np.eye(2)
        staged=[];checks=[];prefixes={}
        for algorithm in ['hhl','qsvt']:
            selected=native_screen['result']['selected'][algorithm];index=selected['index']
            source=circuits[index];entry=compiled['result']['entries'][index]
            if entry['algorithm']!=algorithm or entry['candidate_id']!=selected['candidate_id']:
                raise ValueError('Native selection identity differs from its compiled entry.')
            metadata=entry['solver_metadata'];layout=entry['final_layout']
            validate_measurement_map(source,entry)
            if metadata['system_qubits']!=[0,1] or len(layout)!=source.num_clbits:
                raise ValueError('The paired bias pilot requires the archived two-system-qubit output map.')
            measurement_indices=[i for i,item in enumerate(source.data) if item.operation.name=='measure']
            if len(measurement_indices)!=source.num_clbits:raise ValueError('Each output bit must be measured.')
            boundary=measurement_indices[0]
            if any(item.operation.name not in ('measure','barrier') for item in source.data[boundary:]):
                raise ValueError('Only terminal measurements are supported.')
            prefix=source.copy(name=algorithm+'_frozen_prefix');del prefix.data[boundary:]
            signature=_semantic_circuit_signature(prefix);prefix_id=content_hash(signature);prefixes[algorithm]=signature
            baseline=_native_output_probabilities(prefix,layout)
            baseline_conditional,acceptance=_conditional_exact_probabilities(baseline,metadata)
            if abs(acceptance-entry['success_probability'])>1e-9:
                raise ValueError('Native synthesis changed ideal strict acceptance.')
            # The projected logical result saved during screening is an independent reference.
            candidate_ref=next(d for d in compiled['manifest']['dependencies'] if
                               store.lookup(d['stage'],d['key'])['result'].get('candidate_id')==entry['candidate_id'])
            candidate=store.lookup(candidate_ref['stage'],candidate_ref['key'])
            with np.load(candidate['directory']/'projected_operator.npz') as data:
                if not np.array_equal(data['A'],A) or not np.array_equal(data['b'],b):
                    raise ValueError('Screened candidate belongs to a different mathematical problem.')
                state=data['selected_state']
            if np.max(abs(baseline_conditional-abs(state)**2))>1e-9:
                raise ValueError('Native conditional output differs from the verified logical transformation.')
            for setting in ['ZZ','XX','XZ']:
                template=prefix.copy(name=algorithm+'_'+setting)
                template.barrier(*layout,label='common_preparation_boundary')
                ideal_rotation=np.kron(H if setting[0]=='X' else I,H if setting[1]=='X' else I)
                for q,axis in enumerate(reversed(setting)):
                    if axis=='X':
                        physical=layout[q]
                        for gate in ['rz','sx']:
                            if not _common_supported_instruction(target_snapshot,gate,[physical]):
                                raise ValueError('Archived Target does not support a basis suffix.')
                        template.rz(np.pi/2,physical);template.sx(physical);template.rz(np.pi/2,physical)
                for logical,physical in enumerate(layout):template.measure(physical,logical)
                check=template.copy();del check.data[boundary:]
                if _semantic_circuit_signature(check)!=signature:raise ValueError('Basis suffix changed the preparation prefix.')
                observed,observed_acceptance=_conditional_exact_probabilities(_native_output_probabilities(template,layout),metadata)
                expected=abs(ideal_rotation@state)**2
                error=float(np.max(abs(observed-expected)))
                if error>1e-9 or abs(observed_acceptance-acceptance)>1e-9:
                    raise ValueError('Coherent basis change failed the independent logical-state comparison.')
                expected_truth=(abs(ideal_rotation@target)**2).tolist()
                resources=circuit_resources(template)
                if resources['two_qubit_gates']>300:raise ValueError('Native basis circuit exceeds 300 two-qubit gates.')
                for repeat in range(2):
                    value={**entry,'kind':'coherent_phase_diagnostic','setting':setting,'repeat':repeat,
                        'physical_preparation_sha256':prefix_id,'preparation_layout':layout,
                        'prefix_instruction_count':boundary,'resources':resources,
                        'target_probabilities':expected_truth,'calibration_id':calibration_id,
                        'independent_calibration_block':False}
                    staged.append((template.copy(),value))
                checks.append({'algorithm':algorithm,'setting':setting,'probability_error':error,
                               'strict_ideal_acceptance':observed_acceptance,'shared_prefix_verified':True})
        rng=np.random.default_rng(seed);order=[]
        for repeat in range(2):
            indices=[i for i,(_,e) in enumerate(staged) if e['repeat']==repeat]
            order.extend(rng.permutation(indices).tolist())
        ordered=[staged[i][0] for i in order]
        entries=[{**staged[i][1],'schedule_position':position} for position,i in enumerate(order)]
        for circuit,entry in zip(ordered,entries):validate_measurement_map(circuit,entry)
        save_qpy(directory/'isa_circuits.qpy',ordered)
        recovered=load_qpy(directory/'isa_circuits.qpy')
        if len(recovered)!=12:raise AssertionError('Paired bias confirmation inventory must be 12.')
        for original,restored in zip(ordered,recovered):
            if _semantic_circuit_signature(original)!=_semantic_circuit_signature(restored):raise ValueError('QPY round trip changed a circuit.')
        atomic_json(directory/'prefix_signatures.json',prefixes);atomic_json(directory/'local_checks.json',checks)
        return {'status':'prepared','protocol':'coherent_bias','entries':entries,'calibration_id':calibration_id,
            'isa_circuits_sha256':file_hash(directory/'isa_circuits.qpy'),
            'scientific_identity':{'problem':store.dependency(problem_record),'native_screen':store.dependency(native_screen)},
            'local_checks':checks,'hardware_jobs_submitted':0,
            'declared_budget':{'jobs':1,'circuits':12,'shots_per_circuit':4096,'raw_shots':49152,'max_execution_time_seconds':60},
            'scope':'One paired coherent-circuit bias pilot; two technical repeats; three bases; no readout mitigation or complete state tomography.'}
    return store.run('coherent_confirmation_preparation',spec,prepare,
        roles=['coherent_confirmation','backends','noise_helpers','common_preparation','metrics'],
        dependencies=[store.dependency(problem_record),store.dependency(native_screen),store.dependency(compiled)],
        calibration_ids=[calibration_id])

def analyze_coherent_confirmation(store,raw_record,analysis_configuration):
    evidence=audit_saved_hardware_record(raw_record,store)
    if not evidence['result']['all_counts_and_maps_valid']:raise CacheIntegrityError('Raw paired confirmation invalid.')
    def analyze(directory):
        grouped={};per_repeat=[]
        for row in raw_record['result']['rows']:
            e=row['entry'];counts=read_json(raw_record['directory']/row['counts_file']);m=e['solver_metadata']
            metric=probability_metrics(counts,e['target_probabilities'],m['system_qubits'],m['postselect_bits'],**analysis_configuration)
            per_repeat.append({'algorithm':e['algorithm'],'candidate_id':e['candidate_id'],'setting':e['setting'],'repeat':e['repeat'],'metrics':metric})
            key=(e['algorithm'],e['setting'])
            if key not in grouped:grouped[key]={'entry':e,'counts':{}}
            elif grouped[key]['entry']['physical_preparation_sha256']!=e['physical_preparation_sha256']:
                raise ValueError('Cannot combine different preparations.')
            for bits,value in counts.items():grouped[key]['counts'][bits]=grouped[key]['counts'].get(bits,0)+value
        rows=[]
        for (algorithm,setting),group in grouped.items():
            e=group['entry'];m=e['solver_metadata']
            metric=probability_metrics(group['counts'],e['target_probabilities'],m['system_qubits'],m['postselect_bits'],**analysis_configuration)
            rows.append({'algorithm':algorithm,'candidate_id':e['candidate_id'],'setting':setting,
                         'metrics':metric,'target_probabilities':e['target_probabilities']})
        return {'status':'analyzed','rows':rows,'technical_repeats':per_repeat,'certified_state_fidelity':None,
                'scope':'Three conditional basis distributions, one job, two technical repeats; raw flag errors and measurement error remain.'}
    return store.run('coherent_confirmation_analysis',analysis_configuration,analyze,
        roles=['coherent_confirmation','metrics'],dependencies=[store.dependency(raw_record),store.dependency(evidence)],
        calibration_ids=raw_record['manifest']['calibration_ids'])


In [ ]:
# CELL_ID: cache_validation
import tempfile

def validate_notebook_cache(notebook_path):
    checks=[]
    def expect_failure(fn,label):
        try:fn()
        except (CacheIntegrityError,ValueError,RuntimeError):checks.append(label);return
        raise AssertionError('Guard did not reject '+label)
    with tempfile.TemporaryDirectory(prefix='qls_cache_validation_') as temporary:
        folder=Path(temporary)
        test=StageStore(folder/'original',notebook_path,{'replication_id':'test'})
        ideal=test.register_calibration({'kind':'ideal','noise_mode':'ideal'})
        expect_failure(lambda:test.register_calibration({'kind':'hardware','properties':{}}),'empty_hardware_calibration_rejected')
        expect_failure(lambda:test.register_calibration({'kind':'fake','properties':{'qubits':[[]]}}),'empty_fake_calibration_rejected')
        assert test.key('fixture',{'a':1,'b':2})==test.key('fixture',{'b':2,'a':1})
        checks.append('canonical_dictionary_identity')
        calls=[]
        def calculate(directory):
            calls.append(1);atomic_json(directory/'raw_counts.json',{'00':11,'01':5})
            return {'counts':{'00':11,'01':5}}
        first=test.run('fixture',{'shots':16,'seed':3},calculate,roles=['cache'],calibration_ids=[ideal])
        second=test.run('fixture',{'shots':16,'seed':3},calculate,roles=['cache'],calibration_ids=[ideal])
        assert len(calls)==1 and second['cache_hit']
        checks.append('repeated_run_no_execution')
        assert first['key']!=test.key('fixture',{'shots':32,'seed':3},roles=['cache'],calibration_ids=[ideal])
        checks.append('shot_change_invalidates_samples')
        # One invocation export restores every checksum and cache hit in another folder.
        package=test.export_run([first])
        with zipfile.ZipFile(package) as archive:
            assert not any(n.endswith('.zip') for n in archive.namelist())
            assert 'Quantum_Diffusion_Master.ipynb' in archive.namelist()
        restored=restore_run_export(package,folder/'restored')
        loaded=restored.lookup(first['stage'],first['key'])
        assert loaded['result']==first['result']
        checks.append('portable_offline_restore_without_nested_archives')
        raw=first['directory']/'raw_counts.json';raw.write_text('{}',encoding='utf8')
        expect_failure(lambda:test.lookup(first['stage'],first['key']),'corrupt_counts_rejected')
        # Missing calibration is checked even on cache lookup.
        calibration_path=restored.root/'calibrations'/ideal/'snapshot.json'
        calibration_path.write_text('{"payload":{"kind":"ideal","noise_mode":"changed"}}',encoding='utf8')
        expect_failure(lambda:restored.lookup(first['stage'],first['key']),'altered_calibration_rejected')
        # Test literal-secret detection without placing any real credential in a file.
        candidate='api_'+'key = '+repr('synthetic-sentinel-not-a-real-credential')
        expect_failure(lambda:assert_source_has_no_credentials(candidate),'source_credential_guard')
        key=test.key('uncertain',{'id':1})
        interrupted=test.root/'cache'/'uncertain'/key/'attempts'/'previous'
        atomic_json(interrupted/'submission_intent.json',{'status':'unknown'})
        expect_failure(lambda:test.run('uncertain',{'id':1},calculate),'uncertain_submission_no_duplicate')
        assert len(calls)==1
    return {'status':'passed','checks':checks,'checks_passed':len(checks),'hardware_calls':0}


In [ ]:
# CELL_ID: cache_portability
"""Reuse completed stages across numerically equivalent environments.

Stage keys include the executing environment. A notebook copied to another
operating system therefore misses every historical stage, and stages that once
needed a device would then try to contact IBM again. This store keeps the
original key schema for every new computation, but a lookup that misses under
the current environment also tries each archived environment (from
invocations/*/environment.json) whose scientific packages are identical and
whose Python minor version matches. Only the operating-system string and the
Python patch level may differ. Every such reuse is logged as an event with both
environments; nothing is recomputed, relabeled or copied.
"""
_EQUIVALENCE_PACKAGES = ('numpy', 'scipy', 'qiskit', 'qiskit-aer', 'qiskit-ibm-runtime')

def environments_equivalent(first, second):
    try:
        same_minor = str(first['python']).split('.')[:2] == str(second['python']).split('.')[:2]
        packages = all(first['packages'].get(name) == second['packages'].get(name) is not None
                       for name in _EQUIVALENCE_PACKAGES)
        return bool(same_minor and packages and first['packages'] == second['packages'])
    except (KeyError, TypeError, AttributeError):
        return False

class PortableStageStore(StageStore):
    def __init__(self, root, notebook_path, configuration):
        super().__init__(root, notebook_path, configuration)
        candidates = {}
        for path in sorted((self.root/'invocations').glob('*/environment.json')):
            try:
                environment = read_json(path)
            except (OSError, ValueError):
                continue
            if isinstance(environment, dict) and environment != self.environment and environments_equivalent(environment, self.environment):
                candidates[content_hash(environment)] = environment
        self.equivalent_environments = [candidates[key] for key in sorted(candidates)]
        atomic_json(self.root/'invocations'/self.invocation_id/'environment_equivalence.json',
            {'current': self.environment, 'equivalent_archived_environments': self.equivalent_environments,
             'rule': 'identical scientific package versions and Python minor version; OS and Python patch may differ',
             'use': 'lookup of already completed stages only; new stages are keyed with the current environment'})

    def key_for_environment(self, environment, stage, spec, roles=(), dependencies=(), calibration_ids=()):
        _safe_component(stage, 'stage')
        assert_no_credentials(spec)
        return content_hash({'schema': 3, 'stage': stage, 'spec': spec,
            'code': code_fingerprint(roles), 'environment': environment,
            'dependencies': list(dependencies), 'calibration_ids': sorted(calibration_ids)})

    def run(self, stage, spec, function, roles=(), dependencies=(), calibration_ids=()):
        for identifier in calibration_ids:
            self.calibration(identifier)
        own = self.key(stage, spec, roles, dependencies, calibration_ids)
        if self._load(stage, own) is None:
            for environment in self.equivalent_environments:
                key = self.key_for_environment(environment, stage, spec, roles, dependencies, calibration_ids)
                hit = self._load(stage, key)
                if hit is not None:
                    self._event({'stage': stage, 'key': key, 'action': 'loaded_existing',
                        'original_completed_utc': hit['manifest']['completed_utc'],
                        'original_invocation_id': hit['manifest']['invocation_id'],
                        'attempt_id': hit['manifest']['attempt_id'],
                        'environment_equivalence': {'computed_in': {k: environment[k] for k in ('python', 'platform')},
                                                    'reused_in': {k: self.environment[k] for k in ('python', 'platform')}}})
                    return hit
        return super().run(stage, spec, function, roles, dependencies, calibration_ids)

## Journal confirmation study: definitions

The cells below define the frozen journal design, explicit held-out instances, calibration-derived predictions, common-preparation conditional tomography for every solver, and the guarded campaign. They change no earlier cell. `cache_portability` lets a copy of this notebook on another operating system reuse completed stages computed with identical scientific package versions; every such reuse is logged.

In [ ]:
# CELL_ID: journal_design
"""Frozen design of the journal confirmation study (declared 11 October 2026).

This cell is data, not an experiment. Its source hash is part of every journal
stage identity, so editing any entry below creates new stage identities instead
of silently changing archived results. The design was fixed before any hardware
observation of the held-out instances H1-H6. D0 is the existing development
instance; its earlier jobs remain development data and are never relabeled.

Design changes relative to the 4 October proposal, decided from local
(non-hardware) diagnostics only:
* input-state overlap F_in=|<b_hat|x_hat>|^2 is recorded for every instance.
  D0, H1, H2 and H3 have F_in between 0.92 and 0.98, so a circuit that merely
  prepares b already scores a high fidelity. Two inversion-sensitive instances
  (H4, H5; F_in about 0.57 and 0.67) and a 16-node VQLS scale probe (H6) were
  added, and the inversion gain G=(F-F_in)/(1-F_in) is a primary metric;
* conditional state tomography replaces three-basis TV for every solver;
* coherent candidates were extended to six phase qubits and QSVT degree 25
  because kappa near 5 is not reachable with the earlier candidate set;
* circuits above the 300 two-qubit-gate primary cap enter a separately labeled
  depth-probe arm (cap 1500); they can never be promoted to the primary arm.
"""
import itertools
_SQ2 = 1/np.sqrt(2)
JOURNAL_DESIGN = {
    'design_id': 'journal_confirmation_v1',
    'declared_utc': '2026-10-11T23:59:00Z',
    'instances': {
        'D0': {'role': 'development control; already measured on 14 Sep and 5 Oct 2026',
               'construction': 'graph_problem', 'graph_problem': {'n': 4, 'family': 'cycle', 'gamma': 8.0, 'seed': 17}},
        'H1': {'role': 'held-out, declared 4 Oct: weighted path with localized source', 'construction': 'explicit',
               'n': 4, 'edges': [[0, 1, 0.7], [1, 2, 1.1], [2, 3, 1.4]], 'gamma': 2.0, 'source': [1.0, 0.0, 0.0, 0.0]},
        'H2': {'role': 'held-out, declared 4 Oct: weighted cycle with signed source', 'construction': 'explicit',
               'n': 4, 'edges': [[0, 1, 0.65], [1, 2, 1.15], [2, 3, 0.85], [3, 0, 1.35]], 'gamma': 2.0,
               'source': [_SQ2, 0.0, -_SQ2, 0.0]},
        'H3': {'role': 'held-out size transfer, declared 4 Oct: weighted 8-cycle', 'construction': 'explicit',
               'n': 8, 'edges': [[i, (i+1) % 8, w] for i, w in enumerate([0.6, 1.2, 0.9, 1.4, 0.75, 1.05, 1.3, 0.8])],
               'gamma': 2.0, 'source': [1.0]+[0.0]*7},
        'H4': {'role': 'held-out inversion-sensitive, declared 11 Oct: star with hub source', 'construction': 'explicit',
               'n': 4, 'edges': [[0, 1, 1.0], [0, 2, 1.0], [0, 3, 1.0]], 'gamma': 1.0, 'source': [1.0, 0.0, 0.0, 0.0]},
        'H5': {'role': 'held-out inversion-sensitive, declared 11 Oct: weighted cycle, opposite-node source', 'construction': 'explicit',
               'n': 4, 'edges': [[0, 1, 0.65], [1, 2, 1.15], [2, 3, 0.85], [3, 0, 1.35]], 'gamma': 1.0,
               'source': [0.0, 0.0, 0.0, 1.0]},
        'H6': {'role': 'exploratory VQLS scale probe, declared 11 Oct: weighted 16-cycle', 'construction': 'explicit',
               'n': 16, 'edges': [[i, (i+1) % 16, 0.5+((7*i) % 11)/10] for i in range(16)], 'gamma': 1.0,
               'source': [1.0]+[0.0]*15},
    },
    'coherent': {
        'instances': ['D0', 'H1', 'H2', 'H3', 'H4', 'H5'],
        'candidates': {'hhl_qpe_bits': [3, 4, 5, 6], 'hhl_scalings': ['bin_spacing', 'bounded_support'],
                       'qsvt_degrees': [3, 5, 7, 9, 11, 13, 15, 17, 21, 25]},
        'screening': {'phase_tolerance': 0.02, 'phase_seed': 11, 'operator_relative_tolerance': 0.03,
                      'residual_tolerance': 0.03, 'seed_transpiler': 1729, 'selection_max_two_qubit_gates': 300},
        'selection': {'primary_cap_two_qubit_gates': 300, 'probe_cap_two_qubit_gates': 1500,
                      'objective': 'native two-qubit gates divided by ideal strict acceptance',
                      'tie_breakers': ['native two-qubit gates', 'candidate_id'],
                      'probe_rule': 'if no operator-passing candidate of a method fits the primary cap, '
                                    'the passing candidate with the smallest objective under the probe cap'},
        'historical_ablation': {'instance': 'D0', 'candidates': ['hhl_m5_bin_spacing', 'qsvt_d15'],
                                'purpose': 'same-calibration comparison of the September implementations with the selected ones'},
    },
    'vqls': {
        'two_qubit': {'instances': ['D0', 'H1', 'H2', 'H4', 'H5'], 'treatment': 'vqls_measured',
                      'parameters': {'reps': 2, 'max_evals': 60, 'seed': 11, 'shots': 2048, 'tol': 0.001},
                      'checkpoint': 'independently confirmed best measured point of seed 11 (D0 rule)'},
        'larger': {'instances': ['H3', 'H6'], 'treatment': 'vqls_statevector', 'seeds': [11, 22, 33],
                   'reps': {'3': 2, '4': 2}, 'max_evals': 1000,
                   'checkpoint': 'lowest exact VQLS objective among the seeds; the objective uses A and b only'},
        'matched_optimizer_study': {'instances': ['D0', 'H1', 'H2', 'H4', 'H5'], 'seeds': [11, 22, 33, 44, 55],
                                    'budgets': [15, 30, 60], 'reps': 2, 'shots_per_setting': 2048,
                                    'confirmation_shots_per_setting': 8192, 'tol': 1e-6,
                                    'treatments': ['exact_objective', 'counted_objective']},
    },
    'measurement': {
        'shots': 4096,
        'settings': {'2': 'full', '3': 'even_y', '4': 'even_y'},
        'controls': {'input_state': [2], 'solution_state': [2, 3, 4], 'setting_set': 'even_y',
                     'layout': 'the VQLS inference layout of the same instance'},
        'readout_calibration': {'factorized_patterns': ['all_zero', 'all_one'], 'correlated_two_qubit_system': True},
        'order': 'seeded permutation within each job; seed = 20261011 + block index',
        'repeats_per_block': 1,
    },
    'devices': {
        'primary': {'account': 'lab_zhang', 'backend': 'ibm_rensselaer', 'blocks': ['B1', 'B2', 'B3'],
                    'separation': 'at least one provider calibration update between consecutive blocks; '
                                  'otherwise the blocks are reported as time blocks'},
        'transfer': [{'account': 'lab_premium', 'backend': 'ibm_kingston', 'blocks': ['T1'], 'family': 'Heron r2'},
                     {'account': 'lab_premium', 'backend': 'ibm_boston', 'blocks': ['T1'], 'family': 'Heron r3'}],
        'planning_targets': {'ibm_rensselaer': 'latest archived job-start calibration with its compilation Target',
                             'ibm_kingston': 'FakeKingston packaged snapshot', 'ibm_boston': 'FakeBoston packaged snapshot'},
    },
    'jobs': {'groups': {'A': ['D0'], 'B': ['H1', 'H2'], 'C': ['H4', 'H5'], 'D': ['H3'], 'E': ['H6']},
             'mitigation_arm': {'blocks': ['B1'], 'instances': ['D0', 'H1', 'H2'], 'arms': ['primary', 'vqls'],
                                'options': {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4'},
                                            'twirling': {'enable_gates': True, 'enable_measure': True,
                                                         'num_randomizations': 32, 'shots_per_randomization': 128}}},
             'raw_options': {'dynamical_decoupling': {'enable': False},
                             'twirling': {'enable_gates': False, 'enable_measure': False}},
             'execution': {'init_qubits': True, 'meas_type': 'classified'}},
    'ceilings': {'max_circuits_per_job': 150, 'max_estimated_qpu_seconds_per_job': 300,
                 'max_estimated_qpu_seconds_per_block': 900, 'max_total_estimated_qpu_seconds': 3600,
                 'max_premium_estimated_qpu_seconds': 1200, 'max_two_qubit_gates_any_circuit': 1500,
                 'execution_cap_factor': 1.5, 'execution_cap_floor_seconds': 30},
    'analysis': {
        'primary_metric': 'raw conditional state fidelity: linear estimator with the exact normalized solution, unclipped',
        'secondary_metrics': ['factorized inverse-readout fidelity', 'inversion gain G=(F-F_in)/(1-F_in)',
                              'hardware-normalized gain using the input/solution controls', 'strict acceptance (Wilson)',
                              'Z-basis conditional TV', 'p(node 0) and p(node 2) with Hoeffding radius'],
        'uncertainty': 'multinomial bootstrap of every count record within a job; block-level mean and spread across B1-B3',
        'prediction_models': ['logical_ideal', 'depolarizing_readout', 'thermal_idle_readout', 'esp_global_depolarizing'],
        'prediction_endpoints': ['|F_hat-F_pred| per cell', 'Spearman rank correlation across cells',
                                 'fraction of cells whose bootstrap 95% interval contains F_pred'],
        'claims_not_tested': ['quantum advantage', 'scalable oracle access', 'on-device training', 'security efficacy'],
    },
}

def journal_settings(n, kind):
    """Tensor-product Pauli settings, written with the highest system qubit first."""
    every = [''.join(p) for p in itertools.product('XYZ', repeat=n)]
    if kind == 'full':
        return every
    if kind == 'even_y':
        # Real target states have <P>=0 for every Pauli with an odd number of Y.
        # Each even-Y Pauli is measured by the setting that replaces I with Z.
        return [s for s in every if s.count('Y') % 2 == 0]
    raise ValueError('Unknown setting family.')

def journal_design_summary():
    counts = {str(n): len(journal_settings(n, JOURNAL_DESIGN['measurement']['settings'][str(n)])) for n in (2, 3, 4)}
    return {'design_id': JOURNAL_DESIGN['design_id'], 'design_sha256': content_hash(JOURNAL_DESIGN),
            'settings_per_system_width': counts, 'instances': sorted(JOURNAL_DESIGN['instances'])}

In [ ]:
# CELL_ID: journal_instances
"""Explicit held-out instances and their classical reference data."""
import time

def journal_explicit_problem(identifier, spec):
    n = int(spec['n'])
    if n < 2 or n & (n-1):
        raise ValueError('Journal instances use power-of-two node counts; no padding is applied.')
    W = np.zeros((n, n))
    for u, v, w in spec['edges']:
        u, v, w = int(u), int(v), float(w)
        if u == v or not (0 <= u < n and 0 <= v < n) or not np.isfinite(w) or w <= 0 or W[u, v]:
            raise ValueError('Edges must be distinct, positive and between distinct nodes.')
        W[u, v] = W[v, u] = w
    L = np.diag(W.sum(axis=1)) - W
    if np.sum(np.linalg.eigvalsh(L) < 1e-10) != 1:
        raise ValueError('Each declared graph must be connected.')
    gamma = float(spec['gamma'])
    A = L + gamma*np.eye(n)
    b = np.asarray(spec['source'], dtype=float)
    if b.shape != (n,) or not np.isfinite(b).all() or np.linalg.norm(b) == 0:
        raise ValueError('The source must be finite, nonzero and of length n.')
    eig = np.linalg.eigvalsh(A)
    metadata = {'identifier': identifier, 'role': spec['role'], 'family': 'explicit weighted graph', 'n': n,
        'edges': [[int(u), int(v), float(w)] for u, v, w in spec['edges']], 'gamma': gamma,
        'lambda_min': float(eig[0]), 'lambda_max': float(eig[-1]), 'condition_number_2': float(eig[-1]/eig[0]),
        'eigenvalues': eig.tolist(), 'rhs': 'declared explicitly in JOURNAL_DESIGN',
        'matrix_definition': 'A=diag(W1)-W+gamma I'}
    return Problem(f'{identifier}_n{n}_gamma{gamma:g}', A, b, metadata)

def journal_input_diagnostics(A, b):
    """How much of a solution's structure is already present in the source."""
    A = np.asarray(A, dtype=float); b = np.asarray(b, dtype=float)
    x = np.linalg.solve(A, b)
    xhat, bhat = x/np.linalg.norm(x), b/np.linalg.norm(b)
    overlap = float(abs(bhat @ xhat)**2)
    scaled = reconstruct_from_state(A, b, bhat)
    eig = np.linalg.eigvalsh(A); kappa = eig[-1]/eig[0]
    return {'input_state_fidelity_F_in': overlap,
        'input_state_residual_reference_free_scale': float(np.linalg.norm(A@scaled-b)/np.linalg.norm(b)),
        'input_target_z_tv': float(.5*np.abs(bhat**2-xhat**2).sum()),
        'kantorovich_minimum_F_in_at_this_kappa': float(4*kappa/(1+kappa)**2),
        'target_state_real': xhat.tolist(), 'source_state_real': bhat.tolist(),
        'target_probabilities': (xhat**2).tolist(),
        'interpretation': 'A circuit that only prepares b reaches fidelity F_in; inversion gain is measured relative to it.'}

def journal_classical_timing(A, b, repetitions=201):
    """Median local solve times; a context number, not a speedup comparison."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.linalg import cg, spsolve
    sparse = csr_matrix(A)
    def timed(function):
        samples = []
        for _ in range(repetitions):
            started = time.perf_counter(); value = function(); samples.append(time.perf_counter()-started)
        return value, samples
    dense_x, dense_t = timed(lambda: np.linalg.solve(A, b))
    sparse_x, sparse_t = timed(lambda: spsolve(sparse, b))
    cg_x, cg_t = timed(lambda: cg(sparse, b, rtol=1e-11, atol=0.0, maxiter=10*len(b))[0])
    residual = lambda x: float(np.linalg.norm(A@x-b)/np.linalg.norm(b))
    summary = lambda s: {'median_seconds': float(np.median(s)), 'p10_seconds': float(np.quantile(s, .1)),
                         'p90_seconds': float(np.quantile(s, .9)), 'repetitions': len(s)}
    return {'dense_lapack': dict(summary(dense_t), residual=residual(dense_x)),
            'sparse_direct': dict(summary(sparse_t), residual=residual(sparse_x)),
            'conjugate_gradient': dict(summary(cg_t), residual=residual(cg_x), rtol=1e-11),
            'scope': 'single-thread local CPU timings of this kernel after conversion; excludes Python import and data loading'}

def make_journal_problem_stage(store, identifier, spec, development_problem=None):
    if spec['construction'] == 'graph_problem':
        if development_problem is None:
            raise ValueError('D0 reuses the existing development problem record.')
        return development_problem
    def compute(directory):
        problem = journal_explicit_problem(identifier, spec)
        baseline = classical_baselines(problem)[1]
        diagnostics = journal_input_diagnostics(problem.A, problem.b)
        timing = journal_classical_timing(problem.A, problem.b)
        np.savez_compressed(directory/'problem_arrays.npz', A=problem.A, b=problem.b)
        return {'name': problem.name, 'identifier': identifier, 'instance_id': problem.instance_id,
                'A': problem.A, 'b': problem.b, 'metadata': problem.metadata, 'classical_baselines': baseline,
                'classical_timing': timing, 'input_diagnostics': diagnostics,
                'declaration': JOURNAL_DESIGN['instances'][identifier]['role']}
    return store.run('journal_problem', {'identifier': identifier, 'spec': spec}, compute,
                     roles=['journal_instances', 'problems', 'metrics'])

In [ ]:
# CELL_ID: journal_noise
"""Calibration-derived predictions of increasing completeness, computed without shots.

Every model uses only archived public calibration data for the qubits a circuit
actually touches. Missing values stop the prediction; nothing is filled in.
  logical_ideal          exact native circuit, no noise;
  depolarizing_readout   reported gate errors as depolarizing channels and the
                         reported readout assignment probabilities (the model of
                         the earlier pilot analyses);
  thermal_idle_readout   T1/T2 relaxation during each gate, depolarizing noise
                         topping each gate up to its reported error, relaxation
                         during ALAP-scheduled idle periods, and readout;
  esp_global_depolarizing  one parameter-free global depolarizing weight, the
                         product of per-gate depolarizing survival factors,
                         plus readout. It needs no circuit simulation.
Readout is applied analytically to exact pre-measurement probabilities, so the
predictions contain no sampling noise. Crosstalk, leakage, drift, coherent
over-rotation and non-Markovian effects are excluded from all models.
"""
from qiskit.circuit import Delay
from qiskit.quantum_info import average_gate_fidelity
from qiskit.transpiler import InstructionDurations, PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay
from qiskit_aer.noise import depolarizing_error, thermal_relaxation_error
from qiskit_aer.noise.passes import RelaxationNoisePass

_JN_UNITS = {'s': 1.0, 'ms': 1e-3, 'us': 1e-6, 'µs': 1e-6, 'ns': 1e-9}
_JN_GATES = ('cz', 'sx', 'x', 'id', 'ecr', 'cx')
JOURNAL_NOISE_MODELS = ('logical_ideal', 'depolarizing_readout', 'thermal_idle_readout', 'esp_global_depolarizing')

def _jn_field(fields, name):
    for field in fields:
        if field.get('name') == name and field.get('value') is not None:
            value = float(field['value'])
            return value*_JN_UNITS.get(field.get('unit', ''), 1.0) if name in ('T1', 'T2') else value
    return None

def journal_calibration_tables(payload, physical_to_compact):
    """Per-qubit relaxation/readout and per-gate error/duration for active qubits."""
    properties = payload['properties']; target = payload.get('target') or {}
    qubits = {}
    for physical, compact in physical_to_compact.items():
        fields = properties['qubits'][physical]
        p10, p01 = _jn_field(fields, 'prob_meas1_prep0'), _jn_field(fields, 'prob_meas0_prep1')
        symmetric = _jn_field(fields, 'readout_error')
        source = 'asymmetric assignment probabilities'
        if p10 is None or p01 is None:
            p10 = p01 = symmetric; source = 'symmetric readout_error'
        qubits[compact] = {'physical': physical, 'T1': _jn_field(fields, 'T1'), 'T2': _jn_field(fields, 'T2'),
                           'p_meas1_prep0': p10, 'p_meas0_prep1': p01, 'readout_source': source}
    gates = {}
    for row in target.get('instructions', []):
        locations = row.get('qubits')
        if locations is None or row['name'] not in (*_JN_GATES, 'measure') or not all(q in physical_to_compact for q in locations):
            continue
        gates[(row['name'], tuple(physical_to_compact[q] for q in locations))] = {
            'error': row.get('error'), 'duration': row.get('duration_seconds'), 'error_source': 'Target'}
    for gate in properties.get('gates', []):
        if gate['gate'] not in _JN_GATES or not all(q in physical_to_compact for q in gate['qubits']):
            continue
        entry = gates.setdefault((gate['gate'], tuple(physical_to_compact[q] for q in gate['qubits'])),
                                 {'error': None, 'duration': None})
        fields = {p['name']: p for p in gate.get('parameters', [])}
        if fields.get('gate_error', {}).get('value') is not None:
            entry['error'] = float(fields['gate_error']['value']); entry['error_source'] = 'BackendProperties'
        if entry['duration'] is None and fields.get('gate_length', {}).get('value') is not None:
            entry['duration'] = float(fields['gate_length']['value'])*_JN_UNITS.get(fields['gate_length'].get('unit', 'ns'), 1e-9)
    return qubits, gates

def _jn_used_gates(compact):
    used = set()
    for item in compact.data:
        name = item.operation.name
        if name in ('barrier', 'measure', 'rz', 'delay', 'save_density_matrix'):
            continue
        if name not in _JN_GATES:
            raise ValueError(f'Unsupported native operation {name} in a prediction circuit.')
        used.add((name, tuple(compact.find_bit(q).index for q in item.qubits)))
    return used

def journal_depolarizing_parameter(error, dimension):
    """Aer convention: E(rho)=(1-l)rho+l I/d has average infidelity l(d-1)/d."""
    return min(dimension*float(error)/(dimension-1), dimension**2/(dimension**2-1))

def journal_gate_noise_model(qubits, gates, used, *, thermal):
    model = NoiseModel(basis_gates=['cz', 'sx', 'x', 'rz', 'id', 'measure', 'delay'])
    report = []
    for name, location in sorted(used):
        item = gates.get((name, location))
        if item is None or item['error'] is None:
            raise ValueError(f'Calibration lacks an error for {name}{location}; prediction withheld.')
        dimension = 2**len(location)
        if not thermal:
            channel = depolarizing_error(journal_depolarizing_parameter(item['error'], dimension), len(location))
            report.append({'gate': name, 'qubits': list(location), 'reported_error': item['error'], 'model': 'depolarizing'})
        else:
            if item['duration'] is None or any(qubits[q]['T1'] is None or qubits[q]['T2'] is None for q in location):
                raise ValueError(f'Missing duration or T1/T2 for {name}{location}; thermal prediction withheld.')
            relaxation = None
            for q in location:
                t1 = qubits[q]['T1']; t2 = min(qubits[q]['T2'], 2*t1)
                single = thermal_relaxation_error(t1, t2, item['duration'])
                relaxation = single if relaxation is None else relaxation.expand(single)
            relax_fidelity = float(average_gate_fidelity(relaxation))
            relax_infidelity = 1-relax_fidelity
            if item['error'] > relax_infidelity:
                parameter = dimension*(item['error']-relax_infidelity)/(dimension*relax_fidelity-1)
                parameter = min(parameter, dimension**2/(dimension**2-1))
                channel = depolarizing_error(parameter, len(location)).compose(relaxation)
            else:
                parameter = 0.0; channel = relaxation
            report.append({'gate': name, 'qubits': list(location), 'reported_error': item['error'],
                           'duration_seconds': item['duration'], 'relaxation_infidelity': relax_infidelity,
                           'added_depolarizing_parameter': parameter, 'model': 'thermal relaxation plus depolarizing'})
        model.add_quantum_error(channel, name, list(location))
    return model, report

def journal_schedule_idle(compact, gates, qubits, dt=None):
    """ALAP schedule with explicit delays, then relaxation on every delay."""
    dt = float(dt) if dt else 1e-12
    durations = []
    for (name, location), item in gates.items():
        if item['duration'] is not None:
            durations.append((name, list(location), float(item['duration']), 's'))
    for q in range(compact.num_qubits):
        durations.append(('rz', [q], 0.0, 's'))
    table = InstructionDurations(durations, dt=dt)
    scheduled = PassManager([ALAPScheduleAnalysis(table), PadDelay(durations=table)]).run(compact)
    t1s = [qubits[q]['T1'] for q in range(compact.num_qubits)]
    t2s = [min(qubits[q]['T2'], 2*qubits[q]['T1']) for q in range(compact.num_qubits)]
    noisy = PassManager([RelaxationNoisePass(t1s, t2s, dt=dt, op_types=[Delay])]).run(scheduled)
    idle = sum(float(i.operation.duration)*(dt if i.operation.unit == 'dt' else 1.0)
               for i in scheduled.data if i.operation.name == 'delay')
    return noisy, {'total_idle_qubit_seconds': idle, 'dt_seconds': dt,
                   'schedule': 'ALAP from archived durations; thermal relaxation on every inserted delay'}

def _jn_measured_map(circuit):
    """classical bit -> physical qubit, requiring exactly one terminal measurement each."""
    measured = {}
    for item in circuit.data:
        if item.operation.name == 'measure':
            measured[circuit.find_bit(item.clbits[0]).index] = circuit.find_bit(item.qubits[0]).index
    if set(measured) != set(range(circuit.num_clbits)):
        raise ValueError('Every classical bit must be measured exactly once.')
    return measured

def journal_prefix(circuit):
    """Strip terminal measurements; reject mid-circuit measurement."""
    positions = [i for i, item in enumerate(circuit.data) if item.operation.name == 'measure']
    if not positions:
        raise ValueError('The prepared circuit has no measurement.')
    if any(item.operation.name not in ('measure', 'barrier') for item in circuit.data[positions[0]:]):
        raise ValueError('Only terminal measurements are supported.')
    prefix = circuit.copy(); del prefix.data[positions[0]:]
    return prefix, _jn_measured_map(circuit)

def journal_compact(prefix, measured, max_active=12):
    active = sorted({prefix.find_bit(q).index for item in prefix.data if item.operation.name != 'barrier'
                     for q in item.qubits} | set(measured.values()))
    if len(active) > max_active:
        raise ValueError(f'{len(active)} active qubits exceed the density-matrix cap {max_active}.')
    mapping = {physical: i for i, physical in enumerate(active)}
    compact = QuantumCircuit(len(active))
    compact.global_phase = prefix.global_phase
    for item in prefix.data:
        if item.operation.name == 'barrier':
            continue
        compact.append(item.operation, [mapping[prefix.find_bit(q).index] for q in item.qubits])
    return compact, mapping

def journal_readout_matrices(qubits, measured, mapping):
    """Assignment matrix per classical bit: M[o,i]=P(measured o | prepared i)."""
    matrices = []
    for bit in range(len(measured)):
        info = qubits[mapping[measured[bit]]]
        p10, p01 = info['p_meas1_prep0'], info['p_meas0_prep1']
        if p10 is None or p01 is None:
            raise ValueError('Readout probabilities are missing; raw-readout prediction withheld.')
        matrices.append(np.array([[1-p10, p01], [p10, 1-p01]]))
    return matrices

def journal_apply_bit_matrices(probabilities, matrices):
    """Apply one 2x2 matrix per bit; integer index bit j is classical bit j."""
    k = len(matrices)
    tensor = np.asarray(probabilities, dtype=float).reshape((2,)*k)
    for bit, matrix in enumerate(matrices):
        axis = k-1-bit
        tensor = np.moveaxis(np.tensordot(matrix, tensor, axes=([1], [axis])), 0, axis)
    return tensor.reshape(-1)

def journal_measured_density(circuit, payload, model, *, max_active=12):
    """Exact density matrix of the measured register, classical bit j = index bit j.

    Returns (rho, readout matrices or None, details). Readout matrices are the
    archived assignment probabilities of the measured physical qubits.
    """
    from qiskit_aer import AerSimulator
    if model not in JOURNAL_NOISE_MODELS:
        raise ValueError('Unknown prediction model.')
    prefix, measured = journal_prefix(circuit)
    compact, mapping = journal_compact(prefix, measured, max_active)
    order = [mapping[measured[bit]] for bit in range(len(measured))]
    details = {'model': model, 'active_physical_qubits': sorted(mapping, key=mapping.get),
               'classical_bit_to_physical': [measured[bit] for bit in range(len(measured))]}
    qubits, gates = journal_calibration_tables(payload, mapping) if model != 'logical_ideal' else ({}, {})
    noise = None; simulated = compact
    if model in ('depolarizing_readout', 'thermal_idle_readout'):
        noise, details['gate_channels'] = journal_gate_noise_model(qubits, gates, _jn_used_gates(compact),
                                                                   thermal=(model == 'thermal_idle_readout'))
        if model == 'thermal_idle_readout':
            simulated, details['idle'] = journal_schedule_idle(compact, gates, qubits, (payload.get('target') or {}).get('dt'))
    simulated = simulated.copy()
    simulated.save_density_matrix(qubits=order, label='rho')
    result = AerSimulator(method='density_matrix', noise_model=noise, max_parallel_threads=1).run(
        simulated, shots=1, seed_simulator=1).result()
    if not result.success:
        raise RuntimeError('Density-matrix prediction failed.')
    rho = np.asarray(result.data(0)['rho'])
    if model == 'esp_global_depolarizing':
        survival = 1.0
        for item in compact.data:
            name = item.operation.name
            if name in ('rz', 'delay', 'barrier'):
                continue
            location = tuple(compact.find_bit(q).index for q in item.qubits)
            entry = gates.get((name, location))
            if entry is None or entry['error'] is None:
                raise ValueError(f'Calibration lacks an error for {name}{location}; ESP withheld.')
            survival *= 1-journal_depolarizing_parameter(entry['error'], 2**len(location))
        dimension = rho.shape[0]
        rho = survival*rho + (1-survival)*np.eye(dimension)/dimension
        details['esp_survival_weight'] = survival
    readout = None if model == 'logical_ideal' else journal_readout_matrices(qubits, measured, mapping)
    return rho, readout, details

In [ ]:
# CELL_ID: journal_measurement
"""Common-preparation conditional tomography for every solver, controls and readout.

A cell is one prepared native circuit (solver, control or VQLS) on fixed physical
qubits. Its preparation prefix is never retranspiled: each Pauli setting appends
RZ/SX rotations to the system output qubits and measures every original
classical bit, including success flags. Setting strings list the highest system
qubit first; classical bit j of the system register is logical system qubit j.
"""
_JM_H = np.array([[1, 1], [1, -1]], dtype=complex)/np.sqrt(2)
_JM_ROTATION = {'X': _JM_H, 'Y': _JM_H@np.diag([1, -1j]), 'Z': np.eye(2, dtype=complex)}
_JM_PAULI = {'I': np.eye(2, dtype=complex), 'X': np.array([[0, 1], [1, 0]], dtype=complex),
             'Y': np.array([[0, -1j], [1j, 0]]), 'Z': np.diag([1.0+0j, -1.0])}

def journal_kron(labels, table):
    matrix = np.ones((1, 1), dtype=complex)
    for label in labels:
        matrix = np.kron(matrix, table[label])
    return matrix

def journal_setting_rotation(setting):
    return journal_kron(setting, _JM_ROTATION)

def journal_pauli_labels(settings):
    """Every Pauli measurable from the settings, including identity."""
    labels = set()
    for setting in settings:
        for mask in itertools.product([0, 1], repeat=len(setting)):
            labels.add(''.join(axis if keep else 'I' for axis, keep in zip(setting, mask)))
    return sorted(labels)

def journal_statevector_outputs(circuit, max_active=18):
    """Exact probabilities over the classical register of a measured native circuit."""
    prefix, measured = journal_prefix(circuit)
    compact, mapping = journal_compact(prefix, measured, max_active)
    for physical in measured.values():
        compact.id(mapping[physical])
    state = Statevector.from_instruction(compact)
    return state.probabilities(qargs=[mapping[measured[bit]] for bit in range(len(measured))])

def journal_condition(probabilities, system_bits, postselect_bits):
    """Joint outcome vector -> (system vector indexed by system bit order, accepted mass)."""
    k = int(round(np.log2(len(probabilities))))
    index = np.arange(1 << k)
    accept = np.ones(1 << k, dtype=bool)
    for bit, value in postselect_bits.items():
        accept &= ((index >> int(bit)) & 1) == int(value)
    system = np.zeros(1 << len(system_bits))
    node = np.zeros(1 << k, dtype=np.int64)
    for j, bit in enumerate(system_bits):
        node |= ((index >> int(bit)) & 1) << j
    np.add.at(system, node[accept], np.asarray(probabilities, dtype=float)[accept])
    return system, float(system.sum())

def journal_append_setting(prefix, setting, layout, system_bits, label):
    circuit = prefix.copy(name=label)
    circuit.barrier(*sorted(set(layout)), label='common_preparation_boundary')
    n = len(system_bits)
    for j, bit in enumerate(system_bits):
        axis = setting[n-1-j]; physical = layout[int(bit)]
        if axis == 'Y':
            circuit.rz(-np.pi/2, physical)
        if axis in ('X', 'Y'):
            circuit.rz(np.pi/2, physical); circuit.sx(physical); circuit.rz(np.pi/2, physical)
    for bit, physical in enumerate(layout):
        circuit.measure(physical, bit)
    return circuit

def journal_cell_circuits(native, entry, settings, ideal_conditional_state, label, target_snapshot=None,
                          tolerance=1e-9):
    """Append every setting to one unmodified native prefix and verify each against exact algebra."""
    meta = entry['solver_metadata']
    system_bits = [int(b) for b in meta['system_qubits']]
    post = {int(b): int(v) for b, v in meta['postselect_bits'].items()}
    layout = list(entry['final_layout'])[:native.num_clbits]
    validate_measurement_map(native, entry)
    prefix, measured = journal_prefix(native)
    if [measured[b] for b in range(native.num_clbits)] != layout:
        raise ValueError('Native measurements differ from the archived final layout.')
    signature = _semantic_circuit_signature(prefix)
    prefix_id = content_hash(signature)
    psi = normalized(ideal_conditional_state)
    baseline, acceptance = journal_condition(journal_statevector_outputs(native), system_bits, post)
    if acceptance <= 0 or np.max(abs(baseline/acceptance-abs(psi)**2)) > 1e-8:
        raise ValueError(f'{label}: native output differs from the verified logical state.')
    circuits, rows = [], []
    for setting in settings:
        circuit = journal_append_setting(prefix, setting, layout, system_bits, f'{label}:{setting}')
        check = circuit.copy(); del check.data[len(prefix.data):]
        if _semantic_circuit_signature(check) != signature:
            raise ValueError('A basis suffix changed the preparation prefix.')
        if target_snapshot is not None:
            for item in circuit.data[len(prefix.data):]:
                if item.operation.name in ('barrier', 'measure'):
                    continue
                if not _common_supported_instruction(target_snapshot, item.operation.name,
                                                     [circuit.find_bit(q).index for q in item.qubits]):
                    raise ValueError('Basis suffix instruction absent from the Target.')
        observed, observed_acceptance = journal_condition(journal_statevector_outputs(circuit), system_bits, post)
        expected = abs(journal_setting_rotation(setting)@psi)**2
        error = float(np.max(abs(observed/observed_acceptance-expected)))
        if error > tolerance or abs(observed_acceptance-acceptance) > 1e-8:
            raise ValueError(f'{label}:{setting} failed exact basis validation ({error}).')
        circuits.append(circuit)
        rows.append({**{k: v for k, v in entry.items() if k not in ('resources',)},
                     'kind': 'tomography', 'cell_label': label, 'setting': setting,
                     'physical_preparation_sha256': prefix_id, 'preparation_layout': layout,
                     'prefix_instruction_count': len(prefix.data), 'resources': circuit_resources(circuit),
                     'exact_setting_probability_error': error, 'ideal_strict_acceptance': acceptance})
    return circuits, rows, {'prefix_sha256': prefix_id, 'ideal_strict_acceptance': acceptance,
                            'layout': layout, 'system_bits': system_bits, 'postselect_bits': post}

def journal_readout_circuits(width, layout, label, correlated=False):
    """All-zero and all-one patterns on the exact measured qubits, in classical-bit order.

    For two measured bits the four computational patterns are used; their first and
    last columns are the all-zero and all-one patterns of the factorized model.
    """
    from qiskit import ClassicalRegister, QuantumRegister
    patterns = [('all_zero', [0]*len(layout)), ('all_one', [1]*len(layout))]
    if correlated:
        if len(layout) != 2:
            raise ValueError('Correlated calibration is declared only for two measured bits.')
        patterns = [(f'basis_{i}', [(i >> j) & 1 for j in range(2)]) for i in range(4)]
    circuits, rows = [], []
    for name, bits in patterns:
        circuit = QuantumCircuit(QuantumRegister(width, 'q'), ClassicalRegister(len(layout), 'meas'), name=f'{label}:{name}')
        for bit, physical in enumerate(layout):
            if bits[bit]:
                circuit.sx(physical); circuit.sx(physical)
        for bit, physical in enumerate(layout):
            circuit.measure(physical, bit)
        circuits.append(circuit)
        rows.append({'kind': 'readout_calibration', 'pattern': name, 'prepared_bits': bits,
                     'cell_label': label, 'final_layout': list(layout), 'preparation_layout': list(layout),
                     'solver_metadata': {'system_qubits': list(range(len(layout))), 'postselect_bits': {}},
                     'algorithm': 'readout_calibration', 'resources': circuit_resources(circuit)})
    return circuits, rows

# ---------------- estimation ----------------

def journal_target_paulis(target, labels):
    x = normalized(target)
    return {label: float(np.vdot(x, journal_kron(label, _JM_PAULI)@x).real) for label in labels}

def journal_factorized_inverse(zero_vector, one_vector):
    """Per-bit inverse assignment matrices from all-zero/all-one calibration outcomes."""
    k = int(round(np.log2(len(zero_vector))))
    index = np.arange(1 << k)
    zero = np.asarray(zero_vector, float)/np.sum(zero_vector); one = np.asarray(one_vector, float)/np.sum(one_vector)
    inverses, rows = [], []
    for bit in range(k):
        e0 = float(zero[((index >> bit) & 1) == 1].sum()); e1 = float(one[((index >> bit) & 1) == 0].sum())
        matrix = np.array([[1-e0, e1], [e0, 1-e1]])
        if abs(np.linalg.det(matrix)) < 0.2:
            raise ValueError('Assignment matrix is too close to singular for inversion.')
        inverses.append(np.linalg.inv(matrix)); rows.append({'bit': bit, 'p_meas1_prep0': e0, 'p_meas0_prep1': e1})
    return inverses, rows

_JOURNAL_SIGN_CACHE = {}

def _journal_sign_table(n, labels):
    key = (n, tuple(labels))
    if key not in _JOURNAL_SIGN_CACHE:
        integers = np.arange(1 << n)
        table = {}
        for label in labels:
            mask = sum(1 << j for j in range(n) if label[n-1-j] != 'I')
            table[label] = 1-2*np.array([bin(int(v) & mask).count('1') % 2 for v in integers])
        _JOURNAL_SIGN_CACHE[key] = table
    return _JOURNAL_SIGN_CACHE[key]

def journal_estimate(records, target, system_bits, postselect_bits, *, inverses=None, full=False):
    """Pooled conditional Pauli expectations and the linear fidelity estimate.

    records: [{'setting', 'vector'}] with counts or probabilities over the full
    classical register. With inverses, each normalized vector is first mapped to
    quasi-probabilities; negative values are retained.
    """
    n = len(system_bits)
    settings = [r['setting'] for r in records]
    labels = journal_pauli_labels(settings)
    chi = journal_target_paulis(target, labels)
    sums = {label: 0.0 for label in labels}; weights = {label: 0.0 for label in labels}
    total = accepted_total = 0.0; conditional = {}
    signs = _journal_sign_table(n, labels)
    for record in records:
        vector = np.asarray(record['vector'], dtype=float)
        shots = float(vector.sum())
        probabilities = vector/shots
        if inverses is not None:
            probabilities = journal_apply_bit_matrices(probabilities, inverses)
        system, acceptance = journal_condition(probabilities, system_bits, postselect_bits)
        total += shots; accepted_total += acceptance*shots
        if acceptance <= 0:
            continue
        q = system/acceptance
        conditional[record['setting']] = q
        weight = acceptance*shots
        for label in labels:
            if not all(p == 'I' or p == s for p, s in zip(label, record['setting'])):
                continue
            sums[label] += weight*float(signs[label]@q); weights[label] += weight
    expectations = {label: (sums[label]/weights[label] if weights[label] > 0 else None) for label in labels}
    expectations[('I'*n)] = 1.0
    missing = [label for label, value in expectations.items() if value is None]
    fidelity = None if missing else float(sum(chi[label]*expectations[label] for label in labels)/2**n)
    result = {'fidelity_linear': fidelity, 'acceptance': accepted_total/total if total else None,
              'accepted_shots_equivalent': accepted_total, 'pauli_expectations': expectations,
              'missing_paulis': missing, 'settings': settings}
    if full and not missing:
        rho = sum(expectations[label]*journal_kron(label, _JM_PAULI) for label in labels)/2**n
        rho = (rho+rho.conj().T)/2
        eigenvalues = np.linalg.eigvalsh(rho)
        result.update(density_matrix_real=rho.real.tolist(), density_matrix_imag=rho.imag.tolist(),
                      eigenvalues=eigenvalues.tolist(), purity=float(np.trace(rho@rho).real),
                      nonphysical=bool(eigenvalues[0] < -1e-10))
    if 'Z'*n in conditional:
        z = conditional['Z'*n]
        x = normalized(target); result['z_basis_tv'] = float(.5*np.abs(z-abs(x)**2).sum())
        result['p_node0'] = float(z[0]); result['p_node2'] = float(z[2]) if len(z) > 2 else None
    return result

def journal_bootstrap(records, target, system_bits, postselect_bits, *, calibration=None, samples=400,
                      seed=0, full=False):
    """Multinomial resampling of every count record (and calibration record when used)."""
    rng = np.random.default_rng(seed)
    values = {'raw': [], 'mitigated': [], 'acceptance': []}
    for _ in range(samples):
        draw = [dict(r, vector=rng.multinomial(int(np.sum(r['vector'])), np.asarray(r['vector'])/np.sum(r['vector'])))
                for r in records]
        raw = journal_estimate(draw, target, system_bits, postselect_bits)
        values['raw'].append(raw['fidelity_linear']); values['acceptance'].append(raw['acceptance'])
        if calibration is not None:
            zero = rng.multinomial(int(np.sum(calibration[0])), np.asarray(calibration[0])/np.sum(calibration[0]))
            one = rng.multinomial(int(np.sum(calibration[1])), np.asarray(calibration[1])/np.sum(calibration[1]))
            try:
                inverses, _ = journal_factorized_inverse(zero, one)
                values['mitigated'].append(journal_estimate(draw, target, system_bits, postselect_bits,
                                                            inverses=inverses)['fidelity_linear'])
            except ValueError:
                values['mitigated'].append(None)
    def interval(sequence):
        usable = [v for v in sequence if v is not None and np.isfinite(v)]
        return {'percentile95': np.quantile(usable, [.025, .975]).tolist() if usable else None,
                'standard_deviation': float(np.std(usable, ddof=1)) if len(usable) > 1 else None,
                'valid_replicates': len(usable), 'replicates': len(sequence)}
    return {name: interval(sequence) for name, sequence in values.items() if sequence}

In [ ]:
# CELL_ID: journal_campaign
"""Journal campaign: planning targets, per-device compilation, solver inventories
and the frozen per-device selection rule. Nothing here contacts IBM or submits.
Block assembly, predictions and the guarded submission live in journal_blocks.
"""
from qiskit.circuit import Parameter
from qiskit.circuit.library import CZGate, ECRGate, CXGate, IGate, RZGate, SXGate, XGate
from qiskit.circuit import Measure, Reset
from qiskit.transpiler import InstructionProperties, Target

_JC_OPERATIONS = {'cz': CZGate(), 'ecr': ECRGate(), 'cx': CXGate(), 'id': IGate(), 'sx': SXGate(), 'x': XGate(),
                  'rz': RZGate(Parameter('theta')), 'measure': Measure(), 'reset': Reset(),
                  'delay': Delay(Parameter('t'))}

def journal_target_from_snapshot(snapshot):
    """Rebuild the archived native Target (standard instructions only)."""
    target = Target(num_qubits=snapshot['num_qubits'], dt=snapshot.get('dt'))
    grouped = {}
    for row in snapshot['instructions']:
        if row['name'] not in _JC_OPERATIONS or row.get('qubits') is None:
            continue
        grouped.setdefault(row['name'], {})[tuple(row['qubits'])] = InstructionProperties(
            duration=row.get('duration_seconds'), error=row.get('error'))
    for name, properties in grouped.items():
        target.add_instruction(_JC_OPERATIONS[name], properties, name=name)
    return target

def journal_planning_devices(store):
    """Pre-registration targets: latest archived Nighthawk calibration and packaged Heron snapshots."""
    devices = {}
    archived = []
    for directory in sorted((store.root/'calibrations').glob('*')):
        try:
            record = store.calibration(directory.name)
        except CacheIntegrityError:
            continue
        payload = record['payload']
        if payload.get('kind') == 'hardware' and payload.get('backend') == 'ibm_rensselaer' and payload.get('target') \
                and not payload.get('job_id'):
            archived.append((str(payload['calibration_last_update']), directory.name))
    if archived:
        stamp, identifier = max(archived)
        snapshot = store.calibration(identifier)['payload']['target']
        devices['ibm_rensselaer'] = {'calibration_id': identifier, 'target': journal_target_from_snapshot(snapshot),
                                     'source': 'latest archived live snapshot', 'calibration_last_update': stamp}
    for name, fake in [('ibm_kingston', 'FakeKingston'), ('ibm_boston', 'FakeBoston')]:
        backend = getattr(fake_provider, fake)()
        identifier = capture_backend(store, backend, kind='fake', observation={'purpose': 'journal pre-registration target', 'represents': name})
        devices[name] = {'calibration_id': identifier, 'target': backend.target, 'source': f'{fake} packaged snapshot',
                         'calibration_last_update': store.calibration(identifier)['payload']['calibration_last_update']}
    return devices

def journal_compile(store, label, circuits, entries, target, calibration_id, dependencies=(), *, cap=1500):
    """Compile logical circuits against one Target; keep over-cap circuits and their counts."""
    if len(circuits) != len(entries):
        raise ValueError('One entry per circuit is required.')
    settings = {'optimization_level': 2, 'seed_transpiler': 1729, 'max_two_qubit_gates': int(cap)}
    spec = {'device_label': label, 'logical_circuits': [_semantic_circuit_signature(c) for c in circuits],
            'entries': entries, 'transpile': settings, 'target_num_qubits': target.num_qubits}
    def compute(directory):
        manager = generate_preset_pass_manager(target=target, optimization_level=2, seed_transpiler=1729)
        compiled, rows = [], []
        for index, (logical, entry) in enumerate(zip(circuits, entries)):
            started = time.perf_counter()
            isa = manager.run(logical)
            resources = circuit_resources(isa)
            row = {**entry, 'resources': resources, 'transpile_seconds': time.perf_counter()-started,
                   'initial_layout': isa.layout.initial_index_layout(), 'final_layout': isa.layout.final_index_layout(),
                   'within_cap': resources['two_qubit_gates'] <= cap,
                   'estimated_duration_seconds': float(isa.estimate_duration(target, unit='s'))}
            compiled.append(isa); rows.append(row)
        save_qpy(directory/'isa_circuits.qpy', compiled)
        return {'status': 'compiled', 'device_label': label, 'calibration_id': calibration_id, 'entries': rows,
                'isa_circuits_sha256': file_hash(directory/'isa_circuits.qpy'), 'limits': settings,
                'hardware_jobs_submitted': 0}
    return store.run('journal_compile', spec, compute, roles=['journal_campaign', 'backends'],
                     dependencies=list(dependencies), calibration_ids=[calibration_id])

# ---------------- solvers per instance ----------------

def journal_coherent_screen(store, problem_record, ideal_id):
    design = JOURNAL_DESIGN['coherent']
    settings = {**design['screening'], **design['candidates']}
    return coherent_screening_stages(store, problem_record, ideal_id, settings)

def journal_vqls(store, identifier, problem_record, ideal_id, development_solver=None):
    n = int(round(np.log2(len(problem_record['result']['b']))))
    if n == 2:
        if identifier == 'D0':
            if development_solver is None:
                raise ValueError('D0 reuses its frozen development checkpoint.')
            return development_solver
        return make_solver_stage(store, problem_record, 'vqls_measured',
                                 dict(JOURNAL_DESIGN['vqls']['two_qubit']['parameters']), ideal_id)
    rule = JOURNAL_DESIGN['vqls']['larger']
    runs = [make_solver_stage(store, problem_record, 'vqls_statevector',
                              {'reps': int(rule['reps'][str(n)]), 'max_evals': int(rule['max_evals']), 'seed': int(seed)}, ideal_id)
            for seed in rule['seeds']]
    chosen = min(runs, key=lambda r: (r['result']['solver_metadata']['best_cost'], r['result']['parameters']['seed']))
    def select(directory):
        return {'status': 'selected', 'rule': rule['checkpoint'], 'selected': store.dependency(chosen),
                'candidates': [{'seed': r['result']['parameters']['seed'], 'best_objective': r['result']['solver_metadata']['best_cost'],
                                'record': store.dependency(r)} for r in runs],
                'reference_fidelity_used_for_selection': False}
    selection = store.run('journal_vqls_selection', {'instance': identifier, 'rule': rule}, select,
                          roles=['journal_campaign', 'journal_design'], dependencies=[store.dependency(r) for r in runs],
                          calibration_ids=[ideal_id])
    return chosen if selection['result']['selected'] == store.dependency(chosen) else None

def journal_logical_inventory(store, identifier, problem_record, screening, vqls_record):
    """All logical circuits a block may need for one instance, with their verified ideal states."""
    data = problem_record['result']; A = np.asarray(data['A']); b = np.asarray(data['b'])
    target = normalized(np.linalg.solve(A, b)).real
    items = []
    if screening is not None:
        for record in screening['candidates']:
            result = record['result']
            if result['status'] != 'accepted':
                continue
            with np.load(record['directory']/'projected_operator.npz') as arrays:
                state = arrays['selected_state']
            items.append({'role': 'coherent_candidate', 'candidate_id': result['candidate_id'], 'algorithm': result['algorithm'],
                          'circuit': load_qpy(record['directory']/'logical_measured.qpy')[0], 'state': state,
                          'solver_metadata': {'system_qubits': result['solver_metadata']['system_qubits'],
                                              'postselect_bits': result['solver_metadata']['postselect_bits']},
                          'success_probability': result['success_probability'], 'record': store.dependency(record),
                          'operator_relative_error': result['operator_relative_error']})
    state = np.load(vqls_record['directory']/'selected_state.npz')['state']
    items.append({'role': 'vqls', 'candidate_id': 'vqls_'+vqls_record['result']['algorithm'], 'algorithm': 'vqls',
                  'circuit': load_qpy(vqls_record['directory']/'logical_measured.qpy')[0], 'state': state,
                  'solver_metadata': {'system_qubits': vqls_record['result']['solver_metadata']['system_qubits'],
                                      'postselect_bits': {}},
                  'success_probability': 1.0, 'record': store.dependency(vqls_record)})
    source = normalized(b).real
    return {'identifier': identifier, 'target_state_real': target, 'source_state_real': source, 'items': items}

def journal_compile_entries(inventory):
    """JSON metadata for each logical circuit in an inventory."""
    return [{k: v for k, v in item.items() if k not in ('circuit', 'state')} for item in inventory['items']]

def journal_select(compiled_entries, design=None):
    """Frozen per-device selection; returns {method: {'arm', 'index', ...}}."""
    rule = (design or JOURNAL_DESIGN)['coherent']['selection']
    selected = {}
    for method in ('hhl', 'qsvt'):
        rows = [dict(e, index=i) for i, e in enumerate(compiled_entries)
                if e['role'] == 'coherent_candidate' and e['algorithm'] == method]
        for row in rows:
            row['objective'] = row['resources']['two_qubit_gates']/row['success_probability']
        key = lambda r: (r['objective'], r['resources']['two_qubit_gates'], r['candidate_id'])
        primary = [r for r in rows if r['resources']['two_qubit_gates'] <= rule['primary_cap_two_qubit_gates']]
        probe = [r for r in rows if r['resources']['two_qubit_gates'] <= rule['probe_cap_two_qubit_gates']]
        if primary:
            choice = min(primary, key=key); selected[method] = {'arm': 'primary', **choice}
        elif probe:
            choice = min(probe, key=key); selected[method] = {'arm': 'depth_probe', **choice}
        else:
            selected[method] = {'arm': 'infeasible', 'reason': 'no operator-passing candidate under the probe cap',
                                'smallest_native_two_qubit_gates': min((r['resources']['two_qubit_gates'] for r in rows), default=None)}
    return selected

In [ ]:
# CELL_ID: journal_blocks
"""Block assembly and pre-registered predictions; no network access.

Kept separate from journal_campaign so that edits here never invalidate saved
compilations. Submission and collection live in journal_submission.
"""
from qiskit.transpiler import CouplingMap

# ---------------- block assembly ----------------

def _jc_rep_delay(payload):
    value = (payload.get('configuration') or {}).get('default_rep_delay')
    if value is None:
        return 0.00025
    value = float(value)
    return value*1e-6 if value > 1e-2 else value

def journal_estimate_seconds(circuits, target, payload, shots):
    """The estimator reproduced the 5 Oct charges within 1 s (27.1 vs 28 s; 15.4 vs 16 s)."""
    durations = [float(c.estimate_duration(target, unit='s')) for c in circuits]
    reset = 0.0
    if 'reset' in target.operation_names:
        reset = max((p.duration or 0.0) for p in target['reset'].values() if p is not None)
    rep_delay = _jc_rep_delay(payload)
    return 2.0+shots*sum(d+rep_delay+reset for d in durations), {'rep_delay_seconds': rep_delay,
        'reset_seconds': reset, 'longest_circuit_seconds': max(durations), 'assumed_job_overhead_seconds': 2.0}

def journal_control_circuit(state, layout, target):
    """StatePreparation of a known state on exactly the given physical qubits (any order).

    The target-aware compilation is used whenever its routing stays on the declared qubits, which
    reproduces every pre-registered control unchanged. When it does not (on 13 Oct the H3 control on
    ibm_rensselaer qubits [11, 10, 20] was routed through a neighbour), routing is restricted to the
    couplers among the declared qubits and every instruction is checked against the Target.
    """
    n = int(round(np.log2(len(state))))
    logical = QuantumCircuit(n)
    logical.append(StatePreparation(np.asarray(state, dtype=complex)), list(range(n)))
    logical.measure_all()
    declared = sorted(int(q) for q in layout)
    manager = generate_preset_pass_manager(target=target, optimization_level=2, seed_transpiler=1729,
                                           initial_layout=list(layout))
    native = manager.run(logical)
    if sorted(native.layout.final_index_layout()[:n]) == declared:
        return native
    restricted = CouplingMap()
    for qubit in range(target.num_qubits):
        restricted.add_physical_qubit(qubit)
    neighbours = {q: set() for q in declared}
    for a, b in target.build_coupling_map().get_edges():
        if a in neighbours and b in neighbours:
            restricted.add_edge(a, b); neighbours[a].add(b); neighbours[b].add(a)
    reached, frontier = {declared[0]}, [declared[0]]
    while frontier:
        for q in neighbours[frontier.pop()] - reached:
            reached.add(q); frontier.append(q)
    if reached != set(declared):
        raise ValueError('The declared control qubits are not connected on the device.')
    # Only plain native gates may be named here: live Targets also carry device-specific instructions
    # (e.g. measure_2, measure_reset_2, xslow on ibm_rensselaer) that the transpiler refuses as basis gates.
    native_basis = [name for name in ('cz', 'ecr', 'cx', 'rz', 'sx', 'x', 'id', 'measure', 'reset', 'delay') if name in target.operation_names]
    manager = generate_preset_pass_manager(optimization_level=2, seed_transpiler=1729, initial_layout=list(layout),
                                           coupling_map=restricted, basis_gates=native_basis)
    native = manager.run(logical)
    if sorted(native.layout.final_index_layout()[:n]) != declared:
        raise ValueError('Control routing left the declared physical qubits.')
    for item in native.data:
        if item.operation.name == 'barrier':
            continue
        qargs = tuple(native.find_bit(q).index for q in item.qubits)
        if not set(qargs) <= set(declared) or not target.instruction_supported(operation_name=item.operation.name, qargs=qargs):
            raise ValueError('Restricted control routing produced an instruction outside the declared qubits or the Target.')
    return native

def journal_cells_for_instance(store, identifier, inventory, compiled, device_label, target, target_snapshot):
    """Selected cells and their verified basis circuits for one instance on one device."""
    circuits_all = load_qpy(compiled['directory']/'isa_circuits.qpy')
    entries = compiled['result']['entries']
    n = int(round(np.log2(len(inventory['target_state_real']))))
    settings_kind = JOURNAL_DESIGN['measurement']['settings'][str(n)]
    solver_settings = journal_settings(n, settings_kind)
    control_settings = journal_settings(n, JOURNAL_DESIGN['measurement']['controls']['setting_set'])
    selection = journal_select(entries)
    chosen = []
    for method, choice in selection.items():
        if choice['arm'] in ('primary', 'depth_probe'):
            chosen.append((choice['index'], choice['arm']))
    if identifier == JOURNAL_DESIGN['coherent']['historical_ablation']['instance']:
        for candidate in JOURNAL_DESIGN['coherent']['historical_ablation']['candidates']:
            index = next((i for i, e in enumerate(entries) if e.get('candidate_id') == candidate), None)
            if index is not None and entries[index]['resources']['two_qubit_gates'] <= JOURNAL_DESIGN['ceilings']['max_two_qubit_gates_any_circuit'] \
                    and index not in [c[0] for c in chosen]:
                chosen.append((index, 'historical_ablation'))
    vqls_index = next(i for i, e in enumerate(entries) if e['role'] == 'vqls')
    chosen.append((vqls_index, 'vqls'))
    cells, circuits, rows = [], [], []
    item_by_id = {item['candidate_id']: item for item in inventory['items']}
    for index, arm in chosen:
        entry = {**entries[index], 'instance': identifier, 'arm': arm, 'device_label': device_label}
        item = item_by_id[entry['candidate_id']]
        label = f"{identifier}:{entry['candidate_id']}:{arm}"
        cs, rs, info = journal_cell_circuits(circuits_all[index], entry, solver_settings, item['state'], label, target_snapshot)
        cells.append({'label': label, 'instance': identifier, 'arm': arm, 'algorithm': entry['algorithm'],
                      'candidate_id': entry['candidate_id'], 'settings': solver_settings, 'n': n,
                      'two_qubit_gates': entries[index]['resources']['two_qubit_gates'],
                      'ideal_strict_acceptance': info['ideal_strict_acceptance'], 'prefix_sha256': info['prefix_sha256'],
                      'layout': info['layout'], 'system_bits': info['system_bits'], 'postselect_bits': info['postselect_bits'],
                      'ideal_conditional_state': normalized(item['state']), 'full_tomography': settings_kind == 'full',
                      'solution_state_real': list(map(float, inventory['target_state_real']))})
        circuits.extend(cs); rows.extend(rs)
    vqls_layout = entries[vqls_index]['final_layout'][:n]
    controls = []
    if n in JOURNAL_DESIGN['measurement']['controls']['input_state']:
        source = np.asarray(inventory['source_state_real'], dtype=float)
        controls.append(('input_state', source))
    if n in JOURNAL_DESIGN['measurement']['controls']['solution_state']:
        controls.append(('solution_state', np.asarray(inventory['target_state_real'], dtype=float)))
    for name, state in controls:
        native = journal_control_circuit(state, vqls_layout, target)
        entry = {'role': 'control', 'algorithm': 'control', 'candidate_id': name, 'instance': identifier, 'arm': 'control',
                 'solver_metadata': {'system_qubits': list(range(n)), 'postselect_bits': {}},
                 'final_layout': native.layout.final_index_layout()[:n], 'device_label': device_label,
                 'classical_solution_access': name == 'solution_state', 'resources': circuit_resources(native)}
        label = f'{identifier}:{name}:control'
        cs, rs, info = journal_cell_circuits(native, entry, control_settings, state, label, target_snapshot)
        cells.append({'label': label, 'instance': identifier, 'arm': 'control', 'algorithm': 'control', 'candidate_id': name,
                      'settings': control_settings, 'n': n, 'two_qubit_gates': entry['resources']['two_qubit_gates'],
                      'ideal_strict_acceptance': 1.0, 'prefix_sha256': info['prefix_sha256'], 'layout': info['layout'],
                      'system_bits': info['system_bits'], 'postselect_bits': {}, 'ideal_conditional_state': normalized(state),
                      'full_tomography': False, 'solution_state_real': list(map(float, inventory['target_state_real']))})
        circuits.extend(cs); rows.extend(rs)
    return cells, circuits, rows, selection

def journal_readout_for(cells, width):
    circuits, rows, seen = [], [], set()
    for cell in cells:
        layout = tuple(cell['layout'])
        if layout in seen:
            continue
        seen.add(layout)
        label = 'readout:'+'-'.join(map(str, layout))
        cs, rs = journal_readout_circuits(width, list(layout), label, correlated=(len(layout) == 2))
        circuits.extend(cs); rows.extend(rs)
    return circuits, rows

def journal_prepare_job(store, *, device_label, block, group, arm_name, instance_records, target, payload,
                        calibration_id, account, backend_name, dependencies):
    """Assemble, validate, order and cost one job; no network access."""
    design = JOURNAL_DESIGN
    spec = {'design_id': design['design_id'], 'design_sha256': content_hash(design), 'device_label': device_label,
            'block': block, 'group': group, 'arm': arm_name, 'account': account, 'backend': backend_name,
            'instances': [r['identifier'] for r in instance_records], 'calibration_id': calibration_id,
            'compilations': [store.dependency(r['compiled']) for r in instance_records]}
    def prepare(directory):
        cells, circuits, rows, selections = [], [], [], {}
        for record in instance_records:
            c, cs, rs, selection = journal_cells_for_instance(store, record['identifier'], record['inventory'],
                record['compiled'], device_label, target, payload.get('target'))
            if arm_name == 'mitigation':
                keep = set(design['jobs']['mitigation_arm']['arms'])
                c = [cell for cell in c if cell['arm'] in keep]
                labels = {cell['label'] for cell in c}
                pairs = [(circ, row) for circ, row in zip(cs, rs) if row['cell_label'] in labels]
                cs, rs = [p[0] for p in pairs], [p[1] for p in pairs]
            cells.extend(c); circuits.extend(cs); rows.extend(rs); selections[record['identifier']] = selection
        ro_circuits, ro_rows = journal_readout_for(cells, target.num_qubits)
        circuits.extend(ro_circuits); rows.extend(ro_rows)
        block_index = int(''.join(ch for ch in block if ch.isdigit()) or 0)
        order = np.random.default_rng(20261011+block_index).permutation(len(circuits)).tolist()
        circuits = [circuits[i] for i in order]
        rows = [{**rows[i], 'schedule_position': position, 'calibration_id': calibration_id,
                 'block': block, 'group': group, 'job_arm': arm_name} for position, i in enumerate(order)]
        for circuit, row in zip(circuits, rows):
            validate_measurement_map(circuit, row)
            if circuit_resources(circuit)['two_qubit_gates'] > design['ceilings']['max_two_qubit_gates_any_circuit']:
                raise ValueError('A circuit exceeds the absolute two-qubit ceiling.')
            if circuit.num_qubits != target.num_qubits:
                raise ValueError('Circuit width differs from the device Target.')
        if len(circuits) > design['ceilings']['max_circuits_per_job']:
            raise ValueError('Job exceeds the circuit ceiling; split the group in the design.')
        shots = design['measurement']['shots']
        seconds, timing = journal_estimate_seconds(circuits, target, payload, shots)
        if seconds > design['ceilings']['max_estimated_qpu_seconds_per_job']:
            raise ValueError(f'Estimated {seconds:.1f} s exceeds the per-job ceiling.')
        save_qpy(directory/'isa_circuits.qpy', circuits)
        recovered = load_qpy(directory/'isa_circuits.qpy')
        for original, restored in zip(circuits, recovered):
            if _semantic_circuit_signature(original) != _semantic_circuit_signature(restored):
                raise ValueError('QPY round trip changed a circuit.')
        cell_rows = [{k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in cell.items() if k != 'ideal_conditional_state'}
                     | {'ideal_conditional_state_real': np.real(cell['ideal_conditional_state']).tolist(),
                        'ideal_conditional_state_imag': np.imag(cell['ideal_conditional_state']).tolist()} for cell in cells]
        atomic_json(directory/'cells.json', cell_rows); atomic_json(directory/'schedule.json', rows)
        options = design['jobs']['raw_options'] if arm_name == 'raw' else design['jobs']['mitigation_arm']['options']
        return {'status': 'prepared', 'design_id': design['design_id'], 'device_label': device_label, 'block': block,
                'group': group, 'arm': arm_name, 'account': account, 'backend': backend_name,
                'entries': rows, 'cells': cell_rows, 'selections': selections, 'calibration_id': calibration_id,
                'isa_circuits_sha256': file_hash(directory/'isa_circuits.qpy'), 'circuit_count': len(circuits),
                'shots_per_circuit': shots, 'raw_shots': shots*len(circuits), 'estimated_qpu_seconds': seconds,
                'timing_model': timing, 'sampler_options': {**options, 'execution': design['jobs']['execution']},
                'hardware_jobs_submitted': 0}
    return store.run('journal_job_preparation', spec, prepare,
                     roles=['journal_blocks', 'journal_campaign', 'journal_measurement', 'journal_design', 'backends', 'noise_helpers'],
                     dependencies=list(dependencies), calibration_ids=[calibration_id])

# ---------------- predictions ----------------

def _jc_prefix_measured(circuit):
    """The verified preparation followed directly by Z measurements of every classical bit."""
    boundary = next(i for i, item in enumerate(circuit.data) if item.operation.name == 'barrier'
                    and getattr(item.operation, 'label', None) == 'common_preparation_boundary')
    prefix = circuit.copy(); del prefix.data[boundary:]
    _, measured = journal_prefix(circuit)
    for bit in range(circuit.num_clbits):
        prefix.measure(measured[bit], bit)
    return prefix

def journal_rotated_probabilities(rho, setting, system_bits):
    k = int(round(np.log2(rho.shape[0])))
    matrices = [np.eye(2, dtype=complex)]*k
    n = len(system_bits)
    for j, bit in enumerate(system_bits):
        matrices[int(bit)] = _JM_ROTATION[setting[n-1-j]]
    U = np.ones((1, 1), dtype=complex)
    for bit in reversed(range(k)):
        U = np.kron(U, matrices[bit])
    return np.clip(np.real(np.diag(U@rho@U.conj().T)), 0, None)

def journal_predict_job(store, prepared, calibration_id, label, models=JOURNAL_NOISE_MODELS):
    spec = {'prepared': store.dependency(prepared), 'calibration_id': calibration_id, 'models': list(models), 'label': label}
    def compute(directory):
        payload = store.calibration(calibration_id)['payload']
        circuits = load_qpy(prepared['directory']/'isa_circuits.qpy'); rows = prepared['result']['entries']
        predictions = []
        for cell in prepared['result']['cells']:
            index = next(i for i, r in enumerate(rows) if r.get('cell_label') == cell['label'] and r['kind'] == 'tomography')
            prefix = _jc_prefix_measured(circuits[index])
            row = {'cell_label': cell['label'], 'models': {}}
            for model in models:
                try:
                    rho, readout, details = journal_measured_density(prefix, payload, model)
                except (ValueError, RuntimeError) as error:
                    row['models'][model] = {'status': 'withheld', 'reason': str(error)}
                    continue
                records_true, records_raw = [], []
                for setting in cell['settings']:
                    p = journal_rotated_probabilities(rho, setting, cell['system_bits'])
                    records_true.append({'setting': setting, 'vector': p})
                    records_raw.append({'setting': setting, 'vector': p if readout is None else journal_apply_bit_matrices(p, readout)})
                truth = cell['solution_state_real']
                true_estimate = journal_estimate(records_true, truth, cell['system_bits'], cell['postselect_bits'], full=cell['full_tomography'])
                raw_estimate = journal_estimate(records_raw, truth, cell['system_bits'], cell['postselect_bits'], full=cell['full_tomography'])
                row['models'][model] = {'status': 'predicted', 'fidelity_state': true_estimate['fidelity_linear'],
                    'fidelity_raw_readout': raw_estimate['fidelity_linear'], 'acceptance_state': true_estimate['acceptance'],
                    'acceptance_raw_readout': raw_estimate['acceptance'], 'z_basis_tv_raw': raw_estimate.get('z_basis_tv'),
                    'p_node0_raw': raw_estimate.get('p_node0'), 'details': {k: v for k, v in details.items() if k != 'gate_channels'},
                    'gate_channel_count': len(details.get('gate_channels', []))}
            predictions.append(row)
        atomic_json(directory/'predictions.json', predictions)
        return {'status': 'predicted', 'label': label, 'calibration_id': calibration_id,
                'calibration_last_update': payload.get('calibration_last_update'), 'predictions': predictions,
                'models': list(models), 'made_before_observation': label == 'pre_registration',
                'scope': 'Exact calibration-model predictions of the measured estimators; no shots, no hardware.'}
    return store.run('journal_prediction', spec, compute, roles=['journal_blocks', 'journal_noise', 'journal_measurement'],
                     dependencies=[store.dependency(prepared)], calibration_ids=[calibration_id])

In [ ]:
# CELL_ID: journal_submission
"""Guarded submission and read-only collection of journal jobs.

No function here submits unless CONFIG['journal']['hardware']['submit'] is True
for that device and block, the account/backend pair is allowlisted, the circuit
inventory matches the prepared job, every ceiling holds and the calibration
captured immediately before submission equals the compilation calibration.
Each (design, device, block, job group, arm) has one atomic claim: a job that
may already exist is collected, never resubmitted. These functions create no
cached stage of their own (collection uses the hardened collector's stage), so
editing this cell does not invalidate prepared jobs or predictions.
"""

def _jc_identity(prepared):
    r = prepared['result']
    return {'journal_job_v1': {k: r[k] for k in ('design_id', 'account', 'backend', 'block', 'group', 'arm')}}

def journal_job_registry(store, prepared):
    identity = _jc_identity(prepared)
    request_id = content_hash(identity)
    return (store.root/'hardware_jobs'/request_id, store.root/'hardware_campaigns'/content_hash({'claim': identity}),
            request_id, identity)

def journal_known_job(store, prepared, *, collect, service_factory=None):
    """Return a saved, pending or uncertain job record; None only when nothing exists."""
    registry, claim, request_id, identity = journal_job_registry(store, prepared)
    if (registry/'request.json').is_file():
        saved = read_json(registry/'request.json')
        if saved.get('journal_identity') != identity:
            raise CacheIntegrityError('Registry identity mismatch; no substitution.')
    for name in ('hardened_collection_reference.json', 'completed_reference.json'):
        if (registry/name).is_file():
            reference = read_json(registry/name)
            record = store.lookup(reference['stage'], reference['key'])
            if record is None:
                raise CacheIntegrityError('Known journal job lost its completion record; nothing resubmitted.')
            return record
    if (registry/'job.json').is_file():
        if not collect or service_factory is None:
            return {'status': 'pending', 'job_id': read_json(registry/'job.json')['job_id'], 'submitted_new_job': False,
                    'registry': str(registry.relative_to(store.root))}
        service, backend = service_factory()
        return _collect_registered_pilot(store, service, backend, registry)
    if (registry/'submission_intent.json').is_file():
        return {'status': 'submission_uncertain', 'registry': str(registry.relative_to(store.root)), 'submitted_new_job': False,
                'action': 'Inspect the IBM job list for this tag before any manual reconciliation; no automatic retry.'}
    if (claim/'claimed_request.json').is_file():
        return {'status': 'claim_requires_reconciliation', 'claim': read_json(claim/'claimed_request.json'), 'submitted_new_job': False}
    return None

def journal_submit_job(store, prepared, backend, configuration):
    """Submit one prepared job after every live check; at most once per identity."""
    registry, claim, request_id, identity = journal_job_registry(store, prepared)
    result = prepared['result']
    design = JOURNAL_DESIGN
    if result['backend'] not in HARDWARE_ALLOWLIST.get(result['account'], []):
        raise ValueError('Account/backend pair outside the allowlist.')
    if _backend_name(backend) != result['backend'] or not backend.status().operational:
        raise ValueError('Backend mismatch or not operational; nothing submitted.')
    path = prepared['directory']/'isa_circuits.qpy'
    if file_hash(path) != result['isa_circuits_sha256']:
        raise CacheIntegrityError('Prepared circuits changed.')
    circuits = load_qpy(path); entries = result['entries']
    if len(circuits) != result['circuit_count'] or len(entries) != len(circuits):
        raise ValueError('Inventory differs from the prepared job.')
    for circuit, entry in zip(circuits, entries):
        validate_measurement_map(circuit, entry)
        if circuit.num_qubits != backend.num_qubits:
            raise ValueError('Circuit width differs from the live backend.')
        for item in circuit.data:
            if item.operation.name == 'barrier':
                continue
            qargs = tuple(circuit.find_bit(q).index for q in item.qubits)
            if not backend.target.instruction_supported(operation_name=item.operation.name, qargs=qargs):
                raise ValueError('A native instruction is unsupported by the live Target.')
    before = capture_backend(store, backend, 'hardware', {'phase': 'journal_immediately_before_submission',
                                                          'block': result['block'], 'group': result['group']})
    if before != result['calibration_id']:
        return {'status': 'calibration_changed_before_submission', 'submitted_new_job': False,
                'compile_calibration_id': result['calibration_id'], 'new_calibration_id': before,
                'action': 'Run again: the notebook recompiles against the new calibration under the same frozen rule.'}
    seconds, timing = journal_estimate_seconds(circuits, backend.target, store.calibration(before)['payload'], result['shots_per_circuit'])
    ceilings = design['ceilings']
    if seconds > ceilings['max_estimated_qpu_seconds_per_job']:
        raise ValueError('Live estimate exceeds the per-job ceiling.')
    # Twirled jobs add per-randomization overhead; give the mitigation arm 1.5x more headroom.
    factor = ceilings['execution_cap_factor']*(1.5 if result['arm'] == 'mitigation' else 1.0)
    cap = int(max(ceilings['execution_cap_floor_seconds'], np.ceil(seconds*factor)))
    options = {'default_shots': result['shots_per_circuit'], 'max_execution_time': cap,
               **copy.deepcopy(result['sampler_options']),
               'environment': {'job_tags': ['diffusion_journal', result['design_id'], result['block'], result['group'],
                                            result['arm'], request_id[:16]]}}
    sampler = SamplerV2(mode=backend, options=options)
    registry.mkdir(parents=True, exist_ok=True); claim.mkdir(parents=True, exist_ok=True)
    with (claim/'claimed_request.json').open('x', encoding='utf8') as stream:
        json.dump({'request_id': request_id, 'claimed_utc': utc_now(), 'max_jobs': 1}, stream)
    dependencies = [store.dependency(prepared)]
    request = {'request_id': request_id, 'journal_identity': identity,
               'settings': {'account': result['account'], 'backend': result['backend'], 'shots': result['shots_per_circuit'],
                            'campaign_id': result['design_id']+':'+result['block']+':'+result['group']+':'+result['arm']},
               'compiled_entries': entries, 'compile_calibration_id': result['calibration_id'],
               'before_submission_calibration_id': before, 'compilation': store.dependency(prepared),
               'dependencies': dependencies, 'isa_circuits_sha256': file_hash(path), 'requested_sampler_options': options,
               'timing_estimate': {'estimated_qpu_seconds': seconds, **timing, 'max_execution_time_seconds': cap},
               'user_authorization': configuration.get('approval_note', 'not recorded')}
    atomic_json(registry/'request.json', request); shutil.copy2(path, registry/'isa_circuits.qpy')
    atomic_json(registry/'submission_intent.json', {'status': 'submitting_or_unknown', 'request_id': request_id,
        'written_before_remote_submission_utc': utc_now(), 'calibration_ids': [result['calibration_id'], before],
        'limits': {'circuits': len(circuits), 'shots_per_circuit': result['shots_per_circuit'], 'max_execution_time_seconds': cap}})
    store.bundle(registry, calibration_ids=[result['calibration_id'], before], dependencies=dependencies)
    job = sampler.run(circuits, shots=result['shots_per_circuit'])
    atomic_json(registry/'job.json', {'job_id': job.job_id(), 'submitted_utc': utc_now()})
    store.bundle(registry, calibration_ids=[result['calibration_id'], before], dependencies=dependencies)
    return {'status': 'submitted', 'job_id': job.job_id(), 'submitted_new_job': True, 'request_id': request_id,
            'estimated_qpu_seconds': seconds, 'max_execution_time_seconds': cap,
            'registry': str(registry.relative_to(store.root))}

In [ ]:
# CELL_ID: journal_analysis
"""Per-cell estimates, uncertainty, inversion gain and prediction tests. No network access."""

def journal_counts_vector(counts, width):
    vector = np.zeros(1 << width, dtype=np.int64)
    for key, value in counts.items():
        key = key.replace(' ', '')
        if len(key) != width or set(key)-{'0', '1'}:
            raise ValueError('Count key width differs from the classical register.')
        vector[int(key, 2)] += int(value)
    return vector

def _ja_rows(raw_record):
    rows = []
    for row in raw_record['result']['rows']:
        path = Path(raw_record['directory'])/row['counts_file']
        if file_hash(path) != row['counts_sha256']:
            raise CacheIntegrityError('Raw counts changed after collection.')
        counts = read_json(path)
        rows.append({'entry': row['entry'], 'vector': journal_counts_vector(counts, len(next(iter(counts))))})
    return rows

def _ja_calibration(rows, layout):
    same = [r for r in rows if r['entry']['kind'] == 'readout_calibration' and list(r['entry']['final_layout']) == list(layout)]
    patterns = {r['entry']['pattern']: r['vector'] for r in same}
    if {'all_zero', 'all_one'} <= set(patterns):
        return patterns['all_zero'], patterns['all_one']
    if {'basis_0', 'basis_3'} <= set(patterns):
        return patterns['basis_0'], patterns['basis_3']
    return None

def journal_cell_analysis(rows, cell, input_fidelity, settings):
    records = [{'setting': r['entry']['setting'], 'vector': r['vector']} for r in rows
               if r['entry']['kind'] == 'tomography' and r['entry'].get('cell_label') == cell['label']]
    if sorted(r['setting'] for r in records) != sorted(cell['settings']):
        raise ValueError(f"{cell['label']}: missing or repeated settings.")
    truth = np.asarray(cell['solution_state_real'], dtype=float)
    system, post = cell['system_bits'], {int(k): int(v) for k, v in cell['postselect_bits'].items()}
    raw = journal_estimate(records, truth, system, post, full=cell['full_tomography'])
    calibration = _ja_calibration(rows, cell['layout'])
    mitigated, assignment = None, None
    if calibration is not None:
        inverses, assignment = journal_factorized_inverse(*calibration)
        mitigated = journal_estimate(records, truth, system, post, inverses=inverses, full=cell['full_tomography'])
    boot = journal_bootstrap(records, truth, system, post, calibration=calibration,
                             samples=settings['bootstrap_samples'], seed=settings['seed'])
    total = int(sum(np.sum(r['vector']) for r in records))
    accepted = int(round(raw['accepted_shots_equivalent']))
    gain = lambda f: None if f is None or input_fidelity >= 1 else (f-input_fidelity)/(1-input_fidelity)
    zz = next(r for r in records if set(r['setting']) == {'Z'})
    zz_system, zz_accept = journal_condition(np.asarray(zz['vector'], float), system, post)
    zz_accepted = int(round(zz_system.sum()))
    result = {'cell_label': cell['label'], 'instance': cell['instance'], 'arm': cell['arm'], 'algorithm': cell['algorithm'],
              'candidate_id': cell['candidate_id'], 'two_qubit_gates': cell['two_qubit_gates'], 'n_system_qubits': cell['n'],
              'settings': len(records), 'raw_shots': total, 'accepted_shots': accepted,
              'acceptance': accepted/total, 'acceptance_wilson95': wilson_interval(accepted, total),
              'ideal_strict_acceptance': cell['ideal_strict_acceptance'],
              'fidelity_raw': raw['fidelity_linear'], 'fidelity_raw_bootstrap95': boot['raw']['percentile95'],
              'fidelity_mitigated': None if mitigated is None else mitigated['fidelity_linear'],
              'fidelity_mitigated_bootstrap95': boot.get('mitigated', {}).get('percentile95'),
              'input_state_fidelity_F_in': input_fidelity,
              'inversion_gain_raw': gain(raw['fidelity_linear']),
              'inversion_gain_raw_bootstrap95': None if boot['raw']['percentile95'] is None else [gain(v) for v in boot['raw']['percentile95']],
              'inversion_gain_mitigated': gain(None if mitigated is None else mitigated['fidelity_linear']),
              'z_basis_tv_raw': raw.get('z_basis_tv'), 'p_node0_raw': raw.get('p_node0'), 'p_node2_raw': raw.get('p_node2'),
              'zz_accepted_shots': zz_accepted,
              'two_output_hoeffding_radius95': float(np.sqrt(np.log(4/.05)/(2*zz_accepted))) if zz_accepted else None,
              'readout_assignment': assignment, 'pauli_expectations_raw': raw['pauli_expectations'],
              'linear_tomography': {k: raw[k] for k in ('eigenvalues', 'purity', 'nonphysical') if k in raw},
              'uncertainty_scope': 'within-job multinomial bootstrap; excludes drift, calibration-model and measurement-axis bias'}
    return result

def journal_analyze_job(store, prepared, raw_record, prediction_records, input_fidelities, settings):
    spec = {'prepared': store.dependency(prepared), 'raw': store.dependency(raw_record),
            'predictions': [store.dependency(p) for p in prediction_records], 'settings': settings,
            'input_fidelities': input_fidelities}
    def compute(directory):
        rows = _ja_rows(raw_record)
        analyses = []
        for cell in prepared['result']['cells']:
            analysis = journal_cell_analysis(rows, cell, input_fidelities[cell['instance']], settings)
            analysis['predictions'] = {}
            for record in prediction_records:
                row = next((p for p in record['result']['predictions'] if p['cell_label'] == cell['label']), None)
                if row is not None:
                    analysis['predictions'][record['result']['label']] = row['models']
            analyses.append(analysis)
        atomic_json(directory/'cell_analyses.json', analyses)
        return {'status': 'analyzed', 'cells': analyses, 'job_id': raw_record['result'].get('job_id'),
                'block': prepared['result']['block'], 'group': prepared['result']['group'], 'arm': prepared['result']['arm'],
                'device': prepared['result']['backend'], 'network_accessed': False}
    return store.run('journal_job_analysis', spec, compute, roles=['journal_analysis', 'journal_measurement', 'metrics'],
                     dependencies=[store.dependency(prepared), store.dependency(raw_record), *[store.dependency(p) for p in prediction_records]],
                     calibration_ids=raw_record['manifest']['calibration_ids'])

# ---------------- retrospective test on the archived September/October jobs ----------------

def journal_retrospective(store, raw_record, calibration_id, label, target_state=None, models=JOURNAL_NOISE_MODELS):
    """Predict every archived physical PUB from its job-start calibration and compare with its counts."""
    spec = {'raw': store.dependency(raw_record), 'calibration_id': calibration_id, 'label': label, 'models': list(models),
            'target_state': None if target_state is None else np.asarray(target_state, dtype=float).tolist()}
    def compute(directory):
        payload = store.calibration(calibration_id)['payload']
        isa = Path(raw_record['directory'])/'isa_circuits.qpy'
        circuits = load_qpy(isa)
        rows = []
        for index, row in enumerate(raw_record['result']['rows']):
            entry = row['entry']
            counts = read_json(Path(raw_record['directory'])/row['counts_file'])
            width = circuits[index].num_clbits
            observed = journal_counts_vector(counts, width).astype(float)
            meta = entry['solver_metadata']
            system = [int(b) for b in meta['system_qubits']]; post = {int(k): int(v) for k, v in meta['postselect_bits'].items()}
            observed_system, observed_accept = journal_condition(observed/observed.sum(), system, post)
            item = {'pub_index': index, 'kind': entry.get('kind'), 'algorithm': entry.get('algorithm'),
                    'setting': entry.get('setting', 'Z'*len(system)), 'two_qubit_gates': circuit_resources(circuits[index])['two_qubit_gates'],
                    'observed_acceptance': observed_accept, 'observed_accepted_shots': int(round(observed_accept*observed.sum())),
                    'models': {}}
            target = entry.get('target_probabilities')
            if target is None and 'expected_probabilities' in entry:
                target = entry['expected_probabilities']
            if target is None and entry.get('kind') in ('tomography', 'cost') and target_state is not None:
                setting = entry.get('setting')
                target = (abs(journal_setting_rotation(setting)@np.asarray(target_state, dtype=complex))**2).tolist()
            if target is not None and observed_accept > 0:
                item['observed_tv_to_target'] = float(.5*np.abs(observed_system/observed_accept-np.asarray(target)).sum())
            for model in models:
                try:
                    rho, readout, details = journal_measured_density(circuits[index], payload, model)
                except (ValueError, RuntimeError) as error:
                    item['models'][model] = {'status': 'withheld', 'reason': str(error)}
                    continue
                p = np.clip(np.real(np.diag(rho)), 0, None)
                p = p if readout is None else journal_apply_bit_matrices(p, readout)
                predicted_system, predicted_accept = journal_condition(p, system, post)
                value = {'status': 'predicted', 'predicted_acceptance': predicted_accept,
                         'esp_survival_weight': details.get('esp_survival_weight')}
                if predicted_accept > 0 and observed_accept > 0:
                    value['tv_predicted_vs_observed'] = float(.5*np.abs(predicted_system/predicted_accept-observed_system/observed_accept).sum())
                    if target is not None:
                        value['predicted_tv_to_target'] = float(.5*np.abs(predicted_system/predicted_accept-np.asarray(target)).sum())
                item['models'][model] = value
            rows.append(item)
        atomic_json(directory/'retrospective_predictions.json', rows)
        return {'status': 'compared', 'label': label, 'job_id': raw_record['result'].get('job_id'), 'rows': rows,
                'calibration_id': calibration_id, 'calibration_last_update': payload.get('calibration_last_update'),
                'scope': 'Retrospective: predictions made after these counts were observed; used to select no model parameter.'}
    return store.run('journal_retrospective', spec, compute, roles=['journal_analysis', 'journal_noise', 'journal_measurement'],
                     dependencies=[store.dependency(raw_record)], calibration_ids=[calibration_id])

# ---------------- archived development tomography under the journal estimator ----------------

def journal_archived_vqls_tomography(store, raw_record, problem_record, solver_record, settings):
    """Re-analyze the 5 Oct 24-PUB VQLS job against x_hat, b_hat and the ideal frozen checkpoint."""
    spec = {'raw': store.dependency(raw_record), 'problem': store.dependency(problem_record),
            'solver': store.dependency(solver_record), 'settings': settings}
    def compute(directory):
        A = np.asarray(problem_record['result']['A']); b = np.asarray(problem_record['result']['b'])
        x = normalized(np.linalg.solve(A, b)).real; source = normalized(b).real
        F_in = float((source@x)**2)
        rows = _ja_rows(raw_record)
        tomography = [r for r in rows if r['entry']['kind'] == 'tomography']
        records = [{'setting': r['entry']['setting'], 'vector': r['vector'], 'repeat': r['entry'].get('repeat')} for r in tomography]
        readout = {r['entry']['prepared_basis']: r['vector'] for r in rows if r['entry']['kind'] == 'readout_calibration'}
        calibration = (readout[0], readout[3])
        inverses, assignment = journal_factorized_inverse(*calibration)
        checkpoint = np.load(Path(solver_record['directory'])/'selected_state.npz')['state']
        if np.max(abs(np.imag(checkpoint))) > 1e-12:
            raise ValueError('The frozen real ansatz state must be real.')
        checkpoint = normalized(checkpoint).real
        out = {'F_in_exact': F_in, 'checkpoint_solution_fidelity_exact': float((checkpoint@x)**2),
               'checkpoint_inversion_gain_exact': float(((checkpoint@x)**2-F_in)/(1-F_in))}
        for name, target in [('solution', x), ('source', source), ('checkpoint', checkpoint)]:
            raw = journal_estimate(records, target, [0, 1], {}, full=True)
            mitigated = journal_estimate(records, target, [0, 1], {}, inverses=inverses, full=True)
            boot = journal_bootstrap(records, target, [0, 1], {}, calibration=calibration,
                                     samples=settings['bootstrap_samples'], seed=settings['seed'])
            out[name] = {'fidelity_raw': raw['fidelity_linear'], 'fidelity_raw_bootstrap95': boot['raw']['percentile95'],
                         'fidelity_mitigated': mitigated['fidelity_linear'],
                         'fidelity_mitigated_bootstrap95': boot['mitigated']['percentile95'],
                         'eigenvalues': raw['eigenvalues'], 'purity': raw['purity'],
                         'pauli_expectations_raw': raw['pauli_expectations']}
        f = out['solution']['fidelity_raw']; interval = out['solution']['fidelity_raw_bootstrap95']
        out['inversion_gain_raw'] = (f-F_in)/(1-F_in)
        out['inversion_gain_raw_bootstrap95'] = [(v-F_in)/(1-F_in) for v in interval]
        fm = out['solution']['fidelity_mitigated']; im = out['solution']['fidelity_mitigated_bootstrap95']
        out['inversion_gain_mitigated'] = (fm-F_in)/(1-F_in)
        out['inversion_gain_mitigated_bootstrap95'] = [(v-F_in)/(1-F_in) for v in im]
        out['per_repeat'] = []
        for repeat in sorted({r['repeat'] for r in records}):
            chosen = [r for r in records if r['repeat'] == repeat]
            value = journal_estimate(chosen, x, [0, 1], {})['fidelity_linear']
            out['per_repeat'].append({'repeat': repeat, 'fidelity_raw': value, 'inversion_gain_raw': (value-F_in)/(1-F_in)})
        out['target_pauli_expectations'] = journal_target_paulis(x, journal_pauli_labels(['XX', 'XY', 'XZ', 'YX', 'YY', 'YZ', 'ZX', 'ZY', 'ZZ']))
        out['checkpoint_pauli_expectations'] = journal_target_paulis(checkpoint, journal_pauli_labels(['XX', 'XY', 'XZ', 'YX', 'YY', 'YZ', 'ZX', 'ZY', 'ZZ']))
        out['source_pauli_expectations'] = journal_target_paulis(source, journal_pauli_labels(['XX', 'XY', 'XZ', 'YX', 'YY', 'YZ', 'ZX', 'ZY', 'ZZ']))
        out['readout_assignment'] = assignment
        out['scope'] = 'Same 24 archived records as the 5 Oct analysis; journal estimator, shot-only bootstrap; development instance.'
        return out
    return store.run('journal_archived_tomography', spec, compute, roles=['journal_analysis', 'journal_measurement', 'metrics'],
                     dependencies=[store.dependency(raw_record), store.dependency(problem_record), store.dependency(solver_record)],
                     calibration_ids=raw_record['manifest']['calibration_ids'])

In [ ]:
# CELL_ID: journal_optimizer
"""Matched local VQLS optimizer study: identical wrapper, initial vectors and budgets."""

def journal_matched_optimizer(A, b, *, treatment, budget, initial, reps, shots, confirmation_shots, tol, seed):
    """One shared COBYLA wrapper for the exact and counted VQLS objectives.

    Both treatments use identical initial vectors, periodic angle wrapping,
    rhobeg=0.5, tolerance and best-observed checkpoint; only the objective differs.
    Counted objectives use local StatevectorSampler shots; no hardware is involved.
    """
    from scipy.optimize import minimize
    model = _mvqls_model(A, b)
    x = normalized(np.linalg.solve(A, b))
    trace, best = [], {'cost': np.inf, 'parameters': None, 'evaluation': None}
    total_shots = 0
    def exact_cost(parameters):
        state = Statevector.from_instruction(_real_ansatz(model['num_qubits'], parameters, reps)).data
        image = np.asarray(A)@state
        return float(1-abs(np.vdot(normalized(b), image))**2/np.vdot(image, image).real), state
    def objective(parameters):
        nonlocal total_shots
        parameters = (np.asarray(parameters, dtype=float)+np.pi) % (2*np.pi)-np.pi
        evaluation = len(trace)+1
        if evaluation > budget:
            raise RuntimeError('budget exceeded')
        if treatment == 'exact_objective':
            value, _ = exact_cost(parameters)
        else:
            plan = measurement_plan(A, b, parameters, reps)
            response = local_measured_vqls_executor(plan, shots=shots, seed=seed*100003+evaluation, evaluation=evaluation)
            analysis = analyze_cost(plan, response['counts'])
            total_shots += analysis['total_shots']
            if analysis['status'] != 'ok':
                raise RuntimeError('denominator guard: '+analysis['status'])
            value = analysis['cost_unclipped']
        trace.append({'evaluation': evaluation, 'observed_objective': value})
        if value < best['cost']:
            best.update(cost=value, parameters=parameters.copy(), evaluation=evaluation)
        return value
    started = time.perf_counter(); failure = None
    try:
        minimize(objective, np.asarray(initial, dtype=float), method='COBYLA',
                 options={'maxiter': int(budget), 'tol': float(tol), 'rhobeg': 0.5})
    except RuntimeError as error:
        failure = str(error)
    final_cost, state = exact_cost(best['parameters'])
    plan = measurement_plan(A, b, best['parameters'], reps)
    confirmation = analyze_cost(plan, local_measured_vqls_executor(plan, shots=confirmation_shots,
                                                                    seed=seed*100003+999983, evaluation=0)['counts'])
    return {'treatment': treatment, 'budget': int(budget), 'seed': int(seed), 'initial_parameters': list(map(float, initial)),
            'evaluations': len(trace), 'trace': trace, 'best_observed_objective': best['cost'],
            'best_evaluation': best['evaluation'], 'best_parameters': best['parameters'].tolist(),
            'checkpoint_exact_cost': final_cost, 'checkpoint_state_fidelity': float(abs(np.vdot(x, state))**2),
            'independent_counted_confirmation_cost': confirmation['cost_unclipped'],
            'independent_confirmation_standard_error': confirmation['cost_standard_error_delta'],
            'training_shots': int(total_shots), 'confirmation_shots': int(confirmation['total_shots']),
            'measurement_settings': len(plan['settings']), 'failure': failure,
            'wall_seconds': time.perf_counter()-started}

def journal_optimizer_study(store, identifier, problem_record, ideal_id):
    rule = JOURNAL_DESIGN['vqls']['matched_optimizer_study']
    spec = {'instance': identifier, 'rule': rule}
    def compute(directory):
        A = np.asarray(problem_record['result']['A']); b = np.asarray(problem_record['result']['b'])
        n = int(round(np.log2(len(b)))); rows = []
        for seed in rule['seeds']:
            initial = np.random.default_rng(seed).uniform(-np.pi, np.pi, n*(rule['reps']+1))
            for budget in rule['budgets']:
                for treatment in rule['treatments']:
                    run = journal_matched_optimizer(A, b, treatment=treatment, budget=budget, initial=initial,
                        reps=rule['reps'], shots=rule['shots_per_setting'], confirmation_shots=rule['confirmation_shots_per_setting'],
                        tol=rule['tol'], seed=seed)
                    atomic_json(directory/f'run_{treatment}_b{budget}_s{seed}.json', run)
                    rows.append({k: v for k, v in run.items() if k not in ('trace',)})
        return {'status': 'complete', 'instance': identifier, 'runs': rows, 'hardware_execution': False,
                'scope': 'Local matched optimizer comparison; identical initial vectors and wrapper; no on-QPU training.'}
    return store.run('journal_optimizer_study', spec, compute, roles=['journal_optimizer', 'measured_cost', 'reference_solvers'],
                     dependencies=[store.dependency(problem_record)], calibration_ids=[ideal_id])

In [ ]:
# CELL_ID: journal_timing
"""Classical reference timing for instances whose problem record predates the journal cells."""

def journal_reference_timing(store, identifier, problem_record):
    if problem_record['result'].get('classical_timing'):
        return problem_record
    def compute(directory):
        A = np.asarray(problem_record['result']['A']); b = np.asarray(problem_record['result']['b'])
        return {'identifier': identifier, 'classical_timing': journal_classical_timing(A, b),
                'input_diagnostics': journal_input_diagnostics(A, b)}
    return store.run('journal_reference_timing', {'identifier': identifier}, compute,
                     roles=['journal_timing', 'journal_instances', 'metrics'], dependencies=[store.dependency(problem_record)])

In [ ]:
# CELL_ID: initialize
store=PortableStageStore(OUTPUTS,NOTEBOOK,CONFIG)
ideal_id=store.register_calibration({'kind':'ideal','noise_mode':'ideal',
    'backend_name':'local ideal sampler','device_calibration_required':False,
    'declaration':'No physical device or noise channels; all stochastic seeds saved.'})
validation=store.run('validation',{'suite':'notebook_validation_v1'},
    lambda directory:{'optimized_equivalence':verify_optimized_solvers(include_resource_counts=False),
                      'measured_cost':validate_measured_vqls_component(),
                      'cache':validate_notebook_cache(store.notebook_path)},
    roles=['cache','cache_validation','reference_solvers','optimized_solvers','measured_cost'],
    calibration_ids=[ideal_id])
print('Validation:', 'loaded existing' if validation['cache_hit'] else 'passed and saved')
historical=history_index(WORKSPACE)
atomic_json(OUTPUTS/'invocations'/store.invocation_id/'historical_results_index.json',historical)
print('Historical reference records available:',len(historical['records']))

In [ ]:
# CELL_ID: phase_validation_run
phase_validation_result=store.run('phase_validation',{'seed':20260917,'shots_per_setting':8192,'mixed_trials':160},
    lambda directory:validate_phase_sensitive_witness(seed=20260917,shots_per_setting=8192,mixed_trials=160),
    roles=['phase_validation','measured_cost'],calibration_ids=[ideal_id])
print('Phase-sensitive validation:',phase_validation_result['result']['status'],
      '(loaded)' if phase_validation_result['cache_hit'] else '(computed and saved)')


In [ ]:
# CELL_ID: local_experiments
problem=make_problem_stage(store,CONFIG['problem'])
solvers={};ideal_samples={};ideal_analyses={}
for algorithm,parameters in CONFIG['solvers'].items():
    solvers[algorithm]=make_solver_stage(store,problem,algorithm,parameters,ideal_id)
    ideal_samples[algorithm]=ideal_samples_stage(store,solvers[algorithm],ideal_id,CONFIG['ideal_samples'])
    ideal_analyses[algorithm]=analysis_stage(store,ideal_samples[algorithm],solvers[algorithm],CONFIG['analysis'])
    metric=ideal_analyses[algorithm]['result']
    print(algorithm, 'loaded' if solvers[algorithm]['cache_hit'] else 'saved',
          'accepted shots=',metric['accepted_shots'],'TV=',metric['total_variation'])
print('Every measured-VQLS evaluation has its own saved raw-count and code package.')


In [ ]:
# CELL_ID: coherent_screening_run
coherent_screening=None
if CONFIG.get('coherent_screening',{}).get('enabled',False):
    coherent_screening=coherent_screening_stages(store,problem,ideal_id)
    print('Coherent screening:',len(coherent_screening['candidates']),'candidates;',
          coherent_screening['summary']['result']['passing_candidates'])


In [ ]:
# CELL_ID: fake_experiments
fake_compilations={};fake_samples={};fake_analyses={};compiled_ideal_samples={}
if CONFIG['fake_backend']['enabled']:
    name=CONFIG['fake_backend']['class_name']
    if not name.startswith('Fake') or not hasattr(fake_provider,name):
        raise ValueError('Choose an installed qiskit_ibm_runtime.fake_provider backend class.')
    fake_backend=getattr(fake_provider,name)()
    fake_id=capture_backend(store,fake_backend,kind='fake',observation={'purpose':'packaged fake snapshot'})
    print('Archived fake calibration:',fake_id)
    for algorithm,solver in solvers.items():
        entry=dict(solver['result'],kind='inference')
        compiled=compile_for_backend(store,load_qpy(solver['directory']/'logical_measured.qpy'),[entry],
            fake_backend,fake_id,CONFIG['transpile'],dependencies=[store.dependency(solver)])
        fake_compilations[algorithm]=compiled
        # Both branches use exactly the same ISA, routing, and classical-bit ordering.
        compiled_ideal_samples[algorithm]=snapshot_samples_stage(store,compiled,fake_id,CONFIG['snapshot_simulation'],noisy=False)
        fake_samples[algorithm]=snapshot_samples_stage(store,compiled,fake_id,CONFIG['snapshot_simulation'],noisy=True)
        fake_analyses[algorithm]=analysis_stage(store,fake_samples[algorithm],solver,CONFIG['analysis'])
        print(algorithm,'fake-snapshot TV=',fake_analyses[algorithm]['result']['total_variation'])


In [ ]:
# CELL_ID: hardware_experiments
# This cell can be rerun to collect a pending job; it never duplicates that job.
hardware_result={'status':'disabled','hardware_jobs_submitted_this_call':0}
hardware_cost_plan=None
if CONFIG['hardware']['enabled']:
    chosen=['hhl','qsvt','vqls_measured']
    if any(name not in solvers for name in chosen):
        raise ValueError('The declared pilot requires hhl, qsvt, and vqls_measured.')
    circuits=[];entries=[]
    for algorithm in chosen:
        solver=solvers[algorithm]
        circuits.extend(load_qpy(solver['directory']/'logical_measured.qpy'))
        entries.append(dict(solver['result'],kind='inference'))
    measured=solvers['vqls_measured']['result']
    hardware_cost_plan=measurement_plan(np.asarray(problem['result']['A']),np.asarray(problem['result']['b']),
        measured['best_parameters'],CONFIG['solvers']['vqls_measured']['reps'])
    for setting,circuit in zip(hardware_cost_plan['settings'],hardware_cost_plan['circuits']):
        circuits.append(circuit)
        entries.append({'kind':'cost','setting':setting,'algorithm':'vqls_measured_cost_witness',
            'plan_id':hardware_cost_plan['plan_id'],'parameters':hardware_cost_plan['parameters'],
            'solver_metadata':{'system_qubits':list(range(circuit.num_qubits)),'postselect_bits':{}},
            'claims_limit':'fixed ansatz cost witness; not on-QPU optimization'})
    atomic_json(OUTPUTS/'invocations'/store.invocation_id/'hardware_measurement_plan.json',plan_record(hardware_cost_plan))
    hardware_mode=CONFIG['hardware'].get('mode','collect_only')
    if hardware_mode not in ('collect_only','submit_new'):
        raise ValueError('Hardware mode must be collect_only or submit_new.')
    hardware_configuration={**CONFIG['hardware'],'replication_id':CONFIG['replication_id'],
                            'enabled':hardware_mode=='submit_new'}
    hardware_result=hardware_pilot(store,circuits,entries,[store.dependency(solvers[n]) for n in chosen],
        hardware_configuration,CONFIG['transpile'])
    status_record=hardware_result.get('result',hardware_result)
    print('Hardware:',status_record.get('status'),status_record.get('job_id',''))
else:
    print('Hardware disabled; all results above are local simulations.')


In [ ]:
# CELL_ID: hardware_evidence
hardware_evidence=None;hardware_metadata=None
if 'result' in hardware_result and hardware_result['result'].get('status')=='completed':
    hardware_evidence=audit_saved_hardware_record(hardware_result,store)
    if not hardware_evidence['result']['all_counts_and_maps_valid']:
        raise CacheIntegrityError('Raw hardware evidence failed validation; immutable counts retained for inspection.')
    print('All submitted PUB counts and physical measurement maps validated.')
    if CONFIG['hardware'].get('archive_job_metadata',True):
        hardware_metadata=archive_saved_hardware_metadata(hardware_result,store,
            service_factory=lambda:QiskitRuntimeService(name=CONFIG['hardware']['account']))
        print('Saved job metadata:',hardware_metadata['result']['availability'])


In [ ]:
# CELL_ID: common_preparation_run
common_preparation_record=None
if 'result' in hardware_result and hardware_result['result'].get('status')=='completed':
    ref=hardware_result['result']['compilation']
    saved_compilation=store.lookup(ref['stage'],ref['key'])
    if saved_compilation is None:raise CacheIntegrityError('Saved compilation is required.')
    selected_index=next(i for i,e in enumerate(saved_compilation['result']['entries'])
                        if e.get('kind')=='inference' and e.get('algorithm')=='vqls_measured')
    common_preparation_record=build_common_preparation_measurements(store,saved_compilation,selected_index,hardware_cost_plan)
    print('Future shared-preparation circuits:',common_preparation_record['result']['status'],
          'layout=',common_preparation_record['result']['ordered_physical_layout'],
          'hardware calls=',common_preparation_record['result']['hardware_calls'])


In [ ]:
# CELL_ID: confirmation_run
confirmation_result={'status':'disabled'};confirmation_analysis_record=None;confirmation_evidence=None
confirmation_analysis_validation=store.run('confirmation_analysis_validation',{'seed':20261004},
    lambda directory:validate_confirmation_analysis(),roles=['confirmation_analysis','metrics','measured_cost'],calibration_ids=[ideal_id])
if CONFIG.get('confirmation',{}).get('prepare',False):
    confirmation_result=run_confirmation_campaign(store,problem,solvers['vqls_measured'],CONFIG['confirmation'])
print('Confirmation:',confirmation_result.get('result',confirmation_result).get('status'),
      confirmation_result.get('result',confirmation_result).get('job_id',''))
if 'result' in confirmation_result and confirmation_result['result'].get('status')=='completed':
    confirmation_evidence=audit_saved_hardware_record(confirmation_result,store)
    if not confirmation_evidence['result']['all_counts_and_maps_valid']:raise CacheIntegrityError('Confirmation raw evidence invalid.')
    ref=confirmation_result['result']['compilation'];preparation=store.lookup(ref['stage'],ref['key'])
    plan=measurement_plan(np.asarray(problem['result']['A']),np.asarray(problem['result']['b']),
        solvers['vqls_measured']['result']['best_parameters'],2)
    confirmation_analysis_record=analyze_confirmation_stage(store,confirmation_result,plan,
        preparation['result']['target_state_real'],preparation_audit=preparation['result']['preparation_audit'])
    print('Raw tomography fidelity diagnostic:',confirmation_analysis_record['result']['raw_tomography']['linear_target_fidelity_unclipped'])


In [ ]:
# CELL_ID: coherent_native_run
coherent_native_record=None
confirmation_data=confirmation_result.get('result',confirmation_result)
if coherent_screening is not None and confirmation_data.get('compile_calibration_id'):
    cal_id=confirmation_data['compile_calibration_id']
    def compile_screened(directory):
        service=QiskitRuntimeService(name=CONFIG['confirmation']['account'])
        backend=service.backend(CONFIG['confirmation']['backend'])
        observed=capture_backend(store,backend,'hardware',{'phase':'coherent_candidate_screening'})
        if observed!=cal_id:raise ValueError('Calibration changed: preserve this attempt and refresh confirmation preparation before compiling candidates.')
        circuits,entries,records=load_coherent_candidates(coherent_screening)
        compiled=compile_for_backend(store,circuits,entries,backend,cal_id,
            {'optimization_level':2,'seed_transpiler':1729,'max_two_qubit_gates':300},
            dependencies=[store.dependency(r) for r in records])
        rows=[]
        for index,e in enumerate(compiled['result']['entries']):
            gates=e['resources']['two_qubit_gates'];rate=e['success_probability']
            rows.append({'index':index,'candidate_id':e['candidate_id'],'algorithm':e['algorithm'],
                'two_qubit_gates':gates,'ideal_success_probability':rate,
                'two_qubit_gates_per_ideal_accepted_shot':gates/rate,
                'operator_relative_error':e['operator_relative_error'],'eligible':gates<=300})
        selected={}
        for algorithm in ['hhl','qsvt']:
            eligible=[r for r in rows if r['algorithm']==algorithm and r['eligible']]
            if eligible:selected[algorithm]=min(eligible,key=lambda r:(r['two_qubit_gates_per_ideal_accepted_shot'],r['two_qubit_gates'],r['candidate_id']))
        atomic_json(directory/'candidate_selection.json',{'rows':rows,'selected':selected})
        return {'status':'screened','rows':rows,'selected':selected,'compilation':store.dependency(compiled),
            'cache_dependencies':[store.dependency(compiled)],'hardware_jobs_submitted':0,
            'scope':'Predeclared local correctness and native gates divided by ideal success; heuristic resource proxy, not measured time or hardware prediction.'}
    coherent_native_record=store.run('coherent_native_screen',{'calibration_id':cal_id,'gate_cap':300},compile_screened,
        roles=['coherent_native_run','coherent_screening','backends'],
        dependencies=[store.dependency(coherent_screening['summary'])],calibration_ids=[cal_id])
    print('Native candidate selection:',coherent_native_record['result']['selected'])


In [ ]:
# CELL_ID: coherent_confirmation_run
coherent_confirmation_result={'status':'disabled'};coherent_confirmation_analysis=None
if coherent_native_record is not None and CONFIG.get('coherent_confirmation',{}).get('prepare',False):
    coherent_config=CONFIG['coherent_confirmation']
    identity={'problem':store.dependency(problem),'native_screen':store.dependency(coherent_native_record)}
    coherent_confirmation_result=load_confirmation_if_known(store,coherent_config,scientific_identity=identity,collect=coherent_config.get('collect_existing',True))
    if coherent_confirmation_result is None:
        prepared=prepare_coherent_confirmation(store,problem,coherent_native_record)
        backend=None
        if coherent_config.get('submit',False):
            service=QiskitRuntimeService(name=coherent_config['account']);backend=service.backend(coherent_config['backend'])
        coherent_confirmation_result=submit_prepared_confirmation(store,prepared,backend,coherent_config)
    data=coherent_confirmation_result.get('result',coherent_confirmation_result)
    if data.get('status')=='completed':
        coherent_confirmation_analysis=analyze_coherent_confirmation(store,coherent_confirmation_result,CONFIG['analysis'])
    print('Coherent bias confirmation:',data.get('status'),data.get('job_id',''))


In [ ]:
# CELL_ID: hardware_matched
hardware_matched={}
hardware_data=hardware_result.get('result',hardware_result)
if hardware_data.get('compilation'):
    reference=hardware_data['compilation']
    compiled=store.lookup(reference['stage'],reference['key'])
    calibration_id=hardware_data.get('calibration_availability',{}).get('job_calibration_id',hardware_data['compile_calibration_id'])
    calibration_scope=('job-start properties' if hardware_data.get('calibration_availability',{}).get('job_calibration_id')
                       else 'preparation snapshot; job-start calibration not available yet')
    physical_circuits=load_qpy(compiled['directory']/'isa_circuits.qpy')
    for index,entry in enumerate(compiled['result']['entries']):
        if entry.get('kind')!='inference':continue
        algorithm=entry['algorithm']
        def select(directory,index=index,entry=entry):
            save_qpy(directory/'isa_circuits.qpy',[physical_circuits[index]])
            return {'entries':[entry],'calibration_id':calibration_id,
                    'isa_circuits_sha256':file_hash(directory/'isa_circuits.qpy'),
                    'scope':'unchanged circuit selected from the saved physical job'}
        selected=store.run('select_physical_circuit',{'index':index},select,roles=['hardware_matched'],
            dependencies=[store.dependency(compiled)],calibration_ids=[calibration_id])
        pair={}
        for noisy,label in [(False,'compiled_ideal'),(True,'snapshot_noise')]:
            samples=snapshot_samples_stage(store,selected,calibration_id,CONFIG['snapshot_simulation'],noisy=noisy)
            analyzed=analysis_stage(store,samples,solvers[algorithm],CONFIG['analysis'])
            pair[label]={'samples':samples,'analysis':analyzed}
        hardware_matched[algorithm]={'selected':selected,'pair':pair,'calibration_scope':calibration_scope,
                                     'calibration_id':calibration_id}
        print(algorithm,'same-ISA ideal/noisy TV=',pair['compiled_ideal']['analysis']['result']['total_variation'],
              pair['snapshot_noise']['analysis']['result']['total_variation'],calibration_scope)


In [ ]:
# CELL_ID: confirmation_matched
# Each exact submitted PUB is simulated separately with an indexed random seed.
# Physical-to-compact relabeling preserves every gate and measured bit.
confirmation_matched_records=[];confirmation_matched_rows=[]
H_basis=np.array([[1,1],[1,-1]],dtype=complex)/np.sqrt(2)
BASIS_ROTATIONS={'Z':np.eye(2),'X':H_basis,'Y':H_basis@np.diag([1,-1j])}
for label,physical_result in [('vqls_tomography',confirmation_result),('coherent_bias',coherent_confirmation_result)]:
    data=physical_result.get('result',physical_result)
    if data.get('status')!='completed':continue
    ref=data['compilation'];prepared=store.lookup(ref['stage'],ref['key'])
    calibration_id=data.get('calibration_availability',{}).get('job_calibration_id',data['compile_calibration_id'])
    physical=load_qpy(prepared['directory']/'isa_circuits.qpy')
    target=normalized(np.linalg.solve(np.asarray(problem['result']['A']),np.asarray(problem['result']['b'])))
    for index,entry in enumerate(prepared['result']['entries']):
        if label=='coherent_bias':target_probs=entry['target_probabilities']
        elif entry['kind']=='tomography':
            setting=entry['setting']
            rotation=np.kron(BASIS_ROTATIONS[setting[0]],BASIS_ROTATIONS[setting[1]])
            target_probs=(abs(rotation@target)**2).tolist()
        else:target_probs=(abs(np.asarray(entry['expected_state_real']))**2).tolist()
        def select(directory,index=index,entry=entry,target_probs=target_probs):
            validate_measurement_map(physical[index],entry)
            save_qpy(directory/'isa_circuits.qpy',[physical[index]])
            return {'entries':[entry],'calibration_id':calibration_id,
                'solver_metadata':entry['solver_metadata'],'target_probabilities':target_probs,
                'isa_circuits_sha256':file_hash(directory/'isa_circuits.qpy'),
                'scope':'Exact archived submitted native circuit, relabeled only within the simulator.'}
        selected=store.run('select_confirmation_circuit',{'index':index,'protocol':label},select,
            roles=['confirmation_matched','noise_helpers'],dependencies=[store.dependency(prepared),store.dependency(problem)],
            calibration_ids=[calibration_id])
        confirmation_matched_records.append(selected)
        for noisy,treatment in [(False,'compiled_ideal'),(True,'snapshot_noise')]:
            settings={**CONFIG['snapshot_simulation'],'seed':CONFIG['snapshot_simulation']['seed']+index}
            samples=snapshot_samples_stage(store,selected,calibration_id,settings,noisy=noisy)
            analyzed=analysis_stage(store,samples,selected,CONFIG['analysis'])
            confirmation_matched_records.extend([samples,analyzed])
            confirmation_matched_rows.append({'protocol':label,'pub_index':index,'algorithm':entry['algorithm'],
                'setting':entry.get('setting'),'kind':entry['kind'],'repeat':entry.get('repeat'),
                'treatment':treatment,'calibration_id':calibration_id,'metrics':analyzed['result'],
                'sample':store.dependency(samples),'analysis':store.dependency(analyzed)})
    print('Completed unchanged-circuit simulator comparison:',label,len(physical),'PUBs')


In [ ]:
# CELL_ID: confirmation_reporting
confirmation_publication=None
if confirmation_analysis_record is not None and coherent_confirmation_analysis is not None:
    def summarize(directory):
        aggregates={}
        for row in confirmation_matched_rows:
            if row['protocol']!='coherent_bias':continue
            ref=row['sample'];sample=store.lookup(ref['stage'],ref['key'])
            key=(row['algorithm'],row['setting'],row['treatment'])
            combined=aggregates.setdefault(key,{})
            for bits,n in sample['result']['counts'].items():combined[bits]=combined.get(bits,0)+n
        ref=coherent_confirmation_result['result']['compilation'];prepared=store.lookup(ref['stage'],ref['key'])
        comparisons=[]
        for physical in coherent_confirmation_analysis['result']['rows']:
            algorithm=physical['algorithm'];setting=physical['setting']
            entry=next(e for e in prepared['result']['entries'] if e['algorithm']==algorithm and e['setting']==setting)
            meta=entry['solver_metadata'];metric=dict(physical['metrics'])
            metric.update(measurement_basis=setting,interpretation='Conditional distribution in the named measurement basis; no state-fidelity certificate.')
            if setting=='ZZ' and metric['accepted_shots']:
                accepted=metric['accepted_shots'];hist=metric['conditional_counts']
                radius=float(np.sqrt(np.log(80)/(2*accepted)))
                metric.update(p0=hist[0]/accepted,p2=hist[2]/accepted,
                    p0_wilson95=wilson_interval(hist[0],accepted),p2_wilson95=wilson_interval(hist[2],accepted),
                    two_output_95_hoeffding_radius=radius,precision_target_003_reached=radius<=.03)
            record={'algorithm':algorithm,'setting':setting,'physical':metric}
            for treatment in ['compiled_ideal','snapshot_noise']:
                metric=probability_metrics(aggregates[(algorithm,setting,treatment)],entry['target_probabilities'],
                    meta['system_qubits'],meta['postselect_bits'],**CONFIG['analysis'])
                metric['interpretation']='Conditional distribution in the named measurement basis; no state-fidelity certificate.'
                metric['measurement_basis']=setting;record[treatment]=metric
            comparisons.append(record)
        jobs=[]
        for record in [confirmation_result,coherent_confirmation_result]:
            metrics=read_json(record['directory']/'job_metrics.json')
            jobs.append({'job_id':record['result']['job_id'],'status':record['result']['status'],
                'raw_shots':sum(sum(read_json(record['directory']/row['counts_file']).values()) for row in record['result']['rows']),
                'charged_qpu_seconds':metrics.get('usage',{}).get('qpu_charge_time_seconds'),
                'timestamps':metrics.get('timestamps'),'calibration_ids':record['manifest']['calibration_ids'],
                'calibration_availability':record['result']['calibration_availability']})
        result={'jobs':jobs,'coherent_comparison':comparisons,'vqls':confirmation_analysis_record['result'],
            'scope':'One development instance and one job per protocol; technical repeats do not establish calibration independence.'}
        atomic_json(directory/'publication_summary.json',result)
        return result
    confirmation_publication=store.run('confirmation_publication',CONFIG['analysis'],summarize,
        roles=['confirmation_reporting','metrics'],
        dependencies=[store.dependency(confirmation_analysis_record),store.dependency(coherent_confirmation_analysis),
                      *[store.dependency(r) for r in confirmation_matched_records]],
        calibration_ids=sorted(set(confirmation_result['manifest']['calibration_ids']+coherent_confirmation_result['manifest']['calibration_ids'])))
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(10.6,4.2),layout='constrained')
    colors=['#4477AA','#EEAA33','#228833'];width=.24
    for axis,algorithm in zip(axes,['hhl','qsvt']):
        rows=[next(r for r in confirmation_publication['result']['coherent_comparison'] if r['algorithm']==algorithm and r['setting']==basis) for basis in ['ZZ','XX','XZ']]
        x=np.arange(3)
        for offset,(treatment,label) in enumerate([('compiled_ideal','Same circuit: ideal'),('snapshot_noise','Job-start noise model'),('physical','IBM device')]):
            values=[r[treatment]['total_variation'] for r in rows]
            axis.bar(x+(offset-1)*width,values,width,color=colors[offset],label=label)
        for i,r in enumerate(rows):
            m=r['physical'];interval=m['tv_plugin_bootstrap95']
            axis.errorbar(i+width,m['total_variation'],yerr=[[max(0,m['total_variation']-interval[0])],[max(0,interval[1]-m['total_variation'])]],fmt='none',ecolor='#222222',capsize=3)
        axis.set_xticks(x,['ZZ','XX','XZ']);axis.set_title('HHL, bounded m=4' if algorithm=='hhl' else 'QSVT, degree 5')
        axis.set_ylim(0,.65);axis.grid(axis='y',alpha=.2);axis.set_axisbelow(True)
        axis.set_ylabel('Conditional distribution TV');axis.set_xlabel('Measurement basis; 8192 submitted shots each')
    axes[0].legend(fontsize=8);fig.suptitle('Development confirmation: unchanged circuits across treatments')
    assets=WORKSPACE/'paper/assets';assets.mkdir(exist_ok=True)
    fig.savefig(assets/'confirmation_coherent_comparison.png',dpi=220)
    fig.savefig(assets/'confirmation_coherent_comparison.pdf');plt.show()


In [ ]:
# CELL_ID: report_export
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

hardware_analyses={};hardware_cost=None
if 'result' in hardware_result and hardware_result['result'].get('status')=='completed':
    if hardware_evidence is None or not hardware_evidence['result']['all_counts_and_maps_valid']:
        raise CacheIntegrityError('Hardware counts and measurement maps must pass the evidence audit before analysis.')
    def analyze_hardware(directory):
        rows={};cost_counts={}
        for index,row in enumerate(hardware_result['result']['rows']):
            entry=row['entry'];counts=read_json(hardware_result['directory']/row['counts_file'])
            if entry['kind']=='cost':cost_counts[entry['setting']]=counts
            else:
                meta=entry['solver_metadata']
                rows[entry['algorithm']]=probability_metrics(counts,entry['target_probabilities'],
                    meta['system_qubits'],meta['postselect_bits'],**CONFIG['analysis'])
        cost=analyze_cost(hardware_cost_plan,cost_counts) if cost_counts else None
        applicability=audit_phase_witness_applicability([row['entry'] for row in hardware_result['result']['rows']])
        if cost is not None:
            cost={**cost,'interpretation':'heterogeneous-layout aggregate diagnostic; not a validated single-state cost',
                  'applicability':applicability,'hardware_fidelity_lower_bound':None}
        return {'distributions':rows,'cost_witness':cost,
                'scope':'physical Z distributions and a multi-basis diagnostic; no state fidelity certification or on-device training'}
    hardware_analysis=store.run('hardware_analysis',CONFIG['analysis'],analyze_hardware,
        roles=['metrics','measured_cost','phase_validation','report_export'],
        dependencies=[store.dependency(hardware_result),store.dependency(hardware_evidence),store.dependency(phase_validation_result)],
        calibration_ids=hardware_result['manifest']['calibration_ids'])
    hardware_analyses=hardware_analysis['result']['distributions']
    hardware_cost=hardware_analysis['result']['cost_witness']

rows=[]
for name,solver in solvers.items():
    rows.append({'algorithm':name,
        'local_exact_state_residual':solver['result']['state_diagnostics']['residual_reference_free_scale'],
        'ideal_logical':ideal_analyses[name]['result'],
        'fake_snapshot':fake_analyses.get(name,{}).get('result'),
        'hardware':hardware_analyses.get(name),
        'physical_ISA_ideal':hardware_matched.get(name,{}).get('pair',{}).get('compiled_ideal',{}).get('analysis',{}).get('result'),
        'physical_ISA_snapshot_noise':hardware_matched.get(name,{}).get('pair',{}).get('snapshot_noise',{}).get('analysis',{}).get('result')})
report_dir=OUTPUTS/'invocations'/store.invocation_id/'report'
report_dir.mkdir(parents=True,exist_ok=True)
atomic_json(report_dir/'comparison.json',{'rows':rows,'hardware_cost':hardware_cost,
    'hardware_status':hardware_result.get('result',hardware_result),
    'scope':'Single small problem/pilot. No scalable advantage or general solver ranking established.'})

fig,axes=plt.subplots(1,2,figsize=(12,4.3),layout='constrained')
names=list(solvers);x=np.arange(len(names));width=.24
for shift,(key,label,color) in enumerate([('ideal_logical','Ideal logical','#4477AA'),
                                        ('fake_snapshot','Fake snapshot noise','#EEAA33'),
                                        ('hardware','Physical device','#228833')]):
    values=[row[key]['total_variation'] if row[key] and row[key]['total_variation'] is not None else np.nan for row in rows]
    rates=[row[key]['postselection_rate'] if row[key] else np.nan for row in rows]
    axes[0].bar(x+(shift-1)*width,values,width,label=label,color=color)
    axes[1].bar(x+(shift-1)*width,rates,width,label=label,color=color)
for ax in axes:
    ax.set_xticks(x,names,rotation=18,ha='right');ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
axes[0].set_ylabel('Conditional distribution total variation (lower is better)')
axes[1].set_ylabel('Accepted shots / submitted shots');axes[1].set_ylim(0,1.08)
axes[0].legend(fontsize=8);fig.suptitle(f"{CONFIG['problem']['family']}, N={CONFIG['problem']['n']}, gamma={CONFIG['problem']['gamma']}: single-run diagnostics")
fig.savefig(report_dir/'comparison.png',dpi=180);fig.savefig(report_dir/'comparison.pdf');plt.show()

# Compare exactly the submitted native circuits, with distinct sampling/noise scopes.
physical_rows=[row for row in rows if row['hardware'] is not None]
if physical_rows:
    fig,axes=plt.subplots(1,2,figsize=(10.2,4.1),layout='constrained')
    x=np.arange(len(physical_rows));width=.24
    names=[{'hhl':'HHL','qsvt':'QSVT','vqls_measured':'Frozen VQLS'}.get(r['algorithm'],r['algorithm']) for r in physical_rows]
    for shift,(key,label,color) in enumerate([('physical_ISA_ideal','Same circuit: ideal','#4477AA'),
            ('physical_ISA_snapshot_noise','Same circuit: job-start noise','#EEAA33'),
            ('hardware','IBM device','#228833')]):
        values=[r[key]['total_variation'] if r[key] else np.nan for r in physical_rows]
        rates=[r[key]['postselection_rate'] if r[key] else np.nan for r in physical_rows]
        axes[0].bar(x+(shift-1)*width,values,width,label=label,color=color)
        axes[1].bar(x+(shift-1)*width,rates,width,label=label,color=color)
    for i,row in enumerate(physical_rows):
        v=row['hardware'];ci=v.get('tv_plugin_bootstrap95')
        if ci is not None:
            axes[0].errorbar(i+width,v['total_variation'],yerr=[[max(0,v['total_variation']-ci[0])],[max(0,ci[1]-v['total_variation'])]],fmt='none',ecolor='#222222',capsize=3)
        label_height=max(.12,(ci[1] if ci else v['total_variation'])+.035)
        axes[0].text(i,label_height,str(v['accepted_shots'])+'/'+str(v['total_shots'])+' accepted',ha='center',fontsize=8)
    for ax in axes:
        ax.set_xticks(x,names);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    axes[0].set_ylabel('Conditional probability TV');axes[0].set_ylim(0,1)
    axes[1].set_ylabel('Accepted / submitted shots');axes[1].set_ylim(0,1.08)
    axes[1].legend(fontsize=8,loc='upper left')
    fig.suptitle('Single IBM pilot: distribution error and postselection')
    fig.savefig(report_dir/'physical_pilot_comparison.png',dpi=220)
    fig.savefig(report_dir/'physical_pilot_comparison.pdf');plt.show()
    atomic_json(report_dir/'physical_pilot_figure_provenance.json',{
        'hardware_job_id':hardware_result['result']['job_id'],'data':physical_rows,
        'calibration_ids':hardware_result['manifest']['calibration_ids'],
        'scope':'Unchanged submitted native circuits; 4096 shots per execution. Noise model uses job-start properties with preparation-time configuration/Target.'})

summary=store.summary()
atomic_json(OUTPUTS/'invocations'/store.invocation_id/'summary.json',summary)
text_lines=['# Notebook run '+store.invocation_id,'',
    f"Loaded stages: {summary['loaded']}; newly computed stages: {summary['computed']}.",'',
    '| Algorithm | Exact-state local residual | Ideal TV | Fake-noise TV | Hardware TV |',
    '|---|---:|---:|---:|---:|']
def number(value):return '—' if value is None else f'{value:.6g}'
for row in rows:
    values=[row['local_exact_state_residual'],row['ideal_logical']['total_variation'],
            None if not row['fake_snapshot'] else row['fake_snapshot']['total_variation'],
            None if not row['hardware'] else row['hardware']['total_variation']]
    text_lines.append('| '+row['algorithm']+' | '+' | '.join(number(v) for v in values)+' |')
text_lines+=['','Local residuals require a complete complex state; TV is a different, count-based quantity.',
    'Fake noise is a model derived from the named archived snapshot. Hardware values appear only after successful physical collection.',
    'This development pilot does not establish algorithmic advantage, drift robustness, or on-device optimization.']
if hardware_cost:
    text_lines+=['',f"Heterogeneous-layout aggregate diagnostic: {number(hardware_cost['cost_unclipped'])}; delta-method standard error {number(hardware_cost['cost_standard_error_delta'])}.",
                 'This is not a validated single-state VQLS cost or a fidelity certificate. The five settings used different ordered physical layouts.',
                 'The ratio is finite-shot biased; its delta-method uncertainty excludes preparation mismatch, calibration drift and readout bias.']
(report_dir/'RUN_REPORT.md').write_text('\n'.join(text_lines)+'\n',encoding='utf8')
display(Markdown('\n'.join(text_lines)))

# All experiments already have a code_and_outputs.zip. This final export adds
# the complete invocation, its selected dependency records, and calibration history.
if hasattr(store,'export_run'):
    used=[validation,phase_validation_result,problem,*solvers.values(),*ideal_samples.values(),*ideal_analyses.values(),
          *fake_compilations.values(),*compiled_ideal_samples.values(),*fake_samples.values(),*fake_analyses.values()]
    if 'result' in hardware_result:used.append(hardware_result)
    if hardware_analyses:used.append(hardware_analysis)
    if hardware_evidence is not None:used.append(hardware_evidence)
    if hardware_metadata is not None:used.append(hardware_metadata)
    if common_preparation_record is not None:used.append(common_preparation_record)
    used.append(confirmation_analysis_validation)
    used.extend(confirmation_matched_records)
    if confirmation_publication is not None:
        used.append(confirmation_publication)
        atomic_json(report_dir/'confirmation_publication.json',confirmation_publication['result'])
    atomic_json(report_dir/'confirmation_matched.json',confirmation_matched_rows)
    if coherent_screening is not None:used.extend([*coherent_screening['candidates'],coherent_screening['summary']])
    if coherent_native_record is not None:used.append(coherent_native_record)
    if 'result' in confirmation_result:used.append(confirmation_result)
    if confirmation_evidence is not None:used.append(confirmation_evidence)
    if confirmation_analysis_record is not None:used.append(confirmation_analysis_record)
    confirmation_status=confirmation_result.get('result',confirmation_result)
    if confirmation_status.get('compilation'):
        ref=confirmation_status['compilation'];used.append(store.lookup(ref['stage'],ref['key']))
    atomic_json(report_dir/'confirmation_status.json',confirmation_status)
    if confirmation_analysis_record is not None:atomic_json(report_dir/'confirmation_analysis.json',confirmation_analysis_record['result'])
    if 'result' in coherent_confirmation_result:used.append(coherent_confirmation_result)
    if coherent_confirmation_analysis is not None:used.append(coherent_confirmation_analysis)
    coherent_status=coherent_confirmation_result.get('result',coherent_confirmation_result)
    if coherent_status.get('compilation'):
        ref=coherent_status['compilation'];used.append(store.lookup(ref['stage'],ref['key']))
    atomic_json(report_dir/'coherent_confirmation_status.json',coherent_status)
    if coherent_confirmation_analysis is not None:atomic_json(report_dir/'coherent_confirmation_analysis.json',coherent_confirmation_analysis['result'])
    for treatment in hardware_matched.values():
        used.append(treatment['selected'])
        for pair in treatment['pair'].values():used.extend([pair['samples'],pair['analysis']])
    extras=[p for p in (OUTPUTS/'invocations'/store.invocation_id).rglob('*')
        if p.is_file() and p.suffix in ('.json','.md','.png','.pdf') and 'portable_source' not in p.parts]
    # Job registries and campaign claims travel with the cache so restoring a
    # package cannot turn an existing or uncertain submission into a new job.
    # Preserve the complete provenance of every exported registry, including old jobs.
    # A broken pointer stops export instead of creating a seemingly portable archive.
    for registry in sorted((OUTPUTS/'hardware_jobs').glob('*')):
        if not registry.is_dir():continue
        request_path=registry/'request.json'
        references=[]
        if request_path.is_file():references.extend(read_json(request_path).get('dependencies',[]))
        for pointer_name in ('completed_reference.json','hardened_collection_reference.json'):
            if (registry/pointer_name).is_file():references.append(read_json(registry/pointer_name))
        for ref in references:
            saved=store.lookup(ref['stage'],ref['key'])
            if saved is None:raise CacheIntegrityError('Exported hardware registry has a missing dependency.')
            used.append(saved)
    for folder in ['hardware_jobs','hardware_campaigns','calibration_observations','calibrations']:
        extras.extend(p for p in (OUTPUTS/folder).rglob('*') if p.is_file() and p.suffix in ('.json','.qpy'))
    package=store.export_run(used,extra_files=extras)
    print('Portable code and output package:',package)
print('Run report:',report_dir/'RUN_REPORT.md')


In [ ]:
# CELL_ID: paper_export
# Optional document rebuild; install reportlab in this kernel if needed.
def render_display_equations():
    """Render display mathematics for the manuscript PDF from its editable source."""
    from pathlib import Path
    import re
    import matplotlib
    
    import matplotlib.pyplot as plt
    
    root=WORKSPACE
    source=(root/'paper/manuscript.md').read_text(encoding='utf8')
    assets=root/'paper/assets';assets.mkdir(exist_ok=True)
    plt.rcParams.update({'mathtext.fontset':'stix','font.family':'STIXGeneral'})
    for i,equation in enumerate(re.findall(r'\$\$(.*?)\$\$',source,re.S)):
        fig=plt.figure(figsize=(8,.7))
        fig.text(.5,.5,'$'+equation.strip()+'$',fontsize=15,ha='center',va='center')
        fig.savefig(assets/f'equation_{i:02d}.png',dpi=240,bbox_inches='tight',pad_inches=.04)
        plt.close(fig)
    print('Rendered display equations.')

def render_editable_manuscript():
    render_display_equations()
    """Build an editable LaTeX companion and a typeset PDF from manuscript.md.
    
    Run this inline notebook cell after editing paper/manuscript.md.
    """
    from pathlib import Path
    import re
    from html import escape
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER,TA_JUSTIFY,TA_LEFT
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.platypus import (SimpleDocTemplate,Paragraph,Spacer,Image,Table,TableStyle,KeepTogether)
    from reportlab.lib.utils import ImageReader
    
    root=WORKSPACE
    paper=root/'paper'
    path=paper/'manuscript.md'
    source=path.read_text(encoding='utf8')
    table=(root/'results/reference_v1/analysis/representative_table.md').read_text(encoding='utf8')
    if '@@REPRESENTATIVE_TABLE@@' in source:
        source=source.replace('@@REPRESENTATIVE_TABLE@@','Table 1. Representative fixed-instance results. Residual and fidelity are dimensionless.\n\n'+table)
        path.write_text(source,encoding='utf8')
    if '@@' in source:raise ValueError('Unresolved manuscript placeholder.')
    
    styles={
     'title':ParagraphStyle('title',fontName='Times-Bold',fontSize=21,leading=25,alignment=TA_CENTER,spaceAfter=16),
     'author':ParagraphStyle('author',fontName='Times-Roman',fontSize=10.5,leading=13,alignment=TA_CENTER,spaceAfter=6),
     'body':ParagraphStyle('body',fontName='Times-Roman',fontSize=10.5,leading=14.1,alignment=TA_JUSTIFY,spaceAfter=7,allowWidows=0,allowOrphans=0),
     'h2':ParagraphStyle('h2',fontName='Times-Bold',fontSize=13,leading=16,spaceBefore=13,spaceAfter=7,keepWithNext=True),
     'h3':ParagraphStyle('h3',fontName='Times-Bold',fontSize=11,leading=14,spaceBefore=9,spaceAfter=5,keepWithNext=True),
     'caption':ParagraphStyle('caption',fontName='Times-Roman',fontSize=9,leading=11.6,alignment=TA_LEFT,spaceAfter=9),
     'cell':ParagraphStyle('cell',fontName='Times-Roman',fontSize=8.6,leading=11,alignment=TA_CENTER),
     'reference':ParagraphStyle('reference',fontName='Times-Roman',fontSize=9.3,leading=12,spaceAfter=7),
    }
    def inline(s):
        s=escape(s)
        s=re.sub(r'\*\*(.*?)\*\*',r'<b>\1</b>',s)
        s=re.sub(r'`(.*?)`',r'<font name="Courier">\1</font>',s)
        return s
    
    def texescape(s):
        replacements={'\\':r'\textbackslash{}','&':r'\&','%':r'\%','$':r'\$','#':r'\#','_':r'\_','{':r'\{','}':r'\}','~':r'\textasciitilde{}','^':r'\textasciicircum{}'}
        return ''.join(replacements.get(c,c) for c in s)
    
    parts=re.split(r'\n\s*\n',source.strip())
    story=[];tex=[];eqindex=0;front=0;references=False
    for part in parts:
        part=part.strip()
        if part.startswith('# '):
            title=part[2:];story.append(Paragraph(inline(title),styles['title']))
            tex.append(r'\title{'+texescape(title)+'}\n'+r'\date{Research revision draft, 6 October 2026}')
        elif part.startswith('### '):
            text=part[4:];story.append(Paragraph(inline(text),styles['h3']))
            tex.append(r'\subsection*{'+texescape(text)+'}')
        elif part.startswith('## '):
            text=part[3:];references=text=='References'
            story.append(Paragraph(inline(text),styles['h2']))
            tex.append(r'\section*{'+texescape(text)+'}')
        elif part.startswith('$$'):
            equation=part.strip('$').strip()
            imagepath=paper/'assets'/f'equation_{eqindex:02d}.png'
            width,height=ImageReader(str(imagepath)).getSize()
            target_width=min(450,width/240*72)
            story.append(Image(str(imagepath),width=target_width,height=height/width*target_width,hAlign='CENTER'))
            story.append(Spacer(1,7));eqindex+=1
            tex.append('\\[\n'+equation+'\n\\]')
        elif part.startswith('|'):
            rows=[ [v.strip() for v in line.strip().strip('|').split('|')] for line in part.splitlines() if line.strip().startswith('|')]
            rows=[r for r in rows if not all(re.match(r'^:?-+:?$',v) for v in r)]
            data=[[Paragraph(inline(v),styles['cell']) for v in row] for row in rows]
            t=Table(data,colWidths=([29,40,40,55,102,102,102] if rows[0][0]=='N' else [470/len(rows[0])]*len(rows[0])),repeatRows=1,hAlign='CENTER')
            t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#e9edf1')),
                ('GRID',(0,0),(-1,-1),.35,colors.HexColor('#c8cdd1')),('VALIGN',(0,0),(-1,-1),'MIDDLE'),
                ('TOPPADDING',(0,0),(-1,-1),4),('BOTTOMPADDING',(0,0),(-1,-1),4)]))
            story.append(t);story.append(Spacer(1,8))
            texrows=[' & '.join(texescape(v) for v in row)+r' \\' for row in rows]
            tex.append('\\begin{center}\n\\small\n\\begin{tabularx}{\\textwidth}{'+('X'*len(rows[0]))+'}\n\\hline\n'+texrows[0]+'\n\\hline\n'+'\n'.join(texrows[1:])+'\n\\hline\n\\end{tabularx}\n\\end{center}')
        elif part.startswith('!['):
            m=re.fullmatch(r'!\[(.*)\]\((.*)\)',part,re.S)
            caption,rel=m.groups();imagepath=(paper/rel).resolve()
            width,height=ImageReader(str(imagepath)).getSize()
            im=Image(str(imagepath),width=475,height=height/width*475)
            story.append(KeepTogether([im,Spacer(1,4),Paragraph(inline(caption),styles['caption'])]))
            tex.append('\\begin{center}\n\\includegraphics[width=\\textwidth]{'+rel.replace('\\','/')+'}\n\n{\\small '+texescape(caption)+'}\n\\end{center}')
        elif front<2:
            story.append(Paragraph(inline(part),styles['author']));front+=1
            if front==1:tex.append(r'\author{'+texescape(part)+r'\\Rensselaer Polytechnic Institute}'+'\n'+r'\maketitle')
        else:
            style=styles['reference'] if references else styles['body']
            if re.match(r'^Table [0-9]+\.',part):
                style=ParagraphStyle('tablecaption',parent=styles['caption'],keepWithNext=not part.startswith('Table 1.'))
            story.append(Paragraph(inline(part),style))
            if references:
                m=re.search(r'https://\S+',part)
                if m:
                    tex.append(texescape(part[:m.start()])+r'\url{'+m.group(0)+'}')
                else:tex.append(texescape(part))
            else:tex.append(texescape(part))
    
    out=root/'output/pdf';out.mkdir(parents=True,exist_ok=True)
    output=out/'verified_quantum_linear_solvers_draft.pdf'
    def footer(canvas,doc):
        canvas.saveState();canvas.setFont('Times-Roman',8)
        canvas.setFillColor(colors.HexColor('#555555'))
        canvas.drawString(54,29,'Research revision draft - development evidence; independent confirmation pending')
        canvas.drawRightString(558,29,str(doc.page));canvas.restoreState()
    doc=SimpleDocTemplate(str(output),pagesize=(612,792),leftMargin=54,rightMargin=54,topMargin=48,bottomMargin=47,
        title='Verified quantum linear solvers for graph diffusion',author='Yalcin and coauthors',allowSplitting=True)
    doc.build(story,onFirstPage=footer,onLaterPages=footer)
    preamble='''\\documentclass[11pt]{article}
    \\usepackage[T1]{fontenc}
    \\usepackage[utf8]{inputenc}
    \\usepackage{amsmath,amssymb,graphicx,geometry,url,tabularx}
    \\geometry{margin=1in}
    \\setlength{\\parskip}{0.4em}
    \\setlength{\\parindent}{0pt}
    \\begin{document}
    '''
    (paper/'manuscript.tex').write_text(preamble+'\n\n'.join(tex)+'\n\\end{document}\n',encoding='utf8')
    print(output)

if CONFIG.get('render_manuscript',False):
    render_editable_manuscript()
else:
    print('Paper export available via render_editable_manuscript(); the current PDF is preserved.')


## Journal confirmation study: runs

`journal_local_run` builds the instances, coherent screens, VQLS checkpoints and the matched optimizer study. `journal_retrospective_run` tests the calibration models on all 44 archived physical PUBs. `journal_planning_run` compiles every planned circuit for the archived Nighthawk calibration and two packaged Heron snapshots, applies the frozen selection and writes pre-registered predictions. `journal_hardware_run` is off by default: set `CONFIG['journal']['hardware']['enabled']=True` to collect existing jobs and additionally `submit=True` to submit the named block once; each submitted job gets a prediction from its compile calibration before any of its results is read, and each collected job a second one from its job-start calibration. `journal_report` writes the manuscript tables and figures, including the hardware cells, prediction endpoints and block reproducibility once jobs are analyzed; `journal_export` writes one portable package with every stage used; `journal_paper` regenerates the manuscript numbers and tables and typesets the PDF when LaTeX is available.

In [ ]:
# CELL_ID: journal_local_run
# Local journal stages: instances, coherent screening, VQLS checkpoints and the
# matched optimizer study. No network access; every stage reloads when unchanged.
journal = {'problems': {}, 'screening': {}, 'vqls': {}, 'inventory': {}, 'optimizer': {}, 'input_fidelity': {}}
journal_settings_config = CONFIG.get('journal', {})
if journal_settings_config.get('enabled', False):
    for identifier, spec in JOURNAL_DESIGN['instances'].items():
        journal['problems'][identifier] = make_journal_problem_stage(store, identifier, spec, development_problem=problem)
        data = journal['problems'][identifier]['result']
        journal['input_fidelity'][identifier] = journal_input_diagnostics(data['A'], data['b'])['input_state_fidelity_F_in']
    for identifier in JOURNAL_DESIGN['coherent']['instances']:
        journal['screening'][identifier] = journal_coherent_screen(store, journal['problems'][identifier], ideal_id)
    for identifier in [*JOURNAL_DESIGN['vqls']['two_qubit']['instances'], *JOURNAL_DESIGN['vqls']['larger']['instances']]:
        journal['vqls'][identifier] = journal_vqls(store, identifier, journal['problems'][identifier], ideal_id,
                                                   development_solver=solvers.get('vqls_measured'))
    for identifier, problem_record in journal['problems'].items():
        journal['inventory'][identifier] = journal_logical_inventory(store, identifier, problem_record,
            journal['screening'].get(identifier), journal['vqls'][identifier])
    if journal_settings_config.get('optimizer_study', {}).get('enabled', True):
        for identifier in JOURNAL_DESIGN['vqls']['matched_optimizer_study']['instances']:
            journal['optimizer'][identifier] = journal_optimizer_study(store, identifier, journal['problems'][identifier], ideal_id)
    print(f"{'instance':8} {'N':>3} {'kappa':>6} {'F_in':>6}  passing coherent candidates (operator and residual <= 0.03)")
    for identifier, record in journal['problems'].items():
        meta = record['result']['metadata']; passing = []
        if identifier in journal['screening']:
            passing = journal['screening'][identifier]['summary']['result']['passing_candidates']
        kappa = meta.get('condition_number_2', meta.get('condition_number'))
        print(f"{identifier:8} {len(record['result']['b']):3d} {kappa:6.2f} {journal['input_fidelity'][identifier]:6.3f}  {', '.join(passing) or '-'}")
    for identifier, record in journal['vqls'].items():
        diagnostics = record['result']['state_diagnostics']
        print(f"VQLS {identifier}: {record['result']['algorithm']}, local exact-state fidelity {diagnostics['state_fidelity_diagnostic']:.5f}")
else:
    print('Journal stages disabled in CONFIG.')

In [ ]:
# CELL_ID: journal_retrospective_run
# Retrospective test of the calibration models on all 44 archived physical PUBs.
# Uses each job's own job-start BackendProperties; no network access.
journal['retrospective'] = {}
if journal_settings_config.get('enabled', False) and journal_settings_config.get('retrospective', {}).get('enabled', True):
    development_target = normalized(np.linalg.solve(np.asarray(problem['result']['A']), np.asarray(problem['result']['b']))).real
    for label, record in [('pilot_2026_09_14', hardware_result), ('vqls_tomography_2026_10_05', confirmation_result),
                          ('coherent_bias_2026_10_05', coherent_confirmation_result)]:
        data = record.get('result', record)
        if data.get('status') != 'completed':
            continue
        calibration_id = data.get('calibration_availability', {}).get('job_calibration_id')
        if calibration_id is None:
            print(label, 'has no job-start calibration; retrospective prediction withheld.')
            continue
        journal['retrospective'][label] = journal_retrospective(store, record, calibration_id, label, target_state=development_target)
    if confirmation_result.get('result', {}).get('status') == 'completed':
        journal['archived_tomography'] = journal_archived_vqls_tomography(store, confirmation_result, problem,
                                                                          solvers['vqls_measured'], journal_settings_config['analysis'])
        archived = journal['archived_tomography']['result']
        print(f"D0 frozen VQLS (5 Oct): F(x_hat)={archived['solution']['fidelity_raw']:.5f} "
              f"{np.round(archived['solution']['fidelity_raw_bootstrap95'], 5).tolist()}, F(b_hat)={archived['source']['fidelity_raw']:.5f}, "
              f"F_in={archived['F_in_exact']:.5f}, gain={archived['inversion_gain_raw']:.3f} "
              f"{np.round(archived['inversion_gain_raw_bootstrap95'], 3).tolist()}; ideal checkpoint F={archived['checkpoint_solution_fidelity_exact']:.5f} "
              f"(gain {archived['checkpoint_inversion_gain_exact']:.3f}); hardware vs checkpoint F={archived['checkpoint']['fidelity_raw']:.5f}")
    for label, record in journal['retrospective'].items():
        rows = record['result']['rows']
        for model in JOURNAL_NOISE_MODELS[1:]:
            values = [r['models'][model].get('tv_predicted_vs_observed') for r in rows if r['models'].get(model, {}).get('status') == 'predicted']
            values = [v for v in values if v is not None]
            if values:
                print(f'{label:28} {model:24} median TV(predicted, observed) = {np.median(values):.4f} over {len(values)} PUBs')

In [ ]:
# CELL_ID: journal_planning_run
# Pre-registration: compile every instance for each planning target, apply the frozen
# selection, assemble the block jobs, estimate QPU time and predict every estimator.
# Planning targets are archived/packaged snapshots; nothing here contacts IBM.
journal['planning'] = {}
journal_accounts = {JOURNAL_DESIGN['devices']['primary']['backend']: JOURNAL_DESIGN['devices']['primary']['account'],
                    **{d['backend']: d['account'] for d in JOURNAL_DESIGN['devices']['transfer']}}

def journal_jobs_for_device(backend_name, label, target, payload, calibration_id, block, compiled):
    jobs = {}
    mitigation = JOURNAL_DESIGN['jobs']['mitigation_arm']
    for group, members in JOURNAL_DESIGN['jobs']['groups'].items():
        arms = ['raw']
        if backend_name == JOURNAL_DESIGN['devices']['primary']['backend'] and block in mitigation['blocks'] \
                and set(members) & set(mitigation['instances']):
            arms.append('mitigation')
        for arm in arms:
            chosen = members if arm == 'raw' else [m for m in members if m in mitigation['instances']]
            records = [{'identifier': m, 'inventory': journal['inventory'][m], 'compiled': compiled[m]} for m in chosen]
            jobs[(block, group, arm)] = journal_prepare_job(store, device_label=label, block=block, group=group, arm_name=arm,
                instance_records=records, target=target, payload=payload, calibration_id=calibration_id,
                account=journal_accounts[backend_name], backend_name=backend_name,
                dependencies=[store.dependency(compiled[m]) for m in chosen])
    return jobs

def journal_compile_all(label, target, calibration_id):
    return {identifier: journal_compile(store, label, [i['circuit'] for i in inventory['items']],
                journal_compile_entries(inventory), target, calibration_id,
                dependencies=[i['record'] for i in inventory['items']])
            for identifier, inventory in journal['inventory'].items()}

planning_config = journal_settings_config.get('planning', {})
# The pre-registration target is frozen: the archived live Nighthawk snapshot that was the latest one on
# 11 Oct 2026. Later live captures (e.g. the 13 Oct B1 compile calibration) never move the planning baseline.
JOURNAL_PLANNING_CALIBRATIONS = {'ibm_rensselaer': '48099db4126fecc825b4ce9079d4fe3e871df934af418cfff8363167a561541f'}
if journal_settings_config.get('enabled', False) and planning_config.get('enabled', True):
    planning_devices = journal_planning_devices(store)
    for backend_name, identifier in planning_config.get('calibration_ids', JOURNAL_PLANNING_CALIBRATIONS).items():
        frozen_payload = store.calibration(identifier)['payload']
        planning_devices[backend_name] = {'calibration_id': identifier, 'target': journal_target_from_snapshot(frozen_payload['target']),
                                          'source': 'latest archived live snapshot at pre-registration (frozen)',
                                          'calibration_last_update': str(frozen_payload['calibration_last_update'])}
    for backend_name in planning_config.get('devices', []):
        device = planning_devices[backend_name]; label = backend_name+'@planning'
        payload = store.calibration(device['calibration_id'])['payload']
        compiled = journal_compile_all(label, device['target'], device['calibration_id'])
        jobs = {}
        for block in planning_config.get('blocks', ['B1']):
            for key, prepared in journal_jobs_for_device(backend_name, label, device['target'], payload,
                                                         device['calibration_id'], block, compiled).items():
                jobs[key] = {'prepared': prepared,
                             'prediction': journal_predict_job(store, prepared, device['calibration_id'], 'pre_registration')}
        journal['planning'][backend_name] = {'source': device['source'], 'calibration_id': device['calibration_id'],
            'calibration_last_update': device['calibration_last_update'], 'compiled': compiled, 'jobs': jobs}
    for backend_name, plan in journal['planning'].items():
        total = sum(job['prepared']['result']['estimated_qpu_seconds'] for job in plan['jobs'].values())
        circuits = sum(job['prepared']['result']['circuit_count'] for job in plan['jobs'].values())
        print(f"{backend_name} ({plan['source']}, calibration {plan['calibration_last_update']}): "
              f"{len(plan['jobs'])} jobs, {circuits} circuits, estimated {total:.0f} QPU s per block")
        for (block, group, arm), job in sorted(plan['jobs'].items()):
            if arm != 'raw':
                continue
            for identifier, selection in job['prepared']['result']['selections'].items():
                text = ', '.join(f"{m}: {s['arm']}" + (f" {s['candidate_id']} ({s['resources']['two_qubit_gates']} CZ)" if 'candidate_id' in s else '')
                                 for m, s in selection.items())
                print(f'   {identifier}: {text}')

In [ ]:
# CELL_ID: journal_hardware_run
# Physical blocks, fully automatic: Run All walks the frozen block order on each device
# (B1 -> B2 -> B3 on ibm_rensselaer; T1 on the Heron devices). For every block:
#   * jobs that already exist are read from the saved records: no device access once they are
#     complete; queued jobs are only polled read-only;
#   * the first block with missing jobs is compiled against the live calibration and each missing job
#     is submitted at most once, and only when the frozen separation rule holds: every earlier block
#     has finished and the device has been recalibrated since every calibration an earlier block was
#     compiled or ran under (design: "at least one provider calibration update between consecutive
#     blocks"). Otherwise the cell records why the block waits and continues with the saved results.
# Nothing is edited between blocks: the next Run All continues where this one stopped. Predictions from
# the compile calibration are saved before any result of a job is read, every device contact archives
# a calibration snapshot, and every decision (with calibration timestamps and job ids) is written to
# outputs/invocations/<run>/journal_blocks.json, packaged with the code by journal_export.
# Transfer block T1 follows the same rule but submits only with hardware['transfer_submit']=True.
journal['hardware'] = {}
hardware_config = journal_settings_config.get('hardware', {})
JOURNAL_FINAL_JOB_STATUSES = ('completed', 'hardware_job_failed')

def journal_registry_for(backend_name, block, group, arm):
    identity = {'journal_job_v1': {'design_id': JOURNAL_DESIGN['design_id'], 'account': journal_accounts[backend_name],
                                   'backend': backend_name, 'block': block, 'group': group, 'arm': arm}}
    return store.root/'hardware_jobs'/content_hash(identity)

def journal_expected_jobs(backend_name, block):
    mitigation = JOURNAL_DESIGN['jobs']['mitigation_arm']; keys = []
    for group, members in JOURNAL_DESIGN['jobs']['groups'].items():
        keys.append((block, group, 'raw'))
        if backend_name == JOURNAL_DESIGN['devices']['primary']['backend'] and block in mitigation['blocks'] \
                and set(members) & set(mitigation['instances']):
            keys.append((block, group, 'mitigation'))
    return keys

def journal_registered_seconds(accounts=None):
    """Estimated QPU seconds of every journal job already registered (submitted or uncertain)."""
    total = 0.0
    for path in sorted((store.root/'hardware_jobs').glob('*/request.json')):
        identity = read_json(path).get('journal_identity', {}).get('journal_job_v1', {})
        if identity.get('design_id') == JOURNAL_DESIGN['design_id'] and (accounts is None or identity.get('account') in accounts):
            total += float(read_json(path)['timing_estimate']['estimated_qpu_seconds'])
    return total

def journal_results_read(registry):
    return any((registry/name).is_file() for name in ('hardened_collection_reference.json', 'completed_reference.json'))

def journal_pre_collection_prediction(prepared, registry):
    """Compile-calibration prediction made before the job's results are read (None if too late)."""
    for name in ('pre_collection_prediction.json', 'pre_submission_prediction.json'):
        if (registry/name).is_file():
            reference = read_json(registry/name)
            return store.lookup(reference['stage'], reference['key'])
    if journal_results_read(registry):
        return None  # a prediction made after the counts were saved would not be pre-observation
    prediction = journal_predict_job(store, prepared, prepared['result']['calibration_id'], 'before_collection')
    atomic_json(registry/'pre_collection_prediction.json',
                {**store.dependency(prediction), 'made_utc': utc_now(), 'job_results_read_before_prediction': False})
    return prediction

def journal_blocks_for(backend_name):
    primary = JOURNAL_DESIGN['devices']['primary']
    if backend_name == primary['backend']:
        return list(primary['blocks'])
    return list(next(d['blocks'] for d in JOURNAL_DESIGN['devices']['transfer'] if d['backend'] == backend_name))

def journal_record_status(record):
    if not isinstance(record, dict):
        return 'missing'
    data = record['result'] if isinstance(record.get('result'), dict) else record
    return str(data.get('status', 'unknown'))

def journal_job_id(item):
    record = item['record']
    data = record['result'] if isinstance(record, dict) and isinstance(record.get('result'), dict) else (record or {})
    if data.get('job_id'):
        return data['job_id']
    registry = journal_registry_for(item['backend'], *item['key'])
    return read_json(registry/'job.json')['job_id'] if (registry/'job.json').is_file() else None

class JournalDeviceUnavailable(RuntimeError):
    pass

def journal_connection(backend_name, device_log):
    """Lazy read/submit connection through the saved account alias; archives one calibration on first contact."""
    cache = {}
    def connect():
        if 'error' in cache:
            raise JournalDeviceUnavailable(cache['error'])
        if 'service' not in cache:
            try:
                service = QiskitRuntimeService(name=journal_accounts[backend_name])
                backend = service.backend(backend_name)
                contact = capture_backend(store, backend, 'hardware', {'phase': 'journal_automatic_run_first_contact'})
            except Exception as error:
                cache['error'] = f'{type(error).__name__}: {error}'
                device_log['device_contact'] = {'status': 'unavailable', 'error': cache['error'], 'utc': utc_now()}
                raise JournalDeviceUnavailable(cache['error']) from error
            cache.update(service=service, backend=backend)
            stamp = str(store.calibration(contact)['payload']['calibration_last_update'])
            device_log['device_contact'] = {'status': 'connected', 'utc': utc_now(), 'calibration_id': contact, 'calibration_last_update': stamp}
            device_log['calibrations_archived'].append({'calibration_id': contact, 'calibration_last_update': stamp, 'phase': 'first_contact'})
        return cache['service'], cache['backend']
    return connect

def journal_read_job(item, connect):
    """Saved record, or a read-only status/collection through the service; never submits."""
    try:
        return journal_known_job(store, item['prepared'], collect=hardware_config.get('collect_existing', True), service_factory=connect)
    except CacheIntegrityError:
        raise
    except Exception as error:  # network or service errors leave the job for the next run
        return {'status': 'read_failed', 'error': f'{type(error).__name__}: {error}', 'submitted_new_job': False}

def journal_load_block(backend_name, block, connect):
    state = {}
    for key in journal_expected_jobs(backend_name, block):
        registry = journal_registry_for(backend_name, *key)
        if not (registry/'request.json').is_file():
            continue
        request = read_json(registry/'request.json')
        prepared = store.lookup(request['compilation']['stage'], request['compilation']['key'])
        if prepared is None:
            raise CacheIntegrityError(f'{key}: the prepared job behind a registered request is missing; nothing resubmitted.')
        journal_pre_collection_prediction(prepared, registry)
        item = {'prepared': prepared, 'request': request, 'backend': backend_name, 'key': key}
        item['record'] = journal_read_job(item, connect)
        state[key] = item
    return state

def journal_block_stamps(state):
    """Provider calibration timestamps the block's jobs were compiled, submitted or started under."""
    stamps = {}
    for item in state.values():
        ids = [item['request'].get('compile_calibration_id'), item['request'].get('before_submission_calibration_id')]
        record = item['record']
        if isinstance(record, dict) and isinstance(record.get('result'), dict):
            ids.append((record['result'].get('calibration_availability') or {}).get('job_calibration_id'))
        for identifier in ids:
            if identifier:
                stamps[identifier] = str(store.calibration(identifier)['payload']['calibration_last_update'])
    return stamps

def journal_submit_block(backend_name, block, missing, state, earlier_stamps, connect, block_log, device_log):
    """Compile under the frozen rule and submit each missing job once; returns the submitted keys."""
    service, backend = connect()
    if not backend.status().operational:
        block_log.update(decision='waiting_device_not_operational', reason=f'{backend_name} reports not operational; nothing submitted.')
        return []
    calibration_id = capture_backend(store, backend, 'hardware', {'phase': 'journal_before_compilation', 'block': block})
    payload = store.calibration(calibration_id)['payload']
    stamp = str(payload['calibration_last_update'])
    device_log['calibrations_archived'].append({'calibration_id': calibration_id, 'calibration_last_update': stamp,
                                                'phase': 'before_compilation', 'block': block})
    block_log.update(live_calibration_id=calibration_id, live_calibration_last_update=stamp,
                     earlier_block_calibrations=sorted(set(earlier_stamps.values())))
    if stamp in set(earlier_stamps.values()):
        if not hardware_config.get('allow_time_block', False):
            block_log.update(decision='waiting_for_recalibration',
                             reason=f'{backend_name} is still on calibration {stamp}, which an earlier block used; '
                                    'the frozen design needs a provider calibration update between blocks. Nothing submitted; '
                                    'Run All again after the next calibration, nothing to change.')
            return []
        block_log['time_block'] = True
    label = backend_name+'@'+block
    compiled = journal_compile_all(label, backend.target, calibration_id)
    prepared_jobs = journal_jobs_for_device(backend_name, label, backend.target, payload, calibration_id, block, compiled)
    total = sum(prepared_jobs[key]['result']['estimated_qpu_seconds'] for key in missing)
    ceilings = JOURNAL_DESIGN['ceilings']
    premium = {d['account'] for d in JOURNAL_DESIGN['devices']['transfer']}
    problems = []
    if total > ceilings['max_estimated_qpu_seconds_per_block']:
        problems.append(f'block estimate {total:.0f} s exceeds the per-block ceiling')
    if journal_registered_seconds()+total > ceilings['max_total_estimated_qpu_seconds']:
        problems.append(f'campaign estimate would exceed {ceilings["max_total_estimated_qpu_seconds"]} s')
    if journal_accounts[backend_name] in premium and journal_registered_seconds(premium)+total > ceilings['max_premium_estimated_qpu_seconds']:
        problems.append(f'premium estimate would exceed {ceilings["max_premium_estimated_qpu_seconds"]} s')
    block_log['estimated_qpu_seconds_missing_jobs'] = total
    if problems:
        block_log.update(decision='blocked_by_ceiling', reason='; '.join(problems)+'. Nothing submitted.')
        return []
    submitted = []
    for key in missing:
        prepared = prepared_jobs[key]
        outcome = journal_submit_job(store, prepared, backend, {'approval_note': hardware_config.get('approval_note')})
        block_log['submissions'].append({'job': list(key), **{k: outcome.get(k) for k in
                                         ('status', 'job_id', 'estimated_qpu_seconds', 'max_execution_time_seconds', 'request_id',
                                          'new_calibration_id', 'compile_calibration_id')}, 'utc': utc_now()})
        print(f'   {backend_name} {key}: {outcome["status"]} {outcome.get("job_id", "")}', flush=True)
        if outcome['status'] != 'submitted':
            break  # calibration changed: the remaining jobs are recompiled under the same frozen rule
        registry = journal_registry_for(backend_name, *key)
        state[key] = {'prepared': prepared, 'request': read_json(registry/'request.json'), 'backend': backend_name,
                      'key': key, 'record': outcome}
        submitted.append(key)
    # Predictions from the compile calibration, saved before any of these results is read.
    for key in submitted:
        journal_pre_collection_prediction(state[key]['prepared'], journal_registry_for(backend_name, *key))
    return submitted

def journal_wait_for_jobs(backend_name, block, state, connect, minutes, block_log):
    """Read this block's unfinished jobs now, then poll them read-only for up to `minutes`.

    Jobs still queued afterwards stay registered; a later Run All collects them without resubmitting."""
    deadline = time.monotonic()+60*max(0.0, minutes)
    announced, first = None, True
    while True:
        waiting = [key for key, item in state.items() if journal_record_status(item['record']) in ('pending', 'submitted')]
        if not waiting:
            break
        if not first:
            if time.monotonic() >= deadline:
                break
            if announced is None or time.monotonic()-announced >= 300:
                print(f'   {backend_name} {block}: {len(waiting)} job(s) queued or running; checking every minute for up to '
                      f'{minutes:g} min (a later Run All collects them otherwise).', flush=True)
                announced = time.monotonic()
            time.sleep(60)
        first = False
        for key in waiting:
            state[key]['record'] = journal_read_job(state[key], connect)
    block_log['wait_minutes_limit'] = minutes

def journal_analyze_block(backend_name, state):
    for key, item in state.items():
        record = item['record']
        if journal_record_status(record) != 'completed' or 'result' not in record:
            continue
        registry = journal_registry_for(backend_name, *key)
        predictions = []
        for name in ('pre_collection_prediction.json', 'pre_submission_prediction.json'):
            if (registry/name).is_file():
                reference = read_json(registry/name); found = store.lookup(reference['stage'], reference['key'])
                if found is not None:
                    predictions.append(found)
                break
        job_calibration = (record['result'].get('calibration_availability') or {}).get('job_calibration_id')
        if job_calibration:
            predictions.append(journal_predict_job(store, item['prepared'], job_calibration, 'job_start'))
        item['analysis'] = journal_analyze_job(store, item['prepared'], record, predictions,
                                               journal['input_fidelity'], journal_settings_config['analysis'])

def journal_run_device(backend_name, may_submit, device_log):
    connect = journal_connection(backend_name, device_log)
    earlier_finished, earlier_stamps, submitted_here = True, {}, False
    for block in journal_blocks_for(backend_name):
        expected = journal_expected_jobs(backend_name, block)
        block_log = {'expected_jobs': len(expected), 'submissions': [], 'checked_utc': utc_now()}
        device_log['blocks'][block] = block_log
        state = journal_load_block(backend_name, block, connect)
        block_log['registered_before_run'] = len(state)
        missing = [key for key in expected if key not in state]
        if missing:
            if not may_submit:
                block_log.update(decision='not_submitted_by_configuration',
                                 reason='Submission for this device is off (hardware.transfer_submit=False for T1).')
            elif submitted_here or not earlier_finished:
                block_log.update(decision='waiting_for_previous_block',
                                 reason='Every earlier block must finish first, then the device must be recalibrated.')
            else:
                try:
                    journal_submit_block(backend_name, block, missing, state, earlier_stamps, connect, block_log, device_log)
                    last = block_log['submissions'][-1]['status'] if block_log['submissions'] else None
                    if last == 'calibration_changed_before_submission':
                        # The provider recalibrated between compilation and submission: one recompilation, same rule.
                        journal_submit_block(backend_name, block, [k for k in expected if k not in state],
                                             state, earlier_stamps, connect, block_log, device_log)
                except CacheIntegrityError:
                    raise
                except JournalDeviceUnavailable as error:
                    block_log.update(decision='device_unreachable', reason=str(error)+'. Nothing submitted; the next Run All retries.')
                except Exception as error:  # live checks refused or the service failed; any job already sent stays registered
                    block_log.update(decision='submission_stopped', reason=f'{type(error).__name__}: {error}. '
                                     'No further job submitted; the next Run All re-checks from the saved registry.')
                if any(s['status'] == 'submitted' for s in block_log['submissions']):
                    submitted_here = True
                    block_log.setdefault('decision', 'submitted')
        if state and any(journal_record_status(i['record']) in ('pending', 'submitted') for i in state.values()):
            journal_wait_for_jobs(backend_name, block, state, connect, float(hardware_config.get('wait_minutes', 30)), block_log)
        # saved completions load with cache_hit=True; a collection made in this run is a new stage (cache_hit=False)
        block_log['collected_this_run'] = sum(isinstance(i['record'], dict) and i['record'].get('cache_hit') is False
                                              and journal_record_status(i['record']) in JOURNAL_FINAL_JOB_STATUSES for i in state.values())
        journal_analyze_block(backend_name, state)
        statuses = {'/'.join(key): journal_record_status(item['record']) for key, item in sorted(state.items())}
        block_log['jobs'] = [{'job': list(key), 'status': journal_record_status(item['record']), 'job_id': journal_job_id(item),
                              'compile_calibration_id': item['request'].get('compile_calibration_id'),
                              'analyzed': 'analysis' in item} for key, item in sorted(state.items())]
        finished = bool(state) and len(state) == len(expected) and all(s in JOURNAL_FINAL_JOB_STATUSES for s in statuses.values())
        if 'decision' not in block_log:
            if finished:
                block_log['decision'] = 'complete'
            elif any(s in ('submission_uncertain', 'claim_requires_reconciliation') for s in statuses.values()):
                block_log.update(decision='needs_reconciliation', reason='A submission without a recorded job id exists; '
                                 'check the IBM job list for its tag. No automatic retry.')
            elif any(s == 'read_failed' for s in statuses.values()):
                error = next(item['record'].get('error') for item in state.values() if journal_record_status(item['record']) == 'read_failed')
                block_log.update(decision='device_unreachable', reason=f'Registered jobs could not be read ({error}); '
                                 'nothing submitted, the next Run All reads them again.')
            elif state:
                block_log['decision'] = 'running'
        elif block_log['decision'] == 'submitted' and finished:
            block_log['decision'] = 'submitted_and_complete'
        if any(s == 'read_failed' for s in statuses.values()):
            block_log['read_failures'] = [item['record'].get('error') for item in state.values() if journal_record_status(item['record']) == 'read_failed']
        block_log['calibration_stamps'] = journal_block_stamps(state)
        if state:
            journal['hardware'][(backend_name, block)] = state
        earlier_stamps.update(block_log['calibration_stamps'])
        earlier_finished = earlier_finished and finished

journal_block_log = {'invocation_id': store.invocation_id, 'started_utc': utc_now(), 'design_id': JOURNAL_DESIGN['design_id'],
                     'rule': JOURNAL_DESIGN['devices']['primary']['separation'], 'configuration': copy.deepcopy(hardware_config), 'devices': {}}
if journal_settings_config.get('enabled', False) and hardware_config.get('enabled', False):
    primary_backend = JOURNAL_DESIGN['devices']['primary']['backend']
    devices = hardware_config.get('devices', [primary_backend]+[d['backend'] for d in JOURNAL_DESIGN['devices']['transfer']])
    for backend_name in devices:
        device_log = {'account': journal_accounts[backend_name], 'blocks': {}, 'calibrations_archived': []}
        journal_block_log['devices'][backend_name] = device_log
        may_submit = hardware_config.get('submit', True) and (backend_name == primary_backend or hardware_config.get('transfer_submit', False))
        journal_run_device(backend_name, may_submit, device_log)
    journal_block_log['finished_utc'] = utc_now()
    log_path = OUTPUTS/'invocations'/store.invocation_id/'journal_blocks.json'
    atomic_json(log_path, journal_block_log)
    atomic_json(OUTPUTS/'journal_block_status.json', journal_block_log)
    labels = {'complete': 'complete (saved results)', 'submitted_and_complete': 'submitted and completed in this run',
              'submitted': 'submitted in this run, results pending', 'running': 'queued or running',
              'waiting_for_recalibration': 'waiting for a provider recalibration', 'waiting_for_previous_block': 'waiting for the previous block',
              'not_submitted_by_configuration': 'not submitted (switched off)', 'device_unreachable': 'device not reachable',
              'blocked_by_ceiling': 'blocked by a budget ceiling', 'needs_reconciliation': 'needs manual reconciliation',
              'waiting_device_not_operational': 'device not operational', 'submission_stopped': 'submission stopped by a live check'}
    for backend_name, device_log in journal_block_log['devices'].items():
        print(f'{backend_name} ({device_log["account"]}):')
        for block, entry in device_log['blocks'].items():
            done = sum(j['status'] in JOURNAL_FINAL_JOB_STATUSES for j in entry.get('jobs', []))
            label = labels.get(entry.get('decision'), entry.get('decision'))
            if entry.get('decision') == 'complete' and entry.get('collected_this_run'):
                label = f'complete ({entry["collected_this_run"]} job(s) collected in this run)'
            line = f'   {block}: {label} - {done}/{entry["expected_jobs"]} jobs finished'
            if entry.get('reason'):
                line += f'. {entry["reason"]}'
            print(line)
    for (backend_name, block), state in sorted(journal['hardware'].items()):
        for key, item in sorted(state.items()):
            for cell in item.get('analysis', {}).get('result', {}).get('cells', []):
                print(f"   {backend_name} {block} {cell['cell_label']:44} CZ={cell['two_qubit_gates']:5d} F_raw={cell['fidelity_raw']:.4f} "
                      f"{np.round(cell['fidelity_raw_bootstrap95'] or [np.nan, np.nan], 4).tolist()} G={cell['inversion_gain_raw']:.3f} "
                      f"acc={cell['acceptance']:.4f}")
    print('Block decisions, calibration timestamps and job ids saved to', log_path.relative_to(WORKSPACE))
else:
    print('Journal hardware blocks disabled; planning and local results above are complete without device access.')

In [ ]:
# CELL_ID: journal_report
# Tables and figures for the manuscript, generated only from saved stage records.
import csv
import matplotlib.pyplot as plt
journal_report_dir = OUTPUTS/'invocations'/store.invocation_id/'report'/'journal'
journal_report_dir.mkdir(parents=True, exist_ok=True)
journal_assets = WORKSPACE/'paper'/'assets'; journal_assets.mkdir(parents=True, exist_ok=True)
journal_tables = {}

def journal_write_table(name, rows):
    journal_tables[name] = rows
    atomic_json(journal_report_dir/f'{name}.json', rows)
    if rows:
        keys = list(dict.fromkeys(k for row in rows for k in row))
        with (journal_report_dir/f'{name}.csv').open('w', newline='', encoding='utf8') as stream:
            writer = csv.DictWriter(stream, fieldnames=keys); writer.writeheader()
            for row in rows:
                writer.writerow({k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in row.items()})

if journal_settings_config.get('enabled', False):
    rows = []
    for identifier, record in journal['problems'].items():
        data = record['result']; meta = data['metadata']
        diagnostics = journal_input_diagnostics(data['A'], data['b'])
        timing = data.get('classical_timing') or journal_reference_timing(store, identifier, record)['result']['classical_timing']
        rows.append({'instance': identifier, 'nodes': len(data['b']), 'gamma': meta.get('gamma'),
                     'kappa': meta.get('condition_number_2'), 'F_in': diagnostics['input_state_fidelity_F_in'],
                     'kantorovich_min_F_in': diagnostics['kantorovich_minimum_F_in_at_this_kappa'],
                     'input_target_z_tv': diagnostics['input_target_z_tv'],
                     'sparse_direct_median_us': 1e6*timing['sparse_direct']['median_seconds'],
                     'cg_median_us': 1e6*timing['conjugate_gradient']['median_seconds'],
                     'role': JOURNAL_DESIGN['instances'][identifier]['role']})
    journal_write_table('instances', rows)

    rows = []
    for identifier, screening in journal['screening'].items():
        for candidate in screening['summary']['result']['candidates']:
            rows.append({'instance': identifier, 'candidate_id': candidate['candidate_id'], 'status': candidate['status'],
                         'operator_relative_error': candidate.get('operator_relative_error'),
                         'residual': candidate.get('state_diagnostics', {}).get('residual_reference_free_scale'),
                         'ideal_acceptance': candidate.get('success_probability'),
                         'offline_two_qubit_gates': candidate.get('offline_resources', {}).get('two_qubit_gates'),
                         'rejection_reasons': candidate.get('rejection_reasons')})
    for backend_name, plan in journal.get('planning', {}).items():
        for identifier, compiled in plan['compiled'].items():
            for entry in compiled['result']['entries']:
                match = next((r for r in rows if r['instance'] == identifier and r['candidate_id'] == entry['candidate_id']), None)
                if match is not None:
                    match[f'native_cz_{backend_name}'] = entry['resources']['two_qubit_gates']
    journal_write_table('coherent_screening', rows)

    rows = []
    for identifier, record in journal['vqls'].items():
        result = record['result']
        rows.append({'instance': identifier, 'treatment': result['algorithm'], 'parameters': result['parameters'],
                     'exact_state_fidelity': result['state_diagnostics']['state_fidelity_diagnostic'],
                     'residual': result['state_diagnostics']['residual_reference_free_scale'],
                     'F_in': journal['input_fidelity'][identifier]})
    journal_write_table('vqls_checkpoints', rows)

    rows = []
    for identifier, record in journal['optimizer'].items():
        runs = record['result']['runs']
        for treatment in JOURNAL_DESIGN['vqls']['matched_optimizer_study']['treatments']:
            for budget in JOURNAL_DESIGN['vqls']['matched_optimizer_study']['budgets']:
                chosen = [r for r in runs if r['treatment'] == treatment and r['budget'] == budget]
                fidelities = [r['checkpoint_state_fidelity'] for r in chosen]
                rows.append({'instance': identifier, 'treatment': treatment, 'budget': budget, 'seeds': len(chosen),
                             'median_checkpoint_fidelity': float(np.median(fidelities)),
                             'min_checkpoint_fidelity': float(np.min(fidelities)),
                             'max_checkpoint_fidelity': float(np.max(fidelities)),
                             'median_exact_cost': float(np.median([r['checkpoint_exact_cost'] for r in chosen])),
                             'median_confirmation_cost': float(np.median([r['independent_counted_confirmation_cost'] for r in chosen])),
                             'failures': sum(r['failure'] is not None for r in chosen),
                             'training_shots_per_run': int(np.median([r['training_shots'] for r in chosen])),
                             'F_in': journal['input_fidelity'][identifier]})
    journal_write_table('optimizer_study', rows)

    rows = []
    for label, record in journal.get('retrospective', {}).items():
        for item in record['result']['rows']:
            row = {'job': label, 'pub': item['pub_index'], 'kind': item['kind'], 'algorithm': item['algorithm'],
                   'setting': item['setting'], 'two_qubit_gates': item['two_qubit_gates'],
                   'observed_acceptance': item['observed_acceptance'], 'observed_tv_to_target': item.get('observed_tv_to_target')}
            for model, value in item['models'].items():
                row[f'{model}:tv_pred_obs'] = value.get('tv_predicted_vs_observed')
                row[f'{model}:predicted_acceptance'] = value.get('predicted_acceptance')
                row[f'{model}:predicted_tv_to_target'] = value.get('predicted_tv_to_target')
            rows.append(row)
    journal_write_table('retrospective_predictions', rows)

    rows = []
    for backend_name, plan in journal.get('planning', {}).items():
        for (block, group, arm), job in sorted(plan['jobs'].items()):
            prepared = job['prepared']['result']
            for cell in prepared['cells']:
                prediction = next(p for p in job['prediction']['result']['predictions'] if p['cell_label'] == cell['label'])
                row = {'device': backend_name, 'block': block, 'group': group, 'arm_job': arm, 'cell': cell['label'],
                       'instance': cell['instance'], 'arm': cell['arm'], 'algorithm': cell['algorithm'],
                       'candidate_id': cell['candidate_id'], 'two_qubit_gates': cell['two_qubit_gates'],
                       'ideal_acceptance': cell['ideal_strict_acceptance'], 'settings': len(cell['settings']),
                       'F_in': journal['input_fidelity'][cell['instance']]}
                for model, value in prediction['models'].items():
                    if value.get('status') != 'predicted':
                        continue
                    row[f'{model}:F_raw'] = value['fidelity_raw_readout']; row[f'{model}:F_state'] = value['fidelity_state']
                    row[f'{model}:acceptance_raw'] = value['acceptance_raw_readout']
                    if row['F_in'] < 1:
                        row[f'{model}:gain_raw'] = (value['fidelity_raw_readout']-row['F_in'])/(1-row['F_in'])
                rows.append(row)
    journal_write_table('preregistered_predictions', rows)
    rows = []
    for backend_name, plan in journal.get('planning', {}).items():
        for (block, group, arm), job in sorted(plan['jobs'].items()):
            prepared = job['prepared']['result']
            rows.append({'device': backend_name, 'block': block, 'group': group, 'arm': arm, 'circuits': prepared['circuit_count'],
                         'raw_shots': prepared['raw_shots'], 'estimated_qpu_seconds': prepared['estimated_qpu_seconds'],
                         'planning_source': plan['source'], 'calibration_last_update': plan['calibration_last_update']})
    journal_write_table('planned_jobs', rows)

    rows = []
    for (backend_name, block), state in journal.get('hardware', {}).items():
        for key, item in state.items():
            if 'analysis' not in item:
                continue
            for cell in item['analysis']['result']['cells']:
                row = {'device': backend_name, 'block': block, 'group': key[1], 'job_arm': key[2],
                       **{k: v for k, v in cell.items() if k not in ('pauli_expectations_raw', 'predictions', 'readout_assignment', 'linear_tomography')}}
                for label, models in cell['predictions'].items():
                    for model, value in models.items():
                        if value.get('status') == 'predicted':
                            row[f'{label}:{model}:F_raw'] = value['fidelity_raw_readout']
                rows.append(row)
    # Pre-registered predictions (planning circuits) are attached by device, job arm and cell label.
    preregistered = {(r['device'], r['arm_job'], r['cell']): r for r in journal_tables['preregistered_predictions']}
    for row in rows:
        planned_row = preregistered.get((row['device'], row['job_arm'], row['cell_label']))
        for model in JOURNAL_NOISE_MODELS:
            if planned_row is not None and planned_row.get(f'{model}:F_raw') is not None:
                row[f'pre_registration:{model}:F_raw'] = planned_row[f'{model}:F_raw']
    journal_write_table('hardware_cells', rows)

    # Confirmation endpoints declared in JOURNAL_DESIGN['analysis'] before any held-out data:
    # |F_hat - F_pred|, Spearman rank correlation and bootstrap-interval coverage, per device and prediction record.
    from scipy.stats import spearmanr
    hardware_rows = journal_tables['hardware_cells']
    prediction_labels = sorted({k.split(':')[0] for r in hardware_rows for k in r if k.endswith(':F_raw') and k.count(':') == 2})
    rows = []
    for device in sorted({r['device'] for r in hardware_rows}):
        for scope in ('all_cells', 'solver_cells'):
            chosen = [r for r in hardware_rows if r['device'] == device and r.get('fidelity_raw') is not None
                      and (scope == 'all_cells' or r['algorithm'] != 'control')]
            for label in prediction_labels:
                for model in JOURNAL_NOISE_MODELS:
                    pairs = [(r['fidelity_raw'], r[f'{label}:{model}:F_raw'], r.get('fidelity_raw_bootstrap95'))
                             for r in chosen if r.get(f'{label}:{model}:F_raw') is not None]
                    if not pairs:
                        continue
                    observed = np.array([p[0] for p in pairs]); predicted = np.array([p[1] for p in pairs])
                    rho = float(spearmanr(observed, predicted).statistic) if len(pairs) >= 3 else float('nan')
                    rows.append({'device': device, 'scope': scope, 'prediction': label, 'model': model, 'cells': len(pairs),
                                 'blocks': sorted({r['block'] for r in chosen}),
                                 'mean_abs_error': float(np.mean(np.abs(observed-predicted))),
                                 'median_abs_error': float(np.median(np.abs(observed-predicted))),
                                 'mean_signed_error_observed_minus_predicted': float(np.mean(observed-predicted)),
                                 'spearman_rho': rho if np.isfinite(rho) else None,
                                 'coverage95': float(np.mean([p[2] is not None and p[2][0] <= p[1] <= p[2][1] for p in pairs]))})
    journal_write_table('prediction_endpoints', rows)
    # Block-to-block reproducibility of every repeated cell.
    rows, grouped = [], {}
    for r in hardware_rows:
        grouped.setdefault((r['device'], r['job_arm'], r['cell_label']), []).append(r)
    for (device, job_arm, label), members in sorted(grouped.items()):
        members = sorted(members, key=lambda m: m['block'])
        values = [m['fidelity_raw'] for m in members if m.get('fidelity_raw') is not None]
        row = {'device': device, 'job_arm': job_arm, 'cell_label': label, 'instance': members[0]['instance'],
               'algorithm': members[0]['algorithm'], 'arm': members[0]['arm'], 'blocks': [m['block'] for m in members]}
        for m in members:
            row[f"F_raw_{m['block']}"] = m.get('fidelity_raw'); row[f"G_raw_{m['block']}"] = m.get('inversion_gain_raw')
        row.update({'F_raw_mean': float(np.mean(values)) if values else None,
                    'F_raw_spread': float(np.ptp(values)) if len(values) > 1 else None,
                    'F_raw_sd': float(np.std(values, ddof=1)) if len(values) > 1 else None})
        rows.append(row)
    journal_write_table('block_reproducibility', rows)

    # Figure: pre-registered inversion gain against native two-qubit count, two models.
    def journal_save(fig, name):
        for target in (journal_report_dir, journal_assets):
            fig.savefig(target/f'{name}.png', dpi=220); fig.savefig(target/f'{name}.pdf')
        plt.close(fig)
    instance_colors = {'D0': '#5B7FA3', 'H1': '#7A9CC0', 'H2': '#9DB8D6', 'H3': '#2E8B57', 'H6': '#66BB6A',
                       'H4': '#D1495B', 'H5': '#EDA03B'}
    method_markers = {'hhl': 'o', 'qsvt': 's', 'vqls': '^', 'control': 'X'}
    prediction_rows = [r for r in journal_tables['preregistered_predictions'] if r['arm_job'] == 'raw']
    if prediction_rows:
        devices = [d for d in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston') if any(r['device'] == d for r in prediction_rows)]
        floor = -2.5
        fig, axes = plt.subplots(1, len(devices), figsize=(4.4*len(devices), 4.0), layout='constrained', sharey=True, squeeze=False)
        for axis, device in zip(axes[0], devices):
            for r in [r for r in prediction_rows if r['device'] == device]:
                thermal, esp = r.get('thermal_idle_readout:gain_raw'), r.get('esp_global_depolarizing:gain_raw')
                if thermal is None:
                    continue
                x = max(r['two_qubit_gates'], 1)*(1.08 if r['algorithm'] == 'control' and r['candidate_id'] == 'solution_state' else 1.0)
                color = instance_colors[r['instance']]; marker = method_markers.get(r['algorithm'], 'o')
                shown = [max(v, floor) for v in (thermal, esp) if v is not None]
                if len(shown) == 2:
                    axis.plot([x, x], shown, color=color, lw=.8, alpha=.7)
                axis.scatter([x], [max(thermal, floor)], marker=marker if thermal >= floor else 'v', color=color, s=42, zorder=3,
                             edgecolors='black', linewidths=.4)
                if esp is not None:
                    axis.scatter([x], [max(esp, floor)], marker=marker if esp >= floor else 'v', facecolors='none',
                                 edgecolors=color, s=42, zorder=3, linewidths=1.1)
            axis.axhline(0, color='#333333', lw=.9, ls='--'); axis.axhline(1, color='#999999', lw=.6)
            axis.axvline(300, color='#bbbbbb', lw=.8, ls=':'); axis.set_xscale('log'); axis.set_ylim(floor-.15, 1.12)
            axis.set_xlabel('Native two-qubit (CZ) gates'); axis.set_title(device.replace('ibm_', '')); axis.grid(alpha=.2)
        axes[0][0].set_ylabel('Predicted inversion gain $G$')
        from matplotlib.lines import Line2D
        handles = [Line2D([], [], marker=m, ls='', color='#555555', label=k.upper() if k != 'control' else 'controls')
                   for k, m in method_markers.items()]
        handles += [Line2D([], [], marker='o', ls='', color=c, label=i) for i, c in instance_colors.items()]
        handles += [Line2D([], [], marker='o', ls='', color='#555555', label='thermal+idle (filled)'),
                    Line2D([], [], marker='o', ls='', markerfacecolor='none', color='#555555', label='global depol. (hollow)'),
                    Line2D([], [], marker='v', ls='', color='#555555', label=f'below {floor}')]
        axes[0][-1].legend(handles=handles, fontsize=6.5, loc='lower left', ncol=2, framealpha=.9)
        fig.suptitle('Pre-registered block-B1 predictions; $G=0$ is input preparation, $G=1$ the exact solution', fontsize=10)
        journal_save(fig, 'journal_preregistered_gain')
    retro = journal_tables.get('retrospective_predictions', [])
    if retro:
        fig, axis = plt.subplots(figsize=(5.6, 4.3), layout='constrained')
        labels = {'depolarizing_readout': 'depolarizing + readout', 'thermal_idle_readout': 'thermal + idle + readout',
                  'esp_global_depolarizing': 'global depolarizing (parameter-free)'}
        for model, color, marker in zip(JOURNAL_NOISE_MODELS[1:], ['#4477AA', '#EE6677', '#228833'], ['o', 's', '^']):
            chosen = [r for r in retro if r.get(f'{model}:predicted_tv_to_target') is not None and r.get('observed_tv_to_target') is not None]
            axis.scatter([r['observed_tv_to_target'] for r in chosen], [r[f'{model}:predicted_tv_to_target'] for r in chosen],
                         s=24, alpha=.8, color=color, marker=marker, label=labels[model])
        for r in retro:
            if r['job'] == 'pilot_2026_09_14' and r['algorithm'] in ('hhl', 'qsvt'):
                height = .05 if r['algorithm'] == 'hhl' else .12  # staggered so the two September labels never overlap
                axis.annotate(f"{r['algorithm'].upper()} {r['two_qubit_gates']} CZ", (r['observed_tv_to_target'], height), fontsize=7, ha='center')
        axis.plot([0, .75], [0, .75], color='#555555', lw=.8)
        axis.set_xlabel('Measured conditional TV from target'); axis.set_ylabel('Predicted conditional TV from target')
        axis.set_title(f'Retrospective test on {len(retro)} archived circuits', fontsize=10); axis.legend(fontsize=7); axis.grid(alpha=.2)
        journal_save(fig, 'journal_retrospective_tv')
    # Figure: resource frontier from the Nighthawk planning compilation.
    plan = journal.get('planning', {}).get('ibm_rensselaer')
    if plan:
        fig, axis = plt.subplots(figsize=(5.4, 3.9), layout='constrained')
        for method, marker, color in [('hhl', 'o', '#4477AA'), ('qsvt', 's', '#EE7733')]:
            xs, ys, labels_ = [], [], []
            for identifier, compiled in plan['compiled'].items():
                rows = [e for e in compiled['result']['entries'] if e['role'] == 'coherent_candidate' and e['algorithm'] == method]
                if not rows:
                    continue
                best = min(rows, key=lambda e: (e['resources']['two_qubit_gates']/e['success_probability'], e['resources']['two_qubit_gates']))
                kappa = journal['problems'][identifier]['result']['metadata']['condition_number_2']
                n = len(journal['problems'][identifier]['result']['b'])
                axis.scatter([kappa], [best['resources']['two_qubit_gates']], marker=marker, s=50,
                             facecolors=color if n == 4 else 'none', edgecolors=color, linewidths=1.2,
                             label=f'{method.upper()}' if not xs else None)
                axis.annotate(identifier, (kappa, best['resources']['two_qubit_gates']), textcoords='offset points',
                              xytext=(-14, 4) if identifier == 'H4' else (5, 3), fontsize=7)  # H4/H5 sit at almost the same kappa
                xs.append(kappa)
        axis.axhline(300, color='#777777', ls='--', lw=.8); axis.text(1.45, 315, 'primary cap 300', fontsize=7, color='#555555')
        axis.axhline(1500, color='#777777', ls=':', lw=.8); axis.text(1.45, 1560, 'probe cap 1500', fontsize=7, color='#555555')
        axis.set_yscale('log'); axis.set_xlabel(r'Condition number $\kappa$'); axis.set_ylabel('Native CZ gates (Nighthawk)')
        axis.set_title('Smallest operator-verified circuits (filled: N=4, hollow: N=8)', fontsize=9)
        axis.legend(fontsize=8); axis.grid(alpha=.2)
        journal_save(fig, 'journal_resource_frontier')
    archived = journal.get('archived_tomography')
    if archived is not None:
        result = archived['result']
        labels_p = [l for l in sorted(result['target_pauli_expectations']) if l != 'II'
                    and (abs(result['target_pauli_expectations'][l]) > 1e-9 or abs(result['solution']['pauli_expectations_raw'][l]) > .02)]
        fig, axis = plt.subplots(figsize=(7.0, 3.3), layout='constrained')
        width = .2; x = np.arange(len(labels_p))
        for offset, (name, values, color) in enumerate([
                ('source $\\hat b$', result['source_pauli_expectations'], '#BBBBBB'),
                ('exact solution $\\hat x$', result['target_pauli_expectations'], '#222222'),
                ('ideal VQLS checkpoint', result['checkpoint_pauli_expectations'], '#4477AA'),
                ('hardware (5 Oct)', result['solution']['pauli_expectations_raw'], '#EE6677')]):
            axis.bar(x+(offset-1.5)*width, [values[l] for l in labels_p], width, color=color, label=name)
        axis.set_xticks(x, labels_p); axis.axhline(0, color='black', lw=.6); axis.set_ylabel('Pauli expectation')
        axis.set_title(f"D0 frozen VQLS tomography: F(hardware, x)={result['solution']['fidelity_raw']:.4f}, "
                       f"F_in={result['F_in_exact']:.4f}, G={result['inversion_gain_raw']:.3f}", fontsize=9)
        axis.legend(fontsize=7, ncol=4, loc='upper center'); axis.grid(axis='y', alpha=.2)
        journal_save(fig, 'journal_d0_vqls_paulis')
    optimizer_rows = journal_tables.get('optimizer_study', [])
    if optimizer_rows:
        instances = sorted({r['instance'] for r in optimizer_rows})
        fig, axes = plt.subplots(1, len(instances), figsize=(2.6*len(instances), 3.2), layout='constrained', sharey=True, squeeze=False)
        for axis, identifier in zip(axes[0], instances):
            for treatment, color in [('exact_objective', '#4477AA'), ('counted_objective', '#EE6677')]:
                chosen = sorted([r for r in optimizer_rows if r['instance'] == identifier and r['treatment'] == treatment], key=lambda r: r['budget'])
                axis.errorbar([r['budget'] for r in chosen], [r['median_checkpoint_fidelity'] for r in chosen],
                              yerr=[[r['median_checkpoint_fidelity']-r['min_checkpoint_fidelity'] for r in chosen],
                                    [r['max_checkpoint_fidelity']-r['median_checkpoint_fidelity'] for r in chosen]],
                              marker='o', capsize=3, color=color, label=treatment.replace('_', ' '))
            axis.axhline(journal['input_fidelity'][identifier], color='#777777', ls='--', lw=.8)
            axis.set_title(identifier); axis.set_xlabel('Objective evaluations'); axis.grid(alpha=.2)
        axes[0][0].set_ylabel('Checkpoint state fidelity (local, exact)'); axes[0][0].legend(fontsize=7)
        journal_save(fig, 'journal_optimizer_study')
    # Figure: measured inversion gain (raw, 95% bootstrap) against native CZ count, with job-start predictions.
    measured = [r for r in journal_tables['hardware_cells'] if r['job_arm'] == 'raw' and r.get('inversion_gain_raw') is not None]
    if measured:
        devices = [d for d in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston') if any(r['device'] == d for r in measured)]
        floor, offsets = -2.5, {'B1': 0.93, 'B2': 1.0, 'B3': 1.07, 'T1': 1.0}
        fig, axes = plt.subplots(1, len(devices), figsize=(max(4.4*len(devices), 6.0), 4.6), layout='constrained', sharey=True, squeeze=False)
        for axis, device in zip(axes[0], devices):
            for r in [r for r in measured if r['device'] == device]:
                x = max(r['two_qubit_gates'], 1)*offsets.get(r['block'], 1.0)*(1.08 if r['candidate_id'] == 'solution_state' else 1.0)
                color = instance_colors.get(r['instance'], '#777777'); marker = method_markers.get(r['algorithm'], 'o')
                gain, interval = r['inversion_gain_raw'], r.get('inversion_gain_raw_bootstrap95')
                if interval:
                    axis.plot([x, x], [max(interval[0], floor), max(interval[1], floor)], color=color, lw=1.0, alpha=.8)
                axis.scatter([x], [max(gain, floor)], marker=marker if gain >= floor else 'v', color=color, s=40, zorder=3,
                             edgecolors='black', linewidths=.4)
                predicted, f_in = r.get('job_start:thermal_idle_readout:F_raw'), r['input_state_fidelity_F_in']
                if predicted is not None and f_in < 1:
                    axis.scatter([x], [max((predicted-f_in)/(1-f_in), floor)], marker='_', color='black', s=90, zorder=4, linewidths=1.2)
            axis.axhline(0, color='#333333', lw=.9, ls='--'); axis.axhline(1, color='#999999', lw=.6)
            axis.set_xscale('log'); axis.set_ylim(floor-.15, 1.12); axis.grid(alpha=.2)
            axis.set_xlabel('Native two-qubit (CZ) gates'); axis.set_title(device.replace('ibm_', '')+' ('+', '.join(sorted({r['block'] for r in measured if r['device'] == device}))+')')
        axes[0][0].set_ylabel('Measured inversion gain $G$ (raw)')
        from matplotlib.lines import Line2D
        handles = [Line2D([], [], marker=m, ls='', color='#555555', label=k.upper() if k != 'control' else 'controls') for k, m in method_markers.items()]
        handles += [Line2D([], [], marker='o', ls='', color=c, label=i) for i, c in instance_colors.items()]
        handles += [Line2D([], [], marker='_', ls='', color='black', markersize=10, label='job-start thermal+idle prediction'),
                    Line2D([], [], marker='v', ls='', color='#555555', label=f'below {floor} (drawn at the floor)')]
        fig.legend(handles=handles, fontsize=6.5, loc='outside lower center', ncol=6, framealpha=.9)
        fig.suptitle('Measured inversion gain (raw, 95% shot-noise intervals)', fontsize=10)
        journal_save(fig, 'journal_hardware_gain')
    else:
        for target in (journal_report_dir, journal_assets):
            for suffix in ('.png', '.pdf'):
                (target/f'journal_hardware_gain{suffix}').unlink(missing_ok=True)  # never leave a stale hardware figure
    print('Journal tables:', ', '.join(f'{k} ({len(v)})' for k, v in journal_tables.items()))


In [ ]:
# CELL_ID: journal_export
# One portable package containing every stage used in this run, old and new.
journal_used = []
if journal_settings_config.get('enabled', False):
    journal_used += list(journal['problems'].values())
    for screening in journal['screening'].values():
        journal_used += [*screening['candidates'], screening['summary']]
    journal_used += [*journal['vqls'].values(), *journal['optimizer'].values(), *journal.get('retrospective', {}).values()]
    if journal.get('archived_tomography') is not None:
        journal_used.append(journal['archived_tomography'])
    for identifier, record in journal['problems'].items():
        if not record['result'].get('classical_timing'):
            journal_used.append(journal_reference_timing(store, identifier, record))
    for plan in journal.get('planning', {}).values():
        journal_used += list(plan['compiled'].values())
        for job in plan['jobs'].values():
            journal_used += [job['prepared'], job['prediction']]
    for state in journal.get('hardware', {}).values():
        for item in state.values():
            for name in ('prepared', 'record', 'analysis'):
                value = item.get(name)
                if isinstance(value, dict) and {'stage', 'key', 'directory'} <= set(value):
                    journal_used.append(value)
    journal_extras = [p for p in (OUTPUTS/'invocations'/store.invocation_id).rglob('*')
                      if p.is_file() and p.suffix in ('.json', '.md', '.png', '.pdf', '.csv') and 'portable_source' not in p.parts]
    # Environment records of every invocation, so that a restored package reloads stages computed under any
    # numerically equivalent environment (see cache_portability), not only those of the exporting environment.
    journal_extras.extend(p for p in sorted((OUTPUTS/'invocations').glob('*/environment.json')) if p.parent.name != store.invocation_id)
    for folder in ['hardware_jobs', 'hardware_campaigns', 'calibration_observations', 'calibrations']:
        journal_extras.extend(p for p in (OUTPUTS/folder).rglob('*') if p.is_file() and p.suffix in ('.json', '.qpy'))
    # Every stage this run loaded or computed, including stages that are not returned to the notebook, such as the
    # VQLS checkpoint selections and the runs of unselected seeds, so that a restored package reloads all of them.
    journal_used += [{'stage': e['stage'], 'key': e['key']} for e in store.events
                     if e.get('action') in ('loaded_existing', 'computed_and_saved')]
    journal_package = store.export_run([*used, *journal_used], extra_files=journal_extras)
    run_summary = store.summary()
    atomic_json(OUTPUTS/'invocations'/store.invocation_id/'summary.json', run_summary)
    print('Portable package with every stage of this run:', journal_package)
    print('Stages loaded:', run_summary['loaded'], '| computed and saved:', run_summary['computed'])

In [ ]:
# CELL_ID: journal_paper
# Regenerate the journal manuscript's numbers and tables from saved records, then
# typeset paper/journal/main.tex when a LaTeX installation is available.
import re
import shutil
import subprocess
journal_paper_dir = WORKSPACE/'paper'/'journal'
journal_generated = journal_paper_dir/'generated'

def _tex(value):
    return str(value).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('%', r'\%').replace('&', r'\&').replace('#', r'\#')

def _f(value, digits=3):
    return '--' if value is None or (isinstance(value, float) and not np.isfinite(value)) else f'{value:.{digits}f}'

def _interval(values, digits=3):
    return '--' if not values else f'[{values[0]:.{digits}f}, {values[1]:.{digits}f}]'

def _table(label, caption, header, rows, spec, wide=False, size=r'\footnotesize'):
    env = 'table*' if wide else 'table'
    lines = [f'\\begin{{{env}}}[t]', r'\centering', size, f'\\caption{{{caption}}}', f'\\label{{{label}}}',
             r'\setlength{\tabcolsep}{3.2pt}', f'\\begin{{tabular}}{{{spec}}}', r'\toprule', ' & '.join(header)+r' \\', r'\midrule']
    minus = lambda cell: re.sub(r'(?<![\w$-])-(?=\d)', r'$-$', str(cell))  # typographic minus for negative numbers
    lines += [' & '.join(map(minus, row))+r' \\' for row in rows]
    lines += [r'\bottomrule', r'\end{tabular}', f'\\end{{{env}}}']
    return '\n'.join(lines)+'\n'

_device_order = {'ibm_rensselaer': 0, 'ibm_kingston': 1, 'ibm_boston': 2}
_device_short = {'ibm_rensselaer': 'N', 'ibm_kingston': 'K', 'ibm_boston': 'B'}
_model_names = {'logical_ideal': 'logical-ideal', 'depolarizing_readout': 'depolarizing-readout',
                'thermal_idle_readout': 'thermal-idle-readout', 'esp_global_depolarizing': 'global-depolarizing'}

def _campaign_jobs(outputs_dir, design_id):
    """Every submitted confirmation job from the registry: identity, pre-submission estimate, charged QPU time, start time."""
    jobs = []
    for request_path in sorted((outputs_dir/'hardware_jobs').glob('*/request.json')):
        request = read_json(request_path)
        identity = (request.get('journal_identity') or {}).get('journal_job_v1')
        if not identity or identity.get('design_id') != design_id:
            continue
        metrics = [read_json(p) for p in sorted(request_path.parent.glob('collections/*/job_metrics.json'))]
        metrics = [m for m in metrics if (m.get('usage') or {}).get('qpu_charge_time_seconds') is not None]
        if not metrics:
            continue
        jobs.append({'device': identity['backend'], 'block': identity['block'], 'group': identity['group'], 'arm': identity['arm'],
                     'estimated_seconds': (request.get('timing_estimate') or {}).get('estimated_qpu_seconds'),
                     'charged_seconds': metrics[-1]['usage']['qpu_charge_time_seconds'],
                     'running_utc': (metrics[-1].get('timestamps') or {}).get('running')})
    return jobs

def _cell_label(candidate):
    match = re.fullmatch(r'hhl_m(\d+)_(bounded_support|bin_spacing)', candidate)
    if match:
        return f"HHL $m{{=}}{match.group(1)}$ " + ('bounded' if match.group(2) == 'bounded_support' else 'bin spacing')
    match = re.fullmatch(r'qsvt_d(\d+)', candidate)
    if match:
        return f'QSVT $d{{=}}{match.group(1)}$'
    return {'vqls_vqls_measured': 'VQLS counted', 'vqls_vqls_statevector': 'VQLS exact', 'input_state': 'input state',
            'solution_state': 'solution state'}.get(candidate, _tex(candidate).replace(r'\_', ' '))

def _hardware_table(hardware_rows):
    """Every analyzed hardware cell: G with its 95% bootstrap interval, one column per device, block and job arm."""
    columns = sorted({(r['device'], r['block'], r['job_arm']) for r in hardware_rows},
                     key=lambda c: (_device_order.get(c[0], 9), c[2] != 'raw', c[1]))
    devices_seen = sorted({r['device'] for r in hardware_rows}, key=lambda d: _device_order.get(d, 9))
    method_rank = {'hhl': 0, 'qsvt': 1, 'vqls': 2, 'control': 3}
    arm_names = {'primary': 'primary', 'depth_probe': 'probe', 'historical_ablation': 'ablation', 'vqls': 'VQLS', 'control': 'control'}
    keys = sorted({(r['instance'], r['algorithm'], r['arm'], r['candidate_id']) for r in hardware_rows},
                  key=lambda k: (k[0], method_rank.get(k[1], 9), k[2], k[3]))
    rows = []
    for key in keys:
        members = {(r['device'], r['block'], r['job_arm']): r for r in hardware_rows
                   if (r['instance'], r['algorithm'], r['arm'], r['candidate_id']) == key}
        cz = []
        for device in devices_seen:
            found = next((m for c, m in members.items() if c[0] == device), None)
            cz.append('--' if found is None else str(found['two_qubit_gates']))
        row = [key[0], _cell_label(key[3]), arm_names.get(key[2], _tex(key[2]).replace(r'\_', ' ')), '/'.join(cz)]
        for column in columns:
            m = members.get(column)
            if m is None or not m.get('inversion_gain_raw_bootstrap95'):
                row.append('--')
                continue
            low, high = m['inversion_gain_raw_bootstrap95']
            value = _f(m['inversion_gain_raw'], 2)
            row.append(f"\\Gci{{{value if low <= 0 else chr(92)+'textbf{'+value+'}'}}}{{{_interval([low, high], 2)}}}")
        rows.append(row)
    header = ['ID', 'cell', 'arm', 'CZ ('+'/'.join(_device_short.get(d, d) for d in devices_seen)+')',
              *[f"{_device_short.get(d, _tex(d))} {b}" + ('' if a == 'raw' else r' DD') for d, b, a in columns]]
    return _table('tab:hardware',
        r'Hardware inversion gain $G$ (raw readout) with its 95\% within-job bootstrap interval for every analyzed cell, '
        r'by device (N: \texttt{ibm\_rensselaer}, K: \texttt{ibm\_kingston}, B: \texttt{ibm\_boston}) and block; DD is the '
        r'dynamical-decoupling and twirling arm of B1. Bold values have an interval entirely above zero. CZ counts are those of '
        r'the compiled circuits on the three devices; controls prepare $\bhat$ (input) or $\xhat$ (solution) directly.',
        header, rows, 'lllc'+'c'*len(columns), wide=True, size=r'\scriptsize')

def _confirmation_numbers(hardware_rows, endpoints, design, reproducibility=(), campaign_jobs=(), block_status=None):
    """Results sentences and campaign facts for the analyzed confirmation blocks, generated from the hardware cells,
    the prediction endpoints, the block-reproducibility table, the job registry and the block status;
    pending wording while no block has been analyzed."""
    out = {k: '' for k in ('CampaignSentence', 'ConfirmationHHL', 'ConfirmationPredictions', 'ConfirmationQSVT',
                           'ConfirmationFidelity', 'ConfirmationVQLS', 'ConfirmationControls', 'ConfirmationMitigation',
                           'ConfirmationRepro', 'ConfirmationModels')}
    out['ConfirmationSentence'] = (r'\pending{The confirmation blocks have not been executed. All circuits, predictions and analysis '
                                   r'code are frozen; their results will be reported here without changing the design.}')
    out['AbstractConfirmation'] = 'The pending Nighthawk and Heron blocks discriminate between the models.'
    out['ConclusionConfirmation'] = ('The frozen campaign on held-out, inversion-sensitive instances will show whether any verified solver '
                                     'produces a measurable inversion gain on current processors and which noise model anticipates it; '
                                     'the same protocol can certify a future positive result.')
    out.update({'CampaignJobs': '--', 'CampaignCells': '--', 'CoherentResolved': '--', 'VQLSResolved': '--', 'PositiveCell': 'none'})
    if not hardware_rows:
        return out
    noise_models = ('depolarizing_readout', 'thermal_idle_readout', 'esp_global_depolarizing')
    words = {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine', 10: 'ten'}
    def num(value, digits=2):
        text = f'{value:.{digits}f}'
        return '$' + (text.lstrip('-') if float(text) == 0 else text) + '$'
    def ci(values, digits=2):
        return f'$[{values[0]:.{digits}f}, {values[1]:.{digits}f}]$'
    def join(items):
        items = list(items)
        return '' if not items else items[0] if len(items) == 1 else ', '.join(items[:-1]) + ' and ' + items[-1]
    def tt(device):
        return '\\texttt{' + device.replace('_', '\\_') + '}'
    def gain(row, model, record='pre_registration'):
        value = row.get(f'{record}:{model}:F_raw')
        return None if value is None else (value-row['input_state_fidelity_F_in'])/(1-row['input_state_fidelity_F_in'])
    def above(r):
        return bool(r.get('inversion_gain_raw_bootstrap95')) and r['inversion_gain_raw_bootstrap95'][0] > 0
    def below(r):
        return bool(r.get('inversion_gain_raw_bootstrap95')) and r['inversion_gain_raw_bootstrap95'][1] < 0
    def cz_range(rows):
        values = sorted({r['two_qubit_gates'] for r in rows})
        return str(values[0]) if len(values) == 1 else f'{values[0]}--{values[-1]}'
    def several(n, noun):
        return f"{words.get(n, n)} {noun}" + ('' if n == 1 else 's')
    def count(n, total, noun):
        return f'none of the {total} {noun}' if n == 0 else f'all {total} {noun}' if n == total else f'{n} of the {total} {noun}'
    def day(text):
        return datetime.fromisoformat(str(text).replace('Z', '+00:00'))
    def day_list(stamps):
        dates = sorted({day(s).date() for s in stamps})
        return join(f'{d.day}' for d in dates) + f' {dates[-1]:%B}' if len({d.month for d in dates}) == 1 else join(f'{d.day} {d:%B}' for d in dates)
    raw = sorted([r for r in hardware_rows if r['job_arm'] == 'raw'], key=lambda r: (_device_order.get(r['device'], 9), r['block'], r['instance']))
    places = sorted({(r['device'], r['block']) for r in hardware_rows}, key=lambda p: (_device_order.get(p[0], 9), p[1]))
    devices = sorted({d for d, _ in places}, key=lambda d: _device_order.get(d, 9))
    blocks_of = {d: [b for dd, b in places if dd == d] for d in devices}
    coherent = [r for r in raw if r['algorithm'] in ('hhl', 'qsvt')]
    vqls = [r for r in raw if r['algorithm'] == 'vqls']
    controls = [r for r in raw if r['algorithm'] == 'control']
    # Campaign-wide counts include the mitigation arm; block-by-block statements use the raw arms.
    all_coherent = [r for r in hardware_rows if r['algorithm'] in ('hhl', 'qsvt')]
    all_vqls = [r for r in hardware_rows if r['algorithm'] == 'vqls']
    coherent_pos, all_vqls_pos = [r for r in all_coherent if above(r)], [r for r in all_vqls if above(r)]
    mitigation = [r for r in hardware_rows if r['job_arm'] != 'raw']
    mitigation_pos = [r for r in mitigation if above(r)]
    jobs = len({(r['device'], r['block'], r['group'], r['job_arm']) for r in hardware_rows})
    out.update({'CampaignJobs': str(jobs), 'CampaignCells': str(len(hardware_rows)),
                'CoherentResolved': f'{len(coherent_pos)} of {len(all_coherent)}', 'VQLSResolved': f'{len(all_vqls_pos)} of {len(all_vqls)}',
                'PositiveCell': join(f"{_method_names.get(r['algorithm'], r['algorithm'])} on {r['instance']} on {tt(r['device'])}"
                                     + ('' if r['job_arm'] == 'raw' else ' (mitigation arm)')
                                     + f" ($G={r['inversion_gain_raw']:.2f}$ {ci(r['inversion_gain_raw_bootstrap95'])})" for r in coherent_pos) or 'none'})
    out['ConfirmationSentence'] = (
        (f"The {words.get(len(places), len(places))} blocks comprise" if len(places) > 1 else 'The single block comprises')
        + f" {len(hardware_rows)} analyzed cells in {jobs} jobs"
        + (f", {len(mitigation)} of them in the mitigation arm" if mitigation else '')
        + f" (Table~\\ref{{tab:hardware}}, Fig.~\\ref{{fig:hardware}}). A raw inversion-gain interval lies entirely above zero for "
        f"{count(len(coherent_pos), len(all_coherent), 'coherent cells')} and for {count(len(all_vqls_pos), len(all_vqls), 'VQLS cells')}"
        + ('' if not mitigation else ', none of them in the mitigation arm.' if not mitigation_pos
           else f", {len(mitigation_pos)} of them in the mitigation arm.") + ('' if mitigation else '.'))

    # Realized schedule, calibrations and QPU charges.
    if campaign_jobs:
        pieces = []
        starts = [day(j['running_utc']) for j in campaign_jobs if j.get('running_utc')]
        expected = sum(b.get('expected_jobs', 0) for v in (block_status or {}).get('devices', {}).values() for b in v.get('blocks', {}).values())
        once = len({(j['device'], j['block'], j['group'], j['arm']) for j in campaign_jobs}) == len(campaign_jobs)
        lead = f"All {len(campaign_jobs)} planned jobs" if expected == len(campaign_jobs) else f"{len(campaign_jobs)} jobs"
        if starts:
            first, last = min(starts).date(), max(starts).date()
            span = (f'on {first.day} {first:%B %Y}' if first == last else
                    f'between {first.day} and {last.day} {last:%B %Y}' if (first.year, first.month) == (last.year, last.month)
                    else f'between {first.day} {first:%B} and {last.day} {last:%B %Y}')
            pieces.append(f"{lead} ran {'once ' if once else ''}{span} (UTC) and were collected.")
        stamps = {}
        for device, value in (block_status or {}).get('devices', {}).items():
            for block, entry in value.get('blocks', {}).items():
                values = [day(s) for s in (entry.get('calibration_stamps') or {}).values()]
                if values:
                    stamps[(device, block)] = values
        primary = design['devices']['primary']['backend']
        primary_blocks = [b for b in design['devices']['primary']['blocks'] if (primary, b) in stamps]
        if len(primary_blocks) > 1:
            separated = all(min(stamps[(primary, b2)]) > max(stamps[(primary, b1)]) for b1, b2 in zip(primary_blocks, primary_blocks[1:]))
            listed = join(f'{day_list(stamps[(primary, b)])} ({b})' for b in primary_blocks)
            pieces.append(f"On {tt(primary)} the provider calibrations in force were last updated on {listed}, so " +
                          ('consecutive blocks are separated by at least one recalibration.' if separated
                           else 'not every pair of consecutive blocks is separated by a recalibration; those blocks are time blocks.'))
        spanning = {}
        for (d, b), values in sorted(stamps.items(), key=lambda kv: (_device_order.get(kv[0][0], 9), kv[0][1])):
            distinct = len({v.replace(microsecond=0) for v in values})
            if distinct > 1:
                spanning.setdefault(distinct, []).append(f"{b}" + ('' if d == primary else f" on {tt(d)}"))
        for distinct, names in spanning.items():
            pieces.append(join(names) + (' spans ' if len(names) == 1 else ' each span ') + f'{words.get(distinct, distinct)} provider calibrations; '
                          'every job is predicted and analyzed against its own.')
        estimated = [j for j in campaign_jobs if j.get('estimated_seconds') is not None]
        if estimated:
            charged = sum(j['charged_seconds'] for j in campaign_jobs)
            planned = sum(j['estimated_seconds'] for j in estimated)
            excess = [j['charged_seconds']-j['estimated_seconds'] for j in estimated]
            pieces.append(f"The jobs were charged {charged:.0f} QPU seconds, {charged-planned:.0f} seconds ({100*(charged-planned)/planned:.1f}\\%) "
                          f"above their pre-submission estimates; per job the charge exceeded the estimate by {min(excess):.1f} to {max(excess):.1f} seconds.")
        out['CampaignSentence'] = ' '.join(pieces)

    # Decision cells: HHL on the inversion-sensitive instances, by device.
    decisive = [r for r in raw if r['instance'] in ('H4', 'H5') and r['algorithm'] == 'hhl' and r.get('inversion_gain_raw_bootstrap95')]
    if decisive:
        sentences = []
        for device in devices:
            rows = sorted([r for r in decisive if r['device'] == device], key=lambda r: (r['instance'], r['block']))
            if not rows:
                continue
            blocks = sorted({r['block'] for r in rows})
            by_instance = {i: [r for r in rows if r['instance'] == i] for i in ('H4', 'H5')}
            if len(blocks) > 1:
                values = join(join(num(r['inversion_gain_raw']) for r in cells) + f' on {i}' for i, cells in by_instance.items() if cells)
                status = ('every interval lies below zero' if all(map(below, rows)) else
                          'every interval lies above zero' if all(map(above, rows)) else 'the intervals do not share a sign')
                sentences.append(f"On {tt(device)} ({cz_range(rows)} CZ) the gains were {values} in {join(blocks)}; {status}.")
            else:
                listed = []
                for i, cells in by_instance.items():
                    for r in cells:
                        note = ''
                        if above(r) and len(coherent_pos) == 1:
                            note = ' (the only coherent cell of the campaign with an interval above zero)'
                        listed.append(f"{num(r['inversion_gain_raw'])} {ci(r['inversion_gain_raw_bootstrap95'])} on {i}{note}")
                sentences.append(f"On {tt(device)} ({cz_range(rows)} CZ) they reached {join(listed)}.")
        out['ConfirmationHHL'] = ('The pre-registered decision cells, HHL on H4 and H5, give a processor-dependent answer. ' + ' '.join(sentences))
        # Sign record of the pre-registered predictions, and the job-start prediction for any resolved cell.
        correct = {m: sum((gain(r, m) > 0) == (r['inversion_gain_raw'] > 0) for r in decisive if gain(r, m) is not None) for m in noise_models}
        n = len(decisive)
        local = ('depolarizing_readout', 'thermal_idle_readout')
        if correct[local[0]] == correct[local[1]]:
            local_text = f"the gate-local models had each predicted the sign of the HHL gain for {words.get(correct[local[0]], correct[local[0]])} of these {words.get(n, n)} measurements"
        else:
            local_text = (f"the depolarizing-readout and thermal-idle-readout models had predicted the sign of the HHL gain for "
                          f"{correct[local[0]]} and {correct[local[1]]} of these {n} measurements")
        text = f"Before any of these runs, {local_text}, and the global-depolarizing model for {words.get(correct['esp_global_depolarizing'], correct['esp_global_depolarizing'])}."
        for r in [r for r in decisive if above(r)]:
            pre = [gain(r, m) for m in ('esp_global_depolarizing', 'depolarizing_readout', 'thermal_idle_readout')]
            start = gain(r, 'thermal_idle_readout', 'job_start')
            others = [q for q in decisive if q['instance'] == r['instance'] and q['device'] != r['device'] and gain(q, 'thermal_idle_readout', 'job_start') is not None]
            text += (f" For {r['instance']} on {tt(r['device'])} the pre-registered gains were {num(pre[0])} (global), {num(pre[1])} "
                     f"(depolarizing-readout) and {num(pre[2])} (thermal-idle-readout).")
            if start is not None:
                text += f" From the job-start calibration the thermal-idle-readout model predicted {num(start)}"
                if others:
                    by_device = {}
                    for q in others:
                        by_device.setdefault(q['device'], []).append(gain(q, 'thermal_idle_readout', 'job_start'))
                    text += (' for this cell against ' + join(
                        (f"{num(min(v))} to {num(max(v))}" if len(v) > 1 else num(v[0])) + f" on {tt(d)}"
                        for d, v in sorted(by_device.items(), key=lambda kv: _device_order.get(kv[0], 9)))
                        + ' for the same instance, so the difference between the processors is already visible in their calibration data')
                text += '.'
                for q in [q for q in decisive if q['device'] == r['device'] and q['instance'] != r['instance']]:
                    q_start = gain(q, 'thermal_idle_readout', 'job_start')
                    if q_start is not None:
                        text += (f" For {q['instance']} on the same processor it predicted {num(q_start)}, against the measured "
                                 f"{num(q['inversion_gain_raw'])}.")
        out['ConfirmationPredictions'] = text

    # Deeper coherent circuits and the fidelity of the coherent outputs.
    qsvt = [r for r in raw if r['instance'] in ('H4', 'H5') and r['algorithm'] == 'qsvt' and r.get('inversion_gain_raw_bootstrap95')]
    if qsvt:
        predicted = [v for r in qsvt for m in noise_models for v in [gain(r, m)] if v is not None]
        beyond = sum(r['inversion_gain_raw'] < min(gain(r, m) for m in noise_models) for r in qsvt if None not in [gain(r, m) for m in noise_models])
        gains = [r['inversion_gain_raw'] for r in qsvt]
        out['ConfirmationQSVT'] = (
            f"QSVT on the same instances ({cz_range(qsvt)} CZ) stayed far below input preparation on every processor and block, "
            f"with $G$ from {num(min(gains))} to {num(max(gains))},"
            + (f" against pre-registered predictions of {num(min(predicted))} to {num(max(predicted))}; {count(beyond, len(qsvt), 'cells').replace('the ', '')} "
               f"fall below all three of their own model predictions." if predicted else '.'))
    sensitive = [r for r in coherent if r['instance'] in ('H4', 'H5') and r.get('fidelity_raw_bootstrap95')]
    if sensitive:
        mixed = {r['instance']: 1/2**r['n_system_qubits'] for r in sensitive}
        coherent_like = [r for r in sensitive if r['fidelity_raw'] > (mixed[r['instance']] + r['input_state_fidelity_F_in'])/2]
        near = [r for r in sensitive if r not in coherent_like]
        under = [r for r in near if r['fidelity_raw_bootstrap95'][1] < mixed[r['instance']]]
        floors = {i: (mixed[i]-next(r['input_state_fidelity_F_in'] for r in sensitive if r['instance'] == i))/(1-next(r['input_state_fidelity_F_in'] for r in sensitive if r['instance'] == i))
                  for i in sorted(mixed)}
        if near:
            values = [r['fidelity_raw'] for r in near]
            groups = {}
            for r in coherent_like:
                groups.setdefault((r['algorithm'], r['device']), []).append(r)
            lead = ('Apart from ' + join(f"{_method_names.get(a, a)} on {tt(d)} (" + join(f"$F={r['fidelity_raw']:.2f}$ on {r['instance']}" for r in rows) + ')'
                                         for (a, d), rows in groups.items()) + ', the') if coherent_like else 'The'
            widths = sorted({r['n_system_qubits'] for r in sensitive})
            state = (f"$1/{2**widths[0]}$ of a maximally mixed {words.get(widths[0], widths[0])}-qubit state" if len(widths) == 1
                     else 'of a maximally mixed state')
            text = (f"{lead} raw fidelities of the coherent H4 and H5 cells lie between {num(min(values))} and {num(max(values))}, around the value "
                    f"{state}, whose inversion gain would be " + join(f"{num(v)} on {i}" for i, v in floors.items())
                    + '; these outputs retain little of the prepared state.')
            if under:
                groups = {}
                for r in under:
                    groups.setdefault((r['algorithm'], r['device'], r['instance']), []).append(r['block'])
                described = join(f"{_method_names.get(a, a)} on {i} in {join(sorted(blocks))} on {tt(d)}" for (a, d, i), blocks in groups.items())
                text += (f" The interval{'s' if len(under) > 1 else ''} of {described} lie{'' if len(under) > 1 else 's'} entirely below that value, "
                         'which depolarizing noise alone cannot produce.')
            out['ConfirmationFidelity'] = text

    # Frozen VQLS states, controls and the mitigation arm.
    if vqls:
        parts, missed = [], []
        for instance in sorted({r['instance'] for r in vqls}):
            cells = [r for r in vqls if r['instance'] == instance]
            pos = [r for r in cells if above(r)]
            if not pos:
                missed.append(instance)
                continue
            gains = [r['inversion_gain_raw'] for r in pos]
            span = f"{num(min(gains))} to {num(max(gains))}"
            if len(pos) == len(cells):
                parts.append(f"on {instance} in all {words.get(len(cells), len(cells))} blocks ({span})")
            else:
                notes = []
                for r in [c for c in cells if not above(c)]:
                    control = next((c for c in controls if c['candidate_id'] == 'solution_state' and c['instance'] == instance
                                    and c['device'] == r['device'] and c['block'] == r['block']), None)
                    note = (f", where the solution-state control also reached only $G={control['inversion_gain_raw']:.2f}$"
                            if control and control['inversion_gain_raw'] < 0.5 else '')
                    notes.append(f"{num(r['inversion_gain_raw'])} on {tt(r['device'])}{note}")
                parts.append(f"on {instance} in {words.get(len(pos), len(pos))} of {words.get(len(cells), len(cells))} blocks ({span}; {join(notes)})")
        text = (f"The frozen VQLS states resolved positive gains in {len(all_vqls_pos)} of {len(all_vqls)} cells: " + join(parts) + '.') if parts else ''
        missed = [i for i in missed if not any(above(r) for r in all_vqls if r['instance'] == i)]
        if missed:
            room = max(1-r['input_state_fidelity_F_in'] for r in all_vqls if r['instance'] in missed)
            text += f" On {join(missed)}, where $1-\\Fin\\le{room:.3f}$, no VQLS interval lies above zero."
        out['ConfirmationVQLS'] = text.strip()
    inputs = [r for r in controls if r['candidate_id'] == 'input_state' and r['instance'] in ('H4', 'H5')]
    solutions = [r for r in controls if r['candidate_id'] == 'solution_state']
    pieces = []
    if inputs:
        values = [r['inversion_gain_raw'] for r in inputs]
        pieces.append(f"input-state controls on H4 and H5 gave $G$ between {num(min(values))} and {num(max(values))} in every block, "
                      "close to the value $0$ of ideal input preparation")
    if solutions:
        values = [r['fidelity_raw'] for r in solutions]
        pieces.append(f'solution-state controls on all {words.get(len({r["instance"] for r in solutions}), len({r["instance"] for r in solutions}))} instances reached raw fidelity {num(min(values))} to {num(max(values))}')
    if pieces:
        sentence = ', and '.join(pieces) + '.'
        out['ConfirmationControls'] = sentence[0].upper() + sentence[1:]
    differences = []
    for r in hardware_rows:
        if r['job_arm'] != 'raw' and r['algorithm'] != 'control':
            twin = next((x for x in hardware_rows if x['job_arm'] == 'raw' and x['device'] == r['device']
                         and x['block'] == r['block'] and x['cell_label'] == r['cell_label']), None)
            if twin:
                differences.append(r['fidelity_raw']-twin['fidelity_raw'])
    if differences:
        out['ConfirmationMitigation'] = (f'The dynamical-decoupling and twirling arm changed the raw fidelity of its {len(differences)} solver cells by '
                                         f'{num(float(np.mean(differences)), 3)} on average (from {num(min(differences), 3)} to '
                                         f'{num(max(differences), 3)}) relative to the raw arm of the same block.')

    # Block-to-block reproducibility on the primary device.
    primary = design['devices']['primary']['backend']
    repeated = [r for r in reproducibility if r['device'] == primary and r['job_arm'] == 'raw' and len(r.get('blocks', [])) > 1 and r.get('F_raw_sd') is not None]
    if repeated:
        def half_width(r):
            cells = [c for c in raw if c['device'] == primary and c['cell_label'] == r['cell_label'] and c.get('fidelity_raw_bootstrap95')]
            return float(np.mean([(c['fidelity_raw_bootstrap95'][1]-c['fidelity_raw_bootstrap95'][0])/2 for c in cells])) if cells else None
        stats = [(r, r['F_raw_sd'], half_width(r)) for r in repeated]
        stats = [s for s in stats if s[2]]
        deep = [s for s in stats if s[0]['algorithm'] in ('hhl', 'qsvt')]
        shallow = [s for s in stats if s[0]['algorithm'] not in ('hhl', 'qsvt')]
        exceed = lambda group: sum(sd > hw for _, sd, hw in group)
        widest = max(stats, key=lambda s: s[1])
        n_blocks = max(len(r['blocks']) for r in repeated)
        text = (f"Across the {words.get(n_blocks, n_blocks)} {tt(primary)} blocks the cell fidelity varied by more than its mean shot-noise half-width "
                f"for {exceed(deep)} of the {len(deep)} coherent cells but for {exceed(shallow)} of the {len(shallow)} shallow cells (VQLS states and controls); "
                f"the largest standard deviation, {num(widest[1])}, belongs to {_method_names.get(widest[0]['algorithm'], widest[0]['algorithm'])} on {widest[0]['instance']}.")
        decision = sorted([s for s in deep if s[0]['instance'] in ('H4', 'H5') and s[0]['algorithm'] == 'hhl'], key=lambda s: s[0]['instance'])
        if decision:
            text += (' The decision cells varied ' + ('by less than' if all(sd < hw for _, sd, hw in decision) else 'by more than' if all(sd > hw for _, sd, hw in decision) else 'comparably to')
                     + ' their shot-noise intervals: the HHL fidelities on ' + join(s[0]['instance'] for s in decision) + ' had standard deviations of '
                     + join(num(s[1], 3) for s in decision) + ' against mean half-widths of ' + join(num(s[2], 3) for s in decision) + '.')
        out['ConfirmationRepro'] = text

    # Prediction accuracy by device.
    chosen = [r for r in endpoints if r.get('scope') == 'all_cells' and r.get('model') in _model_names]
    if chosen:
        best = {}
        for record in ('pre_registration', 'job_start'):
            for device in devices:
                candidates = [r for r in chosen if r['prediction'] == record and r['device'] == device and r['model'] != 'logical_ideal']
                if candidates:
                    best[(record, device)] = min(candidates, key=lambda r: r['mean_abs_error'])
        winners = {r['model'] for r in best.values()}
        start = [r for r in chosen if r['prediction'] == 'job_start']
        if len(winners) == 1:
            model = winners.pop()
            def series(record):
                return join(num(best[(record, d)]['mean_abs_error'], 3) for d in devices if (record, d) in best)
            starts = [best[('job_start', d)] for d in devices if ('job_start', d) in best]
            text = (f"On every processor the {_model_names[model]} model has the smallest mean absolute fidelity error, both as pre-registered "
                    f"({series('pre_registration')} on {join(tt(d) for d in devices)}) and from job-start calibrations ({series('job_start')}), "
                    f"with Spearman rank correlations of {num(min(r['spearman_rho'] for r in starts))} to {num(max(r['spearman_rho'] for r in starts))}; "
                    f"only {100*min(r['coverage95'] for r in starts):.0f} to {100*max(r['coverage95'] for r in starts):.0f}\\% of its job-start predictions fall inside the measured intervals.")
        else:
            text = ' '.join(f"Against job-start calibrations on {tt(d)} the {_model_names[best[('job_start', d)]['model']]} model has the smallest mean absolute fidelity error "
                            f"({num(best[('job_start', d)]['mean_abs_error'], 3)} over {best[('job_start', d)]['cells']} cells)." for d in devices if ('job_start', d) in best)
        signed = [r for r in start if r.get('mean_signed_error_observed_minus_predicted') is not None]
        if signed:
            low = min(signed, key=lambda r: r['mean_signed_error_observed_minus_predicted'])
            high = max(signed, key=lambda r: r['mean_signed_error_observed_minus_predicted'])
            text += ((' Every model is optimistic on average: ' if high['mean_signed_error_observed_minus_predicted'] < 0 else ' ')
                     + f"the observed minus predicted job-start fidelity ranges from {num(low['mean_signed_error_observed_minus_predicted'], 3)} "
                     f"({_model_names[low['model']]}) to {num(high['mean_signed_error_observed_minus_predicted'], 3)} ({_model_names[high['model']]}).")
        out['ConfirmationModels'] = text

    # Abstract and conclusion.
    if decisive:
        instance_names = {'H4': 'the star instance', 'H5': 'the weighted-cycle instance'}
        primary = design['devices']['primary']['backend']
        transfer = [d for d in devices if d != primary]
        positive = [r for r in decisive if above(r)]
        negative_devices = [d for d in devices if any(r['device'] == d for r in decisive) and all(below(r) for r in decisive if r['device'] == d)]
        correct = {m: sum((gain(r, m) > 0) == (r['inversion_gain_raw'] > 0) for r in decisive if gain(r, m) is not None) for m in noise_models}
        n = len(decisive)
        primary_blocks = len(blocks_of.get(primary, []))
        setting = (f"{several(primary_blocks, 'block')} on {tt(primary)}" if primary_blocks else '')
        if transfer:
            setting += (' and ' if setting else '') + ('one on each of ' if len(transfer) > 1 else 'one on ') + join(tt(d) for d in transfer) if all(len(blocks_of[d]) == 1 for d in transfer) \
                       else (' and ' if setting else '') + join(f"{words.get(len(blocks_of[d]), len(blocks_of[d]))} on {tt(d)}" for d in transfer)
        place = join(f"on {tt(d)}" + (f" in all {words.get(len(blocks_of[d]), len(blocks_of[d]))} blocks" if len(blocks_of[d]) > 1 else '') for d in negative_devices)
        text = f"In {jobs} jobs ({setting}), the HHL measurements pre-registered as decisive stayed below input preparation {place}"
        if positive:
            text += ', whereas on ' + join(f"{tt(r['device'])} a circuit with {r['two_qubit_gates']} controlled-Z gates reached $G={r['inversion_gain_raw']:.2f}$ "
                                          f"(95\\% interval ${r['inversion_gain_raw_bootstrap95'][0]:.2f}$ to ${r['inversion_gain_raw_bootstrap95'][1]:.2f}$) "
                                          f"on {instance_names.get(r['instance'], r['instance'])}" for r in positive)
            if len(coherent_pos) == len(positive):
                text += f", the only {'one' if len(positive) == 1 else words.get(len(positive), len(positive))} of {len(all_coherent)} coherent-solver measurements resolved above zero"
        d_ok, t_ok = correct['depolarizing_readout'], correct['thermal_idle_readout']
        local_text = (f"each gate-local calibration model had predicted the sign of {words.get(d_ok, d_ok)}" if d_ok == t_ok else
                      f"the two gate-local calibration models had predicted the sign of {words.get(d_ok, d_ok)} and {words.get(t_ok, t_ok)}")
        text += (f"; {local_text} of the {words.get(n, n)} decisive measurements, "
                 f"a global-depolarizing model only {words.get(correct['esp_global_depolarizing'], correct['esp_global_depolarizing'])}.")
        if all_vqls_pos:
            text += f" Frozen variational states resolved positive gains in {len(all_vqls_pos)} of {len(all_vqls)} measurements, reflecting faithful preparation of classically trained states."
        out['AbstractConfirmation'] = text
        sensitive_coherent = [r for r in coherent if r['instance'] in ('H4', 'H5')]
        others_below = all(below(r) for r in sensitive_coherent if not above(r))
        if positive:
            single = all(len(blocks_of[r['device']]) == 1 for r in positive)
            start = [gain(r, 'thermal_idle_readout', 'job_start') for r in positive]
            anticipated = all(v is not None and r['inversion_gain_raw_bootstrap95'][0] <= v <= r['inversion_gain_raw_bootstrap95'][1] for r, v in zip(positive, start))
            outcome = (('verified coherent solvers fell below input preparation on every inversion-sensitive cell except ' if others_below
                        else 'verified coherent solvers did not exceed input preparation on any inversion-sensitive cell except ')
                       + join(f"HHL on {r['instance']} on {tt(r['device'])}, $G={r['inversion_gain_raw']:.2f}$ {ci(r['inversion_gain_raw_bootstrap95'])}" for r in positive)
                       + (', a result that rests on one job under one calibration' if single and len(positive) == 1 else '')
                       + (' and that the thermal-idle-readout model anticipated from the job-start calibration' if anticipated else ''))
        else:
            outcome = 'no verified coherent solver produced a resolved inversion gain on any processor'
        conclusion = f"{'Across' if len(places) > 1 else 'In'} {several(len(places), 'block')} on {several(len(devices), 'processor')}, {outcome}."
        if all_vqls_pos:
            conclusion += (f" The frozen VQLS states, which need only a few CZ gates, resolved positive gains in {len(all_vqls_pos)} of {len(all_vqls)} cells; "
                           'they demonstrate faithful preparation of classically trained states, not hardware solving.')
        conclusion += (' Repeating the resolved cell under independent calibrations is the most informative next experiment, and the same protocol certifies its outcome.'
                       if positive else ' The same protocol can certify a future positive result.')
        out['ConclusionConfirmation'] = conclusion
    return out

_short_roles = {'D0': 'development control', 'H1': 'path, localized source', 'H2': 'cycle, signed source',
                'H3': 'size transfer, 8-cycle', 'H4': 'star, hub source', 'H5': 'cycle, opposite node', 'H6': 'VQLS scale probe'}
_method_names = {'hhl': 'HHL', 'qsvt': 'QSVT', 'vqls': 'VQLS', 'control': 'control'}

if journal_settings_config.get('enabled', False) and journal_tables:
    journal_generated.mkdir(parents=True, exist_ok=True)
    numbers = {}
    numbers['ManuscriptDate'] = datetime.now(timezone.utc).strftime('%d %B %Y')
    numbers['DzeroFin'] = _f(journal['input_fidelity']['D0'], 4)
    archived = journal.get('archived_tomography', {}).get('result') if journal.get('archived_tomography') else None
    if archived:
        numbers['DzeroVqlsF'] = _f(archived['solution']['fidelity_raw'], 4)
        numbers['DzeroVqlsFci'] = _interval(archived['solution']['fidelity_raw_bootstrap95'], 4)
        numbers['DzeroVqlsFinput'] = _f(archived['source']['fidelity_raw'], 4)
        numbers['DzeroVqlsG'] = _f(archived['inversion_gain_raw'], 3)
        numbers['DzeroVqlsGci'] = _interval(archived['inversion_gain_raw_bootstrap95'], 2)
        numbers['DzeroVqlsMinEig'] = _f(archived['solution']['eigenvalues'][0], 4)
        numbers['DzeroCkptF'] = _f(archived['checkpoint_solution_fidelity_exact'], 4)
        numbers['DzeroCkptG'] = _f(archived['checkpoint_inversion_gain_exact'], 2)
        numbers['DzeroHwCkptF'] = _f(archived['checkpoint']['fidelity_raw'], 4)
        numbers['DzeroVqlsFmit'] = _f(archived['solution']['fidelity_mitigated'], 4)
    retro = journal_tables.get('retrospective_predictions', [])
    numbers['RetroPubs'] = str(len(retro))
    # Resource frontier from the Nighthawk planning compilation.
    plan = journal.get('planning', {}).get('ibm_rensselaer')
    frontier = {}
    if plan:
        for identifier, compiled in plan['compiled'].items():
            entries = [e for e in compiled['result']['entries'] if e['role'] == 'coherent_candidate']
            for method in ('hhl', 'qsvt'):
                rows = [e for e in entries if e['algorithm'] == method]
                if rows:
                    best = min(rows, key=lambda e: (e['resources']['two_qubit_gates']/e['success_probability'], e['resources']['two_qubit_gates']))
                    frontier[(identifier, method)] = best
        sensitive = [frontier[(i, m)]['resources']['two_qubit_gates'] for i in ('H4', 'H5') for m in ('hhl', 'qsvt') if (i, m) in frontier]
        numbers['HoldoutMinCZ'] = str(min(sensitive)) if sensitive else '--'
        numbers['HoldoutMaxCZ'] = str(max(sensitive)) if sensitive else '--'
        numbers['HthreeHHLCZ'] = str(frontier[('H3', 'hhl')]['resources']['two_qubit_gates']) if ('H3', 'hhl') in frontier else '--'
        numbers['NighthawkPlanningDate'] = _tex(str(plan['calibration_last_update'])[:16])

    # Table: instances.
    rows = []
    for r in journal_tables['instances']:
        rows.append([r['instance'], r['nodes'], _f(r['gamma'], 2), _f(r['kappa'], 2), _f(r['F_in'], 3),
                     _f(r['kantorovich_min_F_in'], 3), _f(r['cg_median_us'], 1), _short_roles[r['instance']]])
    (journal_generated/'table_instances.tex').write_text(_table('tab:instances',
        r'Benchmark instances. $\Fin$ is the source-state fidelity with the normalized solution and $4\kappa/(1+\kappa)^2$ its Kantorovich lower bound. CG is the median single-core conjugate-gradient solve time (201 repetitions) to relative tolerance $10^{-11}$.',
        ['ID', '$N$', r'$\gamma$', r'$\kappa$', r'$\Fin$', 'bound', r'CG ($\mu$s)', 'role'], rows, 'lrrrrrrl'), encoding='utf8')

    # Table: screening and resources (minimum-objective passing candidate per method and device).
    rows = []
    selection_by = {}
    for backend_name, device_plan in journal.get('planning', {}).items():
        for (block, group, arm), job in device_plan['jobs'].items():
            if arm == 'raw':
                for identifier, selection in job['prepared']['result']['selections'].items():
                    selection_by[(backend_name, identifier)] = selection
    screening_rows = journal_tables['coherent_screening']
    for identifier in JOURNAL_DESIGN['coherent']['instances']:
        for method in ('hhl', 'qsvt'):
            candidates = [r for r in screening_rows if r['instance'] == identifier and r['candidate_id'].startswith(method)]
            passing = [r for r in candidates if r['status'] == 'accepted']
            selection = selection_by.get(('ibm_rensselaer', identifier), {}).get(method, {})
            chosen_id = selection.get('candidate_id')
            chosen = next((r for r in passing if r['candidate_id'] == chosen_id), None)
            cz = [str(selection_by.get((d, identifier), {}).get(method, {}).get('resources', {}).get('two_qubit_gates', '--'))
                  for d in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston')]
            rows.append([identifier, _method_names[method], f'{len(passing)}/{len(candidates)}',
                         _tex(chosen_id) if chosen_id else '--', _f(chosen['operator_relative_error'], 3) if chosen else '--',
                         _f(chosen['ideal_acceptance'], 3) if chosen else '--', ' / '.join(cz),
                         _tex(selection.get('arm', 'infeasible')).replace(r'\_', ' ')])
    (journal_generated/'table_screening.tex').write_text(_table('tab:screen',
        r'Operator screen and frozen selection. Passing counts operator and residual tests ($\le0.03$). The selected candidate minimizes native CZ gates per ideal accepted shot on Nighthawk; CZ counts are for the planning targets of \texttt{ibm\_rensselaer}, \texttt{ibm\_kingston} and \texttt{ibm\_boston}. Primary: $\le300$ CZ; depth probe: $\le1500$.',
        ['ID', 'method', 'pass', 'selected', 'op. err.', 'accept.', 'CZ (N / K / B)', 'arm'], rows, 'llrlrrcl', wide=True), encoding='utf8')

    # Table: matched optimizer study.
    rows = []
    for identifier in JOURNAL_DESIGN['vqls']['matched_optimizer_study']['instances']:
        for treatment in JOURNAL_DESIGN['vqls']['matched_optimizer_study']['treatments']:
            chosen = sorted([r for r in journal_tables['optimizer_study'] if r['instance'] == identifier and r['treatment'] == treatment],
                            key=lambda r: r['budget'])
            if not chosen:
                continue
            cells = [f"{_f(r['median_checkpoint_fidelity'], 3)} ({_f(r['min_checkpoint_fidelity'], 2)}--{_f(r['max_checkpoint_fidelity'], 2)})" for r in chosen]
            rows.append([identifier, 'exact' if treatment.startswith('exact') else 'counted', *cells])
    budgets = JOURNAL_DESIGN['vqls']['matched_optimizer_study']['budgets']
    counted_shots = sorted({r['training_shots_per_run'] for r in journal_tables['optimizer_study']
                            if r['treatment'] == 'counted_objective' and r['budget'] == max(budgets)})
    shots_caption = (f" Counted runs used {', '.join(f'{v:,}'.replace(',', r'{,}') for v in counted_shots)} simulator shots at {max(budgets)} evaluations."
                     if counted_shots else '')
    (journal_generated/'table_optimizer.tex').write_text(_table('tab:optimizer',
        r'Matched VQLS optimizer study (local). Median (range) over five shared initial vectors of the checkpoint state fidelity after 15, 30 and 60 objective evaluations; $\Fin$ is listed in Table~\ref{tab:instances}. Counted objectives use 2048 shots per setting.' + shots_caption,
        ['ID', 'objective', *[f'{b} evals' for b in budgets]], rows, 'll'+'c'*len(budgets), size=r'\scriptsize'), encoding='utf8')
    exact60 = [r for r in journal_tables['optimizer_study'] if r['treatment'] == 'exact_objective' and r['budget'] == max(budgets)]
    counted60 = [r for r in journal_tables['optimizer_study'] if r['treatment'] == 'counted_objective' and r['budget'] == max(budgets)]
    if exact60 and counted60:
        shots_text = f"{int(np.median([r['training_shots_per_run'] for r in counted60])):,}".replace(',', r'\,')
        numbers['OptimizerSentence'] = (
            f"With {max(budgets)} evaluations the median checkpoint fidelity across instances was "
            f"{_f(np.median([r['median_checkpoint_fidelity'] for r in exact60]), 3)} for the exact objective and "
            f"{_f(np.median([r['median_checkpoint_fidelity'] for r in counted60]), 3)} for the counted objective, whose runs used about "
            + shots_text + " simulator shots each; the worst counted seed reached "
            f"{_f(min(r['min_checkpoint_fidelity'] for r in counted60), 3)}.")
    # Table: archived development hardware.
    rows = []
    for algorithm, label, cz in [('hhl', 'HHL m=5 bin spacing', 223), ('qsvt', 'QSVT d=15', 302), ('vqls_measured', 'VQLS frozen', 2)]:
        value = hardware_analyses.get(algorithm) if hardware_analyses else None
        if value:
            rows.append(['14 Sep', label, cz, 'Z', f"{value['accepted_shots']}/{value['total_shots']}", _f(value['total_variation'], 3), '--'])
    if coherent_confirmation_analysis is not None:
        for row in sorted(coherent_confirmation_analysis['result']['rows'], key=lambda r: (r['algorithm'], ['ZZ', 'XX', 'XZ'].index(r['setting']))):
            m = row['metrics']; cz = 143 if row['algorithm'] == 'hhl' else 104
            label = 'HHL m=4 bounded' if row['algorithm'] == 'hhl' else 'QSVT d=5'
            rows.append(['5 Oct', label, cz, row['setting'], f"{m['accepted_shots']}/{m['total_shots']}", _f(m['total_variation'], 3), '--'])
    if archived:
        rows.append(['5 Oct', 'VQLS frozen, 9 settings', 2, 'tomo.', '73728/73728', '--',
                     f"$F$={archived['solution']['fidelity_raw']:.4f}, $G$={archived['inversion_gain_raw']:.3f}"])
    (journal_generated/'table_d0.tex').write_text(_table('tab:d0',
        r'Archived development jobs on D0 (\texttt{ibm\_rensselaer}). TV is the conditional total variation from the target distribution in the named basis. The tomography row gives the raw fidelity with $\xhat$ and the inversion gain.',
        ['date', 'circuit', 'CZ', 'basis', 'accepted', 'TV', 'tomography'], rows, 'llrlrrl', wide=True), encoding='utf8')

    # Table: retrospective prediction test by circuit class.
    classes = {}
    for r in retro:
        if r['kind'] in ('readout_calibration',):
            continue
        family = 'VQLS' if 'vqls' in str(r['algorithm']) else ('control' if r['kind'] == 'control' else _method_names.get(r['algorithm'], r['algorithm']))
        classes.setdefault((r['job'], family), []).append(r)
    job_labels = {'pilot_2026_09_14': '14 Sep', 'coherent_bias_2026_10_05': '5 Oct', 'vqls_tomography_2026_10_05': '5 Oct'}
    rows = []; summary = {}
    for (job, family), items in sorted(classes.items(), key=lambda kv: (kv[0][0] != 'pilot_2026_09_14', kv[0][1])):
        observed = [r['observed_tv_to_target'] for r in items if r.get('observed_tv_to_target') is not None]
        row = [job_labels[job], family, len(items), f"{np.median([r['two_qubit_gates'] for r in items]):.0f}", _f(np.median(observed), 3)]
        for model in JOURNAL_NOISE_MODELS[1:]:
            values = [r.get(f'{model}:predicted_tv_to_target') for r in items]
            pairs = [(v, r['observed_tv_to_target']) for v, r in zip(values, items) if v is not None and r.get('observed_tv_to_target') is not None]
            distance = [r.get(f'{model}:tv_pred_obs') for r in items if r.get(f'{model}:tv_pred_obs') is not None]
            row.append(f"{_f(np.median([p for p, _ in pairs]), 3)} / {_f(np.median(distance), 3)}" if pairs else '--')
            summary[(job, family, model)] = {'predicted': np.median([p for p, _ in pairs]) if pairs else None,
                                             'distance': np.median(distance) if distance else None,
                                             'observed': np.median(observed) if observed else None}
        rows.append(row)
    (journal_generated/'table_retro.tex').write_text(_table('tab:retro',
        r'Retrospective model test on archived circuits, by job and circuit class. Each model entry gives the median predicted conditional TV from the target and the median TV between predicted and measured outcome distributions; the observed column is the measured TV from the target.',
        ['job', 'class', 'circuits', 'CZ', 'observed', 'depol.+readout', 'thermal+idle', 'global depol.'], rows, 'llrrrccc', wide=True), encoding='utf8')
    deep = [(f, s) for (job, f, m), s in summary.items() if job == 'pilot_2026_09_14' and f in ('HHL', 'QSVT')]
    if summary:
        def pick(job, family, model, key):
            value = summary.get((job, family, model), {}).get(key)
            return _f(value, 3)
        numbers['RetroSentence'] = (
            'For the two-CZ VQLS circuits every model predicts the measured distributions to within the shot noise '
            f"(median distance {pick('vqls_tomography_2026_10_05', 'VQLS', 'thermal_idle_readout', 'distance')}). "
            'For the 104--143 CZ circuits of 5 October the thermal-idle model is closest '
            f"(median distance {pick('coherent_bias_2026_10_05', 'HHL', 'thermal_idle_readout', 'distance')} for HHL and "
            f"{pick('coherent_bias_2026_10_05', 'QSVT', 'thermal_idle_readout', 'distance')} for QSVT), "
            'whereas for the 223 and 302 CZ circuits of 14 September all gate-local models underpredict the measured error; '
            'the parameter-free global-depolarizing model comes closest '
            f"(predicted TV {pick('pilot_2026_09_14', 'HHL', 'esp_global_depolarizing', 'predicted')} and "
            f"{pick('pilot_2026_09_14', 'QSVT', 'esp_global_depolarizing', 'predicted')} against measured "
            f"{pick('pilot_2026_09_14', 'HHL', 'esp_global_depolarizing', 'observed')} and "
            f"{pick('pilot_2026_09_14', 'QSVT', 'esp_global_depolarizing', 'observed')}). "
            'No model reproduces the QSVT XZ error, which suggests coherent or crosstalk errors outside all four models.')

    # Table: pre-registered predictions, one row per cell, three devices.
    pre = [r for r in journal_tables.get('preregistered_predictions', []) if r['arm_job'] == 'raw']
    keys = sorted({(r['instance'], r['algorithm'], r['arm'], r['candidate_id']) for r in pre},
                  key=lambda k: (k[0], ['hhl', 'qsvt', 'vqls', 'control'].index(k[1]) if k[1] in ('hhl', 'qsvt', 'vqls', 'control') else 9, k[3]))
    rows = []
    for key in keys:
        by_device = {r['device']: r for r in pre if (r['instance'], r['algorithm'], r['arm'], r['candidate_id']) == key}
        row = [key[0], _tex(key[3]).replace(r'\_', ' '), key[2].replace('_', ' ')]
        for column in ('two_qubit_gates', 'thermal_idle_readout:F_raw', 'thermal_idle_readout:gain_raw'):
            values = []
            for device in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston'):
                value = by_device.get(device, {}).get(column)
                values.append('--' if value is None else (str(value) if column == 'two_qubit_gates' else _f(value, 2)))
            row.append(' / '.join(values))
        rows.append(row)
    (journal_generated/'table_prereg.tex').write_text(_table('tab:prereg',
        r'Pre-registered block-B1 predictions (thermal-idle-readout model, raw readout) for Nighthawk / Kingston / Boston planning targets. $F$ is the predicted raw tomographic fidelity with $\xhat$ and $G$ the predicted inversion gain. Controls prepare $\bhat$ (input) or $\xhat$ (solution) on the VQLS qubits.',
        ['ID', 'cell', 'arm', 'CZ', '$F$', '$G$'], rows, 'lllccc', wide=True), encoding='utf8')
    coherent = [r for r in pre if r['algorithm'] in ('hhl', 'qsvt') and r.get('thermal_idle_readout:gain_raw') is not None]
    vqls_rows = [r for r in pre if r['algorithm'] == 'vqls' and r.get('thermal_idle_readout:gain_raw') is not None]
    planned = journal_tables.get('planned_jobs', [])
    nighthawk_seconds = sum(r['estimated_qpu_seconds'] for r in planned if r['device'] == 'ibm_rensselaer')
    if coherent:
        negative = sum(r['thermal_idle_readout:gain_raw'] < 0 for r in coherent)
        numbers['PreregSentence'] = (
            f"Under the thermal-idle-readout model {negative} of {len(coherent)} coherent cells have negative predicted gain. "
            f"The Nighthawk block, including the mitigation arm, is estimated at {nighthawk_seconds:.0f} QPU seconds.")
    def _range(rows, keys):
        values = [r.get(k) for r in rows for k in keys if r.get(k) is not None]
        return (min(values), max(values)) if values else (None, None)
    gain_keys = ['depolarizing_readout:gain_raw', 'thermal_idle_readout:gain_raw', 'esp_global_depolarizing:gain_raw']
    sensitive_hhl = [r for r in pre if r['instance'] in ('H4', 'H5') and r['algorithm'] == 'hhl']
    sensitive_qsvt = [r for r in pre if r['instance'] in ('H4', 'H5') and r['algorithm'] == 'qsvt']
    local_hhl = _range(sensitive_hhl, gain_keys[:2]); global_hhl = _range(sensitive_hhl, gain_keys[2:])
    qsvt_range = _range(sensitive_qsvt, gain_keys)
    vqls_sensitive = [r for r in pre if r['instance'] in ('H3', 'H4', 'H5', 'H6') and r['algorithm'] == 'vqls']
    vqls_range = _range(vqls_sensitive, gain_keys)
    numbers['PreregHHLlocal'] = f"{_f(local_hhl[0], 2)} to {_f(local_hhl[1], 2)}"
    numbers['PreregHHLglobal'] = f"{_f(global_hhl[0], 2)} to {_f(global_hhl[1], 2)}"
    # The verb follows the numbers: 'disagree in sign' only if every gate-local value is negative and every global value positive.
    if None not in (*local_hhl, *global_hhl):
        numbers['PreregHHLverdict'] = ('disagree in sign for HHL' if local_hhl[1] < 0 < global_hhl[0]
                                       else 'differ for HHL' if abs(np.mean(local_hhl)-np.mean(global_hhl)) > .1 else 'agree for HHL')
    # Device-level statement for H4/H5 HHL, generated from the same records.
    device_tex = {d: r'\texttt{'+_tex(d)+'}' for d in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston')}
    split, mixed = [], []
    for device in ('ibm_rensselaer', 'ibm_kingston', 'ibm_boston'):
        chosen = [r for r in sensitive_hhl if r['device'] == device]
        local_d, global_d = _range(chosen, gain_keys[:2]), _range(chosen, gain_keys[2:])
        if chosen and None not in (*local_d, *global_d):
            (split if local_d[1] < 0 < global_d[0] else mixed).append((device, chosen))
    parts = []
    if split:
        parts.append('on ' + ' and '.join(device_tex[d] for d, _ in split) +
                     ' every gate-local HHL prediction for H4 and H5 is negative and every global-depolarizing prediction positive')
    for device, chosen in mixed:
        for key, name in ((gain_keys[0], 'depolarizing-readout'), (gain_keys[1], 'thermal-idle-readout')):
            values = [r[key] for r in chosen if r.get(key) is not None]
            if values and max(values) > 0:
                parts.append(f'on {device_tex[device]} the {name} model also predicts a positive gain (${_f(min(values), 2)}$ to ${_f(max(values), 2)}$)')
    if parts:
        sentence = '; '.join(parts) + '.'
        numbers['PreregHHLdevices'] = sentence[0].upper() + sentence[1:]
    numbers['PreregQSVTsensitive'] = f"{_f(qsvt_range[0], 2)} to {_f(qsvt_range[1], 2)}"
    numbers['PreregVQLSsensitive'] = f"{_f(vqls_range[0], 2)} to {_f(vqls_range[1], 2)}"
    checkpoints = {r['instance']: r for r in journal_tables.get('vqls_checkpoints', [])}
    if 'H2' in checkpoints:
        h2 = checkpoints['H2']
        numbers['HtwoVqlsIdealG'] = _f((h2['exact_state_fidelity']-h2['F_in'])/(1-h2['F_in']), 2)
    if 'H5' in checkpoints:
        numbers['HfiveVqlsF'] = _f(checkpoints['H5']['exact_state_fidelity'], 3)
    numbers['PlannedNighthawkSeconds'] = f"{nighthawk_seconds:.0f}"
    numbers['PlannedHeronSeconds'] = f"{sum(r['estimated_qpu_seconds'] for r in planned if r['device'] == 'ibm_kingston'):.0f}"
    numbers['PlannedCampaignMinutes'] = f"{(3*sum(r['estimated_qpu_seconds'] for r in planned if r['device'] == 'ibm_rensselaer' and r['arm'] == 'raw') + sum(r['estimated_qpu_seconds'] for r in planned if r['device'] == 'ibm_rensselaer' and r['arm'] != 'raw') + sum(r['estimated_qpu_seconds'] for r in planned if r['device'] in ('ibm_kingston', 'ibm_boston')))/60:.0f}"
    # Confirmation blocks: table, campaign facts and results sentences, generated from the analyzed cells only.
    hardware_rows = journal_tables.get('hardware_cells', [])
    hardware_table = journal_generated/'table_hardware.tex'
    if hardware_rows:
        hardware_table.write_text(_hardware_table(hardware_rows), encoding='utf8')
    else:
        hardware_table.unlink(missing_ok=True)  # never typeset a stale hardware table
    block_status_path = OUTPUTS/'journal_block_status.json'
    numbers.update(_confirmation_numbers(
        hardware_rows, journal_tables.get('prediction_endpoints', []), JOURNAL_DESIGN, journal_tables.get('block_reproducibility', []),
        _campaign_jobs(OUTPUTS, JOURNAL_DESIGN['design_id']) if hardware_rows else [],
        read_json(block_status_path) if block_status_path.is_file() else None))
    linux = []
    for path in sorted((OUTPUTS/'invocations').glob('*/environment.json')):
        environment = read_json(path)
        if str(environment.get('platform', '')).startswith('Linux') and (path.parent/'summary.json').is_file() and (path.parent/'environment_equivalence.json').is_file():
            linux.append(path.parent)
    if linux:
        first = read_json(linux[0]/'summary.json')
        # The sentence reports stages computed on Windows and reused on Linux, so count exactly those.
        numbers['ReproLoaded'] = str(sum(1 for e in first['events'] if 'environment_equivalence' in e and
                                         str(e['environment_equivalence']['computed_in']['platform']).startswith('Windows')))
        numbers['ReproComputed'] = str(first['computed'])
    numbers.setdefault('PreregHHLdevices', '')
    for name in ['PreregHHLlocal', 'PreregHHLglobal', 'PreregHHLverdict', 'PreregQSVTsensitive', 'PreregVQLSsensitive', 'HtwoVqlsIdealG', 'HfiveVqlsF',
                 'PlannedNighthawkSeconds', 'PlannedHeronSeconds', 'PlannedCampaignMinutes', 'DzeroVqlsF', 'DzeroVqlsFci', 'DzeroVqlsFinput', 'DzeroVqlsG', 'DzeroVqlsGci', 'DzeroVqlsMinEig', 'DzeroCkptF',
                 'DzeroCkptG', 'DzeroHwCkptF', 'DzeroVqlsFmit', 'HoldoutMinCZ',
                 'HoldoutMaxCZ', 'HthreeHHLCZ', 'NighthawkPlanningDate', 'OptimizerSentence', 'RetroSentence', 'PreregSentence',
                 'ReproLoaded', 'ReproComputed']:
        numbers.setdefault(name, r'\pending{n/a}')
    import re as _re
    for key, value in list(numbers.items()):
        if _re.fullmatch(r'[-\[\]\d., ]+', value) and any(ch.isdigit() for ch in value):
            numbers[key] = '$'+value+'$'
        elif _re.fullmatch(r'-?[\d.]+ to -?[\d.]+', value):
            first, second = value.split(' to ')
            numbers[key] = f'${first}$ to ${second}$'
    (journal_generated/'numbers.tex').write_text(''.join(f'\\newcommand{{\\{k}}}{{{v}}}\n' for k, v in numbers.items()), encoding='utf8')
    atomic_json(journal_report_dir/'manuscript_numbers.json', numbers)
    journal_pdf = None
    if shutil.which('pdflatex') and (journal_paper_dir/'main.tex').is_file():
        log = []
        for command in (['pdflatex', '-interaction=nonstopmode', 'main.tex'], ['bibtex', 'main'],
                        ['pdflatex', '-interaction=nonstopmode', 'main.tex'], ['pdflatex', '-interaction=nonstopmode', 'main.tex']):
            completed = subprocess.run(command, cwd=journal_paper_dir, capture_output=True, text=True, timeout=300)
            log.append({'command': command, 'returncode': completed.returncode})
        if (journal_paper_dir/'main.pdf').is_file():
            target = WORKSPACE/'output'/'pdf'; target.mkdir(parents=True, exist_ok=True)
            journal_pdf = target/'journal_manuscript_draft.pdf'
            shutil.copy2(journal_paper_dir/'main.pdf', journal_pdf)
        atomic_json(journal_report_dir/'latex_build.json', log)
    print('Manuscript numbers:', len(numbers), '| PDF:', journal_pdf if journal_pdf else 'LaTeX not available; tables regenerated only')
